> **Developed by Farhan Soomro** · Free community utility from [SemanticModelBPforAI](https://github.com/SoomroFarhanH/SemanticModelBPforAI) · Licensed under the [MIT License](https://github.com/SoomroFarhanH/SemanticModelBPforAI/blob/main/LICENSE)
>
> ⚠️ **Disclaimer:** Provided "AS IS", without warranty of any kind, express or implied, including but not limited to the warranties of merchantability, fitness for a particular purpose, and non-infringement. This is a personal community project. It is not an official Microsoft product and is not supported or endorsed by Microsoft or any employer. Review the code and test it in a non-production environment before use. You use it at your own risk. In no event shall the author be liable for any claim, damages, data loss, service disruption, capacity or licensing cost, or other liability arising from its use. Results are guidance only and should be validated by a qualified person.


## 🤖 Fabric Data Agent Analyzer v3.0.0

> 💡 **Core Principle:** *"AI is only as good as the data behind it."* A data agent can only be as accurate as the sources, metadata, instructions, and examples it is grounded on.

> 📘 **Documentation snapshot:** 2026-09-29 · Aligned with [Add and configure data sources](https://learn.microsoft.com/en-us/fabric/data-science/data-agent-add-datasources) | [Configuration best practices](https://learn.microsoft.com/en-us/fabric/data-science/data-agent-configuration-best-practices) | [Data agent runtime](https://learn.microsoft.com/en-us/fabric/data-science/data-agent-runtime) | [Evaluate your data agent](https://learn.microsoft.com/en-us/fabric/data-science/evaluate-data-agent) | [Semantic model best practices](https://learn.microsoft.com/en-us/fabric/data-science/semantic-model-best-practices) | [Prepare your data for AI](https://learn.microsoft.com/power-bi/create-reports/copilot-prepare-data-ai)

This notebook assesses a **Fabric data agent**, its shared configuration, and **every connected source type** (up to five): Lakehouse, Warehouse, Fabric SQL Database, Mirrored Database, Eventhouse KQL Database, Power BI semantic model, Graph Model *(preview)*, Fabric Ontology *(preview)*, and Azure AI Search Index *(preview)*. The analyzer is **read-only by default**. It never changes the agent or its sources.

### What's New in v3.0.0 (Multi-Source Refactor)

- 🆕 **Multi-source architecture:** data agent discovery → normalized source inventory → common agent checks → per-source adapters → cross-source routing → evaluation → scorecard.
- 🆕 **Three collection modes:** `LIVE_SDK` (Fabric data agent Python SDK, preview), `GIT_EXPORT` (Fabric Git item definition), and `MANUAL_MANIFEST` (JSON/YAML fallback).
- 🆕 **Normalized `Finding` records:** each record has a stable rule ID, source, object path, evidence, status, confidence, remediation, and documentation URL. The old `(severity, check, description)` tuples are gone.
- 🆕 **Status model:** `PASS · WARN · FAIL · MANUAL · NOT_APPLICABLE · NOT_EVALUATED · UNKNOWN`. Features a source doesn't support are marked **N/A** and aren't failed. Manual checks are **no longer scored as zero**.
- 🆕 **Readiness and coverage are reported separately.** Blocker-based **release gates** always override the numeric average.
- 🆕 **Source adapters:** SQL family (with Lakehouse/Warehouse/SQL DB/Mirrored subtype checks), KQL, Graph, Ontology, and Azure AI Search. Each adapter has capability-aware example-query validation and read-only static lint.
- ♻️ **Semantic model checks kept:** all v2.2.2 checks now run through the `SemanticModelAdapter` with stable IDs (`SM-001` … `SM-027`). The rule-key drift is fixed, and the missing naming check (1.5) is restored.
- 🔐 **Auth fix:** the Prep for AI validator now sends a proper bearer token. The token is never printed or persisted.
- 📤 **Exports:** HTML, Markdown, CSV, and JSON are all built from one normalized findings set.
- 🧭 **Source run plan (4.1):** as soon as discovery finishes, it shows which sources the agent actually uses and which source sections to run or skip. It also flags sources that are connected but have nothing selected. `SKIP_UNSELECTED_SOURCES` skips the deep checks for those sources, and each source section can be skipped on its own.

### Notebook map

| # | Section | # | Section |
|---|---|---|---|
| 1 | Parameters | 10 | Ontology *(preview)* |
| 2 | Framework, rule catalog & preflight | 11 | Azure AI Search *(preview)* |
| 3 | Data agent discovery | 12 | Cross-source routing & consistency |
| 4 | Source inventory, run plan & adapter framework | 13 | Example-query validation summary |
| 5 | Common agent checks | 14 | End-to-end evaluation |
| 6 | SQL sources (Lakehouse · Warehouse · SQL DB · Mirrored) | 15 | Security & permissions |
| 7 | Eventhouse KQL Database | 16 | ALM, deployment & service/admin |
| 8 | Power BI semantic model | 17 | Scorecard, release gates & exports |
| 9 | Graph Model *(preview)* | | |

> ⚖️ **Rule provenance:** rules marked *Proposed* in the specification are validation heuristics, not Microsoft product requirements. Preview capabilities are feature-detected because their behavior and contracts can change.

### 📋 Prerequisites

- Run this notebook inside a **Microsoft Fabric** notebook. Standalone Python works only for `GIT_EXPORT` or `MANUAL_MANIFEST` static analysis.
- Packages are installed in the next cell: `fabric-data-agent-sdk` *(preview)*, `semantic-link-labs`, `httpx`, and `pyyaml`.
- **Tenant settings:** Fabric data agent, cross-geo processing/storing for AI (when applicable), and Copilot/Azure OpenAI must be enabled. See [data agent tenant settings](https://learn.microsoft.com/en-us/fabric/data-science/data-agent-tenant-settings).
- **Read-only by default:** the notebook reads configuration and metadata. Live queries only run when `RUN_LIVE_QUERY_TESTS = True`, and a statement is blocked unless it passes the read-only lint first.

#### Minimum documented permissions for the identity that runs the analysis

| Source | Minimum documented query permission |
|---|---|
| Data agent | Permission to view the agent configuration (the draft is visible to editors; consumers see the published version) — see [sharing and permissions](https://learn.microsoft.com/en-us/fabric/data-science/data-agent-sharing) |
| Power BI semantic model | Read (Build is required for metadata, BPA, and Memory Analyzer checks) |
| Lakehouse | Read on the lakehouse item, plus table access if enforced |
| Warehouse | Read / `SELECT` on relevant tables |
| KQL Database | Reader role on the database |
| Ontology *(preview)* | Read on the ontology and Read on each underlying bound semantic model, lakehouse, or KQL database |
| Graph Model *(preview)* | Read on the graph item and its underlying data |
| Azure AI Search *(preview)* | Role-based access (Search Index Data Reader / Contributor for the connecting identity) |

> ⚠️ **Preview notice:** the Fabric data agent SDK, evaluation SDK, source control, Graph Model, Ontology, and Azure AI Search integration all include preview capabilities as of the 2026-09-29 documentation snapshot. The analyzer feature-detects them. When a capability can't be inspected, the check is reported as `UNKNOWN` or `NOT_EVALUATED`, never as a pass.

In [ ]:
# Install required packages (fabric-data-agent-sdk is preview)
%pip install semantic-link-labs fabric-data-agent-sdk httpx pyyaml --quiet --upgrade

---
# 1️⃣ PARAMETERS

Set the values below, then choose **Run all**.

| Scenario | Set |
|---|---|
| Assess a live data agent | `DATA_AGENT_NAME_OR_ID`, `WORKSPACE_ID_OR_NAME`, `COLLECTION_MODE = "LIVE_SDK"` |
| Review a Git-synced agent definition (PR / CI) | `COLLECTION_MODE = "GIT_EXPORT"`, `GIT_EXPORT_ROOT` = the data agent item folder |
| Preview sources or no API access | `COLLECTION_MODE = "MANUAL_MANIFEST"`, `MANUAL_MANIFEST_PATH` |
| Standalone semantic model (v2.x behavior) | Leave `DATA_AGENT_NAME_OR_ID` blank and set `dataset` + `workspace` |

> 🧭 **Not sure which sections apply?** Run through section 4 first. The **Source Run Plan** (4.1) lists the source sections to run for this agent and the ones you can skip.

`MANUAL_ATTESTATIONS` records manual checks that you verified yourself. They're counted as evaluated with `USER_SUPPLIED` evidence instead of staying `MANUAL`.

In [ ]:
# ============================================================
# 🔧  PARAMETERS — update these values
# ============================================================
# --- Target ---------------------------------------------------
WORKSPACE_ID_OR_NAME  = ""            # Workspace containing the data agent (blank = current notebook workspace)
DATA_AGENT_NAME_OR_ID = ""            # Blank = standalone semantic-model mode (uses `dataset` below)
DATA_AGENT_STAGE      = "sandbox"     # sandbox (draft) | production (published)
COLLECTION_MODE       = "LIVE_SDK"    # LIVE_SDK | GIT_EXPORT | MANUAL_MANIFEST
GIT_EXPORT_ROOT       = ""            # Data agent item folder that contains files/config/...
MANUAL_MANIFEST_PATH  = ""            # Optional JSON/YAML manifest (fallback / preview sources)

# --- Standalone semantic-model compatibility (v2.x contract) ----
dataset   = "ContosoSalesData"        # Semantic model name or ID — used only when DATA_AGENT_NAME_OR_ID is blank
workspace = "YourWorkspaceName"       # Workspace of that semantic model

# --- Behaviour (read-only by default) --------------------------
RUN_LIVE_QUERY_TESTS   = False        # Bounded read-only smoke tests + example-query execution
RUN_PERFORMANCE_TESTS  = False        # Latency sampling of example queries (requires RUN_LIVE_QUERY_TESTS)
RUN_FEW_SHOT_EVALUATOR = False        # SDK evaluate_few_shots (LLM-based; documented for SQL sources only)
RUN_EVALUATION         = False        # SDK evaluate_data_agent against EVALUATION_DATASET_PATH
SKIP_UNSELECTED_SOURCES = True        # Skip deep checks for sources that are connected but have no objects selected (see 4.1)
FAIL_ON_PREVIEW_SOURCE = False        # Treat any preview source/runtime as a release blocker
EXPORT_REPORTS         = True
OUTPUT_ROOT            = "/lakehouse/default/Files/DataAgentReadiness"

# --- Evaluation / routing / persona inputs (optional CSV paths) --
EVALUATION_DATASET_PATH   = ""        # question, expected_answer (+ case_id, expected_source, expected_language, persona, category, criticality)
EVALUATION_TABLE_NAME     = "data_agent_readiness_eval"
ROUTING_MATRIX_PATH       = ""        # topic, primary_source, secondary_source, reason, default_date, grain, metric_owner, clarification_rule
PERSONA_TEST_RESULTS_PATH = ""        # persona, source_name, expected_access, observed_access, notes

# --- Thresholds (configurable; published in the report) --------
SEVERITY_WEIGHTS = {"BLOCKER": 8, "CRITICAL": 5, "IMPORTANT": 3, "RECOMMENDED": 1, "INFO": 0}
COVERAGE_RELEASE_THRESHOLD = 0.70     # Minimum evaluation coverage for a release decision
MAX_SELECTED_OBJECTS_WARN  = 25       # Per source; broader selections are flagged for review
FRESHNESS_MAX_AGE_HOURS    = 48
SMOKE_TEST_MAX_OBJECTS     = 10
LATENCY_TARGET_SECONDS     = 10

# --- Manual attestations (recorded as USER_SUPPLIED evidence) ---
# Keys: rule ID, or "RULE-ID@Source Name" for a single source.
# Example: {"DA-017": {"status": "PASS", "evidence": "Tenant settings verified by Fabric admin on 2026-09-29"}}
MANUAL_ATTESTATIONS = {}
PREVIEW_ACCEPTED    = False           # Owner explicitly accepts preview sources/runtime for this release

# --- Legacy semantic-model attestations (used only when not readable live) --
prep_for_ai_configured = False        # Prep for AI configured (AI Data Schema, AI Instructions, Verified Answers)
is_direct_lake         = False        # Direct Lake storage mode guidance
# ============================================================

---
# 2️⃣ FRAMEWORK, RULE CATALOG & PREFLIGHT

These cells set up the **normalized data contract** that every section shares. It includes the `Status` and `Severity` enums, the `Finding`, `DataSourceProfile`, and `AgentProfile` records, the source-type alias table, the capability registry, the `KNOWN_LIMITATIONS` watch list, and the rule catalog.

**Design rules enforced by the helpers:**

- Every finding has a stable `rule_id`. IDs never depend on notebook cell numbers.
- When a check raises an exception, it's recorded as `NOT_EVALUATED` or `UNKNOWN`, never as a pass.
- Unsupported features are `NOT_APPLICABLE` and are left out of the denominator.
- Manual confirmation is never silently converted to zero. Checks stay `MANUAL` until you attest them.
- Raw source-type values are preserved and mapped to canonical families through a versioned alias table.

In [ ]:
import json
import os
import re
import sys
import time
import warnings
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Optional

import pandas as pd
from IPython.display import display, HTML, Markdown

warnings.filterwarnings('ignore')

try:
    import sempy.fabric as fabric
    _SEMPY_IMPORT_ERROR = None
except Exception as _e:  # Outside Fabric: static analysis modes still work
    fabric = None
    _SEMPY_IMPORT_ERROR = f"{type(_e).__name__}: {_e}"

ANALYZER_VERSION  = "3.0.0"
DOC_SNAPSHOT_DATE = "2026-09-29"
RUN_STARTED_UTC   = datetime.now(timezone.utc)


class Status(str, Enum):
    PASS = "PASS"
    WARN = "WARN"
    FAIL = "FAIL"
    MANUAL = "MANUAL"
    NOT_APPLICABLE = "NOT_APPLICABLE"
    NOT_EVALUATED = "NOT_EVALUATED"
    UNKNOWN = "UNKNOWN"


class Severity(str, Enum):
    BLOCKER = "BLOCKER"
    CRITICAL = "CRITICAL"
    IMPORTANT = "IMPORTANT"
    RECOMMENDED = "RECOMMENDED"
    INFO = "INFO"


EVALUATED_STATUSES = {Status.PASS, Status.WARN, Status.FAIL}
STATUS_ICON = {
    "PASS": "✅", "WARN": "⚠️", "FAIL": "❌", "MANUAL": "📝",
    "NOT_APPLICABLE": "➖", "NOT_EVALUATED": "⏸️", "UNKNOWN": "❔",
}
SEVERITY_ORDER = {"BLOCKER": 0, "CRITICAL": 1, "IMPORTANT": 2, "RECOMMENDED": 3, "INFO": 4}


@dataclass
class DataSourceProfile:
    source_id: str
    display_name: str
    source_type_raw: str
    source_family: str                  # SQL, KQL, SEMANTIC, GRAPH, ONTOLOGY, AI_SEARCH, UNKNOWN
    source_kind: str = "UNKNOWN"        # LAKEHOUSE, WAREHOUSE, FABRIC_SQL_DATABASE, MIRRORED_DATABASE, KQL_DATABASE, ...
    workspace_id: Optional[str] = None
    artifact_id: Optional[str] = None
    runtime: Optional[str] = None
    selected_objects: list = field(default_factory=list)
    instructions: Optional[str] = None
    description: Optional[str] = None
    example_queries: list = field(default_factory=list)
    metadata: dict = field(default_factory=dict)
    provenance: str = "UNKNOWN"         # SDK | GIT | USER_SUPPLIED


@dataclass
class AgentProfile:
    name: str
    agent_id: Optional[str]
    workspace: Optional[str]
    stage: str
    collection_mode: str
    runtime: str = "UNKNOWN"            # STANDARD | PREVIEW | UNKNOWN
    instructions: Optional[str] = None
    description: Optional[str] = None
    sources: list = field(default_factory=list)
    published_snapshot: Optional[dict] = None   # {"instructions":..., "source_names": [...]} of the other stage
    metadata: dict = field(default_factory=dict)
    errors: list = field(default_factory=list)
    provenance: str = "UNKNOWN"
    standalone_semantic: bool = False


@dataclass
class Finding:
    rule_id: str
    title: str
    source_id: Optional[str]
    source_name: Optional[str]
    source_family: str                  # AGENT, SQL, KQL, SEMANTIC, GRAPH, ONTOLOGY, AI_SEARCH, CROSS_SOURCE
    object_path: Optional[str]
    status: Status
    severity: Severity
    score_earned: float
    score_possible: float
    evidence: str
    recommendation: str
    automation: str                     # AUTOMATED, PARTIAL, MANUAL
    confidence: str                     # HIGH, MEDIUM, LOW
    docs_url: Optional[str] = None
    source_type_raw: Optional[str] = None
    observed_at_utc: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat(timespec="seconds"))


# ------------------------------------------------------------------
# Source-type normalization (versioned alias table — snapshot 2026-09-29)
# Values observed in the fabric-data-agent-sdk and Git item definition.
# Unknown values are preserved and reported as UNKNOWN (rule DA-004).
# ------------------------------------------------------------------
SOURCE_TYPE_ALIAS_VERSION = "2026-09-29"
SOURCE_TYPE_ALIASES = {
    "lakehousetables": "LAKEHOUSE", "lakehouse": "LAKEHOUSE",
    "datawarehouse": "WAREHOUSE", "warehouse": "WAREHOUSE", "warehousetables": "WAREHOUSE",
    "sqldatabase": "FABRIC_SQL_DATABASE", "fabricsqldatabase": "FABRIC_SQL_DATABASE", "sqldatabaseinfabric": "FABRIC_SQL_DATABASE",
    "mirroreddatabase": "MIRRORED_DATABASE", "mirroredazuredatabrickscatalog": "MIRRORED_DATABASE",
    "kusto": "KQL_DATABASE", "kqldatabase": "KQL_DATABASE", "eventhouse": "KQL_DATABASE", "eventhousekqldatabase": "KQL_DATABASE",
    "semanticmodel": "SEMANTIC_MODEL", "powerbisemanticmodel": "SEMANTIC_MODEL", "dataset": "SEMANTIC_MODEL",
    "graph": "GRAPH_MODEL", "graphmodel": "GRAPH_MODEL", "microsoftgraphinfabric": "GRAPH_MODEL", "fabricgraph": "GRAPH_MODEL",
    "ontology": "ONTOLOGY", "fabricontology": "ONTOLOGY",
    "azureaisearch": "AZURE_AI_SEARCH", "azureaisearchindex": "AZURE_AI_SEARCH", "aisearch": "AZURE_AI_SEARCH",
    "azuresearch": "AZURE_AI_SEARCH", "azuresearchindex": "AZURE_AI_SEARCH", "searchindex": "AZURE_AI_SEARCH",
}
# Git folder prefixes documented for the data agent item definition (others follow the same pattern)
GIT_FOLDER_PREFIXES = [
    ("lakehouse-tables-", "LAKEHOUSE"), ("warehouse-tables-", "WAREHOUSE"), ("semantic-model-", "SEMANTIC_MODEL"),
    ("kusto-", "KQL_DATABASE"), ("ontology-", "ONTOLOGY"), ("mirrored-database-", "MIRRORED_DATABASE"),
    ("sql-database-", "FABRIC_SQL_DATABASE"), ("graph-model-", "GRAPH_MODEL"), ("graph-", "GRAPH_MODEL"),
    ("azure-ai-search-", "AZURE_AI_SEARCH"), ("ai-search-", "AZURE_AI_SEARCH"),
]
SOURCE_KIND_TO_FAMILY = {
    "LAKEHOUSE": "SQL", "WAREHOUSE": "SQL", "FABRIC_SQL_DATABASE": "SQL", "MIRRORED_DATABASE": "SQL",
    "KQL_DATABASE": "KQL", "SEMANTIC_MODEL": "SEMANTIC", "GRAPH_MODEL": "GRAPH",
    "ONTOLOGY": "ONTOLOGY", "AZURE_AI_SEARCH": "AI_SEARCH",
}
PREVIEW_SOURCE_KINDS = {"GRAPH_MODEL", "ONTOLOGY", "AZURE_AI_SEARCH"}


def normalize_source_type(raw) -> str:
    """Map a raw SDK/Git/manifest type value to a canonical source kind (or UNKNOWN)."""
    if raw is None:
        return "UNKNOWN"
    key = re.sub(r"[\s_\-\.]", "", str(raw).lower())
    if key in SOURCE_TYPE_ALIASES:
        return SOURCE_TYPE_ALIASES[key]
    return "UNKNOWN"


# ------------------------------------------------------------------
# Capability registry (static snapshot; overridden by live evidence)
# True = documented support · False = documented as unsupported · None = not listed / feature-detect
# ------------------------------------------------------------------
CAPABILITIES = {
    "SQL":       {"query_language": "SQL", "example_queries": True, "source_instructions": True, "source_description": True,
                  "schema_object_descriptions": "PREVIEW_RUNTIME", "topics": "PREVIEW_RUNTIME", "few_shot_evaluator": True, "preview": False},
    "KQL":       {"query_language": "KQL", "example_queries": True, "source_instructions": True, "source_description": True,
                  "few_shot_evaluator": False, "preview": False},
    "SEMANTIC":  {"query_language": "DAX", "example_queries": False, "source_instructions": False, "source_description": False,
                  "prep_for_ai": True, "few_shot_evaluator": False, "preview": False},
    "ONTOLOGY":  {"query_language": "ONTOLOGY_NATIVE", "example_queries": False, "source_instructions": None, "source_description": None,
                  "few_shot_evaluator": False, "preview": True},
    "GRAPH":     {"query_language": "GQL", "example_queries": None, "source_instructions": None, "source_description": None,
                  "few_shot_evaluator": False, "preview": True},
    "AI_SEARCH": {"query_language": "SEARCH", "example_queries": None, "source_instructions": None, "source_description": None,
                  "few_shot_evaluator": False, "preview": True},
}
# Dedicated example-query support table (per source kind). None = not listed → never failed for missing examples.
EXAMPLE_QUERY_SUPPORT = {
    "LAKEHOUSE": True, "WAREHOUSE": True, "KQL_DATABASE": True, "SEMANTIC_MODEL": False, "ONTOLOGY": False,
    "FABRIC_SQL_DATABASE": None, "MIRRORED_DATABASE": None, "GRAPH_MODEL": None, "AZURE_AI_SEARCH": None,
}


def capability(source, key):
    """Live evidence (source.metadata['capabilities']) overrides the static registry."""
    live = (source.metadata or {}).get("capabilities", {}) if source is not None else {}
    if key in live:
        return live[key]
    if key == "example_queries" and source is not None:
        return EXAMPLE_QUERY_SUPPORT.get(source.source_kind, None)
    fam = source.source_family if source is not None else None
    return CAPABILITIES.get(fam, {}).get(key)


# ------------------------------------------------------------------
# Documentation map (last reviewed = DOC_SNAPSHOT_DATE)
# ------------------------------------------------------------------
_L = "https://learn.microsoft.com/en-us/fabric"
DOCS = {
    "datasources":    f"{_L}/data-science/data-agent-add-datasources",
    "create":         f"{_L}/data-science/how-to-create-data-agent",
    "config":         f"{_L}/data-science/data-agent-configurations",
    "best_practices": f"{_L}/data-science/data-agent-configuration-best-practices",
    "runtime":        f"{_L}/data-science/data-agent-runtime",
    "examples":       f"{_L}/data-science/data-agent-example-queries",
    "sharing":        f"{_L}/data-science/data-agent-sharing",
    "sdk":            f"{_L}/data-science/fabric-data-agent-sdk",
    "evaluate":       f"{_L}/data-science/evaluate-data-agent",
    "alm":            f"{_L}/data-science/data-agent-source-control",
    "tenant":         f"{_L}/data-science/data-agent-tenant-settings",
    "sql":            f"{_L}/data-science/data-agent-sql-sources",
    "schema_desc":    f"{_L}/data-science/data-agent-schema-object-descriptions",
    "topics":         f"{_L}/data-science/data-agent-topics",
    "semantic":       f"{_L}/data-science/semantic-model-best-practices",
    "prep_ai":        "https://learn.microsoft.com/en-us/power-bi/create-reports/copilot-prepare-data-ai",
    "copilot_sm":     "https://learn.microsoft.com/en-us/power-bi/create-reports/copilot-semantic-models",
    "copilot_eval":   "https://learn.microsoft.com/en-us/power-bi/create-reports/copilot-evaluate-data",
    "star":           "https://learn.microsoft.com/en-us/power-bi/guidance/star-schema",
    "kql":            "https://learn.microsoft.com/en-us/kusto/query/best-practices?view=microsoft-fabric",
    "gql":            f"{_L}/graph/gql-language-guide",
    "graph_schema":   f"{_L}/graph/design-graph-schema",
    "gql_perf":       f"{_L}/graph/gql-query-performance",
    "ontology_agent": f"{_L}/iq/ontology/how-to-create-data-agent",
    "ontology_integ": f"{_L}/iq/ontology/concepts-agent-integration",
    "ontology_tut":   f"{_L}/iq/ontology/tutorial-1-create-ontology",
    "ai_search":      f"{_L}/data-science/data-agent-ai-search-index",
}

# ------------------------------------------------------------------
# Versioned watch list of current limitations (§24 of the update guide)
# ------------------------------------------------------------------
KNOWN_LIMITATIONS = pd.DataFrame([
    ("KL-01", "A data agent supports up to five data sources.", "DA-003", "datasources"),
    ("KL-02", "Preview features and their configuration contracts can change.", "DA-013", "runtime"),
    ("KL-03", "SQL schema object descriptions and topics require the preview runtime.", "SQL-021|SQL-022", "schema_desc"),
    ("KL-04", "The example-query evaluator (evaluate_few_shots) is documented as SQL-only.", "SQL-013|KQL-014", "examples"),
    ("KL-05", "Semantic models and Ontology are documented as not supporting example queries.", "EXQ-001", "examples"),
    ("KL-06", "Docs disagree on top-3 vs typically top-4 examples passed to the query tool — validate via run steps.", "DA-024", "examples"),
    ("KL-07", "Column-level source selection is not supported in the Git item definition; table selection controls inclusion.", "DA-011", "alm"),
    ("KL-08", "Graph schema structural changes currently require a full reload.", "GRAPH-009", "graph_schema"),
    ("KL-09", "Data agent has a known issue with ontologies bound to semantic models.", "ONT-011", "ontology_agent"),
    ("KL-10", "Ontology guidance includes a temporary 'Support group by in GQL' agent instruction.", "ONT-012", "ontology_agent"),
    ("KL-11", "Q&A is documented for advanced-DAX value indexing, but Q&A retirement is scheduled for December 2026.", "SM-026", "semantic"),
    ("KL-12", "SDK, evaluation SDK, source control, Graph, Ontology, and Azure AI Search include preview capabilities.", "DA-013", "sdk"),
    ("KL-13", "Querying with the Fabric OpenAI client migrates from Assistants API to Responses API (Aug 2026); management code unaffected.", "DA-023", "sdk"),
], columns=["limitation_id", "limitation", "rule_ids", "docs_key"])
KNOWN_LIMITATIONS["docs_url"] = KNOWN_LIMITATIONS["docs_key"].map(DOCS)
KNOWN_LIMITATIONS["last_reviewed"] = DOC_SNAPSHOT_DATE


# ------------------------------------------------------------------
# Generic helpers
# ------------------------------------------------------------------
def _first(d, *keys, default=None):
    """Return the first non-empty value among keys (tolerates camelCase / snake_case contracts)."""
    if not isinstance(d, dict):
        return default
    for k in keys:
        if k in d and d[k] not in (None, "", [], {}):
            return d[k]
    return default


def _text(v) -> str:
    if v is None:
        return ""
    if isinstance(v, float) and pd.isna(v):
        return ""
    return str(v).strip()


_REDACTIONS = [
    (re.compile(r"(?i)bearer\s+[a-z0-9\-\._~\+/]+=*"), "Bearer ***"),
    (re.compile(r"eyJ[a-zA-Z0-9_\-]{10,}\.[a-zA-Z0-9_\-]{10,}\.[a-zA-Z0-9_\-]{5,}"), "***JWT***"),
    (re.compile(r"(?i)(password|pwd|secret|api[_-]?key|sig|accountkey)\s*[=:]\s*[^;\s,'\"]+"), r"\1=***"),
    (re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}"), "***@***"),
    (re.compile(r"\b\d{3}[-. ]\d{2}[-. ]\d{4}\b"), "***-**-****"),
]


def redact(text) -> str:
    """Mask tokens, secrets, and literal PII while preserving schema-level evidence."""
    s = _text(text)
    for rx, repl in _REDACTIONS:
        s = rx.sub(repl, s)
    return s


def short(text, n=160) -> str:
    s = re.sub(r"\s+", " ", _text(text))
    return s if len(s) <= n else s[: n - 1] + "…"


# ------------------------------------------------------------------
# Findings registry — keyed by section so re-running a cell never duplicates rows
# ------------------------------------------------------------------
_FINDINGS_BY_SECTION: dict = {}


def begin_section(section: str):
    _FINDINGS_BY_SECTION[section] = []


def record(section: str, items):
    if items is None:
        return
    if isinstance(items, Finding):
        items = [items]
    _FINDINGS_BY_SECTION.setdefault(section, []).extend(items)


def all_findings() -> list:
    return [f for sec in _FINDINGS_BY_SECTION.values() for f in sec]


def findings_df(items=None) -> pd.DataFrame:
    items = all_findings() if items is None else items
    cols = ["rule_id", "source_family", "source_type_raw", "source_id", "source_name", "object_path", "status",
            "severity", "title", "evidence", "recommendation", "automation", "confidence", "score_earned",
            "score_possible", "docs_url", "analyzer_version", "docs_snapshot_date", "observed_at_utc"]
    if not items:
        return pd.DataFrame(columns=cols)
    rows = []
    for f in items:
        r = asdict(f)
        r["status"] = f.status.value
        r["severity"] = f.severity.value
        r["analyzer_version"] = ANALYZER_VERSION
        r["docs_snapshot_date"] = DOC_SNAPSHOT_DATE
        rows.append(r)
    return pd.DataFrame(rows)[cols]


print(f"✅ Framework loaded — analyzer v{ANALYZER_VERSION}, documentation snapshot {DOC_SNAPSHOT_DATE}")
print(f"   Source-type alias table v{SOURCE_TYPE_ALIAS_VERSION}: {len(SOURCE_TYPE_ALIASES)} aliases → {len(SOURCE_KIND_TO_FAMILY)} source kinds")
if _SEMPY_IMPORT_ERROR:
    print(f"   ℹ️ sempy not available ({_SEMPY_IMPORT_ERROR}). Live Fabric checks will be NOT_EVALUATED.")

In [ ]:
# ------------------------------------------------------------------
# RULE CATALOG — stable IDs, default severity, automation level, docs, remediation
# (rule_id, family, title, severity, automation, docs_key, recommendation)
# ------------------------------------------------------------------
_RULE_ROWS = [
    # ---------------- Common agent checks (§6) ----------------
    ("DA-001", "AGENT", "Purpose and audience defined", "CRITICAL", "PARTIAL", "best_practices", "State intended users, supported decisions, and representative questions in the agent description or instructions."),
    ("DA-002", "AGENT", "Narrow mission", "CRITICAL", "PARTIAL", "best_practices", "Scope the agent to one domain; split 'answer anything' agents into domain-specific agents."),
    ("DA-003", "AGENT", "Supported source count (1–5)", "BLOCKER", "AUTOMATED", "datasources", "Connect between one and five data sources; split the agent if more are needed."),
    ("DA-004", "AGENT", "Known source types", "CRITICAL", "AUTOMATED", "datasources", "Update the source-type normalization map after verifying the current SDK/Git contract."),
    ("DA-005", "AGENT", "Descriptive, unique source names", "IMPORTANT", "AUTOMATED", "best_practices", "Give each source a unique business-readable name; avoid generic names such as Source1 or Database1."),
    ("DA-006", "AGENT", "Source description quality", "IMPORTANT", "PARTIAL", "best_practices", "Describe what the source contains, which questions it answers, and what distinguishes it from other sources."),
    ("DA-007", "AGENT", "Agent instruction structure", "IMPORTANT", "PARTIAL", "best_practices", "Structure agent instructions with Objective, Data sources and routing, Shared terminology, Response guidelines, and Out-of-scope sections."),
    ("DA-008", "AGENT", "Source-specific instructions", "CRITICAL", "PARTIAL", "best_practices", "Document objects, grain, joins, date rules, calculations, business values, and exceptions in data-source instructions."),
    ("DA-009", "AGENT", "Cross-source routing", "CRITICAL", "PARTIAL", "best_practices", "Name the authoritative source and precedence rule for every overlapping domain in agent instructions."),
    ("DA-010", "AGENT", "Cross-source metric consistency", "CRITICAL", "PARTIAL", "best_practices", "Align shared metric, date, currency, and status definitions across sources or document the differences."),
    ("DA-011", "AGENT", "Minimal schema selection", "CRITICAL", "AUTOMATED", "config", "Select only objects required by the declared scope; remove staging, audit, and unrelated objects."),
    ("DA-012", "AGENT", "Runtime recorded", "IMPORTANT", "AUTOMATED", "runtime", "Record whether the standard or preview runtime is used and republish after changing it."),
    ("DA-013", "AGENT", "Preview-risk disclosure", "IMPORTANT", "AUTOMATED", "runtime", "Label every preview source/configuration and record an explicit production decision (PREVIEW_ACCEPTED)."),
    ("DA-014", "AGENT", "Read-only contract", "BLOCKER", "AUTOMATED", "config", "Remove data-changing statements (INSERT/UPDATE/DELETE/MERGE/DDL/.set/.append/.drop) from examples and test queries."),
    ("DA-015", "AGENT", "Minimum source access", "BLOCKER", "PARTIAL", "sharing", "Verify the consumer identity can read required metadata and data without elevated authoring permissions."),
    ("DA-016", "AGENT", "Security propagation", "CRITICAL", "PARTIAL", "sharing", "Test RLS, CLS, source ACLs, and user-identity behavior with representative personas."),
    ("DA-017", "AGENT", "Tenant and cross-geo settings", "BLOCKER", "MANUAL", "tenant", "Enable and approve the required data agent, Copilot, and cross-geo AI tenant settings for the environment."),
    ("DA-018", "AGENT", "Outbound access rules", "CRITICAL", "MANUAL", "datasources", "Allow external connections required by the agent in workspace outbound data-connection rules."),
    ("DA-019", "AGENT", "Draft/published version separation", "CRITICAL", "AUTOMATED", "alm", "Publish an approved version for consumers and keep draft changes out of production until re-published."),
    ("DA-020", "AGENT", "Git and deployment workflow", "IMPORTANT", "PARTIAL", "alm", "Source-control the draft configuration and promote through dev/test/prod with reviews."),
    ("DA-021", "AGENT", "Source refresh / drift detection", "IMPORTANT", "AUTOMATED", "config", "Refresh the agent's source schema after changes and remove selections that no longer exist."),
    ("DA-022", "AGENT", "Ground-truth evaluation set", "CRITICAL", "AUTOMATED", "evaluate", "Maintain an evaluation set with question and expected_answer (plus expected_source, persona, criticality)."),
    ("DA-023", "AGENT", "End-to-end evaluation", "BLOCKER", "PARTIAL", "evaluate", "Evaluate the published sandbox/production stage and classify every failure."),
    ("DA-024", "AGENT", "Run-step inspection", "IMPORTANT", "PARTIAL", "evaluate", "Review generated SQL/DAX/KQL/GQL, the selected source, and retrieved examples/topics in run steps."),
    ("DA-025", "AGENT", "Operational ownership", "IMPORTANT", "MANUAL", "best_practices", "Document the owner, review cadence, incident path, and change approvers."),
    ("DA-026", "AGENT", "Capacity monitoring", "IMPORTANT", "MANUAL", "best_practices", "Monitor capacity consumption and query latency before production rollout."),
    # ---------------- SQL family (§7.2) ----------------
    ("SQL-001", "SQL", "Selected schema is focused", "CRITICAL", "AUTOMATED", "sql", "Select only the tables/views needed; exclude staging, audit, archive, and unrelated subject areas."),
    ("SQL-002", "SQL", "Selected objects resolve", "BLOCKER", "AUTOMATED", "sql", "Re-select or refresh objects that no longer exist or cannot be queried."),
    ("SQL-003", "SQL", "Business-readable names", "IMPORTANT", "AUTOMATED", "sql", "Rename or wrap objects with generic names, unexplained abbreviations, or near-duplicates in curated views."),
    ("SQL-004", "SQL", "Table/object description coverage", "IMPORTANT", "PARTIAL", "schema_desc", "Describe each selected object's entity/event, grain, purpose, and authoritative use."),
    ("SQL-005", "SQL", "Column description coverage", "IMPORTANT", "PARTIAL", "schema_desc", "Describe codes, flags, dates, units, status values, and ambiguous columns first."),
    ("SQL-006", "SQL", "Grain documented", "CRITICAL", "PARTIAL", "sql", "State 'one row per …' for every fact/event table in instructions or descriptions."),
    ("SQL-007", "SQL", "Join path documented", "CRITICAL", "PARTIAL", "sql", "Document valid join keys and required paths; flag fan-out and fact-to-fact joins."),
    ("SQL-008", "SQL", "Authoritative objects identified", "CRITICAL", "PARTIAL", "sql", "Add usage rules for overlapping current/archive, raw/curated, and snapshot/transaction objects."),
    ("SQL-009", "SQL", "Business values documented", "CRITICAL", "PARTIAL", "sql", "Define codes, booleans, units, nulls, and sentinel values in data-source instructions."),
    ("SQL-010", "SQL", "Time logic documented", "CRITICAL", "PARTIAL", "sql", "Define the default event date, fiscal calendar, timezone, and period boundaries."),
    ("SQL-011", "SQL", "Calculation rules are executable", "CRITICAL", "PARTIAL", "sql", "Represent key metrics as stable views or exact instructions/examples rather than vague prose."),
    ("SQL-012", "SQL", "Example questions are distinct", "IMPORTANT", "AUTOMATED", "examples", "Remove duplicate-intent, contradictory, or multi-intent example questions."),
    ("SQL-013", "SQL", "Example SQL passes validation", "CRITICAL", "AUTOMATED", "examples", "Fix example queries that fail syntax, permission, or result-shape validation."),
    ("SQL-014", "SQL", "Question-to-SQL mapping", "CRITICAL", "PARTIAL", "examples", "Ensure literals, dates, filters, grouping, and metrics in the question appear in the query."),
    ("SQL-015", "SQL", "Avoid broad projection", "IMPORTANT", "AUTOMATED", "examples", "Replace SELECT * with explicit columns in examples and test queries."),
    ("SQL-016", "SQL", "Result size bounded where appropriate", "IMPORTANT", "PARTIAL", "examples", "Use TOP/filters for detail/list examples; aggregations need not be limited."),
    ("SQL-017", "SQL", "Query plans are practical", "IMPORTANT", "MANUAL", "sql", "Test representative complex joins, grouping, and filters against latency goals."),
    ("SQL-018", "SQL", "Data quality smoke tests", "CRITICAL", "AUTOMATED", "sql", "Check key uniqueness, null rates, referential gaps, and duplicate business rows on selected objects."),
    ("SQL-019", "SQL", "Freshness and availability", "CRITICAL", "AUTOMATED", "sql", "Record the latest data timestamp and endpoint query success against FRESHNESS_MAX_AGE_HOURS."),
    ("SQL-020", "SQL", "Least privilege", "BLOCKER", "PARTIAL", "sharing", "Grant consumers only the read/query access they need; verify row/column restrictions."),
    ("SQL-021", "SQL", "Preview descriptions are purposeful", "IMPORTANT", "PARTIAL", "schema_desc", "Write schema object descriptions that add business context instead of repeating the name."),
    ("SQL-022", "SQL", "Topics are well structured", "IMPORTANT", "PARTIAL", "topics", "Use descriptive, self-contained, non-conflicting topics and verify them through run steps."),
    ("SQL-023", "SQL", "Runtime behavior is intentional", "IMPORTANT", "AUTOMATED", "runtime", "Default production to the standard runtime unless preview features are accepted and regression-tested."),
    ("LH-001", "SQL", "Lakehouse: selected Delta tables appear in the SQL analytics endpoint", "BLOCKER", "AUTOMATED", "sql", "Verify each selected table is visible through the SQL analytics endpoint."),
    ("LH-002", "SQL", "Lakehouse: schema synchronization is current", "CRITICAL", "AUTOMATED", "sql", "Refresh SQL endpoint metadata and the agent's schema after lakehouse changes."),
    ("LH-003", "SQL", "Lakehouse: shortcut-backed tables are reachable", "CRITICAL", "PARTIAL", "sql", "Record shortcut target, ownership, permission, and freshness; fix broken shortcuts."),
    ("LH-004", "SQL", "Lakehouse: curated tables preferred over raw/bronze", "IMPORTANT", "AUTOMATED", "sql", "Select curated (silver/gold) tables rather than raw/bronze/staging tables."),
    ("LH-005", "SQL", "Lakehouse: Delta health and file layout reviewed", "RECOMMENDED", "MANUAL", "sql", "Review OPTIMIZE/V-Order and small-file health (operational, not a product requirement)."),
    ("WH-001", "SQL", "Warehouse: consumer can SELECT required objects", "BLOCKER", "PARTIAL", "sharing", "Grant Read / SELECT on the required objects to consumers."),
    ("WH-002", "SQL", "Warehouse: curated views encapsulate fragile joins/calculations", "IMPORTANT", "AUTOMATED", "sql", "Expose curated views for fragile joins and calculations."),
    ("WH-003", "SQL", "Warehouse: schema and object names are business-readable", "IMPORTANT", "AUTOMATED", "sql", "Avoid exposing technical/staging schemas and object names."),
    ("WH-004", "SQL", "Warehouse: representative queries meet latency target", "IMPORTANT", "PARTIAL", "sql", "Measure p50/p95 latency of representative queries against LATENCY_TARGET_SECONDS."),
    ("SQLDB-001", "SQL", "SQL DB: required schemas and tables are discoverable", "BLOCKER", "AUTOMATED", "sql", "Validate metadata discovery and a bounded read for each selected object."),
    ("SQLDB-002", "SQL", "SQL DB: transactional objects narrowed to analytics-safe views", "IMPORTANT", "AUTOMATED", "sql", "Expose analytics-safe views instead of raw OLTP tables."),
    ("SQLDB-003", "SQL", "SQL DB: status, soft-delete, and current-row semantics documented", "CRITICAL", "PARTIAL", "sql", "Document IsDeleted/IsActive/ValidTo semantics so inactive rows are excluded."),
    ("MIR-001", "SQL", "Mirrored DB: mirroring state is healthy", "BLOCKER", "PARTIAL", "sql", "Verify mirroring is running and selected objects are replicated before testing the agent."),
    ("MIR-002", "SQL", "Mirrored DB: replication lag is measured", "CRITICAL", "PARTIAL", "sql", "Compare source update time with mirrored availability against a configured threshold."),
    ("MIR-003", "SQL", "Mirrored DB: source-to-mirror schema drift detected", "CRITICAL", "AUTOMATED", "sql", "Flag missing/renamed objects and type changes between source and mirror."),
    ("MIR-004", "SQL", "Mirrored DB: external-source and outbound rules satisfied", "CRITICAL", "MANUAL", "datasources", "Confirm workspace outbound protection allows the mirrored connection."),
    # ---------------- KQL (§8) ----------------
    ("KQL-001", "KQL", "Source scope is focused", "CRITICAL", "AUTOMATED", "datasources", "Select only required tables, materialized views, functions, and shortcuts."),
    ("KQL-002", "KQL", "Selected objects resolve and are readable", "BLOCKER", "AUTOMATED", "datasources", "Run bounded schema and query smoke tests; re-select missing objects."),
    ("KQL-003", "KQL", "Names and source description are clear", "IMPORTANT", "PARTIAL", "best_practices", "Explain the event domain, retention window, and authoritative tables."),
    ("KQL-004", "KQL", "Event time is typed datetime", "CRITICAL", "AUTOMATED", "kql", "Store primary event time as datetime rather than Unix epoch long values."),
    ("KQL-005", "KQL", "Default time column and timezone defined", "CRITICAL", "PARTIAL", "best_practices", "State the default time column and timezone in data-source instructions."),
    ("KQL-006", "KQL", "Early selective filters", "IMPORTANT", "AUTOMATED", "kql", "Put where (datetime first) immediately after the table reference."),
    ("KQL-007", "KQL", "Specific table and column references", "IMPORTANT", "AUTOMATED", "kql", "Avoid union * / search * when named tables and columns can be used."),
    ("KQL-008", "KQL", "Efficient string predicates", "IMPORTANT", "AUTOMATED", "kql", "Prefer has / == over contains when token semantics allow."),
    ("KQL-009", "KQL", "Bounded exploratory results", "IMPORTANT", "AUTOMATED", "kql", "Use take/limit/count/summarize on detail queries."),
    ("KQL-010", "KQL", "Dynamic-field extraction strategy", "IMPORTANT", "PARTIAL", "kql", "Materialize frequently extracted dynamic fields at ingestion where appropriate."),
    ("KQL-011", "KQL", "Repeated heavy aggregations", "IMPORTANT", "PARTIAL", "kql", "Consider materialized views for common aggregations."),
    ("KQL-012", "KQL", "Join strategy reviewed", "IMPORTANT", "PARTIAL", "kql", "Check join input sizes and cross-cluster joins; encode stable patterns in examples."),
    ("KQL-013", "KQL", "Functions callable for intended questions", "CRITICAL", "AUTOMATED", "datasources", "Validate required parameters, permissions, and bounded sample results of selected functions."),
    ("KQL-014", "KQL", "Example question/KQL pairs are valid", "CRITICAL", "AUTOMATED", "examples", "Fix examples with invalid syntax, literals, time filters, grouping, or result shape."),
    ("KQL-015", "KQL", "Example conflicts are absent", "CRITICAL", "PARTIAL", "examples", "Resolve examples with the same intent but different sources, filters, or aggregation."),
    ("KQL-016", "KQL", "Retention and freshness documented", "CRITICAL", "PARTIAL", "best_practices", "Record the retention policy window and the newest event timestamp."),
    ("KQL-017", "KQL", "Shortcut targets are healthy", "CRITICAL", "PARTIAL", "datasources", "Verify shortcut availability, permission, and expected latency."),
    ("KQL-018", "KQL", "Reader permissions are sufficient", "BLOCKER", "PARTIAL", "sharing", "Grant the Reader role on the KQL database to consumers."),
    # ---------------- Graph (§9, preview) ----------------
    ("GRAPH-001", "GRAPH", "Preview accepted", "CRITICAL", "MANUAL", "datasources", "Record explicit owner acceptance of preview support and regression risk."),
    ("GRAPH-002", "GRAPH", "Node labels are business-readable", "IMPORTANT", "AUTOMATED", "graph_schema", "Label nodes with recognizable entities, not raw table names."),
    ("GRAPH-003", "GRAPH", "Edge labels express business relationships", "CRITICAL", "AUTOMATED", "graph_schema", "Use meaningful verbs such as purchases, owns, or reportsTo."),
    ("GRAPH-004", "GRAPH", "Every node type has a valid key", "BLOCKER", "AUTOMATED", "graph_schema", "Define a present, non-null, unique (or compound) key for every node type."),
    ("GRAPH-005", "GRAPH", "Node/edge source mappings resolve", "BLOCKER", "AUTOMATED", "graph_schema", "Fix mappings whose source table or columns no longer exist."),
    ("GRAPH-006", "GRAPH", "Edge endpoints and direction are correct", "CRITICAL", "PARTIAL", "graph_schema", "Align origin/target and matched key columns with business meaning."),
    ("GRAPH-007", "GRAPH", "Property selection is minimal", "IMPORTANT", "AUTOMATED", "graph_schema", "Include only properties needed for query and analysis."),
    ("GRAPH-008", "GRAPH", "Orphan and dangling relationships", "CRITICAL", "PARTIAL", "graph_schema", "Measure and disclose unmatched origin/target keys."),
    ("GRAPH-009", "GRAPH", "Graph schema is stable (reload after structural change)", "CRITICAL", "MANUAL", "graph_schema", "Verify a full reload completed after structural schema changes."),
    ("GRAPH-010", "GRAPH", "Representative traversals pass", "CRITICAL", "PARTIAL", "gql", "Test one-hop, multi-hop, optional, and aggregation questions."),
    ("GRAPH-011", "GRAPH", "GQL projection is narrow", "IMPORTANT", "AUTOMATED", "gql_perf", "Avoid RETURN *; return only required properties."),
    ("GRAPH-012", "GRAPH", "High-cardinality results are bounded", "IMPORTANT", "AUTOMATED", "gql_perf", "Use LIMIT or restrictive patterns where matches can expand broadly."),
    ("GRAPH-013", "GRAPH", "Traversals are shallow and targeted", "IMPORTANT", "AUTOMATED", "gql_perf", "Avoid unnecessary broad or variable-length traversals."),
    ("GRAPH-014", "GRAPH", "Filter placement preserves semantics", "IMPORTANT", "PARTIAL", "gql_perf", "Distinguish pattern constraints from post-match filters (shortest paths, optional matches)."),
    ("GRAPH-015", "GRAPH", "Current graph limitations checked", "IMPORTANT", "MANUAL", "gql", "Record current size, timeout, and feature limits from the active platform release."),
    ("GRAPH-016", "GRAPH", "Security and source permissions tested", "BLOCKER", "PARTIAL", "sharing", "Test representative users against the graph and its underlying data."),
    # ---------------- Ontology (§10, preview) ----------------
    ("ONT-001", "ONTOLOGY", "Preview accepted", "CRITICAL", "MANUAL", "ontology_agent", "Record explicit owner acceptance of preview status."),
    ("ONT-002", "ONTOLOGY", "Entity names are business-friendly", "IMPORTANT", "AUTOMATED", "ontology_tut", "Rename generated technical names (dim/fact table names)."),
    ("ONT-003", "ONTOLOGY", "Entity definitions are present", "CRITICAL", "AUTOMATED", "ontology_tut", "Define the real-world object and intended use of every entity."),
    ("ONT-004", "ONTOLOGY", "Every entity has a key", "BLOCKER", "AUTOMATED", "ontology_tut", "Define a key that uniquely identifies ingested records."),
    ("ONT-005", "ONTOLOGY", "Property bindings resolve", "BLOCKER", "AUTOMATED", "ontology_tut", "Fix bindings whose columns are missing or type-incompatible."),
    ("ONT-006", "ONTOLOGY", "Relationship names express business meaning", "CRITICAL", "AUTOMATED", "ontology_tut", "Rename generated relationship labels to meaningful verbs."),
    ("ONT-007", "ONTOLOGY", "Relationship matched columns align to keys", "BLOCKER", "AUTOMATED", "ontology_tut", "Match origin/target mapping columns to the entity key properties."),
    ("ONT-008", "ONTOLOGY", "Relationship coverage and orphans", "CRITICAL", "PARTIAL", "ontology_tut", "Identify missing required relationships and unmatched keys."),
    ("ONT-009", "ONTOLOGY", "Definitions, rules, and metrics are non-conflicting", "CRITICAL", "PARTIAL", "ontology_integ", "Align concept definitions across the ontology and other agent sources."),
    ("ONT-010", "ONTOLOGY", "Bound-source permissions are complete", "BLOCKER", "PARTIAL", "ontology_agent", "Grant Read on the ontology and Read/query on every bound source."),
    ("ONT-011", "ONTOLOGY", "Unsupported binding known issue (semantic-model bindings)", "BLOCKER", "AUTOMATED", "ontology_agent", "Rebind the ontology to lakehouse/KQL sources until the semantic-model binding issue is resolved."),
    ("ONT-012", "ONTOLOGY", "Aggregation workaround present ('Support group by in GQL')", "IMPORTANT", "AUTOMATED", "ontology_agent", "Add 'Support group by in GQL' to agent instructions while the documented aggregation issue applies."),
    ("ONT-013", "ONTOLOGY", "Relationship and aggregation prompts pass", "CRITICAL", "PARTIAL", "ontology_agent", "Test traversal, grouping, and aggregation questions in the evaluation set."),
    ("ONT-014", "ONTOLOGY", "Initialization/reload state is ready", "IMPORTANT", "MANUAL", "ontology_agent", "Wait for documented initialization before treating 'no data' as a schema failure."),
    ("ONT-015", "ONTOLOGY", "Provenance and governance are visible", "IMPORTANT", "PARTIAL", "ontology_integ", "Report underlying source bindings and permission dependencies."),
    # ---------------- Azure AI Search (§11, preview) ----------------
    ("AIS-001", "AI_SEARCH", "Preview accepted", "CRITICAL", "MANUAL", "ai_search", "Record explicit owner acceptance of preview status."),
    ("AIS-002", "AI_SEARCH", "Resource URL resolves", "BLOCKER", "PARTIAL", "ai_search", "Configure a reachable https://<service>.search.windows.net endpoint and index name."),
    ("AIS-003", "AI_SEARCH", "Role-based access is enabled", "BLOCKER", "MANUAL", "ai_search", "Enable role-based access control on the Azure AI Search service."),
    ("AIS-004", "AI_SEARCH", "Required roles are present", "BLOCKER", "MANUAL", "ai_search", "Assign Search Index Data Reader and Search Index Data Contributor to the connecting identity."),
    ("AIS-005", "AI_SEARCH", "User-identity access is tested", "CRITICAL", "MANUAL", "ai_search", "Verify query-time access respects the asking user's identity and index permissions."),
    ("AIS-006", "AI_SEARCH", "Source context is complete", "CRITICAL", "PARTIAL", "ai_search", "Describe index content, key fields, intended questions, and routing boundaries."),
    ("AIS-007", "AI_SEARCH", "Search mode is intentional", "IMPORTANT", "AUTOMATED", "ai_search", "Choose full-text, hybrid, or semantic search to match index capability and evaluation results."),
    ("AIS-008", "AI_SEARCH", "Document count within supported range (3–20)", "CRITICAL", "AUTOMATED", "ai_search", "Set the retrieval document count between 3 and 20 and tune with evaluation."),
    ("AIS-009", "AI_SEARCH", "Citation field is available", "CRITICAL", "AUTOMATED", "ai_search", "Expose one exact, case-sensitive field: url, sourceUrl, filePath, path, or folderPath."),
    ("AIS-010", "AI_SEARCH", "Citation values are populated and usable", "CRITICAL", "PARTIAL", "ai_search", "Populate valid source references on sampled documents."),
    ("AIS-011", "AI_SEARCH", "Searchable content field is populated", "BLOCKER", "PARTIAL", "ai_search", "Ensure representative documents have non-empty searchable text (proposed check)."),
    ("AIS-012", "AI_SEARCH", "Chunk metadata is traceable", "IMPORTANT", "PARTIAL", "ai_search", "Map each chunk to document, section/page, and source reference (proposed check)."),
    ("AIS-013", "AI_SEARCH", "Duplicate and empty chunks are controlled", "IMPORTANT", "PARTIAL", "ai_search", "Report duplicate content, empty chunks, and extreme chunk-size outliers (proposed check)."),
    ("AIS-014", "AI_SEARCH", "Index freshness is measured", "CRITICAL", "PARTIAL", "ai_search", "Record the latest indexed timestamp and source-to-index lag."),
    ("AIS-015", "AI_SEARCH", "Retrieval relevance evaluation", "CRITICAL", "PARTIAL", "ai_search", "Verify expected documents/chunks are retrieved for ground-truth prompts."),
    ("AIS-016", "AI_SEARCH", "Grounded answer and citation evaluation", "CRITICAL", "PARTIAL", "ai_search", "Verify final-answer facts are supported by retrieved content and citations are correct."),
    ("AIS-017", "AI_SEARCH", "Cross-source routing is explicit", "CRITICAL", "PARTIAL", "ai_search", "Route policy/prose to search and structured aggregations to SQL/KQL/DAX/GQL in instructions."),
    # ---------------- Cross-source (§14) ----------------
    ("XSR-001", "CROSS_SOURCE", "Domain overlap inventory", "CRITICAL", "PARTIAL", "best_practices", "Map every business topic to candidate sources (ROUTING_MATRIX_PATH)."),
    ("XSR-002", "CROSS_SOURCE", "Authoritative source per topic", "CRITICAL", "PARTIAL", "best_practices", "Name one preferred source per topic or an explicit reconciliation rule."),
    ("XSR-003", "CROSS_SOURCE", "Metric-definition consistency", "CRITICAL", "PARTIAL", "best_practices", "Compare formula, grain, currency, status, and date logic of shared metrics."),
    ("XSR-004", "CROSS_SOURCE", "Time semantics consistency", "CRITICAL", "PARTIAL", "best_practices", "Make event/transaction/snapshot time, fiscal calendar, and timezone explicit per source."),
    ("XSR-005", "CROSS_SOURCE", "Entity-key consistency", "CRITICAL", "PARTIAL", "best_practices", "Map customer/product/order identifiers across sources."),
    ("XSR-006", "CROSS_SOURCE", "Structured vs unstructured routing", "CRITICAL", "PARTIAL", "ai_search", "Route numerical aggregation to governed structured sources and document interpretation to search."),
    ("XSR-007", "CROSS_SOURCE", "Real-time vs historical routing", "CRITICAL", "PARTIAL", "best_practices", "Route real-time/time-series questions to Eventhouse and stable reporting to curated sources."),
    ("XSR-008", "CROSS_SOURCE", "Graph/ontology traversal routing", "IMPORTANT", "PARTIAL", "best_practices", "Route relationship-rich questions to graph or ontology when appropriate."),
    ("XSR-009", "CROSS_SOURCE", "Ambiguity handling", "CRITICAL", "PARTIAL", "best_practices", "Instruct the agent to ask a clarifying question instead of assuming a source."),
    ("XSR-010", "CROSS_SOURCE", "Cross-source answer reconciliation", "CRITICAL", "PARTIAL", "evaluate", "Test two-source questions for units, join keys, and time alignment."),
    ("XSR-011", "CROSS_SOURCE", "Permission mismatch behavior", "CRITICAL", "PARTIAL", "sharing", "Verify a user lacking one source gets a clear authorization outcome."),
    ("XSR-012", "CROSS_SOURCE", "Source failure behavior", "IMPORTANT", "MANUAL", "best_practices", "Verify the agent discloses unavailable sources and doesn't silently substitute."),
    # ---------------- Example queries (§13) ----------------
    ("EXQ-001", "AGENT", "Example-query capability respected", "IMPORTANT", "AUTOMATED", "examples", "Only author examples for sources that support them; move semantic-model guidance to Prep for AI."),
    # ---------------- Semantic model adapter (§12, v2.2.2 checks with stable IDs) ----------------
    ("SM-000", "SEMANTIC", "Semantic model metadata is readable", "BLOCKER", "AUTOMATED", "semantic", "Verify dataset/workspace names and Build permission on the semantic model."),
    ("SM-001", "SEMANTIC", "Star schema validation", "CRITICAL", "AUTOMATED", "star", "Refactor flat models into star schemas; resolve M:M, bidirectional filters, and isolated tables."),
    ("SM-002", "SEMANTIC", "Best Practice Analyzer", "IMPORTANT", "AUTOMATED", "copilot_eval", "Address BPA errors and warnings (performance, DAX quality, formatting)."),
    ("SM-003", "SEMANTIC", "Memory Analyzer", "RECOMMENDED", "AUTOMATED", "copilot_eval", "Remove unused columns and optimize high-cardinality columns."),
    ("SM-004", "SEMANTIC", "Data type validation", "IMPORTANT", "AUTOMATED", "copilot_eval", "Convert date/numeric values stored as text to proper types."),
    ("SM-005", "SEMANTIC", "Business-friendly naming", "IMPORTANT", "AUTOMATED", "copilot_eval", "Use plain business names with spaces; remove prefixes, abbreviations, and CamelCase/snake_case."),
    ("SM-006", "SEMANTIC", "Descriptions coverage and quality", "CRITICAL", "AUTOMATED", "copilot_eval", "Describe every object in the AI Data Schema, front-loading purpose, units, and calculation."),
    ("SM-007", "SEMANTIC", "Synonyms / linguistic schema", "RECOMMENDED", "AUTOMATED", "prep_ai", "Add targeted synonyms (prefer AI Instructions for acronyms); avoid over-synonymization."),
    ("SM-008", "SEMANTIC", "Row labels", "IMPORTANT", "AUTOMATED", "copilot_eval", "Set row labels on dimension tables."),
    ("SM-009", "SEMANTIC", "Explicit measures (no implicit aggregations)", "CRITICAL", "AUTOMATED", "copilot_eval", "Create explicit DAX measures and set numeric columns to 'Do not summarize'."),
    ("SM-010", "SEMANTIC", "Report-scoped measures", "RECOMMENDED", "MANUAL", "copilot_eval", "Move report-scoped measures into the model."),
    ("SM-011", "SEMANTIC", "Ambiguous date fields / marked date table", "IMPORTANT", "AUTOMATED", "copilot_sm", "Mark a date table and disambiguate multiple date columns."),
    ("SM-012", "SEMANTIC", "Hidden objects and private tables", "IMPORTANT", "AUTOMATED", "copilot_sm", "Hide PK/FK/sort/GUID columns and mark internal tables private."),
    ("SM-013", "SEMANTIC", "Model complexity / grounding truncation risk", "IMPORTANT", "AUTOMATED", "copilot_eval", "Reduce exposed objects via AI Data Schema; hide helpers."),
    ("SM-014", "SEMANTIC", "Performance Analyzer for AI Data Schema measures", "RECOMMENDED", "MANUAL", "copilot_eval", "Run Performance Analyzer against AI Data Schema measures before deployment."),
    ("SM-015", "SEMANTIC", "Duplicate column names across tables", "IMPORTANT", "AUTOMATED", "copilot_eval", "Rename duplicated visible column names to be table-specific."),
    ("SM-016", "SEMANTIC", "Copilot grounding completeness", "IMPORTANT", "AUTOMATED", "copilot_sm", "Add format strings, data categories, and calculation-group descriptions."),
    ("SM-017", "SEMANTIC", "Hierarchies (logical groupings)", "RECOMMENDED", "AUTOMATED", "copilot_eval", "Add hierarchies for natural drill paths (date, geography, product)."),
    ("SM-018", "SEMANTIC", "Description length — Copilot 200-character window", "RECOMMENDED", "AUTOMATED", "copilot_eval", "Front-load key information within the first 200 characters."),
    ("SM-019", "SEMANTIC", "AI Data Schema selection", "CRITICAL", "PARTIAL", "prep_ai", "Select only business-relevant tables/columns/measures in Prep for AI → Simplify data schema."),
    ("SM-020", "SEMANTIC", "Measure dependencies", "IMPORTANT", "AUTOMATED", "prep_ai", "Include every dependent object of selected measures in the AI Data Schema."),
    ("SM-021", "SEMANTIC", "Helper measures excluded", "IMPORTANT", "AUTOMATED", "prep_ai", "Hide helper/intermediate measures or exclude them from the AI Data Schema."),
    ("SM-022", "SEMANTIC", "Verified Answers", "IMPORTANT", "PARTIAL", "prep_ai", "Configure verified answers for top questions with 5–7 trigger questions and up to three filters."),
    ("SM-023", "SEMANTIC", "Prep for AI — AI Instructions (incl. negative scope)", "CRITICAL", "PARTIAL", "prep_ai", "Put semantic definitions, metric preferences, date defaults, synonyms, and negative scope in Prep for AI."),
    ("SM-024", "SEMANTIC", "Prep for AI configuration retrievable", "IMPORTANT", "AUTOMATED", "prep_ai", "Verify the Power BI MCP endpoint returns the model's Prep for AI configuration."),
    ("SM-025", "SEMANTIC", "Data agent table selection aligns with AI Data Schema", "CRITICAL", "AUTOMATED", "semantic", "Select the same tables in the data agent as in Prep for AI → AI Data Schema."),
    ("SM-026", "SEMANTIC", "DAX generation mode and value indexing recorded", "IMPORTANT", "AUTOMATED", "runtime", "Record standard vs advanced DAX (runtime) and re-check the Q&A value-indexing dependency after Dec 2026."),
    ("SM-027", "SEMANTIC", "Power BI service & admin readiness", "RECOMMENDED", "MANUAL", "copilot_sm", "Mark model Prepped for AI, endorse/tag it, confirm Copilot capacity and Q&A settings."),
]

RULES = {
    r[0]: {"family": r[1], "title": r[2], "severity": r[3], "automation": r[4], "docs_url": DOCS.get(r[5], DOCS["best_practices"]), "recommendation": r[6]}
    for r in _RULE_ROWS
}
assert len(RULES) == len(_RULE_ROWS), "Duplicate rule IDs in catalog"

_DEFAULT_RATIO = {Status.PASS: 1.0, Status.WARN: 0.5, Status.FAIL: 0.0}


def make_finding(rule_id, status, evidence, *, source=None, object_path=None, ratio=None, severity=None,
                 recommendation=None, confidence="HIGH", automation=None, title=None, weight=None, family=None):
    """Create a normalized Finding. `weight=0` creates an unscored detail row (one row per object)."""
    rule = RULES.get(rule_id, {"family": family or "AGENT", "title": title or rule_id, "severity": "INFO",
                               "automation": "MANUAL", "docs_url": DOCS["best_practices"], "recommendation": ""})
    status = Status(status)
    sev = Severity(severity or rule["severity"])
    possible = float(SEVERITY_WEIGHTS.get(sev.value, 0) if weight is None else weight)
    if status in EVALUATED_STATUSES:
        r = _DEFAULT_RATIO[status] if ratio is None else max(0.0, min(1.0, float(ratio)))
        earned = round(possible * r, 3)
    else:
        earned = 0.0
    if source is not None:
        fam = source.source_family if source.source_family != "UNKNOWN" else (family or rule["family"])
        sid, sname, sraw = source.source_id, source.display_name, source.source_type_raw
    else:
        fam, sid, sname, sraw = (family or rule["family"]), None, None, None
    return Finding(
        rule_id=rule_id, title=title or rule["title"], source_id=sid, source_name=sname, source_family=fam,
        object_path=object_path, status=status, severity=sev, score_earned=earned, score_possible=possible,
        evidence=redact(evidence), recommendation=recommendation or rule["recommendation"],
        automation=automation or rule["automation"], confidence=confidence, docs_url=rule["docs_url"], source_type_raw=sraw,
    )


def detail(rule_id, status, evidence, *, source=None, object_path=None, severity=None, confidence="HIGH"):
    """Unscored per-object detail row for a rule (does not change the score)."""
    return make_finding(rule_id, status, evidence, source=source, object_path=object_path, severity=severity,
                        confidence=confidence, weight=0)


def attestation_for(rule_id, source=None):
    keys = [f"{rule_id}@{source.display_name}"] if source is not None else []
    keys.append(rule_id)
    for k in keys:
        if k in MANUAL_ATTESTATIONS:
            return MANUAL_ATTESTATIONS[k]
    return None


def manual_check(rule_id, guidance, *, source=None, object_path=None):
    """MANUAL unless the user attested it in MANUAL_ATTESTATIONS (then USER_SUPPLIED evidence)."""
    att = attestation_for(rule_id, source)
    if att:
        st = str(att.get("status", "PASS")).upper()
        st = st if st in Status.__members__ else "PASS"
        return make_finding(rule_id, st, f"USER_SUPPLIED: {att.get('evidence', 'attested')}", source=source,
                            object_path=object_path, confidence="MEDIUM")
    return make_finding(rule_id, Status.MANUAL, guidance, source=source, object_path=object_path, confidence="LOW")


def safe_check(rule_id, fn, *, source=None, object_path=None):
    """Run a check function; exceptions become NOT_EVALUATED (never a passing score)."""
    try:
        out = fn()
        if out is None:
            return []
        return out if isinstance(out, list) else [out]
    except Exception as e:
        return [make_finding(rule_id, Status.NOT_EVALUATED, f"Check raised {type(e).__name__}: {short(e, 200)}",
                             source=source, object_path=object_path, confidence="LOW")]


def show_findings(items, title=None, max_rows=60):
    """Compact renderer for a list of findings (rendering is separate from checks)."""
    if title:
        print(title)
    if not items:
        print("   (no findings)")
        return
    df = findings_df(items)
    df = df.assign(_o=df["severity"].map(SEVERITY_ORDER), _s=(df["score_possible"] == 0).astype(int))
    df = df.sort_values(["_s", "_o", "rule_id"]).drop(columns=["_o", "_s"])
    view = df[["rule_id", "status", "severity", "source_name", "object_path", "title", "evidence"]].copy()
    view["status"] = view["status"].map(lambda s: f"{STATUS_ICON.get(s, '')} {s}")
    view["evidence"] = view["evidence"].map(lambda s: short(s, 140))
    display(view.head(max_rows))
    if len(view) > max_rows:
        print(f"   … {len(view) - max_rows} more rows (see exports)")


print(f"✅ Rule catalog loaded: {len(RULES)} rules")
print("   " + ", ".join(f"{fam}={n}" for fam, n in pd.Series([r['family'] for r in RULES.values()]).value_counts().items()))

In [ ]:
# ------------------------------------------------------------------
# QUERY LINT HELPERS — static, read-only heuristics shared by all adapters
# (need an allow-list in practice; findings are WARN-level unless noted)
# ------------------------------------------------------------------
import difflib

_SQL_COMMENT = re.compile(r"--[^\n]*|/\*.*?\*/", re.S)
_SQL_STRING = re.compile(r"'(?:''|[^'])*'")
_KQL_COMMENT = re.compile(r"//[^\n]*")
_DQ_STRING = re.compile(r'"(?:\\.|[^"\\])*"')

_SQL_MUTATING = re.compile(r"\b(insert|update|delete|merge|drop|alter|create|truncate|grant|revoke|deny|exec|execute|"
                           r"sp_[a-z_]+|bulk\s+insert|copy\s+into|into\s+#?\w+\s+from)\b", re.I)
_KQL_CONTROL = re.compile(r"(^|\n|;)\s*\.(?!show\b)[a-z\-]+", re.I)
_GQL_MUTATING = re.compile(r"\b(insert|create|delete|detach\s+delete|set|remove|merge|drop)\b", re.I)


def strip_sql(q: str) -> str:
    return _SQL_STRING.sub("''", _SQL_COMMENT.sub(" ", q or ""))


def strip_kql(q: str) -> str:
    q = _KQL_COMMENT.sub(" ", q or "")
    return _SQL_STRING.sub("''", _DQ_STRING.sub('""', q))


def query_language_for(source) -> str:
    return CAPABILITIES.get(source.source_family, {}).get("query_language", "UNKNOWN") if source else "UNKNOWN"


def read_only_violations(query: str, language: str) -> list:
    """Return data-changing tokens found in a query (empty list = read-only)."""
    lang = (language or "").upper()
    if lang == "SQL":
        return sorted({m.group(1).lower() for m in _SQL_MUTATING.finditer(strip_sql(query))})
    if lang == "KQL":
        return sorted({m.group(0).strip(" ;\n") for m in _KQL_CONTROL.finditer(strip_kql(query))})
    if lang == "GQL":
        return sorted({m.group(1).lower() for m in _GQL_MUTATING.finditer(strip_kql(query))})
    return []


# ---------- SQL ----------
_SQL_OBJ_REF = re.compile(r"\b(?:from|join)\s+((?:\[[^\]]+\]|[\w#]+)(?:\s*\.\s*(?:\[[^\]]+\]|\w+)){0,3})", re.I)
_SQL_AGG = re.compile(r"\b(count|sum|avg|min|max|stdev|var|string_agg|approx_count_distinct)\s*\(|\bgroup\s+by\b", re.I)


def sql_object_refs(q: str) -> set:
    refs = set()
    for m in _SQL_OBJ_REF.finditer(strip_sql(q)):
        ref = re.sub(r"[\[\]\s]", "", m.group(1)).lower()
        if ref and not ref.startswith(("(", "#")) and ref not in ("select",):
            refs.add(ref)
    return refs


def sql_cte_names(q: str) -> set:
    return {m.group(1).lower() for m in re.finditer(r"(?:\bwith|,)\s*(\w+)\s+as\s*\(", strip_sql(q), re.I)}


def lint_sql(q: str) -> list:
    """(rule_id, message) tuples for one SQL example."""
    s = strip_sql(q)
    out = []
    if not s.strip():
        return [("SQL-013", "Empty query")]
    if not re.match(r"^\s*(select|with|\()", s, re.I):
        out.append(("SQL-013", "Query does not start with SELECT/WITH"))
    if s.count("(") != s.count(")"):
        out.append(("SQL-013", "Unbalanced parentheses"))
    if re.search(r"\bselect\s+(top\s+\d+\s+)?(distinct\s+)?\*", s, re.I) or re.search(r",\s*\*\s*(,|\bfrom\b)", s, re.I):
        out.append(("SQL-015", "SELECT * — use explicit columns"))
    is_agg = bool(_SQL_AGG.search(s))
    bounded = bool(re.search(r"\btop\s*\(?\s*\d+|\boffset\s+\d+\s+rows|\bfetch\s+(first|next)\b|\bwhere\b", s, re.I))
    if not is_agg and not bounded:
        out.append(("SQL-016", "Detail query without TOP/WHERE/OFFSET-FETCH bound"))
    return out


# ---------- KQL ----------
def kql_first_source(q: str) -> str:
    s = strip_kql(q)
    body = [ln for ln in re.split(r";\s*\n?", s) if ln.strip() and not re.match(r"^\s*(let|declare|set)\b", ln.strip(), re.I)]
    head = (body[-1] if body else s).strip()
    m = re.match(r"^([\w\-\.\[\]'\"]+(?:\([^)]*\))?)", head)
    return re.sub(r"[\[\]'\"]", "", m.group(1)).split("(")[0].lower() if m else ""


def lint_kql(q: str) -> list:
    s = strip_kql(q)
    out = []
    if not s.strip():
        return [("KQL-014", "Empty query")]
    ops = [p.strip() for p in s.split("|")]
    if re.search(r"\bunion\s+\*|\bunion\s+[\w]*\*", s, re.I) or re.search(r"\bsearch\s+\*|^\s*search\b", s, re.I | re.M):
        out.append(("KQL-007", "Wildcard union/search — reference specific tables/columns"))
    if re.search(r"\bcontains\b", s, re.I):
        out.append(("KQL-008", "'contains' used — prefer 'has' / '==' when token semantics allow"))
    if len(ops) > 1:
        first_op = ops[1].lower()
        has_time_filter = bool(re.search(r"\b(ago|between|datetime|startofday|startofweek|startofmonth|now)\s*\(", s, re.I))
        if not first_op.startswith(("where", "take", "limit", "count", "summarize", "project")):
            out.append(("KQL-006", "First operator is not a selective 'where'"))
        elif first_op.startswith("where") and not has_time_filter and "summarize" not in s.lower():
            out.append(("KQL-006", "No datetime predicate on an event query"))
    bounded = bool(re.search(r"\|\s*(take|limit|top|count|summarize|distinct|sample)\b", s, re.I))
    if not bounded:
        out.append(("KQL-009", "Unbounded detail query — add take/limit/top/summarize"))
    return out


# ---------- GQL ----------
def lint_gql(q: str) -> list:
    s = strip_kql(q)
    out = []
    if re.search(r"\breturn\s+\*", s, re.I):
        out.append(("GRAPH-011", "RETURN * — return only required properties"))
    is_agg = bool(re.search(r"\b(count|sum|avg|min|max|collect)\s*\(|\bgroup\s+by\b", s, re.I))
    if re.search(r"\bmatch\b", s, re.I) and not is_agg and not re.search(r"\blimit\s+\d+", s, re.I):
        out.append(("GRAPH-012", "MATCH without LIMIT or aggregation"))
    if re.search(r"\]\s*->\s*\{\s*\d*\s*,\s*\}|\*\s*\d*\s*\.\.\s*\]|->\s*\{\s*\d+\s*,\s*\d{2,}\s*\}|\]\s*-\s*\{\s*\d*\s*,\s*\}", s):
        out.append(("GRAPH-013", "Unbounded or deep variable-length traversal"))
    if re.search(r"\boptional\s+match\b[^;]*?\bwhere\b", s, re.I | re.S):
        out.append(("GRAPH-014", "WHERE after OPTIONAL MATCH — confirm pattern vs post-match filter semantics"))
    return out


# ---------- Example-set helpers ----------
def _norm_q(q: str) -> str:
    return re.sub(r"[^a-z0-9 ]", "", re.sub(r"\s+", " ", (q or "").lower())).strip()


def duplicate_questions(examples: list, threshold: float = 0.9) -> list:
    """Pairs (i, j, ratio) of near-duplicate example questions."""
    qs = [_norm_q(e.get("question")) for e in examples]
    pairs = []
    for i in range(len(qs)):
        for j in range(i + 1, len(qs)):
            if qs[i] and qs[j]:
                r = 1.0 if qs[i] == qs[j] else difflib.SequenceMatcher(None, qs[i], qs[j]).ratio()
                if r >= threshold:
                    pairs.append((i, j, round(r, 2)))
    return pairs


def question_literals(question: str) -> list:
    """Literals a query should reflect: quoted strings and standalone numbers/years."""
    lits = re.findall(r"['\"]([^'\"]{2,})['\"]", question or "")
    lits += re.findall(r"(?<![\w.])(\d{2,}(?:\.\d+)?)(?![\w.])", question or "")
    return lits


def unmapped_literals(question: str, query: str) -> list:
    ql = (query or "").lower()
    return [l for l in question_literals(question) if l.lower() not in ql]


def is_multi_intent(question: str) -> bool:
    q = (question or "").lower()
    return q.count("?") > 1 or bool(re.search(r"\b(and also|as well as|and then)\b", q))


print("✅ Query lint helpers loaded (SQL · KQL · GQL · read-only contract · example-set checks)")

In [ ]:
print("=" * 70)
print("PREFLIGHT — ENVIRONMENT, DEPENDENCIES & PARAMETERS")
print("=" * 70)

from importlib import metadata as _md


def _pkg_version(name):
    try:
        return _md.version(name)
    except Exception:
        return None


try:
    import notebookutils  # noqa: F401  (available inside Fabric notebooks)
    IN_FABRIC = True
except Exception:
    IN_FABRIC = False

PACKAGES = {p: _pkg_version(p) for p in [
    "semantic-link-sempy", "semantic-link-labs", "fabric-data-agent-sdk", "httpx", "pyyaml", "azure-kusto-data", "pandas"]}

try:
    from fabric.dataagent.client import FabricDataAgentManagement  # noqa: F401
    SDK_AVAILABLE, SDK_IMPORT_ERROR = True, None
except Exception as _e:
    SDK_AVAILABLE, SDK_IMPORT_ERROR = False, f"{type(_e).__name__}: {_e}"

MGMT_STAGE = {"sandbox": "staging", "production": "published"}.get(str(DATA_AGENT_STAGE).lower())
OTHER_MGMT_STAGE = {"staging": "published", "published": "staging"}.get(MGMT_STAGE)
STANDALONE_SEMANTIC = not str(DATA_AGENT_NAME_OR_ID).strip()

param_errors = []
if str(COLLECTION_MODE).upper() not in {"LIVE_SDK", "GIT_EXPORT", "MANUAL_MANIFEST"}:
    param_errors.append(f"COLLECTION_MODE '{COLLECTION_MODE}' must be LIVE_SDK, GIT_EXPORT or MANUAL_MANIFEST")
if MGMT_STAGE is None:
    param_errors.append(f"DATA_AGENT_STAGE '{DATA_AGENT_STAGE}' must be 'sandbox' or 'production'")
if str(COLLECTION_MODE).upper() == "GIT_EXPORT" and not os.path.isdir(str(GIT_EXPORT_ROOT)):
    param_errors.append(f"GIT_EXPORT_ROOT '{GIT_EXPORT_ROOT}' is not a folder")
if str(COLLECTION_MODE).upper() == "MANUAL_MANIFEST" and not os.path.isfile(str(MANUAL_MANIFEST_PATH)):
    param_errors.append(f"MANUAL_MANIFEST_PATH '{MANUAL_MANIFEST_PATH}' is not a file")
if RUN_PERFORMANCE_TESTS and not RUN_LIVE_QUERY_TESTS:
    param_errors.append("RUN_PERFORMANCE_TESTS requires RUN_LIVE_QUERY_TESTS = True (performance tests skipped)")
if STANDALONE_SEMANTIC and str(COLLECTION_MODE).upper() == "LIVE_SDK" and not str(dataset).strip():
    param_errors.append("Set DATA_AGENT_NAME_OR_ID (agent mode) or dataset (standalone semantic-model mode)")

env_rows = [
    ("Analyzer version", ANALYZER_VERSION), ("Documentation snapshot", DOC_SNAPSHOT_DATE),
    ("Python", sys.version.split()[0]), ("Running in Fabric notebook", IN_FABRIC),
    ("Collection mode", COLLECTION_MODE),
    ("Mode", "Standalone semantic model (v2.x compatible)" if STANDALONE_SEMANTIC and str(COLLECTION_MODE).upper() == "LIVE_SDK" else "Data agent"),
    ("Data agent / stage", f"{DATA_AGENT_NAME_OR_ID or '—'} / {DATA_AGENT_STAGE} → management stage '{MGMT_STAGE}'"),
    ("fabric-data-agent-sdk importable", SDK_AVAILABLE if SDK_AVAILABLE else f"No ({SDK_IMPORT_ERROR})"),
    ("Live query tests", RUN_LIVE_QUERY_TESTS), ("Evaluation run", RUN_EVALUATION),
] + [(f"pkg: {k}", v or "not installed") for k, v in PACKAGES.items()]
display(pd.DataFrame(env_rows, columns=["Item", "Value"]))

if param_errors:
    print("\n⚠️  Parameter issues:")
    for e in param_errors:
        print(f"   • {e}")
else:
    print("\n✅ Parameters valid.")
if str(COLLECTION_MODE).upper() == "LIVE_SDK" and not STANDALONE_SEMANTIC and not SDK_AVAILABLE:
    print("\n❗ LIVE_SDK mode needs fabric-data-agent-sdk. Discovery will record UNKNOWN evidence.")
print("\n🔒 Read-only mode: no configuration or data will be modified.")

---
# 3️⃣ DATA AGENT DISCOVERY

Discovery reads the agent and its connected sources, then normalizes them into an `AgentProfile` with one `DataSourceProfile` per source.

| Mode | Reads | Notes |
|---|---|---|
| `LIVE_SDK` | `FabricDataAgentManagement.get_settings / list_datasources / get_configuration / get_fewshots` (public API), falling back to the legacy `get_configuration / get_datasources` | The SDK is **preview**. Optional methods are feature-detected. Nothing is written. |
| `GIT_EXPORT` | `files/config/{draft\|published}/stage_config.json`, `<source folder>/datasource.json`, `fewshots.json`, and `publish_info.json` | Table-level `is_selected` controls inclusion. Column-level selection isn't supported. |
| `MANUAL_MANIFEST` | JSON/YAML `agent:` manifest | Values are labeled `USER_SUPPLIED`. Missing values stay `UNKNOWN`. |

<details><summary>Manual manifest example (YAML)</summary>

```yaml
agent:
  name: SalesAndOperationsAgent
  runtime: standard            # standard | preview
  stage: sandbox
  description: Sales and delivery KPIs for regional sales managers.
  instructions: |
    ## Objective
    ...
  sources:
    - name: SalesWarehouse
      type: warehouse
      workspace: Analytics-Dev
      selected_objects: [dbo.FactSales, dbo.DimCustomer]
      description: Curated sales transactions and customer dimensions.
      instructions: |
        ...
      example_queries_file: examples/sales_warehouse.json   # or example_queries: [{question, query}]
    - name: PolicySearchIndex
      type: azure_ai_search
      endpoint: https://contoso.search.windows.net
      index_name: policies
      search_type: hybrid
      document_count: 5
      fields: [{name: content, type: Edm.String, searchable: true}, {name: sourceUrl, type: Edm.String}]
```
</details>

In [ ]:
print("=" * 70)
print("DATA AGENT DISCOVERY")
print("=" * 70)

SDK_HANDLES = {}   # source_id → SDK datasource handle (kept out of exported metadata)

_OBJECT_SUFFIXES = {"table", "view", "function", "materializedview", "shortcut", "externaltable", "entity", "entitytype",
                    "node", "nodetype", "edge", "edgetype", "relationship", "relationshiptype", "index", "storedprocedure"}
_COLUMN_SUFFIXES = {"column", "field", "property", "measure"}


def _as_dict(obj):
    if obj is None:
        return {}
    if isinstance(obj, dict):
        return obj
    v = getattr(obj, "value", None)            # SDK TaggedValue
    if isinstance(v, dict):
        return v
    if hasattr(obj, "__dict__"):
        return {k: v for k, v in vars(obj).items() if not k.startswith("_")}
    return {"_repr": repr(obj)}


def _suffix(etype) -> str:
    return str(etype or "").split(".")[-1].lower().replace("_", "")


def summarize_elements(elements) -> list:
    """Walk a datasource element tree → list of objects with columns. Table-level is_selected controls inclusion."""
    objects = []

    def walk(nodes, path):
        for el in nodes or []:
            if not isinstance(el, dict):
                continue
            name = _text(_first(el, "display_name", "displayName", "name", "id"))
            suf = _suffix(_first(el, "type"))
            children = list(_first(el, "children", "elements", default=[]) or [])
            if suf in _OBJECT_SUFFIXES or (not suf and _first(el, "columns")):
                cols = []
                for ch in children + list(_first(el, "columns", default=[]) or []):
                    if not isinstance(ch, dict):
                        continue
                    if _suffix(_first(ch, "type")) in _COLUMN_SUFFIXES or not _first(ch, "type"):
                        cols.append({
                            "name": _text(_first(ch, "display_name", "displayName", "name")),
                            "data_type": _text(_first(ch, "data_type", "dataType", "column_type", "columnType", "sql_type", "kusto_type")),
                            "description": _text(_first(ch, "description", "user_description", "userDescription")),
                        })
                objects.append({
                    "path": ".".join(path + (name,)), "name": name, "object_type": suf or "table",
                    "type_raw": _text(_first(el, "type")),
                    "is_selected": bool(_first(el, "is_selected", "isSelected", default=False)),
                    "description": _text(_first(el, "description", "user_description", "userDescription")),
                    "columns": cols,
                })
            else:
                walk(children, path + ((name,) if suf == "schema" and name else ()))

    walk(elements if isinstance(elements, list) else [elements] if isinstance(elements, dict) else [], ())
    return objects


def normalize_fewshots(fs) -> list:
    rows = []
    if fs is None:
        return rows
    if isinstance(fs, pd.DataFrame):
        fs = fs.to_dict("records")
    if isinstance(fs, dict):
        inner = _first(fs, "fewShots", "fewshots", "value", "examples")
        fs = inner if isinstance(inner, list) else [{"question": k, "query": v} for k, v in fs.items() if isinstance(v, str)]
    for i, item in enumerate(fs or []):
        if not isinstance(item, dict):
            continue
        q = _text(_first(item, "question", "Question", "natural language", "naturalLanguage"))
        s = _text(_first(item, "query", "Query", "sql", "kql", "gql"))
        if q or s:
            rows.append({"id": _text(_first(item, "id", "Id")) or f"ex-{i + 1}", "question": q, "query": s})
    return rows


def detect_runtime(settings) -> str:
    if not isinstance(settings, dict) or not settings:
        return "UNKNOWN"
    exp = settings.get("experimental") if isinstance(settings.get("experimental"), dict) else {}
    for v in (exp.get("enableExperimentalFeatures"), settings.get("enable_preview_runtime"),
              settings.get("enablePreviewRuntime"), settings.get("previewRuntime")):
        if isinstance(v, bool):
            return "PREVIEW" if v else "STANDARD"
    rt = _text(_first(settings, "runtime", "runtimeVersion", "runtime_version")).lower()
    if "preview" in rt:
        return "PREVIEW"
    if rt in ("standard", "ga", "default"):
        return "STANDARD"
    return "UNKNOWN"


# The public data agent API returns non-lakehouse sources as a generic "FabricItem" with an itemReference,
# so the real source kind has to be resolved from other evidence.
_ITEM_TYPE_KEYS = ("itemType", "item_type", "artifactType", "artifact_type", "artifactKind", "artifact_kind")
_ITEM_REF_KEYS = ("itemReference", "item_reference", "lakehouseReference", "lakehouse_reference", "datasourceReference")
_ITEM_LOOKUP_CACHE = {}


def _item_reference(cfg) -> dict:
    ref = _first(cfg, *_ITEM_REF_KEYS)
    return ref if isinstance(ref, dict) else {}


def _kind_from_elements(elements) -> str:
    """Element types carry the source prefix (e.g. semantic_model.table, lakehouse_tables.table)."""
    stack = list(elements if isinstance(elements, list) else [elements] if isinstance(elements, dict) else [])
    while stack:
        el = stack.pop()
        if not isinstance(el, dict):
            continue
        etype = _text(_first(el, "type"))
        if "." in etype:
            kind = normalize_source_type(etype.split(".")[0])
            if kind != "UNKNOWN":
                return kind
        stack.extend(_first(el, "children", "elements", default=[]) or [])
    return "UNKNOWN"


def _lookup_fabric_item(item_id, workspace_id):
    """Resolve an item ID in its workspace → (item type, display name, error)."""
    key = (_text(item_id).lower(), _text(workspace_id))
    if not key[0]:
        return None, None, None
    if key not in _ITEM_LOOKUP_CACHE:
        try:
            from sempy.fabric import list_items
            items = list_items(workspace=workspace_id or None)
            id_col = next((c for c in ("Id", "Item Id", "id") if c in items.columns), None)
            match = items[items[id_col].astype(str).str.lower() == key[0]] if id_col else items.iloc[0:0]
            if match.empty:
                _ITEM_LOOKUP_CACHE[key] = (None, None, f"item {item_id} not found in workspace {workspace_id or 'current'}")
            else:
                row = match.iloc[0]
                _ITEM_LOOKUP_CACHE[key] = (_text(row.get("Type")) or None, _text(row.get("Display Name")) or None, None)
        except Exception as e:
            _ITEM_LOOKUP_CACHE[key] = (None, None, f"item lookup failed: {short(e, 160)}")
    return _ITEM_LOOKUP_CACHE[key]


def build_source_profile(cfg, *, elements=None, fewshots=None, provenance="UNKNOWN", folder_kind=None,
                         source_id=None, default_workspace=None) -> DataSourceProfile:
    cfg = _as_dict(cfg)
    ref = _item_reference(cfg)
    raw_type = _first(cfg, "type", "datasource_type", "dataSourceType", "source_type", "kind")
    item_id = (_text(_first(cfg, "artifact_id", "artifactId", "item_id", "itemId"))
               or _text(_first(ref, "itemId", "item_id", "artifactId", "artifact_id")) or None)
    ws_id = (_text(_first(cfg, "workspace_id", "workspaceId", "workspace", "workspace_name"))
             or _text(_first(ref, "workspaceId", "workspace_id")) or default_workspace)
    element_src = elements if elements is not None else _first(cfg, "elements", default=[])
    name = _text(_first(cfg, "display_name", "displayName", "name"))

    kind, type_resolution, lookup_error = normalize_source_type(raw_type), "type", None
    if kind == "UNKNOWN":
        alt = _first(cfg, *_ITEM_TYPE_KEYS) or _first(ref, *_ITEM_TYPE_KEYS)
        if normalize_source_type(alt) != "UNKNOWN":
            kind, type_resolution = normalize_source_type(alt), f"item type '{_text(alt)}'"
    if kind == "UNKNOWN" and folder_kind:
        kind, type_resolution = folder_kind, "git folder"
    if kind == "UNKNOWN":
        k = _kind_from_elements(element_src)
        if k != "UNKNOWN":
            kind, type_resolution = k, "element types"
    if item_id and (kind == "UNKNOWN" or not name) and provenance == "SDK":
        item_type, item_name, lookup_error = _lookup_fabric_item(item_id, ws_id)
        name = name or _text(item_name)
        if kind == "UNKNOWN" and normalize_source_type(item_type) != "UNKNOWN":
            kind, type_resolution = normalize_source_type(item_type), f"workspace item type '{item_type}'"
    if kind == "UNKNOWN":
        type_resolution = "unresolved"

    name = name or _text(source_id) or "unnamed-source"
    objs = summarize_elements(element_src)
    selected = [o for o in objs if o["is_selected"]]
    explicit = _first(cfg, "selected_objects", "selectedObjects")
    if explicit and not selected:
        selected = [o if isinstance(o, dict) else {"path": str(o), "name": str(o).split(".")[-1], "object_type": "table",
                                                    "is_selected": True, "description": "", "columns": []}
                    for o in explicit]
        for o in selected:
            o.setdefault("path", o.get("name", "")); o.setdefault("columns", []); o.setdefault("description", "")
            o.setdefault("object_type", "table"); o["is_selected"] = True
    raw = {k: v for k, v in cfg.items() if k not in ("elements", "selected_objects", "selectedObjects")}
    return DataSourceProfile(
        source_id=_text(_first(cfg, "id", "datasource_id", "artifact_id", "artifactId")) or _text(source_id) or name,
        display_name=name,
        source_type_raw=_text(raw_type) or (folder_kind or "UNKNOWN"),
        source_family=SOURCE_KIND_TO_FAMILY.get(kind, "UNKNOWN"),
        source_kind=kind,
        workspace_id=ws_id,
        artifact_id=item_id,
        selected_objects=selected,
        instructions=_text(_first(cfg, "dataSourceInstructions", "data_source_instructions", "additional_instructions",
                                  "additionalInstructions", "instructions")) or None,
        description=_text(_first(cfg, "userDescription", "user_description", "description", "dataSourceDescription")) or None,
        example_queries=normalize_fewshots(fewshots if fewshots is not None else _first(cfg, "example_queries", "fewShots", "fewshots")),
        metadata={"all_objects": objs, "raw_config": raw, "element_tree_available": bool(objs),
                  "type_resolution": type_resolution, "item_lookup_error": lookup_error},
        provenance=provenance,
    )


# ------------------------------ LIVE_SDK ------------------------------
def _paged_elements(handle, stage, max_pages=50):
    items, token = [], None
    for _ in range(max_pages):
        resp = _as_dict(handle.get_elements(stage=stage, continuation_token=token))
        items.extend(resp.get("value", []) or [])
        token = _first(resp, "continuationToken", "continuation_token")
        if not token:
            break
    return items


def collect_live_sdk() -> AgentProfile:
    from fabric.dataagent.client import FabricDataAgentManagement
    mgmt = FabricDataAgentManagement(DATA_AGENT_NAME_OR_ID, WORKSPACE_ID_OR_NAME or None)
    api = getattr(mgmt, "_client", None)
    prof = AgentProfile(name=_text(getattr(api, "data_agent_name", None)) or str(DATA_AGENT_NAME_OR_ID),
                        agent_id=_text(getattr(api, "data_agent_id", None)) or None,
                        workspace=_text(getattr(api, "workspace_id", None)) or WORKSPACE_ID_OR_NAME or None,
                        stage=DATA_AGENT_STAGE, collection_mode="LIVE_SDK", provenance="SDK")
    prof.metadata["sdk_version"] = PACKAGES.get("fabric-data-agent-sdk")

    settings = None
    if hasattr(mgmt, "get_settings"):
        try:
            settings = _as_dict(mgmt.get_settings(MGMT_STAGE))
            prof.metadata["settings_api"] = "public"
        except Exception as e:
            prof.errors.append(f"get_settings({MGMT_STAGE}) failed: {short(e, 200)}")
    if not settings and hasattr(mgmt, "get_configuration"):
        try:
            c = mgmt.get_configuration()
            settings = {"aiInstructions": getattr(c, "instructions", None),
                        "enable_preview_runtime": getattr(c, "enable_preview_runtime", None)}
            prof.metadata["settings_api"] = "legacy"
        except Exception as e:
            prof.errors.append(f"legacy get_configuration failed: {short(e, 200)}")
    settings = settings or {}
    prof.instructions = _text(_first(settings, "aiInstructions", "ai_instructions", "additionalInstructions", "instructions")) or None
    prof.description = _text(_first(settings, "description", "publishDescription", "publishingDescription", "userDescription")) or None
    prof.runtime = detect_runtime(settings)
    prof.metadata["settings_keys"] = sorted(settings.keys())

    sources = []
    if hasattr(mgmt, "list_datasources"):
        try:
            for h in mgmt.list_datasources(MGMT_STAGE):
                hid = _text(getattr(h, "_id", None))
                cfg = _as_dict(h.get_configuration(MGMT_STAGE))
                elements = _first(cfg, "elements")
                if elements is None and hasattr(h, "get_elements"):
                    try:
                        elements = _paged_elements(h, MGMT_STAGE)
                    except Exception as e:
                        prof.errors.append(f"get_elements failed for {hid}: {short(e, 160)}")
                fs = None
                try:
                    fs = h.get_fewshots(MGMT_STAGE)
                except Exception:
                    pass
                sp = build_source_profile(cfg, elements=elements, fewshots=fs, provenance="SDK", source_id=hid,
                                          default_workspace=prof.workspace)
                SDK_HANDLES[sp.source_id] = h
                sources.append(sp)
        except Exception as e:
            prof.errors.append(f"list_datasources({MGMT_STAGE}) failed: {short(e, 200)}")
            sources = []
    if not sources and hasattr(mgmt, "get_datasources"):
        try:
            for ds in mgmt.get_datasources():
                cfg = _as_dict(ds.get_configuration())
                fs = None
                try:
                    fs = ds.get_fewshots()
                except Exception:
                    pass
                sp = build_source_profile(cfg, fewshots=fs, provenance="SDK", source_id=getattr(ds, "_id", None),
                                          default_workspace=prof.workspace)
                SDK_HANDLES[sp.source_id] = ds
                sources.append(sp)
            prof.metadata["datasource_api"] = "legacy"
        except Exception as e:
            prof.errors.append(f"legacy get_datasources failed: {short(e, 200)}")
    prof.sources = sources

    # Other stage snapshot (draft vs published) for DA-019
    if OTHER_MGMT_STAGE and hasattr(mgmt, "get_settings"):
        try:
            other = _as_dict(mgmt.get_settings(OTHER_MGMT_STAGE))
            names = []
            if hasattr(mgmt, "list_datasources"):
                for h in mgmt.list_datasources(OTHER_MGMT_STAGE):
                    names.append(_text(_first(_as_dict(h.get_configuration(OTHER_MGMT_STAGE)), "display_name", "displayName", "name")))
            prof.published_snapshot = {
                "stage": OTHER_MGMT_STAGE,
                "instructions": _text(_first(other, "aiInstructions", "ai_instructions", "additionalInstructions", "instructions")),
                "runtime": detect_runtime(other), "source_names": sorted(n for n in names if n),
            }
        except Exception as e:
            prof.published_snapshot = {"stage": OTHER_MGMT_STAGE, "error": short(e, 200)}
    return prof


# ------------------------------ GIT_EXPORT ------------------------------
def _read_json(path):
    with open(path, encoding="utf-8-sig") as fh:
        return json.load(fh)


def _find_config_dir(root):
    for cand in (os.path.join(root, "files", "config"), os.path.join(root, "config"), root):
        if os.path.isdir(os.path.join(cand, "draft")) or os.path.isfile(os.path.join(cand, "data_agent.json")):
            return cand
    for dirpath, dirnames, _ in os.walk(root):
        if "draft" in dirnames and os.path.basename(dirpath) == "config":
            return dirpath
    return None


def _folder_kind(folder_name):
    low = folder_name.lower()
    for prefix, kind in GIT_FOLDER_PREFIXES:
        if low.startswith(prefix):
            return kind
    return None


def _read_stage(stage_dir, default_workspace=None):
    out = {"settings": {}, "sources": []}
    if not os.path.isdir(stage_dir):
        return out
    sc = os.path.join(stage_dir, "stage_config.json")
    if os.path.isfile(sc):
        out["settings"] = _read_json(sc)
    for entry in sorted(os.listdir(stage_dir)):
        p = os.path.join(stage_dir, entry)
        dsj = os.path.join(p, "datasource.json")
        if os.path.isdir(p) and os.path.isfile(dsj):
            cfg = _read_json(dsj)
            fsj = os.path.join(p, "fewshots.json")
            fs = _read_json(fsj) if os.path.isfile(fsj) else None
            sp = build_source_profile(cfg, fewshots=fs, provenance="GIT", folder_kind=_folder_kind(entry),
                                      source_id=entry, default_workspace=default_workspace)
            sp.metadata["git_folder"] = entry
            sp.metadata["fewshots_file_present"] = os.path.isfile(fsj)
            out["sources"].append(sp)
    return out


def collect_git_export() -> AgentProfile:
    cfg_dir = _find_config_dir(GIT_EXPORT_ROOT)
    prof = AgentProfile(name=DATA_AGENT_NAME_OR_ID or os.path.basename(os.path.normpath(GIT_EXPORT_ROOT)), agent_id=None,
                        workspace=WORKSPACE_ID_OR_NAME or None, stage=DATA_AGENT_STAGE, collection_mode="GIT_EXPORT",
                        provenance="GIT")
    if cfg_dir is None:
        prof.errors.append(f"No files/config folder with draft/ or data_agent.json under {GIT_EXPORT_ROOT}")
        return prof
    prof.metadata["config_dir"] = cfg_dir
    for fname in ("data_agent.json", "publish_info.json"):
        fp = os.path.join(cfg_dir, fname)
        if os.path.isfile(fp):
            try:
                prof.metadata[fname] = _read_json(fp)
            except Exception as e:
                prof.errors.append(f"{fname}: {short(e, 160)}")
    pub = prof.metadata.get("publish_info.json") or {}
    prof.description = _text(_first(pub, "description", "publishDescription", "publishingDescription")) or None
    stage_folder = "draft" if MGMT_STAGE == "staging" else "published"
    other_folder = "published" if stage_folder == "draft" else "draft"
    cur = _read_stage(os.path.join(cfg_dir, stage_folder), prof.workspace)
    prof.sources = cur["sources"]
    prof.instructions = _text(_first(cur["settings"], "aiInstructions", "ai_instructions", "instructions")) or None
    prof.runtime = detect_runtime(cur["settings"])
    prof.metadata["stage_folder"] = stage_folder
    oth = _read_stage(os.path.join(cfg_dir, other_folder), prof.workspace)
    if oth["settings"] or oth["sources"]:
        prof.published_snapshot = {
            "stage": other_folder, "instructions": _text(_first(oth["settings"], "aiInstructions", "ai_instructions")),
            "runtime": detect_runtime(oth["settings"]), "source_names": sorted(s.display_name for s in oth["sources"]),
        }
    return prof


# ------------------------------ MANUAL_MANIFEST ------------------------------
def collect_manual_manifest() -> AgentProfile:
    path = MANUAL_MANIFEST_PATH
    with open(path, encoding="utf-8-sig") as fh:
        text = fh.read()
    if path.lower().endswith((".yaml", ".yml")):
        import yaml
        doc = yaml.safe_load(text)
    else:
        doc = json.loads(text)
    agent = (doc or {}).get("agent", doc or {})
    base = os.path.dirname(os.path.abspath(path))
    prof = AgentProfile(name=_text(agent.get("name")) or "manual-agent", agent_id=_text(agent.get("id")) or None,
                        workspace=_text(agent.get("workspace")) or WORKSPACE_ID_OR_NAME or None,
                        stage=_text(agent.get("stage")) or DATA_AGENT_STAGE, collection_mode="MANUAL_MANIFEST",
                        provenance="USER_SUPPLIED")
    rt = _text(agent.get("runtime")).lower()
    prof.runtime = "PREVIEW" if "preview" in rt else "STANDARD" if rt in ("standard", "ga") else "UNKNOWN"
    prof.instructions = _text(agent.get("instructions")) or None
    prof.description = _text(agent.get("description")) or None
    for k in ("owner", "review_cadence", "incident_path", "change_approvers", "git_connected", "deployment_pipeline"):
        if k in agent:
            prof.metadata[k] = agent[k]
    if isinstance(agent.get("published"), dict):
        prof.published_snapshot = agent["published"]
    for i, s in enumerate(agent.get("sources", []) or []):
        s = dict(s)
        exf = s.pop("example_queries_file", None)
        if exf:
            p = exf if os.path.isabs(exf) else os.path.join(base, exf)
            try:
                s["example_queries"] = _read_json(p)
            except Exception as e:
                prof.errors.append(f"example_queries_file '{exf}': {short(e, 120)}")
        sp = build_source_profile(s, provenance="USER_SUPPLIED", source_id=f"manual-{i + 1}",
                                  default_workspace=prof.workspace)
        sources_caps = s.get("capabilities")
        if isinstance(sources_caps, dict):
            sp.metadata["capabilities"] = sources_caps
        prof.sources.append(sp)
    return prof


# ------------------------------ Standalone semantic model ------------------------------
def collect_standalone_semantic() -> AgentProfile:
    ws = workspace or WORKSPACE_ID_OR_NAME or None
    src = DataSourceProfile(source_id=str(dataset), display_name=str(dataset), source_type_raw="semantic_model",
                            source_family="SEMANTIC", source_kind="SEMANTIC_MODEL", workspace_id=ws,
                            provenance="USER_SUPPLIED", metadata={"standalone": True})
    return AgentProfile(name=f"(standalone semantic model) {dataset}", agent_id=None, workspace=ws, stage=DATA_AGENT_STAGE,
                        collection_mode="STANDALONE_SEMANTIC", sources=[src], provenance="USER_SUPPLIED",
                        standalone_semantic=True)


_mode = str(COLLECTION_MODE).upper()
try:
    if _mode == "GIT_EXPORT":
        AGENT_PROFILE = collect_git_export()
    elif _mode == "MANUAL_MANIFEST":
        AGENT_PROFILE = collect_manual_manifest()
    elif STANDALONE_SEMANTIC:
        AGENT_PROFILE = collect_standalone_semantic()
    elif SDK_AVAILABLE:
        AGENT_PROFILE = collect_live_sdk()
    else:
        AGENT_PROFILE = AgentProfile(name=str(DATA_AGENT_NAME_OR_ID), agent_id=None, workspace=WORKSPACE_ID_OR_NAME or None,
                                     stage=DATA_AGENT_STAGE, collection_mode="LIVE_SDK",
                                     errors=[f"fabric-data-agent-sdk not importable: {SDK_IMPORT_ERROR}"])
except Exception as e:
    AGENT_PROFILE = AgentProfile(name=str(DATA_AGENT_NAME_OR_ID or dataset), agent_id=None, workspace=WORKSPACE_ID_OR_NAME or None,
                                 stage=DATA_AGENT_STAGE, collection_mode=_mode, errors=[f"Discovery failed: {type(e).__name__}: {short(e, 300)}"])

SOURCES = AGENT_PROFILE.sources
print(f"\n  Agent        : {AGENT_PROFILE.name}")
print(f"  Mode         : {AGENT_PROFILE.collection_mode}  (evidence: {AGENT_PROFILE.provenance})")
print(f"  Stage        : {AGENT_PROFILE.stage}")
print(f"  Runtime      : {AGENT_PROFILE.runtime}")
print(f"  Sources      : {len(SOURCES)}")
print(f"  Instructions : {'present (' + str(len(AGENT_PROFILE.instructions)) + ' chars)' if AGENT_PROFILE.instructions else 'not found'}")
if AGENT_PROFILE.errors:
    print("\n⚠️  Discovery warnings (affected checks become UNKNOWN / NOT_EVALUATED):")
    for e in AGENT_PROFILE.errors:
        print(f"   • {redact(e)}")

---
# 4️⃣ NORMALIZED SOURCE INVENTORY

Every connected source is mapped to a canonical **source kind** and **family**. The original product value is kept in `source_type_raw`. The capability columns show what the current documentation says each source supports. Live configuration evidence overrides this static registry.

> ℹ️ Some pages call the graph source **Graph Model** and others call it **Microsoft Graph in Fabric**. Both labels normalize to `GRAPH_MODEL`. Don't confuse it with the Microsoft Graph API.

In [ ]:
print("=" * 70)
print("NORMALIZED SOURCE INVENTORY")
print("=" * 70)


def _cap_label(v):
    return {True: "✅", False: "➖ n/a", None: "❔ not listed"}.get(v, str(v))


inventory_rows = []
for s in SOURCES:
    inventory_rows.append({
        "source_name": s.display_name,
        "source_kind": s.source_kind,
        "family": s.source_family,
        "source_type_raw": s.source_type_raw,
        "type_resolved_from": s.metadata.get("type_resolution", "type"),
        "preview": "⚠️ preview" if s.source_kind in PREVIEW_SOURCE_KINDS else "",
        "selected_objects": len(s.selected_objects),
        "has_instructions": bool(s.instructions),
        "has_description": bool(s.description),
        "example_queries": len(s.example_queries),
        "query_language": CAPABILITIES.get(s.source_family, {}).get("query_language", "UNKNOWN"),
        "examples_supported": _cap_label(capability(s, "example_queries")),
        "source_instructions_supported": _cap_label(capability(s, "source_instructions")),
        "evidence": s.provenance,
    })
SOURCE_INVENTORY = pd.DataFrame(inventory_rows)

if SOURCE_INVENTORY.empty:
    print("\n❔ No sources discovered. Source-level checks will be UNKNOWN. Review the discovery warnings above.")
else:
    display(SOURCE_INVENTORY)
    fam_counts = SOURCE_INVENTORY["family"].value_counts().to_dict()
    print(f"\n  Families: {fam_counts}")
    unknown = SOURCE_INVENTORY[SOURCE_INVENTORY["source_kind"] == "UNKNOWN"]
    if not unknown.empty:
        print(f"  ❔ {len(unknown)} source(s) with unrecognized type: {', '.join(unknown['source_type_raw'])}")
        for s in SOURCES:
            if s.source_kind == "UNKNOWN" and s.metadata.get("item_lookup_error"):
                print(f"     • {s.display_name}: {redact(s.metadata['item_lookup_error'])}")
    prev = SOURCE_INVENTORY[SOURCE_INVENTORY["preview"] != ""]
    if not prev.empty:
        print(f"  ⚠️ Preview sources: {', '.join(prev['source_name'])}  (PREVIEW_ACCEPTED = {PREVIEW_ACCEPTED})")

print("\n📋 Known limitations watch list (versioned):")
display(KNOWN_LIMITATIONS[["limitation_id", "limitation", "rule_ids", "last_reviewed"]])

---
## 4.1 🧭 SOURCE RUN PLAN

Check this as soon as discovery finishes. It shows which sources the agent **actually uses** and which source sections apply, so you don't have to run the whole notebook to find out.

| Usage | Meaning | What the source sections do |
|---|---|---|
| ✅ in use | Connected, and objects are selected (or selection isn't exposed) | Assess it |
| ⚠️ connected, nothing selected | Connected, but no tables or objects are selected, so the agent can't query it | Skip its deep checks when `SKIP_UNSELECTED_SOURCES = True` (recorded as `NOT_EVALUATED`). `DA-011` still reports it as `FAIL`. |
| ❔ unresolved type | The source type isn't in the alias table | No adapter runs. `DA-004` reports it. |

Sections marked **▶️ RUN** are the ones to run for this agent. You can skip sections marked **⏭️ SKIP**. Each source section (§6–§11, §16.2) can be skipped on its own. **Run all** is still safe, because skipped sections finish instantly with `NOT_APPLICABLE`.


In [ ]:
print("=" * 70)
print("SOURCE RUN PLAN — WHAT THE AGENT USES · WHICH SECTIONS TO RUN")
print("=" * 70)

# Source family → notebook sections that only do work when that family is connected.
SOURCE_SECTIONS = {
    "SQL":       "§6 SQL sources (Lakehouse · Warehouse · SQL DB · Mirrored)",
    "KQL":       "§7 Eventhouse KQL Database",
    "SEMANTIC":  "§8 Power BI semantic model (8.0–8.5) + §16.2 Power BI service & admin",
    "GRAPH":     "§9 Graph Model (preview)",
    "ONTOLOGY":  "§10 Fabric Ontology (preview)",
    "AI_SEARCH": "§11 Azure AI Search (preview)",
}
ALWAYS_RUN_SECTIONS = ("§4.2 Adapter framework · §5 Common agent checks · §12 Cross-source · §13 Example queries · "
                       "§14 Evaluation · §15 Security · §16 ALM · §17 Scorecard & exports")
_USAGE_ICON = {"IN_USE": "✅ in use", "NOT_SELECTED": "⚠️ connected, nothing selected", "UNRESOLVED": "❔ unresolved type"}


def source_usage(s) -> tuple:
    """(state, detail): IN_USE, NOT_SELECTED (connected, but the agent can't query it), or UNRESOLVED. Mirrors DA-011."""
    if s.source_family not in SOURCE_SECTIONS:
        return "UNRESOLVED", f"type '{s.source_type_raw}' not recognized, so no adapter can assess it"
    if s.source_family == "AI_SEARCH":
        return "IN_USE", "search index (schema selection doesn't apply)"
    n_sel, n_all = len(s.selected_objects), len(s.metadata.get("all_objects", []))
    if n_sel == 0 and n_all > 0:
        return "NOT_SELECTED", f"0 of {n_all} objects selected, so the agent can't query it"
    if n_sel:
        return "IN_USE", f"{n_sel} of {n_all} objects selected" if n_all else f"{n_sel} objects selected"
    return "IN_USE", "selection not exposed; assumed in use"


def skip_reason(s):
    """Why the source sections skip this source, or None when it should be assessed."""
    state, why = source_usage(s)
    if state == "NOT_SELECTED" and SKIP_UNSELECTED_SOURCES:
        return f"connected but not used by the agent ({why}); set SKIP_UNSELECTED_SOURCES = False to assess it anyway"
    return None


usage_rows = []
for s in SOURCES:
    state, why = source_usage(s)
    usage_rows.append({"source_name": s.display_name, "source_kind": s.source_kind, "usage": _USAGE_ICON[state], "detail": why,
                       "assessed_in": SOURCE_SECTIONS.get(s.source_family, "— (no adapter)") if state != "UNRESOLVED" and not skip_reason(s)
                       else "— (skipped)"})
SOURCE_USAGE = pd.DataFrame(usage_rows)

plan_rows, run_sections, skip_sections = [], [], []
for fam, sec in SOURCE_SECTIONS.items():
    connected = [s for s in SOURCES if s.source_family == fam]
    assessed = [s for s in connected if not skip_reason(s)]
    if assessed:
        action, why = "▶️ RUN", ", ".join(s.display_name for s in assessed)
        run_sections.append(sec)
    else:
        action = "⏭️ SKIP"
        why = (f"connected but not used: {', '.join(s.display_name for s in connected)}" if connected
               else "no source of this type is connected")
        skip_sections.append(sec)
    plan_rows.append({"section": sec, "action": action, "sources": why})
SOURCE_RUN_PLAN = pd.DataFrame(plan_rows)

if SOURCE_USAGE.empty:
    print("\n❗ No sources were discovered. Fix the discovery warnings in section 3 first;")
    print("   until then every source section only reports NOT_APPLICABLE.")
else:
    print("\nSources connected to the agent:")
    display(SOURCE_USAGE)
print("\nSection plan:")
display(SOURCE_RUN_PLAN)

print("\n▶️ Run for this agent :", "; ".join(run_sections) if run_sections else "none (no source section applies)")
print("⏭️ Safe to skip      :", "; ".join(skip_sections) if skip_sections else "none")
print("🔁 Always run        :", ALWAYS_RUN_SECTIONS)

unused = [s.display_name for s in SOURCES if source_usage(s)[0] == "NOT_SELECTED"]
if unused:
    print(f"\n⚠️ Connected but never queried (no objects selected): {', '.join(unused)}")
    print("   Select the objects the agent needs, or remove the source. DA-011 reports this as FAIL.")
    print("   Deep checks for these sources are " + ("skipped (SKIP_UNSELECTED_SOURCES = True)." if SKIP_UNSELECTED_SOURCES
                                                     else "still run (SKIP_UNSELECTED_SOURCES = False)."))
unresolved = [f"{s.display_name} ('{s.source_type_raw}')" for s in SOURCES if source_usage(s)[0] == "UNRESOLVED"]
if unresolved:
    print(f"\n❔ Unresolved source types: {', '.join(unresolved)}")
    print("   Add the raw type to SOURCE_TYPE_ALIASES (section 2), then re-run discovery.")


---
## 4.2 ⚙️ ADAPTER FRAMEWORK *(always run)*

This cell holds the shared adapter base class, safe dispatch, the read-only query executors, and the helpers used by more than one source section. Sections §6–§11 and §16 need it. Run it even when you skip source sections.


In [ ]:
# ------------------------------------------------------------------
# ADAPTER FRAMEWORK — base class, safe dispatch, read-only executors
# ------------------------------------------------------------------
ADAPTERS = {}          # source_kind → adapter instance (each section registers its adapters)
ADAPTER_RESULTS = {}   # source_id → metadata collected by the adapter (for summaries)


class SourceAdapter:
    source_family = "UNKNOWN"
    primary_rule = "DA-004"          # rule used when a whole step fails to evaluate

    def collect_metadata(self, source: DataSourceProfile) -> dict:
        return {}

    def run_schema_checks(self, source, metadata) -> list:
        return []

    def run_configuration_checks(self, source, metadata) -> list:
        return []

    def validate_examples(self, source, metadata) -> list:
        return []

    def smoke_test(self, source, metadata) -> list:
        return []

    def live_rules(self) -> list:
        """Rules that need RUN_LIVE_QUERY_TESTS; reported NOT_EVALUATED when live tests are off."""
        return []

    def run(self, source) -> list:
        out = []
        try:
            md = self.collect_metadata(source) or {}
        except Exception as e:
            md = {"collect_error": f"{type(e).__name__}: {short(e, 200)}"}
        ADAPTER_RESULTS[source.source_id] = md
        for step in (self.run_schema_checks, self.run_configuration_checks, self.validate_examples):
            out += safe_check(self.primary_rule, lambda step=step: step(source, md), source=source)
        if RUN_LIVE_QUERY_TESTS:
            out += safe_check(self.primary_rule, lambda: self.smoke_test(source, md), source=source)
        else:
            for rid in self.live_rules():
                out.append(make_finding(rid, Status.NOT_EVALUATED, "Live test not run (RUN_LIVE_QUERY_TESTS = False)",
                                        source=source, confidence="LOW"))
        return self._one_scored_row_per_rule(out)

    @staticmethod
    def _one_scored_row_per_rule(out):
        """When static and live results exist for the same rule, keep the last evaluated one scored; others become detail rows."""
        by_rule = {}
        for i, f in enumerate(out):
            if f.score_possible > 0:
                by_rule.setdefault(f.rule_id, []).append(i)
        for idxs in by_rule.values():
            if len(idxs) < 2:
                continue
            evaluated = [i for i in idxs if out[i].status in EVALUATED_STATUSES]
            keep = evaluated[-1] if evaluated else idxs[-1]
            for i in idxs:
                if i != keep:
                    out[i].score_possible, out[i].score_earned = 0.0, 0.0
        return out


def run_family(family: str, section: str, heading: str):
    """Dispatch every source of a family to its adapter; record findings under `section`."""
    begin_section(section)
    fam_sources = [s for s in SOURCES if s.source_family == family]
    print("=" * 70)
    print(heading)
    print("=" * 70)
    if not fam_sources:
        print(f"\n➖ NOT_APPLICABLE — no {family} source is connected to this agent (see 4.1 Source Run Plan).")
        return []
    all_out = []
    for s in fam_sources:
        adapter = ADAPTERS.get(s.source_kind)
        print(f"\n▶ {s.display_name}  [{s.source_kind} · raw type '{s.source_type_raw}' · evidence {s.provenance}]")
        reason = skip_reason(s)
        if reason:
            print(f"   ⏭️ Skipped — {reason}")
            f = [make_finding(adapter.primary_rule if adapter else "DA-004", Status.NOT_EVALUATED, f"Skipped: {reason}",
                              source=s, confidence="LOW")]
        elif adapter is None:
            f = [make_finding("DA-004", Status.UNKNOWN, f"No adapter registered for {s.source_kind}", source=s)]
        else:
            f = adapter.run(s)
        record(section, f)
        all_out += f
        show_findings(f)
    return all_out


# ---------------- Helpers shared by several source sections ----------------
# Kept here so each source section (6–11) can be skipped without breaking the others.
def _mentioned(name: str, text: str) -> bool:
    return bool(name) and bool(text) and name.split(".")[-1].lower() in text.lower()


_TECH_LABEL = re.compile(r"(?i:^(dim|fact|fct|tbl|stg|vw)_|_(id|key|sk|fk|pk)$|^(t|n|e|node|edge|table)\d+$)|^[A-Z0-9_]{4,}$")
_GENERIC_EDGE = re.compile(r"(?i:^(edge|rel|relationship|link|fk|ref|related|to|has)\d*$|_to_|^fk_)|^[A-Z0-9_]+$")


def _items(raw: dict, *keys) -> list:
    v = _first(raw, *keys, default=[])
    if isinstance(v, dict):
        v = [{**(val if isinstance(val, dict) else {"value": val}), "name": (val.get("name") if isinstance(val, dict) else None) or k}
             for k, val in v.items()]
    return [x for x in (v or []) if isinstance(x, dict)]


def _schema_part(raw, *keys):
    schema = _first(raw, "schema", "graph_schema", "graphSchema", "model", default={})
    return _items(raw, *keys) or (_items(schema, *keys) if isinstance(schema, dict) else [])


def preview_acceptance(rule_id, s):
    att = attestation_for(rule_id, s)
    if att or not PREVIEW_ACCEPTED:
        return manual_check(rule_id, "Record explicit owner acceptance of preview support (PREVIEW_ACCEPTED or MANUAL_ATTESTATIONS).", source=s)
    return make_finding(rule_id, Status.PASS, "USER_SUPPLIED: PREVIEW_ACCEPTED = True", source=s, confidence="MEDIUM")


# ---------------- Fabric REST (read-only GET / documented read POSTs) ----------------
def fabric_rest(method: str, path: str, payload=None):
    if fabric is None:
        raise RuntimeError("sempy not available")
    client = fabric.FabricRestClient()
    resp = client.get(path) if method.upper() == "GET" else client.post(path, json=payload or {})
    if resp.status_code >= 400:
        raise RuntimeError(f"HTTP {resp.status_code} for {method} {path}")
    try:
        return resp.json()
    except Exception:
        return {}


def resolve_workspace_id(ws):
    if not ws or fabric is None:
        return ws
    try:
        return fabric.resolve_workspace_id(ws)
    except Exception:
        return ws


# ---------------- Read-only query executors ----------------
_SQL_CONNECTORS = {
    "LAKEHOUSE": [("ConnectLakehouse", "lakehouse")],
    "WAREHOUSE": [("ConnectWarehouse", "warehouse")],
    "FABRIC_SQL_DATABASE": [("ConnectSQLDatabase", "sql_database")],
    "MIRRORED_DATABASE": [("ConnectMirroredDatabase", "mirrored_database"), ("ConnectSQLEndpoint", "item")],
}


def sql_connection(source):
    import sempy_labs as labs
    item = source.artifact_id or source.display_name
    ws = source.workspace_id or AGENT_PROFILE.workspace or None
    for cls_name, kw in _SQL_CONNECTORS.get(source.source_kind, []):
        cls = getattr(labs, cls_name, None)
        if cls is None:
            continue
        try:
            return cls(**{kw: item, "workspace": ws})
        except TypeError:
            return cls(item, workspace=ws)
    raise NotImplementedError(f"No semantic-link-labs SQL connector available for {source.source_kind}")


def run_sql(source, query: str, max_seconds=None):
    """Execute a read-only T-SQL statement. Blocks anything that fails the read-only lint."""
    bad = read_only_violations(query, "SQL")
    if bad:
        raise PermissionError(f"Blocked non-read-only statement: {bad}")
    t0 = time.perf_counter()
    with sql_connection(source) as conn:
        df = conn.query(query)
    return df, time.perf_counter() - t0


def run_kql(source, query: str):
    """Execute a read-only KQL query (request_readonly, row cap) through the SDK source wrapper."""
    bad = read_only_violations(query, "KQL")
    if bad:
        raise PermissionError(f"Blocked control command: {bad}")
    cfg = dict(source.metadata.get("raw_config", {}))
    cfg.setdefault("type", "kusto")
    cfg.setdefault("display_name", source.display_name)
    cfg.setdefault("database_name", _first(cfg, "database_name", "databaseName", "database") or source.display_name)
    cfg.setdefault("endpoint", _first(cfg, "endpoint", "query_uri", "queryUri", "cluster", "cluster_uri"))
    if not cfg.get("endpoint") and not cfg.get("workspace_id"):
        raise RuntimeError("No Eventhouse query endpoint in the source configuration")
    from fabric.dataagent.datasources import make_source
    t0 = time.perf_counter()
    df = make_source(cfg).execute(query)
    return df, time.perf_counter() - t0


def example_set_findings(source, dup_rule: str) -> list:
    """Distinctness / conflict checks shared by SQL and KQL example sets (conflicting duplicates → FAIL)."""
    ex = source.example_queries
    sup = capability(source, "example_queries")
    if not ex:
        if sup is True:
            return [make_finding(dup_rule, Status.WARN, "No example queries configured", source=source)]
        return [make_finding(dup_rule, Status.NOT_APPLICABLE, "No example queries (support not documented or not applicable)", source=source)]
    out = []
    pairs = duplicate_questions(ex)
    multi = [e for e in ex if is_multi_intent(e.get("question"))]
    conflicts = [(i, j, r) for i, j, r in pairs if _norm_q(ex[i]["query"]) != _norm_q(ex[j]["query"])]
    for i, j, r in pairs:
        kind = "Conflicting" if (i, j, r) in conflicts else "Near-duplicate"
        out.append(detail(dup_rule, Status.FAIL if kind == "Conflicting" else Status.WARN,
                          f"{kind} questions ({r}): '{short(ex[i]['question'], 60)}' ~ '{short(ex[j]['question'], 60)}'",
                          source=source, object_path=f"{ex[i]['id']}|{ex[j]['id']}"))
    for e in multi:
        out.append(detail(dup_rule, Status.WARN, f"Multi-intent question: '{short(e['question'], 80)}'", source=source, object_path=e["id"]))
    st = Status.FAIL if conflicts else Status.WARN if (pairs or multi) else Status.PASS
    out.insert(0, make_finding(dup_rule, st, f"{len(ex)} examples; {len(pairs)} near-duplicate pair(s) ({len(conflicts)} with different query logic); {len(multi)} multi-intent",
                               source=source))
    return out


print("✅ Adapter framework ready (SourceAdapter, run_family, read-only SQL/KQL executors)")

---
# 5️⃣ COMMON AGENT CHECKS

These checks run **once per agent**, or once per source where they're capability-gated. For example, source descriptions and data-source instructions are `NOT_APPLICABLE` for semantic models under current documented behavior. They aren't failures.

> 🧭 **Instruction layering:** agent-level instructions cover the objective, cross-source routing, shared terminology, response format, and tone. Source-specific query logic stays with the source configuration. For semantic models, DAX guidance belongs in **Prep for AI**, because the DAX generator **ignores data-agent-level instructions**.

<details><summary>Agent instruction template</summary>

```markdown
## Objective
State the focused outcome and intended users.

## Data sources and routing
- Use `<source A>` for ...
- Use `<source B>` for ...
- If both sources can answer ..., prefer ... because ...

## Shared terminology
Define only terms that apply across sources.

## Response guidelines
Specify concise output structure, units, currency, and tone.

## Out-of-scope requests
State what the agent cannot answer and when it should ask a clarifying question.
```
</details>

---
## 5.1 🎯 Scoping & Design Assessment · `DA-001` · `DA-002`

**Before diving into technical validation, ensure your Data Agent has a clear, narrow scope.**

> ⚠️ **Key Insight from Microsoft:** "Agents with laser-focused missions deliver value. General-purpose 'everything agents' fall short due to performance issues, confused logic, low trust, and maintenance challenges."

### Why Narrow Scope Matters

**Avoid:** ❌ General-purpose "everything agent" covering:
- Sales & Revenue Analytics
- Inventory & Supply Chain  
- Marketing & Campaign Performance
- Customer Support & Experience
- Finance & Forecasting
- Order Fulfillment & Logistics

**Prefer:** ✅ Domain-specific agents like:
- **Sales Intelligence Agent** — real-time order data, sales KPIs only
- **RevOps Intelligence Agent** — marketing campaigns + sales correlation only
- **Customer Experience Agent** — support tickets + satisfaction metrics only

### Scoping Checklist (Manual Assessment Required)

Before building your Data Agent, answer these questions:


In [ ]:
print("=" * 70)
print("CHECK 0: SCOPING & DESIGN ASSESSMENT (MANUAL)")
print("=" * 70)
print("\n🎯 CRITICAL: Complete this scoping exercise BEFORE technical configuration.\n")
print("✅ SCOPING CHECKLIST:")
print()
print("   📝 1. WHO ARE THE PRIMARY USERS?")
print("      Define specific user personas (e.g., 'sales analysts', 'regional managers')")
print("      NOT generic 'business users' or 'everyone in the company'")
print()
print("   📝 2. WHAT ARE THE MAIN FUNCTIONS? (List 3-5 specific tasks)")
print("      Examples:")
print("        ✅ 'Monitor real-time order volume by region'")
print("        ✅ 'Compare campaign ROI across marketing channels'")
print("        ✅ 'Identify top complaint categories by geography'")
print("      NOT:")
print("        ❌ 'Answer any business question'")
print("        ❌ 'Provide insights from all data'")
print()
print("   📝 3. TOP 10-15 QUESTIONS USERS WILL ASK")
print("      List actual expected questions:")
print("        • 'What are our top-selling products this week?'")
print("        • 'Which warehouses are running low on stock?'")
print("        • 'What's the ROI of Q1 influencer campaigns in APAC?'")
print()
print("   📝 4. WHAT'S OUT OF SCOPE?")
print("      Clearly define what this agent WON'T handle:")
print("        • HR/employee queries → not included")
print("        • Financial forecasting → separate agent")
print("        • Unstructured documents → not in scope")
print()
print("   📝 5. DATA AVAILABILITY CHECK")
print("      For each key question, verify:")
print("        ☐ Data exists in Fabric (Lakehouse/Warehouse/SQL DB/Mirrored/Eventhouse/Semantic Model/Graph/Ontology/AI Search)")
print("        ☐ Data is current (refresh frequency meets requirements)")
print("        ☐ Data quality is acceptable (no major gaps/errors)")
print("        ☐ Questions are answerable with existing data")
print()
print("   📝 6. SECURITY REQUIREMENTS")
print("      Define who can access this Data Agent:")
print("        ☐ Identify user personas who will query this agent")
print("        ☐ Document Row Level Security (RLS) requirements")
print("        ☐ List sensitive data fields to exclude from AI Data Schema")
print("        ☐ Confirm compliance with data privacy policies (GDPR, HIPAA, etc.)")
print("        ☐ Define if agent should be internal-only or externally accessible")
print("        ☐ Verify workspace-level permissions are appropriately scoped")
print()
print("⚠️  WARNING SIGNS OF SCOPE PROBLEMS:")
print("   • More than 20 tables selected in AI Data Schema")
print("   • Agent serves multiple unrelated business domains")
print("   • Users from 5+ different departments")
print("   • Mix of real-time and historical analytics")
print("   • Questions require 'why' or open-ended analysis")
print()
print("💡 DESIGN PATTERNS:")
print()
print("   ┌─────────────────────────────────────────────────────────┐")
print("   │ PATTERN 1: Domain-Specific Agents (RECOMMENDED)        │")
print("   ├─────────────────────────────────────────────────────────┤")
print("   │ • Sales Intelligence Agent → Eventhouse + Sales Model  │")
print("   │ • RevOps Agent → Marketing Data + Sales KPIs           │")
print("   │ • Cx Intelligence Agent → Support Lakehouse only        │")
print("   │                                                         │")
print("   │ Benefits: High accuracy, clear ownership, maintainable │")
print("   └─────────────────────────────────────────────────────────┘")
print()
print("   ┌─────────────────────────────────────────────────────────┐")
print("   │ PATTERN 2: Multi-Agent Orchestration (ADVANCED)        │")
print("   ├─────────────────────────────────────────────────────────┤")
print("   │ Main Orchestrator Agent routes to:                     │")
print("   │   → Sales Intelligence Agent                            │")
print("   │   → Marketing Intelligence Agent                        │")
print("   │   → Customer Support Agent                              │")
print("   │   → Finance Agent                                       │")
print("   │                                                         │")
print("   │ Use when: Users need cross-domain questions            │")
print("   └─────────────────────────────────────────────────────────┘")
print()
print()

print("📊 Scoping is scored automatically by DA-001 (purpose & audience) and DA-002 (narrow mission) in the next cell,")
print("   using the agent description and instructions. Record manual answers with MANUAL_ATTESTATIONS.")
print()
print("=" * 70)


In [ ]:
print("=" * 70)
print("COMMON AGENT CHECKS (DA-001 … DA-014, EXQ-001)")
print("=" * 70)

begin_section("agent")
P = AGENT_PROFILE
_F = []
AGENT_TEXT = f"{P.instructions or ''}\n{P.description or ''}"
SETTINGS_READ = (P.collection_mode in ("GIT_EXPORT", "MANUAL_MANIFEST")) or bool(P.metadata.get("settings_keys"))


def _absent_or_unknown(rule_id, what):
    """Absence is FAIL only when the configuration was positively read; otherwise UNKNOWN."""
    if SETTINGS_READ:
        return make_finding(rule_id, Status.FAIL, f"{what} not found in the {P.collection_mode} configuration")
    return make_finding(rule_id, Status.UNKNOWN, f"{what} could not be inspected (configuration not readable)", confidence="LOW")


if P.standalone_semantic:
    for rid in ["DA-001", "DA-002", "DA-003", "DA-004", "DA-005", "DA-006", "DA-007", "DA-008", "DA-009",
                "DA-010", "DA-011", "DA-012", "DA-013", "DA-014", "EXQ-001"]:
        _F.append(make_finding(rid, Status.NOT_APPLICABLE, "Standalone semantic-model mode — no data agent in scope"))
else:
    # DA-001 Purpose and audience
    def _da001():
        if not AGENT_TEXT.strip():
            return _absent_or_unknown("DA-001", "Agent description/instructions")
        audience = re.search(r"\b(users?|analysts?|managers?|team|personas?|audience|executives?|leaders?|reps?|stakeholders?)\b", AGENT_TEXT, re.I)
        purpose = re.search(r"\b(objective|purpose|goal|helps?|answers? questions|use this agent|designed to|intended)\b", AGENT_TEXT, re.I)
        found = [n for n, ok in (("audience", audience), ("purpose", purpose)) if ok]
        st = Status.PASS if len(found) == 2 else Status.WARN if found else Status.FAIL
        return make_finding("DA-001", st, f"Signals found: {found or 'none'}", confidence="MEDIUM")
    _F += safe_check("DA-001", _da001)

    # DA-002 Narrow mission
    def _da002():
        if not AGENT_TEXT.strip():
            return _absent_or_unknown("DA-002", "Agent description/instructions")
        broad = re.findall(r"\b(?:any|all|every)\s+(?:business\s+)?(?:questions?|data|topics?)\b|answer anything|everything|general[- ]purpose", AGENT_TEXT, re.I)
        domains = ["sales", "finance", "marketing", "inventory", "supply chain", "hr", "human resources", "support",
                   "logistics", "manufacturing", "procurement", "legal", "operations", "customer service"]
        hit = sorted({d for d in domains if re.search(rf"\b{re.escape(d)}\b", AGENT_TEXT, re.I)})
        if broad or len(hit) >= 4:
            return make_finding("DA-002", Status.WARN, f"Broad wording: {broad[:3]}; domains mentioned: {hit}", confidence="MEDIUM")
        return make_finding("DA-002", Status.PASS, f"No broad wording; domains mentioned: {hit or 'n/a'}", confidence="MEDIUM")
    _F += safe_check("DA-002", _da002)

    # DA-003 Source count
    n_src = len(SOURCES)
    if n_src == 0 and P.errors:
        _F.append(make_finding("DA-003", Status.UNKNOWN, f"No sources discovered; discovery errors: {short('; '.join(P.errors), 200)}", confidence="LOW"))
    else:
        _F.append(make_finding("DA-003", Status.PASS if 1 <= n_src <= 5 else Status.FAIL, f"{n_src} connected source(s); supported range is 1–5"))

    # DA-004 Known source types
    unknown_src = [s for s in SOURCES if s.source_kind == "UNKNOWN"]
    for s in unknown_src:
        _F.append(make_finding("DA-004", Status.UNKNOWN, f"Unrecognized source type: {s.source_type_raw}", source=s))
    if SOURCES and not unknown_src:
        _F.append(make_finding("DA-004", Status.PASS, f"All {n_src} source types recognized: {sorted({s.source_kind for s in SOURCES})}"))

    # DA-005 Descriptive, unique names
    def _da005():
        if not SOURCES:
            return make_finding("DA-005", Status.UNKNOWN, "No sources discovered")
        names = [s.display_name for s in SOURCES]
        dups = sorted({n for n in names if [x.lower() for x in names].count(n.lower()) > 1})
        generic = [n for n in names if re.match(r"^(source|database|db|datasource|data|lakehouse|warehouse|kql ?db|model|dataset|test|new|untitled)[ _\-]?\d*$", n, re.I) or len(n) < 3]
        out = [detail("DA-005", Status.WARN, "Generic source name", object_path=n) for n in generic]
        out += [detail("DA-005", Status.FAIL, "Duplicate source name", object_path=n) for n in dups]
        st = Status.FAIL if dups else Status.WARN if generic else Status.PASS
        return [make_finding("DA-005", st, f"{len(names)} names; duplicates={dups or 'none'}; generic={generic or 'none'}")] + out
    _F += safe_check("DA-005", _da005)

    # DA-006 Source description quality / DA-008 Source-specific instructions (capability-gated)
    for s in SOURCES:
        cap_desc = capability(s, "source_description")
        if cap_desc is False:
            _F.append(make_finding("DA-006", Status.NOT_APPLICABLE, f"{s.source_kind} does not support source descriptions (use Prep for AI / item metadata)", source=s))
        elif not s.description:
            _F.append(make_finding("DA-006", Status.FAIL if (cap_desc is True and SETTINGS_READ) else Status.UNKNOWN,
                                   "No source description found" if cap_desc is True else "Source description support not documented for this source; none found", source=s))
        else:
            d = s.description
            cues = [c for c, rx in (("contains", r"\b(contains?|includes?|stores?|holds?)\b"),
                                    ("answers", r"\b(answers?|questions?|use (this|for)|for questions)\b"),
                                    ("distinguishes", r"\b(only|not|instead|unlike|authoritative|primary|preferred)\b"))
                    if re.search(rx, d, re.I)]
            st = Status.PASS if len(cues) >= 2 and len(d) >= 40 else Status.WARN
            _F.append(make_finding("DA-006", st, f"{len(d)} chars; cues={cues}", source=s, confidence="MEDIUM"))

        cap_ins = capability(s, "source_instructions")
        if cap_ins is False:
            _F.append(make_finding("DA-008", Status.NOT_APPLICABLE, "Source instructions are not supported for this source type — put guidance in Prep for AI (semantic model) or agent-level routing", source=s))
        elif not s.instructions:
            _F.append(make_finding("DA-008", Status.FAIL if (cap_ins is True and SETTINGS_READ) else Status.UNKNOWN,
                                   "No data-source instructions found" if cap_ins is True else "Source-instruction support not documented; none found", source=s))
        else:
            t = s.instructions
            names = [o.get("name", "") for o in s.selected_objects if o.get("name")]
            cues = {
                "objects": any(n.lower() in t.lower() for n in names) if names else bool(re.search(r"\b(table|view|index|entity)\b", t, re.I)),
                "grain": bool(re.search(r"one row per|\bgrain\b|\bper (order|customer|event|day|line)\b", t, re.I)),
                "joins": bool(re.search(r"\bjoin|relationship|foreign key|\bkey\b", t, re.I)),
                "dates": bool(re.search(r"\bdate|time ?zone|\butc\b|fiscal|timestamp", t, re.I)),
                "values": bool(re.search(r"\bmeans\b|=\s*'?\w+|\bcodes?\b|\bstatus\b|\bflag\b|\bunits?\b|\bcurrency\b", t, re.I)),
            }
            found = [k for k, v in cues.items() if v]
            st = Status.PASS if len(found) >= 4 else Status.WARN if len(found) >= 2 else Status.FAIL
            _F.append(make_finding("DA-008", st, f"Covered: {found}; missing: {[k for k in cues if k not in found]}", source=s, confidence="MEDIUM"))

    # DA-007 Agent instruction structure
    def _da007():
        if not P.instructions:
            return _absent_or_unknown("DA-007", "Agent instructions")
        t = P.instructions
        parts = {
            "objective": r"objective|purpose|goal",
            "routing": r"data sources?|routing|\buse\b.+\bfor\b|prefer",
            "terminology": r"terminology|definitions?|glossary|abbreviations?|\bmeans\b",
            "response": r"response|format|tone|units|currency|concise",
            "out_of_scope": r"out[- ]of[- ]scope|cannot answer|do not answer|clarif",
        }
        found = [k for k, rx in parts.items() if re.search(rx, t, re.I)]
        st = Status.PASS if len(found) >= 4 else Status.WARN if len(found) >= 2 else Status.FAIL
        return make_finding("DA-007", st, f"Sections detected: {found}; missing: {[k for k in parts if k not in found]}", confidence="MEDIUM")
    _F += safe_check("DA-007", _da007)

    # DA-009 Cross-source routing
    def _da009():
        if len(SOURCES) <= 1:
            return make_finding("DA-009", Status.NOT_APPLICABLE, "Single-source agent")
        if not P.instructions:
            return _absent_or_unknown("DA-009", "Agent instructions (routing)")
        t = P.instructions.lower()
        named = [s.display_name for s in SOURCES if s.display_name.lower() in t]
        routing = bool(re.search(r"\b(use|prefer|route|authoritative|primary)\b", t))
        ratio = len(named) / len(SOURCES)
        st = Status.PASS if ratio == 1 and routing else Status.WARN if named else Status.FAIL
        return make_finding("DA-009", st, f"Sources named in instructions: {named} of {[s.display_name for s in SOURCES]}; routing verbs={routing}",
                            ratio=ratio if st == Status.WARN else None, confidence="MEDIUM")
    _F += safe_check("DA-009", _da009)

    # DA-010 Metric consistency (manual, attestable) — XSR-003 adds evidence in section 12
    _F.append(make_finding("DA-010", Status.NOT_APPLICABLE, "Single-source agent") if len(SOURCES) <= 1 else
              manual_check("DA-010", "Compare shared metric, date, currency, and status definitions across sources (see XSR-003)."))

    # DA-011 Minimal schema selection
    for s in SOURCES:
        if s.source_family in ("AI_SEARCH",):
            _F.append(make_finding("DA-011", Status.NOT_APPLICABLE, "Schema selection does not apply to search indexes", source=s))
            continue
        n_sel = len(s.selected_objects)
        n_all = len(s.metadata.get("all_objects", []))
        if n_sel == 0 and n_all == 0:
            _F.append(make_finding("DA-011", Status.UNKNOWN, "Schema selection not exposed for this source", source=s, confidence="LOW"))
            continue
        noisy = [o["path"] for o in s.selected_objects if re.search(r"(^|[._])(stg|staging|tmp|temp|bak|backup|audit|log|archive|old|test|raw|bronze|landing)([._]|\d|$)", o["path"], re.I)]
        _F += [detail("DA-011", Status.WARN, "Technical/staging object selected", source=s, object_path=p) for p in noisy]
        if n_sel == 0:
            st, ev = Status.FAIL, f"No objects selected (of {n_all})"
        elif n_sel > MAX_SELECTED_OBJECTS_WARN or noisy:
            st, ev = Status.WARN, f"{n_sel} selected (of {n_all or '?'}); threshold {MAX_SELECTED_OBJECTS_WARN}; technical objects: {len(noisy)}"
        else:
            st, ev = Status.PASS, f"{n_sel} selected (of {n_all or '?'})"
        _F.append(make_finding("DA-011", st, ev, source=s))

    # DA-012 Runtime recorded
    _F.append(make_finding("DA-012", Status.PASS if P.runtime in ("STANDARD", "PREVIEW") else Status.UNKNOWN,
                           f"Runtime = {P.runtime}", confidence="HIGH" if P.runtime != "UNKNOWN" else "LOW"))

    # DA-013 Preview-risk disclosure
    preview_items = [s.display_name for s in SOURCES if s.source_kind in PREVIEW_SOURCE_KINDS] + (["preview runtime"] if P.runtime == "PREVIEW" else [])
    if not preview_items:
        _F.append(make_finding("DA-013", Status.PASS, "No preview sources or preview runtime detected"))
    elif FAIL_ON_PREVIEW_SOURCE:
        _F.append(make_finding("DA-013", Status.FAIL, f"Preview in use: {preview_items}; FAIL_ON_PREVIEW_SOURCE = True", severity="BLOCKER"))
    elif PREVIEW_ACCEPTED:
        _F.append(make_finding("DA-013", Status.PASS, f"USER_SUPPLIED: PREVIEW_ACCEPTED = True for {preview_items}", confidence="MEDIUM"))
    else:
        _F.append(make_finding("DA-013", Status.WARN, f"Preview in use without recorded acceptance: {preview_items}"))

    # DA-014 Read-only contract (examples; evaluation queries are linted in section 14)
    def _da014():
        checked, bad = 0, []
        for s in SOURCES:
            lang = query_language_for(s)
            for ex in s.example_queries:
                if ex.get("query"):
                    checked += 1
                    v = read_only_violations(ex["query"], lang)
                    if v:
                        bad.append(detail("DA-014", Status.FAIL, f"Data-changing tokens {v} in example '{short(ex.get('question'), 80)}'",
                                          source=s, object_path=ex.get("id"), severity="BLOCKER"))
        if checked == 0:
            return make_finding("DA-014", Status.NOT_APPLICABLE, "No example queries to lint")
        return [make_finding("DA-014", Status.FAIL if bad else Status.PASS, f"{checked} example queries linted; {len(bad)} with data-changing statements")] + bad
    _F += safe_check("DA-014", _da014)

    # EXQ-001 Example-query capability respected
    for s in SOURCES:
        sup, n_ex = capability(s, "example_queries"), len(s.example_queries)
        if sup is False and n_ex:
            _F.append(make_finding("EXQ-001", Status.WARN, f"{n_ex} example(s) on a source documented as not supporting examples — they will not be used", source=s))
        elif sup is False:
            _F.append(make_finding("EXQ-001", Status.NOT_APPLICABLE, "Example queries not supported for this source (by design)", source=s))
        elif sup is None:
            _F.append(make_finding("EXQ-001", Status.NOT_APPLICABLE if not n_ex else Status.PASS,
                                   f"Example-query support not listed for {s.source_kind}; {n_ex} example(s) found — not failed", source=s, confidence="LOW"))
        else:
            _F.append(make_finding("EXQ-001", Status.PASS if n_ex else Status.WARN,
                                   f"{n_ex} example(s)" + ("" if n_ex else " — add representative question/query pairs"), source=s))

record("agent", _F)
show_findings(_F, title="\nCommon agent findings:")

---
# 6️⃣ SQL SOURCES — Lakehouse · Warehouse · Fabric SQL Database · Mirrored Database

All SQL sources use T-SQL through the source's SQL endpoint. The **standard runtime** uses NL2SQL. The **preview runtime** can use Advanced NL2SQL, which also uses **schema object descriptions** and **topics**. All SQL sources use schema selection, data-source instructions, and example queries.

**How checks are evaluated**

- **Static checks** (always): schema focus, naming, description coverage, grain, joins, authoritative objects, business values, time logic, and example-query lint (read-only, `SELECT *`, bounds, literal mapping, duplicates).
- **Live checks** (`RUN_LIVE_QUERY_TESTS = True`): object resolution through `INFORMATION_SCHEMA`, bounded `SELECT TOP 1` smoke tests, key/null data-quality probes, freshness (`MAX(<date column>)`), example-query execution, and latency when `RUN_PERFORMANCE_TESTS = True`. Mirrored databases also run the read-only `getMirroringStatus` / `getTablesMirroringStatus` calls.
- **SDK few-shot evaluator** (`RUN_FEW_SHOT_EVALUATOR = True`): `evaluate_few_shots` rates clarity, relatedness, mapping, and conflicts. It's documented as **SQL-only** and uses an LLM.

<details><summary>Data-source instruction template for SQL</summary>

```markdown
## General knowledge
- State the source purpose and authoritative domains.
- State freshness and timezone assumptions.

## Important objects and grain
- `dbo.FactOrders`: one row per order line.
- `dbo.DimCustomer`: one row per customer.

## Relationships
- Join `FactOrders.CustomerId` to `DimCustomer.CustomerId`.
- Aggregate order lines to order level before joining to order-level payment totals.

## Business values
- `OrderStatus = 'C'` means completed.
- `IsActive = 1` means active.
- Monetary columns are stored in USD, not cents.

## Date rules
- Use `OrderSubmittedDate` unless the user explicitly asks about shipment or delivery.
- Fiscal year starts on ...

## When asked about
- Current orders: use `vw_CurrentOrders`, not the archive table.
- Revenue: use ...
```
</details>

In [ ]:
_TECH_NAME = re.compile(r"^(tbl|tmp|temp|stg|t\d+|x|obj|table\d*|view\d*|vw\d*)[_\d]*$|^[a-z]{1,2}\d*$|_(tmp|bak|old|copy\d*)$", re.I)
_TECH_COL = re.compile(r"^(col|column|field|c|f)_?\d+$|^x\d*$|_(tmp|bak|old)$", re.I)
_CODED_COL = re.compile(r"(status|code|flag|type|category|_cd$|cd$|^is_?|^has_?|_ind$|indicator|uom|unit)", re.I)
_DATE_COL = re.compile(r"(date|time|_dt$|dt$|timestamp|_ts$|period|month|year|day)", re.I)
_DATE_TYPES = re.compile(r"date|time", re.I)
_OVERLAP_SUFFIX = re.compile(r"(_|\b)(archive|arch|hist|history|historical|raw|stg|staging|snapshot|snap|current|curr|v\d|old|bak|backup|latest)$", re.I)


def _base_name(name: str) -> str:
    n = re.sub(r"^(vw_|v_|dim_|fact_|fct_)", "", name.split(".")[-1].lower())
    return _OVERLAP_SUFFIX.sub("", n).rstrip("_")


class SqlSourceAdapter(SourceAdapter):
    source_family = "SQL"
    primary_rule = "SQL-002"

    def __init__(self, subtype: str):
        self.subtype = subtype

    def live_rules(self):
        base = ["SQL-002", "SQL-018", "SQL-019"]
        return base + {"LAKEHOUSE": ["LH-001", "LH-002"], "WAREHOUSE": ["WH-004"], "FABRIC_SQL_DATABASE": ["SQLDB-001"],
                       "MIRRORED_DATABASE": ["MIR-001", "MIR-002", "MIR-003"]}.get(self.subtype, [])

    # ---------------- metadata ----------------
    def collect_metadata(self, source):
        sel = source.selected_objects
        cols = [(o["path"], c) for o in sel for c in o.get("columns", [])]
        return {
            "selected": sel, "columns": cols, "column_metadata": bool(cols),
            "text": f"{source.instructions or ''}\n{source.description or ''}",
            "facts": [o for o in sel if re.search(r"(^|[._])(fact|fct|event|transaction|sales|orders?|log|activity|line)s?", o["path"], re.I)],
        }

    # ---------------- static schema checks ----------------
    def run_schema_checks(self, s, md):
        out, sel, text = [], md["selected"], md["text"]
        n = len(sel)
        noisy = [o["path"] for o in sel if re.search(r"(^|[._])(stg|staging|tmp|temp|bak|audit|log|archive|old|test)([._]|\d|$)", o["path"], re.I)]
        out.append(make_finding("SQL-001", Status.FAIL if n == 0 else Status.WARN if (n > MAX_SELECTED_OBJECTS_WARN or noisy) else Status.PASS,
                                f"{n} selected object(s); staging/audit/archive selected: {noisy or 'none'}", source=s))
        if n == 0:
            return out

        tech = [o["path"] for o in sel if _TECH_NAME.search(o["name"])]
        tech_cols = [f"{p}.{c['name']}" for p, c in md["columns"] if c.get("name") and _TECH_COL.search(c["name"])]
        flagged = len(tech) + len(tech_cols)
        denom = n + len(md["columns"])
        ratio = flagged / max(denom, 1)
        out.append(make_finding("SQL-003", Status.PASS if ratio < 0.1 else Status.WARN if ratio < 0.3 else Status.FAIL,
                                f"{flagged}/{denom} technical/generic names; e.g. {(tech + tech_cols)[:5]}", source=s, confidence="MEDIUM"))

        documented = [o for o in sel if o.get("description") or _mentioned(o["name"], s.instructions or "")]
        cov = len(documented) / n
        out.append(make_finding("SQL-004", Status.PASS if cov >= 0.8 else Status.WARN if cov >= 0.5 else Status.FAIL,
                                f"{len(documented)}/{n} objects described (schema description or instructions)", ratio=cov if cov < 0.8 else None,
                                source=s, confidence="MEDIUM"))
        out += [detail("SQL-004", Status.WARN, "Object not described in schema descriptions or instructions", source=s, object_path=o["path"])
                for o in sel if o not in documented][:25]

        if md["column_metadata"]:
            amb = [(p, c) for p, c in md["columns"] if _CODED_COL.search(c.get("name", "")) or _DATE_COL.search(c.get("name", ""))]
            doc = [(p, c) for p, c in amb if c.get("description") or _mentioned(c.get("name", ""), text)]
            cov = len(doc) / len(amb) if amb else 1.0
            out.append(make_finding("SQL-005", Status.PASS if cov >= 0.8 else Status.WARN if cov >= 0.4 else Status.FAIL,
                                    f"{len(doc)}/{len(amb)} ambiguous columns (codes, flags, dates, units) documented",
                                    ratio=cov if cov < 0.8 else None, source=s, confidence="MEDIUM"))
        else:
            out.append(make_finding("SQL-005", Status.NOT_EVALUATED, "Column metadata not exposed by this collection mode", source=s, confidence="LOW"))

        facts = md["facts"]
        if not facts:
            out.append(make_finding("SQL-006", Status.NOT_APPLICABLE, "No fact/event-like objects detected", source=s, confidence="LOW"))
        else:
            has_grain = bool(re.search(r"one row per|\bgrain\b", text, re.I))
            covered = [o["path"] for o in facts if _mentioned(o["name"], text)]
            st = Status.PASS if has_grain and len(covered) == len(facts) else Status.WARN if has_grain or covered else Status.FAIL
            out.append(make_finding("SQL-006", st, f"Grain statement={has_grain}; fact objects mentioned {len(covered)}/{len(facts)}", source=s, confidence="MEDIUM"))

        if n < 2:
            out.append(make_finding("SQL-007", Status.NOT_APPLICABLE, "Single selected object", source=s))
        else:
            ex_joins = sum(1 for e in s.example_queries if re.search(r"\bjoin\b", e.get("query", ""), re.I))
            joins_doc = bool(re.search(r"\bjoin\b|relationship|foreign key", text, re.I))
            st = Status.PASS if joins_doc and (ex_joins or re.search(r"\b\w+(id|key)\b", text, re.I)) else Status.WARN if (joins_doc or ex_joins) else Status.FAIL
            out.append(make_finding("SQL-007", st, f"Join guidance in instructions={joins_doc}; examples with JOIN={ex_joins}", source=s, confidence="MEDIUM"))

        groups = {}
        for o in sel:
            groups.setdefault(_base_name(o["path"]), []).append(o["path"])
        overlaps = {k: v for k, v in groups.items() if len(v) > 1}
        unresolved = {k: v for k, v in overlaps.items() if not all(_mentioned(p, text) for p in v)}
        st = Status.PASS if not overlaps else Status.FAIL if unresolved and len(unresolved) == len(overlaps) else Status.WARN if unresolved else Status.PASS
        out.append(make_finding("SQL-008", st, f"Overlapping object groups: {list(overlaps.values())[:5] or 'none'}; without usage rules: {len(unresolved)}", source=s, confidence="MEDIUM"))

        coded = [f"{p}.{c['name']}" for p, c in md["columns"] if _CODED_COL.search(c.get("name", ""))]
        if not md["column_metadata"]:
            out.append(make_finding("SQL-009", Status.NOT_EVALUATED, "Column metadata not exposed", source=s, confidence="LOW"))
        elif not coded:
            out.append(make_finding("SQL-009", Status.NOT_APPLICABLE, "No coded/flag columns detected", source=s, confidence="LOW"))
        else:
            vals = bool(re.search(r"\bmeans\b|=\s*'?[\w\d]+'?\s*(means|is|=>|→)|\bvalues?\b|\bstored in\b|\bcents\b|\bcurrency\b", text, re.I))
            out.append(make_finding("SQL-009", Status.PASS if vals else Status.WARN, f"{len(coded)} coded columns; value definitions present={vals}", source=s, confidence="MEDIUM"))

        date_cols = [c for _, c in md["columns"] if _DATE_TYPES.search(c.get("data_type", "")) or _DATE_COL.search(c.get("name", ""))]
        if md["column_metadata"] and not date_cols:
            out.append(make_finding("SQL-010", Status.NOT_APPLICABLE, "No date/time columns detected", source=s, confidence="LOW"))
        else:
            cues = [c for c, rx in (("default date", r"\b(use|default)\b[^.\n]{0,60}\bdate\b|\bunless\b"), ("fiscal", r"fiscal"),
                                    ("timezone", r"time ?zone|\butc\b|\blocal time\b"), ("period rule", r"inclusive|exclusive|calendar|week starts|month end"))
                    if re.search(rx, text, re.I)]
            out.append(make_finding("SQL-010", Status.PASS if len(cues) >= 2 else Status.WARN if cues else Status.FAIL, f"Time-logic cues: {cues}", source=s, confidence="MEDIUM"))

        metric_rules = bool(re.search(r"calculated as|defined as|\bis (the )?sum of|\bformula\b|\bexclud(e|ing)\b|\bnet of\b", text, re.I))
        agg_examples = sum(1 for e in s.example_queries if _SQL_AGG.search(e.get("query", "")))
        views = [o for o in sel if o.get("object_type") == "view" or re.match(r"^(vw_|v_)", o["name"], re.I)]
        st = Status.PASS if (metric_rules and (agg_examples or views)) else Status.WARN if (metric_rules or agg_examples or views) else Status.FAIL
        out.append(make_finding("SQL-011", st, f"Metric definitions={metric_rules}; aggregation examples={agg_examples}; curated views={len(views)}", source=s, confidence="LOW"))
        return out

    # ---------------- configuration checks (runtime, preview features, subtype) ----------------
    def run_configuration_checks(self, s, md):
        out, sel, rt = [], md["selected"], AGENT_PROFILE.runtime
        if rt != "PREVIEW":
            out.append(make_finding("SQL-021", Status.NOT_APPLICABLE, f"Schema object descriptions require the preview runtime (runtime = {rt})", source=s))
            out.append(make_finding("SQL-022", Status.NOT_APPLICABLE, f"Topics require the preview runtime (runtime = {rt})", source=s))
        else:
            descs = [(o["path"], o.get("description", "")) for o in sel if o.get("description")]
            descs += [(f"{p}.{c['name']}", c.get("description", "")) for p, c in md["columns"] if c.get("description")]
            weak = [p for p, d in descs if len(d) < 15 or _norm_q(d) == _norm_q(p.split(".")[-1].replace("_", " "))]
            out.append(make_finding("SQL-021", Status.NOT_APPLICABLE if not descs else Status.PASS if not weak else Status.WARN,
                                    f"{len(descs)} descriptions; {len(weak)} repeat the name or are < 15 chars", source=s, confidence="MEDIUM"))
            topics = _first(s.metadata.get("raw_config", {}), "topics", "Topics", default=None)
            if not topics:
                out.append(make_finding("SQL-022", Status.NOT_APPLICABLE, "No topics configured", source=s))
            else:
                items = topics if isinstance(topics, list) else [topics]
                texts = [_text(_first(t, "content", "text", "body") if isinstance(t, dict) else t) for t in items]
                titles = [_text(_first(t, "title", "name", "heading") if isinstance(t, dict) else "") for t in items]
                bad = [i for i, (ti, tx) in enumerate(zip(titles, texts)) if not tx or (not ti and not tx.lstrip().startswith("#"))]
                out.append(make_finding("SQL-022", Status.PASS if not bad else Status.WARN, f"{len(items)} topic(s); {len(bad)} empty or without a descriptive heading", source=s, confidence="MEDIUM"))
        out.append(make_finding("SQL-023", Status.PASS if rt == "STANDARD" or (rt == "PREVIEW" and PREVIEW_ACCEPTED) else
                                Status.WARN if rt == "PREVIEW" else Status.UNKNOWN,
                                f"Runtime = {rt}; PREVIEW_ACCEPTED = {PREVIEW_ACCEPTED}", source=s))
        out.append(manual_check("SQL-017", "Test representative complex joins/grouping/filters against latency goals (see WH-004 when live).", source=s))
        out.append(manual_check("SQL-020", "Verify consumers have only the read/query access needed and RLS/CLS is respected (see section 15).", source=s))

        if self.subtype == "LAKEHOUSE":
            raw = [o["path"] for o in sel if re.search(r"(^|[._])(raw|bronze|landing|stg|staging|ingest)([._]|$)", o["path"], re.I)]
            out.append(make_finding("LH-004", Status.WARN if raw else Status.PASS, f"Raw/bronze/staging tables selected: {raw or 'none'}", source=s))
            out.append(manual_check("LH-003", "Record shortcut targets, ownership, permission, and freshness for shortcut-backed tables.", source=s))
            out.append(manual_check("LH-005", "Review Delta table health (OPTIMIZE / V-Order / small files).", source=s))
        elif self.subtype == "WAREHOUSE":
            views = [o for o in sel if o.get("object_type") == "view" or re.match(r"^(vw_|v_)", o["name"], re.I)]
            joins = sum(1 for e in s.example_queries if re.search(r"\bjoin\b", e.get("query", ""), re.I))
            st = Status.PASS if views or len(sel) < 2 or not joins else Status.WARN
            out.append(make_finding("WH-002", st, f"{len(views)} curated view(s); {joins} example(s) with JOIN", source=s, confidence="MEDIUM"))
            tech_schema = sorted({o["path"].split(".")[0] for o in sel if "." in o["path"] and re.match(r"^(stg|staging|etl|tmp|temp|raw|audit|landing|sys)$", o["path"].split(".")[0], re.I)})
            out.append(make_finding("WH-003", Status.WARN if tech_schema else Status.PASS, f"Technical schemas exposed: {tech_schema or 'none'}", source=s))
            if not RUN_LIVE_QUERY_TESTS:
                out.append(manual_check("WH-001", "Confirm consumers have Read / SELECT on the selected objects (or enable RUN_LIVE_QUERY_TESTS).", source=s))
        elif self.subtype == "FABRIC_SQL_DATABASE":
            views = [o for o in sel if o.get("object_type") == "view" or re.match(r"^(vw_|v_)", o["name"], re.I)]
            st = Status.PASS if views or len(sel) <= 3 else Status.WARN
            out.append(make_finding("SQLDB-002", st, f"{len(views)} view(s) of {len(sel)} selected objects", source=s, confidence="MEDIUM"))
            sd = [f"{p}.{c['name']}" for p, c in md["columns"] if re.search(r"is_?deleted|deleted|is_?active|active|valid_?(to|from)|effective|is_?current|end_?date", c.get("name", ""), re.I)]
            if not sd:
                out.append(make_finding("SQLDB-003", Status.NOT_APPLICABLE if md["column_metadata"] else Status.NOT_EVALUATED,
                                        "No soft-delete/current-row columns detected" if md["column_metadata"] else "Column metadata not exposed", source=s))
            else:
                doc = [c for c in sd if _mentioned(c, md["text"])]
                out.append(make_finding("SQLDB-003", Status.PASS if len(doc) == len(sd) else Status.WARN if doc else Status.FAIL,
                                        f"{len(doc)}/{len(sd)} status/soft-delete columns documented: {sd[:5]}", source=s))
        elif self.subtype == "MIRRORED_DATABASE":
            out.append(manual_check("MIR-004", "Confirm workspace outbound access protection allows the mirrored source connection.", source=s))
        return out

    # ---------------- example validation (static; live execution in smoke_test) ----------------
    def validate_examples(self, s, md):
        out = example_set_findings(s, "SQL-012")
        ex = s.example_queries
        if not ex:
            for rid in ("SQL-013", "SQL-014", "SQL-015", "SQL-016"):
                out.append(make_finding(rid, Status.NOT_APPLICABLE, "No example queries", source=s))
            return out
        sel_names = {o["path"].lower() for o in md["selected"]} | {o["name"].lower() for o in md["selected"]}
        lint_hits = {"SQL-013": [], "SQL-015": [], "SQL-016": []}
        unmapped = []
        for e in ex:
            q = e.get("query", "")
            for rid, msg in lint_sql(q):
                lint_hits[rid].append(detail(rid, Status.WARN if rid != "SQL-013" else Status.FAIL, msg, source=s, object_path=e["id"]))
            refs = sql_object_refs(q) - sql_cte_names(q)
            outside = sorted(r for r in refs if sel_names and r not in sel_names and r.split(".")[-1] not in sel_names)
            if outside:
                lint_hits["SQL-013"].append(detail("SQL-013", Status.WARN, f"References objects outside the selected schema: {outside}", source=s, object_path=e["id"]))
            miss = unmapped_literals(e.get("question", ""), q)
            if miss:
                unmapped.append(detail("SQL-014", Status.WARN, f"Question literals not found in query: {miss}", source=s, object_path=e["id"]))
        n = len(ex)
        for rid in ("SQL-013", "SQL-015", "SQL-016"):
            hits = lint_hits[rid]
            bad_ex = len({h.object_path for h in hits})
            st = Status.PASS if not hits else (Status.FAIL if rid == "SQL-013" and any(h.status == Status.FAIL for h in hits) else Status.WARN)
            out.append(make_finding(rid, st, f"{n - bad_ex}/{n} examples pass static validation",
                                    ratio=(n - bad_ex) / n if st == Status.WARN else None, source=s, confidence="MEDIUM"))
            out += hits
        out.append(make_finding("SQL-014", Status.PASS if not unmapped else Status.WARN, f"{n - len(unmapped)}/{n} examples map all question literals",
                                ratio=(n - len(unmapped)) / n if unmapped else None, source=s, confidence="MEDIUM"))
        out += unmapped

        handle = SDK_HANDLES.get(s.source_id)
        if RUN_FEW_SHOT_EVALUATOR and s.source_kind in ("LAKEHOUSE", "WAREHOUSE"):
            if handle is not None and hasattr(handle, "evaluate_few_shots"):
                try:
                    res = handle.evaluate_few_shots()
                    rate = float(getattr(res, "success_rate", 0.0))
                    conflicts = int(getattr(res, "conflict_count", 0))
                    out.append(make_finding("SQL-013", Status.PASS if rate >= 90 and not conflicts else Status.WARN,
                                            f"evaluate_few_shots: success_rate={rate:.1f}%, conflicts={conflicts}", source=s, weight=0, confidence="MEDIUM"))
                    fc = getattr(res, "failure_cases", None)
                    if isinstance(fc, pd.DataFrame) and not fc.empty:
                        print("   evaluate_few_shots — failure cases:")
                        display(fc.head(20))
                except Exception as e:
                    out.append(detail("SQL-013", Status.NOT_EVALUATED, f"evaluate_few_shots failed: {short(e, 160)}", source=s))
            else:
                out.append(detail("SQL-013", Status.NOT_EVALUATED, "evaluate_few_shots needs a legacy SDK datasource handle (LIVE_SDK)", source=s))
        return out

    # ---------------- live, bounded, read-only smoke tests ----------------
    def smoke_test(self, s, md):
        out, sel = [], md["selected"][:SMOKE_TEST_MAX_OBJECTS]
        try:
            cat, _ = run_sql(s, "SELECT TABLE_SCHEMA, TABLE_NAME, TABLE_TYPE FROM INFORMATION_SCHEMA.TABLES")
            catalog = {f"{r.TABLE_SCHEMA}.{r.TABLE_NAME}".lower() for r in cat.itertuples()} | {str(r.TABLE_NAME).lower() for r in cat.itertuples()}
        except Exception as e:
            ev = f"SQL endpoint not reachable: {short(e, 200)}"
            return [make_finding(r, Status.NOT_EVALUATED, ev, source=s, confidence="LOW") for r in self.live_rules()]

        missing = [o["path"] for o in md["selected"] if o["path"].lower() not in catalog and o["name"].lower() not in catalog]
        out += [detail("SQL-002", Status.FAIL, "Selected object not found in the SQL endpoint catalog", source=s, object_path=p) for p in missing]
        ok, failed = 0, []
        for o in sel:
            ident = ".".join(f"[{p}]" for p in o["path"].split("."))
            try:
                run_sql(s, f"SELECT TOP 1 * FROM {ident}")
                ok += 1
            except Exception as e:
                failed.append(o["path"])
                out.append(detail("SQL-002", Status.FAIL, f"Bounded read failed: {short(e, 140)}", source=s, object_path=o["path"]))
        st = Status.PASS if not missing and not failed else Status.FAIL
        out.append(make_finding("SQL-002", st, f"{len(md['selected']) - len(missing)}/{len(md['selected'])} in catalog; {ok}/{len(sel)} bounded reads succeeded", source=s))
        sub_rule = {"LAKEHOUSE": "LH-001", "FABRIC_SQL_DATABASE": "SQLDB-001", "MIRRORED_DATABASE": "MIR-003"}.get(self.subtype)
        if sub_rule:
            out.append(make_finding(sub_rule, st, f"Missing from SQL endpoint: {missing or 'none'}", source=s))
        if self.subtype == "LAKEHOUSE":
            out.append(make_finding("LH-002", Status.PASS if not missing else Status.WARN, f"Agent selection vs endpoint metadata: {len(missing)} drifted", source=s))
        if self.subtype == "WAREHOUSE":
            out.append(make_finding("WH-001", Status.PASS if not failed else Status.FAIL, f"SELECT succeeded on {ok}/{len(sel)} objects (analysis identity)", source=s, confidence="MEDIUM"))

        # Data quality: key uniqueness / nulls on dimension-like keys
        dq_issues, dq_checked = [], 0
        for o in sel:
            if re.search(r"fact|fct|event|log", o["path"], re.I):
                continue
            keys = [c["name"] for c in o.get("columns", []) if re.search(r"(^id$|id$|key$|_sk$|_nk$)", c.get("name", ""), re.I)][:1]
            if not keys:
                continue
            ident = ".".join(f"[{p}]" for p in o["path"].split("."))
            k = keys[0]
            try:
                df, _ = run_sql(s, f"SELECT COUNT_BIG(*) AS n, COUNT_BIG(DISTINCT [{k}]) AS d, SUM(CASE WHEN [{k}] IS NULL THEN 1 ELSE 0 END) AS nulls FROM {ident}")
                dq_checked += 1
                n, d, nulls = int(df.iloc[0]["n"]), int(df.iloc[0]["d"]), int(df.iloc[0]["nulls"] or 0)
                if n and (d < n - nulls or nulls):
                    dq_issues.append(detail("SQL-018", Status.WARN, f"Key [{k}]: rows={n}, distinct={d}, nulls={nulls}", source=s, object_path=o["path"]))
            except Exception as e:
                dq_issues.append(detail("SQL-018", Status.NOT_EVALUATED, f"DQ probe failed: {short(e, 120)}", source=s, object_path=o["path"]))
        out.append(make_finding("SQL-018", Status.NOT_EVALUATED if not dq_checked else Status.PASS if not [d for d in dq_issues if d.status == Status.WARN] else Status.WARN,
                                f"{dq_checked} key column(s) probed; {len(dq_issues)} issue(s)", source=s))
        out += dq_issues

        # Freshness
        fresh, stale = [], []
        for o in sel:
            dcols = [c["name"] for c in o.get("columns", []) if _DATE_TYPES.search(c.get("data_type", ""))][:1]
            if not dcols:
                continue
            ident = ".".join(f"[{p}]" for p in o["path"].split("."))
            try:
                df, _ = run_sql(s, f"SELECT MAX([{dcols[0]}]) AS latest FROM {ident}")
                latest = pd.to_datetime(df.iloc[0]["latest"], utc=True, errors="coerce")
                if pd.notna(latest):
                    age_h = (datetime.now(timezone.utc) - latest.to_pydatetime()).total_seconds() / 3600
                    fresh.append((o["path"], dcols[0], round(age_h, 1)))
                    if age_h > FRESHNESS_MAX_AGE_HOURS:
                        stale.append(detail("SQL-019", Status.WARN, f"Latest {dcols[0]} is {age_h:.1f} h old (> {FRESHNESS_MAX_AGE_HOURS} h)", source=s, object_path=o["path"]))
            except Exception:
                pass
        out.append(make_finding("SQL-019", Status.NOT_EVALUATED if not fresh else Status.WARN if stale else Status.PASS,
                                f"Freshness sampled on {len(fresh)} object(s): {fresh[:5]}", source=s, confidence="MEDIUM"))
        out += stale

        # Example execution (+ latency)
        lat, ex_fail = [], []
        for e in s.example_queries:
            try:
                _, secs = run_sql(s, e["query"])
                lat.append(secs)
            except Exception as ex:
                ex_fail.append(detail("SQL-013", Status.FAIL, f"Execution failed: {short(ex, 160)}", source=s, object_path=e["id"]))
        if s.example_queries:
            out.append(make_finding("SQL-013", Status.FAIL if ex_fail else Status.PASS,
                                    f"Live execution: {len(s.example_queries) - len(ex_fail)}/{len(s.example_queries)} examples succeeded", source=s))
            out += ex_fail
        if self.subtype == "WAREHOUSE":
            if RUN_PERFORMANCE_TESTS and lat:
                p50, p95 = float(pd.Series(lat).quantile(0.5)), float(pd.Series(lat).quantile(0.95))
                out.append(make_finding("WH-004", Status.PASS if p95 <= LATENCY_TARGET_SECONDS else Status.WARN,
                                        f"Example latency p50={p50:.2f}s p95={p95:.2f}s (target {LATENCY_TARGET_SECONDS}s, n={len(lat)})", source=s))
            else:
                out.append(make_finding("WH-004", Status.NOT_EVALUATED, "Enable RUN_PERFORMANCE_TESTS to sample latency", source=s, confidence="LOW"))

        if self.subtype == "MIRRORED_DATABASE":
            out += self._mirroring(s, md)
        return out

    def _mirroring(self, s, md):
        out = []
        ws_id = resolve_workspace_id(s.workspace_id or AGENT_PROFILE.workspace)
        if not (ws_id and s.artifact_id):
            return [make_finding(r, Status.NOT_EVALUATED, "Mirrored database item/workspace ID not exposed", source=s, confidence="LOW") for r in ("MIR-001", "MIR-002")]
        base = f"/v1/workspaces/{ws_id}/mirroredDatabases/{s.artifact_id}"
        try:
            st = fabric_rest("POST", f"{base}/getMirroringStatus")
            status = _text(_first(st, "status", "Status"))
            out.append(make_finding("MIR-001", Status.PASS if status.lower() == "running" else Status.FAIL, f"Mirroring status = {status or 'unknown'}", source=s))
        except Exception as e:
            out.append(make_finding("MIR-001", Status.NOT_EVALUATED, f"getMirroringStatus failed: {short(e, 160)}", source=s, confidence="LOW"))
        try:
            tb = fabric_rest("POST", f"{base}/getTablesMirroringStatus")
            rows = tb.get("data", []) if isinstance(tb, dict) else []
            lags = []
            for r in rows:
                last = _first(r.get("metrics", {}) if isinstance(r.get("metrics"), dict) else {}, "lastSyncDateTime", "lastSyncTime")
                ts = pd.to_datetime(last, utc=True, errors="coerce")
                if pd.notna(ts):
                    lags.append(((datetime.now(timezone.utc) - ts.to_pydatetime()).total_seconds() / 3600, f"{r.get('sourceSchemaName')}.{r.get('sourceTableName')}"))
            if lags:
                worst = max(lags)
                out.append(make_finding("MIR-002", Status.PASS if worst[0] <= FRESHNESS_MAX_AGE_HOURS else Status.WARN,
                                        f"Max replication lag {worst[0]:.1f} h ({worst[1]}); threshold {FRESHNESS_MAX_AGE_HOURS} h", source=s))
            else:
                out.append(make_finding("MIR-002", Status.NOT_EVALUATED, "No per-table sync timestamps returned", source=s, confidence="LOW"))
        except Exception as e:
            out.append(make_finding("MIR-002", Status.NOT_EVALUATED, f"getTablesMirroringStatus failed: {short(e, 160)}", source=s, confidence="LOW"))
        return out


for _k in ("LAKEHOUSE", "WAREHOUSE", "FABRIC_SQL_DATABASE", "MIRRORED_DATABASE"):
    ADAPTERS[_k] = SqlSourceAdapter(subtype=_k)

SQL_FINDINGS = run_family("SQL", "sql", "SQL SOURCES — SQL-001 … SQL-023, LH-*, WH-*, SQLDB-*, MIR-*")

---
# 7️⃣ EVENTHOUSE KQL DATABASE

Fabric data agents can select Eventhouse KQL **tables, materialized views, functions, and shortcuts**. Eventhouse KQL databases support example queries. However, the documented `evaluate_few_shots` utility is **SQL-only**, so KQL examples are checked with static lint, optional bounded execution (`request_readonly`, row cap), and end-to-end evaluation.

**Static-lint heuristics** (these need an allow-list, and matches are reported as `WARN`):

- `union *`, wildcard tables, or `search *` used where a named table or column would work
- an event query with no time predicate, or a first operator that isn't a selective `where`
- `contains` used where token semantics point to `has`
- detail queries without `take`, `limit`, `top`, or `summarize`
- cross-cluster or cross-Eventhouse references, flagged for review
- dynamic-field extraction that repeats across examples, and repeated heavy aggregations (candidates for materialized views)

In [ ]:
class KqlSourceAdapter(SourceAdapter):
    source_family = "KQL"
    primary_rule = "KQL-002"

    def live_rules(self):
        return ["KQL-002", "KQL-013"]

    def collect_metadata(self, s):
        sel = s.selected_objects
        return {
            "selected": sel,
            "tables": [o for o in sel if o.get("object_type") in ("table", "materializedview", "shortcut", "externaltable")],
            "functions": [o for o in sel if o.get("object_type") == "function"],
            "mviews": [o for o in sel if o.get("object_type") == "materializedview"],
            "shortcuts": [o for o in sel if o.get("object_type") in ("shortcut", "externaltable")],
            "columns": [(o["path"], c) for o in sel for c in o.get("columns", [])],
            "text": f"{s.instructions or ''}\n{s.description or ''}",
        }

    def run_schema_checks(self, s, md):
        out, sel = [], md["selected"]
        out.append(make_finding("KQL-001", Status.FAIL if not sel else Status.WARN if len(sel) > MAX_SELECTED_OBJECTS_WARN else Status.PASS,
                                f"{len(sel)} selected: {len(md['tables'])} tables/MVs/shortcuts, {len(md['functions'])} functions", source=s))
        d = md["text"]
        cues = [c for c, rx in (("domain", r"\b(events?|telemetry|logs?|signals?|measurements?|contains)\b"),
                                ("retention", r"retention|\bdays?\b|\bmonths?\b|history"),
                                ("authoritative", r"authoritative|primary|use\b.+\bfor\b|prefer"))
                if re.search(rx, d, re.I)]
        out.append(make_finding("KQL-003", Status.PASS if len(cues) >= 2 else Status.WARN if cues else Status.FAIL,
                                f"Description/instruction cues: {cues}", source=s, confidence="MEDIUM"))
        cols = md["columns"]
        if not cols:
            out.append(make_finding("KQL-004", Status.NOT_EVALUATED, "Column types not exposed by this collection mode", source=s, confidence="LOW"))
        else:
            time_named = [(p, c) for p, c in cols if re.search(r"(time|timestamp|_ts$|^ts$|date|epoch)", c.get("name", ""), re.I)]
            epoch = [f"{p}.{c['name']}" for p, c in time_named if re.search(r"long|int|real|decimal", c.get("data_type", ""), re.I)]
            tables_wo_dt = [o["path"] for o in md["tables"] if o.get("columns") and not any(re.search(r"datetime", c.get("data_type", ""), re.I) for c in o["columns"])]
            st = Status.FAIL if epoch and tables_wo_dt else Status.WARN if epoch or tables_wo_dt else Status.PASS
            out.append(make_finding("KQL-004", st, f"Epoch-typed time columns: {epoch[:5] or 'none'}; tables without datetime: {tables_wo_dt[:5] or 'none'}", source=s))
        dyn = [f"{p}.{c['name']}" for p, c in cols if "dynamic" in c.get("data_type", "").lower()]
        extracts = sum(len(re.findall(r"parse_json|todynamic|extract_json|bag_unpack|\.\[\s*['\"]|\[\s*['\"][\w]+['\"]\s*\]", e.get("query", ""), re.I)) for e in s.example_queries)
        out.append(make_finding("KQL-010", Status.NOT_APPLICABLE if not dyn else Status.WARN if extracts >= 3 else Status.PASS,
                                f"{len(dyn)} dynamic column(s); {extracts} extraction(s) in examples", source=s, confidence="MEDIUM"))
        if md["shortcuts"]:
            out.append(manual_check("KQL-017", f"Verify availability, permission, and latency of shortcut targets: {[o['path'] for o in md['shortcuts']][:10]}", source=s))
        else:
            out.append(make_finding("KQL-017", Status.NOT_APPLICABLE, "No shortcuts/external tables selected", source=s))
        return out

    def run_configuration_checks(self, s, md):
        out, t = [], md["text"]
        dt_cols = [c.get("name") for _, c in md["columns"] if "datetime" in c.get("data_type", "").lower()]
        named_time = any(_mentioned(c, t) for c in dt_cols) if dt_cols else bool(re.search(r"\b(timestamp|eventtime|ingestion_time|time column)\b", t, re.I))
        tz = bool(re.search(r"\butc\b|time ?zone", t, re.I))
        out.append(make_finding("KQL-005", Status.PASS if named_time and tz else Status.WARN if named_time or tz else Status.FAIL,
                                f"Default time column documented={named_time}; timezone documented={tz}", source=s, confidence="MEDIUM"))
        ret = bool(re.search(r"retention|\bkept for\b|\bdays?\b.*\bdata\b|\bhistory\b", t, re.I))
        out.append(make_finding("KQL-016", Status.PASS if ret else Status.WARN, f"Retention/freshness documented={ret} (live freshness sampled when RUN_LIVE_QUERY_TESTS)",
                                source=s, confidence="MEDIUM"))
        out.append(manual_check("KQL-018", "Confirm consumers have the Reader role on the KQL database (see section 15).", source=s))
        return out

    def validate_examples(self, s, md):
        out = example_set_findings(s, "KQL-015")
        ex = s.example_queries
        if not ex:
            for rid in ("KQL-006", "KQL-007", "KQL-008", "KQL-009", "KQL-011", "KQL-012", "KQL-014"):
                out.append(make_finding(rid, Status.NOT_APPLICABLE, "No example queries", source=s))
            return out
        known = {o["name"].lower() for o in md["selected"]} | {o["path"].lower() for o in md["selected"]}
        hits = {r: [] for r in ("KQL-006", "KQL-007", "KQL-008", "KQL-009", "KQL-014")}
        agg_by_src, joins, xcluster = {}, [], []
        for e in ex:
            q = e.get("query", "")
            for rid, msg in lint_kql(q):
                hits[rid].append(detail(rid, Status.WARN if rid != "KQL-014" else Status.FAIL, msg, source=s, object_path=e["id"]))
            src = kql_first_source(q)
            if known and src and src not in known and not re.match(r"^(print|datatable|range|union|search|materialize|toscalar|let)$", src):
                hits["KQL-014"].append(detail("KQL-014", Status.WARN, f"First source '{src}' is not a selected object", source=s, object_path=e["id"]))
            miss = unmapped_literals(e.get("question", ""), q)
            if miss:
                hits["KQL-014"].append(detail("KQL-014", Status.WARN, f"Question literals not found in query: {miss}", source=s, object_path=e["id"]))
            if re.search(r"\|\s*summarize\b", q, re.I):
                agg_by_src[src] = agg_by_src.get(src, 0) + 1
            if re.search(r"\bjoin\b", q, re.I):
                joins.append(e["id"])
            if re.search(r"\b(cluster|database)\s*\(", q, re.I):
                xcluster.append(e["id"])
        n = len(ex)
        for rid in ("KQL-006", "KQL-007", "KQL-008", "KQL-009"):
            bad = len({h.object_path for h in hits[rid]})
            out.append(make_finding(rid, Status.PASS if not bad else Status.WARN, f"{n - bad}/{n} examples pass", ratio=(n - bad) / n if bad else None, source=s, confidence="MEDIUM"))
            out += hits[rid]
        bad14 = len({h.object_path for h in hits["KQL-014"]})
        out.append(make_finding("KQL-014", Status.PASS if not bad14 else Status.FAIL if any(h.status == Status.FAIL for h in hits["KQL-014"]) else Status.WARN,
                                f"{n - bad14}/{n} examples pass static validation", ratio=(n - bad14) / n if bad14 else None, source=s, confidence="MEDIUM"))
        out += hits["KQL-014"]
        heavy = {k: v for k, v in agg_by_src.items() if v >= 3}
        out.append(make_finding("KQL-011", Status.WARN if heavy and not md["mviews"] else Status.PASS,
                                f"Repeated aggregations per source: {heavy or 'none'}; materialized views selected: {len(md['mviews'])}", source=s, confidence="MEDIUM"))
        if not joins and not xcluster:
            out.append(make_finding("KQL-012", Status.NOT_APPLICABLE, "No joins or cross-cluster references in examples", source=s))
        else:
            out.append(make_finding("KQL-012", Status.WARN if xcluster else Status.PASS,
                                    f"Examples with join: {joins[:5]}; cross-cluster/database references for review: {xcluster[:5]}", source=s, confidence="MEDIUM"))
        return out

    def smoke_test(self, s, md):
        try:
            run_kql(s, "print ok = 1")
        except Exception as e:
            ev = f"Eventhouse not reachable for live tests: {short(e, 200)}"
            return [make_finding(r, Status.NOT_EVALUATED, ev, source=s, confidence="LOW") for r in self.live_rules()]
        out, ok, failed = [], 0, []
        for o in md["tables"][:SMOKE_TEST_MAX_OBJECTS]:
            try:
                run_kql(s, f"['{o['name']}'] | take 1")
                ok += 1
            except Exception as e:
                failed.append(detail("KQL-002", Status.FAIL, f"Bounded read failed: {short(e, 140)}", source=s, object_path=o["path"]))
        if not md["tables"]:
            out.append(make_finding("KQL-002", Status.NOT_APPLICABLE, "No tables/materialized views selected", source=s))
        else:
            out.append(make_finding("KQL-002", Status.PASS if not failed else Status.FAIL, f"{ok}/{min(len(md['tables']), SMOKE_TEST_MAX_OBJECTS)} bounded reads succeeded", source=s))
            out += failed
        fn_fail = []
        for o in md["functions"][:SMOKE_TEST_MAX_OBJECTS]:
            try:
                run_kql(s, f"{o['name']}() | take 1")
            except Exception as e:
                fn_fail.append(detail("KQL-013", Status.WARN, f"Parameterless call failed (may require parameters): {short(e, 120)}", source=s, object_path=o["path"]))
        out.append(make_finding("KQL-013", Status.NOT_APPLICABLE if not md["functions"] else Status.PASS if not fn_fail else Status.WARN,
                                f"{len(md['functions']) - len(fn_fail)}/{len(md['functions'])} functions callable without parameters", source=s))
        out += fn_fail
        fresh, stale = [], []
        for o in md["tables"][:SMOKE_TEST_MAX_OBJECTS]:
            dcol = next((c["name"] for c in o.get("columns", []) if "datetime" in c.get("data_type", "").lower()), None)
            if not dcol:
                continue
            try:
                df, _ = run_kql(s, f"['{o['name']}'] | summarize latest = max(['{dcol}'])")
                latest = pd.to_datetime(df.iloc[0]["latest"], utc=True, errors="coerce")
                if pd.notna(latest):
                    age_h = (datetime.now(timezone.utc) - latest.to_pydatetime()).total_seconds() / 3600
                    fresh.append((o["path"], round(age_h, 1)))
                    if age_h > FRESHNESS_MAX_AGE_HOURS:
                        stale.append(detail("KQL-016", Status.WARN, f"Newest event {age_h:.1f} h old", source=s, object_path=o["path"]))
            except Exception:
                pass
        out.append(make_finding("KQL-016", Status.NOT_EVALUATED if not fresh else Status.WARN if stale else Status.PASS,
                                f"Newest-event age (h) sampled: {fresh[:5]}", source=s, confidence="MEDIUM"))
        out += stale
        ex_fail = []
        for e in s.example_queries:
            try:
                run_kql(s, e["query"])
            except Exception as ex:
                ex_fail.append(detail("KQL-014", Status.FAIL, f"Execution failed: {short(ex, 160)}", source=s, object_path=e["id"]))
        if s.example_queries:
            out.append(make_finding("KQL-014", Status.FAIL if ex_fail else Status.PASS,
                                    f"Live execution: {len(s.example_queries) - len(ex_fail)}/{len(s.example_queries)} examples succeeded", source=s))
            out += ex_fail
        return out


ADAPTERS["KQL_DATABASE"] = KqlSourceAdapter()
KQL_FINDINGS = run_family("KQL", "kql", "EVENTHOUSE KQL DATABASE — KQL-001 … KQL-018")

---
# 8️⃣ POWER BI SEMANTIC MODEL — `SemanticModelAdapter`

All v2.2.2 semantic-model checks are kept. They now run through the **`SemanticModelAdapter`** and have **stable rule IDs** (`SM-000` … `SM-027`). Each check cell runs once for **every connected semantic model**. In standalone mode, it runs once for the `dataset` parameter. When no semantic model is connected, each check reports `NOT_APPLICABLE`.

### ⚠️ Important configuration distinction (current Fabric data agent behavior)

The semantic-model **DAX generation tool** uses semantic-model metadata and **Prep for AI** configuration. It **ignores data-agent-level instructions** for DAX generation. The data agent also doesn't support **source instructions or source descriptions** for semantic models. That means:

- **Prep for AI** holds semantic definitions, metric preferences, date defaults, synonyms, and negative scope.
- **Agent level** holds cross-source routing, shared abbreviations, response format, and tone.
- The tables selected in the data agent should **match the Prep for AI AI Data Schema** (`SM-025`).
- The runtime determines whether standard or **advanced DAX** generation (preview) is active. This is recorded in `SM-026`. The Q&A dependency for advanced-DAX value indexing is version-gated because **Q&A retirement is scheduled for December 2026**.

| Legacy # | Rule ID | Legacy # | Rule ID | Legacy # | Rule ID |
|---|---|---|---|---|---|
| 1.1 Star schema | `SM-001` | 1.10 Report measures | `SM-010` | 2.1 AI Data Schema | `SM-019` |
| 1.2 BPA | `SM-002` | 1.11 Date fields | `SM-011` | 2.2 Dependencies | `SM-020` |
| 1.3 Memory | `SM-003` | 1.12 Hidden/private | `SM-012` | 2.3 Helper measures | `SM-021` |
| 1.4 Data types | `SM-004` | 1.13 Complexity | `SM-013` | 3 Verified Answers | `SM-022` |
| 1.5 Naming | `SM-005` | 1.14 Perf Analyzer | `SM-014` | 4 AI Instructions | `SM-023` |
| 1.6 Descriptions | `SM-006` | 1.15 Duplicate columns | `SM-015` | 4.1 Prep for AI config | `SM-024` |
| 1.7 Synonyms | `SM-007` | 1.16 Grounding | `SM-016` | 5 Agent ↔ AI Schema | `SM-025` |
| 1.8 Row labels | `SM-008` | 1.17 Hierarchies | `SM-017` | DAX mode / runtime | `SM-026` |
| 1.9 Explicit measures | `SM-009` | 1.18 Description length | `SM-018` | 7 Service & admin | `SM-027` |

---

## 8.0 🎯 Microsoft Reference Framework (semantic model)

### Consumption Mode Decision Matrix

**Ask the user first: How will this model be consumed?**

| **Consumption Mode** | **Description** | **Required Investment** |
|---|---|---|
| **📊 Reports Only** | Users access data via pre-built reports & dashboards | Foundation: architecture, naming, star schema |
| **💬 Conversational BI** | Copilot, Data Agents, or Q&A for ad-hoc questions | Full: foundation + AI metadata (descriptions, AI instructions, AI Data Schema) |
| **🔄 Both** | Mixed consumption (reports + conversational) | Full investment + Verified Answers for critical questions |

**Impact:** Determines scope of readiness review. Don't over-invest in Conversational BI features if the model is Reports-only.

---

### 🔑 Four Core Principles

These principles guide every decision in the readiness review:

1. **Foundation First** 
   - A weak model cannot be rescued by AI instructions or Verified Answers
   - Star schema, explicit measures, and correct data types are prerequisites
   - If section 1 (Model Architecture) fails, fixing section 4 (AI Instructions) will not help

2. **Literal Interpretation**
   - Copilot reads names and descriptions literally
   - `AMT` ≠ `Sales Amount` to a language model
   - Use business-friendly, human-readable terminology (Section 2: Naming)

3. **Explicit Over Implicit**
   - Implicit measures, report-scoped measures, and hidden business logic are invisible to Copilot
   - Everything reachable by natural language must live in the model as an explicit measure with metadata
   - Use explicit DAX measures instead of implicit aggregations (Section 1: Model Architecture)

4. **Iterate from Observation**
   - Don't guess. Write descriptions and AI instructions based on real user prompts and observed Copilot failures
   - Test with actual questions; refine based on results
   - Validate with domain experts and end-users before deployment

---

### 🔧 Editing Capability Routing (What Tool to Use?)

| **What to Edit** | **Where It Lives** | **How to Edit** | **Agent Can?** |
|---|---|---|---|
| **Names, descriptions, measures, data types, relationships** | TOM model metadata | Power BI Desktop | ✅ Yes (via MCP) |
| **Synonyms, AI Instructions, AI Data Schema** | PBIP files (.json) | PBIP project files or Fabric `updateDefinition` | ❌ Not directly |
| **Verified Answers** | PBIP + Report layer | Power BI Desktop + live testing | ❌ Suggest only |
| **Hidden flags, summarization, data categories** | TOM model metadata | Power BI Desktop | ✅ Yes (via MCP) |

**Rule:** When you cannot edit something automatically, refer the user to the appropriate tool.

---

---

### 📋 Six-Section Readiness Checklist (Microsoft Reference Structure)

**Complete sections in order — do not skip ahead.** If an earlier section fails, fixing later sections will not help.

| **Section** | **Focus** | **Do** | **Don't** |
|---|---|---|---|
| **0. Business Context** | Gather user personas, key questions, scope | Interview users; list top 10-15 questions | Assume scope; skip discovery |
| **1. Model Architecture** | Foundation quality: star schema, explicit measures, data types | Use star schema; create explicit DAX measures; correct data types | Leave flat models; rely on implicit aggregations |
| **2. Naming** | Business-friendly names for Copilot literal interpretation | Use full words (not abbreviations); avoid jargon | Use technical names, CamelCase, UPPER_CASE |
| **3. Descriptions** | Concise context (200-char limit for Copilot); front-load key info | Lead with purpose; include units/ranges; describe calculations | Generic auto-generated descriptions; bury key info |
| **4. AI Instructions** | Metric routing, terminology, time definitions, clarification rules | Define routing for ambiguous terms; list business abbreviations | Duplicate descriptions; encode should-be-measures as instructions |
| **5. AI Data Schema** | Control Copilot exposure: include business tables, exclude helpers | Include only business-relevant columns; exclude PKs/FKs/GUIDs; add synonyms | Expose everything; dilute signal Copilot uses |
| **6. Verified Answers** | Pre-built trusted responses for top questions | Configure top 3-5 most-asked questions in Desktop; validate with users | Attempt to auto-generate; skip user validation |

**Impact:** Sections 0-5 can be partially automated via MCP. Section 6 requires manual Desktop configuration. Skipping = models that disappoint at scale.

---

In [ ]:
print("=" * 70)
print("SEMANTIC MODEL ADAPTER — METADATA COLLECTION (SM-000)")
print("=" * 70)

SYS_PATTERN = r'^(DateTableTemplate_|LocalDateTable_)'
PREP4AI_MCP_URL = "https://api.fabric.microsoft.com/v1/mcp/powerbi"
_LEGACY_SEVERITY = {"critical": "CRITICAL", "important": "IMPORTANT", "warning": "IMPORTANT", "recommended": "RECOMMENDED"}


def is_hidden_mask(df, col='Is Hidden'):
    if col not in df.columns:
        return pd.Series([False] * len(df), index=df.index)
    return df[col].apply(lambda v: str(v).lower() == 'true' if not isinstance(v, bool) else v)


def missing_desc_mask(df, col='Description'):
    if col not in df.columns:
        return pd.Series([True] * len(df), index=df.index)
    return df[col].isna() | (df[col].astype(str).str.strip() == '')


def fetch_prep_for_ai(model_identifier, workspace_name):
    """Retrieve the Prep for AI configuration through the Power BI remote MCP server.
    Adapted from Sandeep Pawar (Fabric.guru). The bearer token is never printed or persisted.
    """
    import httpx
    from sempy.fabric import list_datasets
    datasets = list_datasets(workspace=workspace_name)
    name_col = next((c for c in ['Dataset Name', 'Name'] if c in datasets.columns), None)
    id_col = next((c for c in ['Dataset ID', 'Id', 'Object ID'] if c in datasets.columns), None)
    if name_col is None or id_col is None:
        raise RuntimeError(f"Could not identify dataset name/id columns: {list(datasets.columns)}")
    model_str = str(model_identifier).strip()
    info = datasets[datasets[name_col].astype(str).str.strip().eq(model_str) | datasets[id_col].astype(str).str.strip().eq(model_str)]
    if info.empty:
        return None
    model_id = str(info[id_col].iloc[0])
    token = notebookutils.credentials.getToken("pbi")
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json", "Accept": "application/json, text/event-stream"}
    payload = {"jsonrpc": "2.0", "id": 1, "method": "tools/call",
               "params": {"name": "GetSemanticModelSchema", "arguments": {"artifactId": model_id}}}
    with httpx.Client(timeout=120.0) as client:
        response = client.post(PREP4AI_MCP_URL, headers=headers, json=payload)
    del token, headers
    if response.status_code >= 400:
        raise RuntimeError(f"Power BI MCP server returned HTTP {response.status_code}")
    data = {}
    for line in response.text.split('\n'):
        if line.startswith('data: '):
            data = json.loads(line[6:])
            break
    if not data:
        data = response.json()
    parsed = json.loads(data['result']['content'][0]['text'])
    return {
        "name": parsed['semanticModel']['Name'],
        "tables": parsed['schema'].get('Tables') or [],
        "relationships": parsed['schema'].get('ActiveRelationships') or [],
        "custom_instructions": parsed['schema'].get('CustomInstructions'),
        "verified_answers": parsed['schema'].get('VerifiedAnswers') or [],
    }


@dataclass
class SemanticContext:
    source: DataSourceProfile
    dataset: str
    workspace: Optional[str]
    frames: dict = field(default_factory=dict)
    prep4ai: Optional[dict] = None
    prep4ai_error: Optional[str] = None
    load_error: Optional[str] = None
    check_scores: dict = field(default_factory=dict)
    check_status: dict = field(default_factory=dict)


class SemanticModelAdapter(SourceAdapter):
    source_family = "SEMANTIC"
    primary_rule = "SM-000"

    def collect_metadata(self, s) -> SemanticContext:
        ds = s.artifact_id or s.display_name
        ws = s.workspace_id or AGENT_PROFILE.workspace or (workspace if AGENT_PROFILE.standalone_semantic else None)
        ctx = SemanticContext(source=s, dataset=ds, workspace=ws)
        if fabric is None:
            ctx.load_error = f"sempy not available ({_SEMPY_IMPORT_ERROR})"
            return ctx
        try:
            tables_df = fabric.list_tables(dataset=ds, workspace=ws)
            columns_df = fabric.list_columns(dataset=ds, workspace=ws)
            measures_df = fabric.list_measures(dataset=ds, workspace=ws)
            relationships_df = fabric.list_relationships(dataset=ds, workspace=ws)
        except Exception as e:
            ctx.load_error = f"{type(e).__name__}: {short(e, 200)}"
            return ctx
        tables_df = tables_df[~tables_df['Name'].str.match(SYS_PATTERN, na=False)].copy()
        visible_tables = tables_df[~is_hidden_mask(tables_df)]
        visible_columns = columns_df[~is_hidden_mask(columns_df)]
        visible_measures = measures_df[~is_hidden_mask(measures_df)]
        if 'Is Row Number' in columns_df.columns:
            rn_mask = columns_df['Is Row Number'].apply(lambda v: str(v).lower() == 'true' if not isinstance(v, bool) else v)
            visible_columns = visible_columns[~rn_mask[visible_columns.index]]
        ctx.frames = dict(tables_df=tables_df, columns_df=columns_df, measures_df=measures_df, relationships_df=relationships_df,
                          visible_tables=visible_tables, visible_columns=visible_columns, visible_measures=visible_measures)
        try:
            ctx.prep4ai = fetch_prep_for_ai(ds, ws)
            if ctx.prep4ai is None:
                ctx.prep4ai_error = f"Model '{ds}' not found in workspace listing"
        except Exception as e:
            ctx.prep4ai_error = f"{type(e).__name__}: {short(redact(e), 200)}"
        return ctx


ADAPTERS["SEMANTIC_MODEL"] = SemanticModelAdapter()
SEMANTIC_SOURCES = [s for s in SOURCES if s.source_family == "SEMANTIC" and not skip_reason(s)]
SEMANTIC_CONTEXTS = [ADAPTERS["SEMANTIC_MODEL"].collect_metadata(s) for s in SEMANTIC_SOURCES]
CURRENT_SEMANTIC_SOURCE = None
check_scores, check_status, all_issues, prep4ai, PREP4AI_ERROR = {}, {}, [], None, None


def _activate_semantic(ctx):
    """Expose one model's metadata under the v2.x global names used by the check cells."""
    g = globals()
    g.update(ctx.frames)
    g.update(dataset=ctx.dataset, workspace=ctx.workspace, prep4ai=ctx.prep4ai, PREP4AI_ERROR=ctx.prep4ai_error,
             check_scores=ctx.check_scores, check_status=ctx.check_status, all_issues=[], CURRENT_SEMANTIC_SOURCE=ctx.source)


def sm_manual(rule_id, guidance, legacy_attested=False):
    """Manual semantic check → MANUAL (not zero) unless attested (MANUAL_ATTESTATIONS or legacy flag)."""
    att = attestation_for(rule_id, CURRENT_SEMANTIC_SOURCE)
    if att:
        st = str(att.get("status", "PASS")).upper()
        st, ev = (st if st in Status.__members__ else "PASS"), f"USER_SUPPLIED: {att.get('evidence', 'attested')}"
    elif legacy_attested:
        st, ev = "PASS", "USER_SUPPLIED: prep_for_ai_configured = True"
    else:
        st, ev = "MANUAL", guidance
    check_status[rule_id] = (Status(st), ev)
    check_scores[rule_id] = {'achieved': {"PASS": 1, "WARN": 0.5}.get(st, 0), 'max': 1, 'weight': 1}
    print(f"\n📊 Status: {STATUS_ICON.get(st, '')} {st} — {ev}")


def _convert_semantic(rule_id, ctx, issues) -> list:
    sc, st = ctx.check_scores.get(rule_id), ctx.check_status.get(rule_id)
    ratio = (sc['achieved'] / sc['max']) if sc and sc.get('max') else None
    issue_text = "; ".join(d for _, _, d in issues)
    if st:
        status, ev = st
    elif sc is None:
        status, ev = Status.NOT_EVALUATED, "Check did not record a result"
    else:
        status = Status.PASS if ratio >= 0.9 else Status.WARN if ratio >= 0.5 else Status.FAIL
        ev = f"Score {sc['achieved']}/{sc['max']}"
    if issue_text and status in EVALUATED_STATUSES:
        ev = f"{ev}; {issue_text}"
    out = [make_finding(rule_id, status, ev, source=ctx.source, ratio=ratio if status in EVALUATED_STATUSES else None)]
    for sev, check, desc in issues:
        out.append(detail(rule_id, Status.FAIL if sev == "critical" else Status.WARN, desc, source=ctx.source,
                          object_path=check, severity=_LEGACY_SEVERITY.get(sev, "IMPORTANT")))
    return out


def run_semantic_check(rule_id, fn):
    """Run a v2.x check body once per semantic model; convert its score/issues into normalized findings."""
    if not SEMANTIC_CONTEXTS:
        print(f"➖ {rule_id}: NOT_APPLICABLE — no Power BI semantic model to assess (see 4.1 Source Run Plan).")
        return
    saved = {k: globals().get(k) for k in ("dataset", "workspace")}
    try:
        for ctx in SEMANTIC_CONTEXTS:
            if len(SEMANTIC_CONTEXTS) > 1:
                print(f"\n──────── Semantic model: {ctx.source.display_name} ────────")
            section = f"semantic:{rule_id}:{ctx.source.source_id}"
            begin_section(section)
            if ctx.load_error:
                print(f"⏸️ {rule_id}: NOT_EVALUATED — metadata not loaded ({ctx.load_error})")
                record(section, make_finding(rule_id, Status.NOT_EVALUATED, f"Metadata not loaded: {ctx.load_error}", source=ctx.source, confidence="LOW"))
                continue
            _activate_semantic(ctx)
            ctx.check_scores.pop(rule_id, None)
            ctx.check_status.pop(rule_id, None)
            try:
                fn()
                record(section, _convert_semantic(rule_id, ctx, globals()["all_issues"]))
            except Exception as e:
                print(f"⏸️ {rule_id}: NOT_EVALUATED — check raised {type(e).__name__}: {short(e, 200)}")
                record(section, make_finding(rule_id, Status.NOT_EVALUATED, f"Check raised {type(e).__name__}: {short(e, 200)}",
                                             source=ctx.source, confidence="LOW"))
    finally:
        globals().update(saved)


begin_section("semantic:SM-000")
for s in SOURCES:
    if s.source_family == "SEMANTIC" and skip_reason(s):
        record("semantic:SM-000", make_finding("SM-000", Status.NOT_EVALUATED, f"Skipped: {skip_reason(s)}", source=s, confidence="LOW"))
        print(f"\n⏭️ {s.display_name}: skipped — {skip_reason(s)}")
if not SEMANTIC_CONTEXTS:
    print("\n➖ No Power BI semantic model to assess — §8 and §16.2 will report NOT_APPLICABLE, so you can skip them.")
for ctx in SEMANTIC_CONTEXTS:
    if ctx.load_error:
        record("semantic:SM-000", make_finding("SM-000", Status.FAIL if fabric is not None else Status.NOT_EVALUATED,
                                               f"Metadata load failed: {ctx.load_error}", source=ctx.source))
        print(f"\n❌ {ctx.source.display_name}: {ctx.load_error}")
        print("   Verify: 1) dataset/workspace names  2) Build permission  3) model is published to Fabric")
        continue
    f = ctx.frames
    record("semantic:SM-000", make_finding("SM-000", Status.PASS,
           f"{len(f['visible_tables'])} visible tables, {len(f['visible_columns'])} visible columns, "
           f"{len(f['visible_measures'])} visible measures, {len(f['relationships_df'])} relationships", source=ctx.source))
    print(f"\n✅ Metadata loaded for: '{ctx.source.display_name}'  (workspace: {ctx.workspace or 'current'})")
    print(f"  Visible tables   : {len(f['visible_tables']):>4}  (total incl. hidden: {len(f['tables_df'])})")
    print(f"  Visible columns  : {len(f['visible_columns']):>4}  (total incl. hidden: {len(f['columns_df'])})")
    print(f"  Visible measures : {len(f['visible_measures']):>4}  (total incl. hidden: {len(f['measures_df'])})")
    print(f"  Relationships    : {len(f['relationships_df']):>4}")
    print(f"  Prep for AI      : {'retrieved' if ctx.prep4ai else 'not retrieved — ' + str(ctx.prep4ai_error)}")

---
## 8.1 SEMANTIC MODEL OPTIMIZATION · `SM-001` … `SM-018`

This section validates foundational model quality that directly impacts Data Agent accuracy and performance.

### 1.1 ⭐ Star Schema Validation · `SM-001`

**Why it matters:** Data Agent's DAX generation relies on clear fact-dimension relationships. Flat/denormalized models produce unreliable results.

**What's checked:**
- No relationships (flat model) — **CRITICAL**
- Many-to-many relationships — **should be rare**
- Bidirectional cross-filtering — **use sparingly**
- Isolated visible tables — **connect or hide**

In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 1.1: STAR SCHEMA VALIDATION")
    print("=" * 70)

    issues = []
    score = 20  # Max points

    if relationships_df.empty:
        print("❌ CRITICAL: No relationships found — flat/denormalized model detected.")
        print("\n   This is a BLOCKING issue for Data Agent accuracy.")
        print("   Refactor into a star schema with clear fact and dimension tables.")
        print("\n   📚 Learn: https://learn.microsoft.com/power-bi/guidance/star-schema")
        issues.append(("critical", "1.1 Star Schema", "No relationships found — flat model"))
        score = 0
    else:
        # Many-to-many relationships
        m2m_col = next((c for c in ['Multiplicity', 'Cardinality'] if c in relationships_df.columns), None)
        if m2m_col:
            m2m = relationships_df[
                relationships_df[m2m_col].astype(str).str.contains('ManyToMany|Many.*Many', case=False, na=False)
            ]
            if not m2m.empty:
                print(f"⚠️  {len(m2m)} many-to-many relationship(s) found:")
                for _, r in m2m.iterrows():
                    ft = r.get('From Table', 'Unknown')
                    tt = r.get('To Table', 'Unknown')
                    print(f"     • {ft}  ↔  {tt}")
                print("\n   M:M relationships reduce DAX accuracy and performance.")
                print("   ACTION: Introduce bridge tables to resolve these.")
                issues.append(("important", "1.1 Star Schema", f"{len(m2m)} many-to-many relationship(s)"))
                score -= min(6, len(m2m) * 3)

        # Bidirectional cross-filter
        cf_col = next((c for c in ['Cross Filter Direction', 'Cross Filtering Behavior'] if c in relationships_df.columns), None)
        if cf_col:
            bidir = relationships_df[
                relationships_df[cf_col].astype(str).str.contains('Both', case=False, na=False)
            ]
            if not bidir.empty:
                print(f"⚠️  {len(bidir)} bidirectional relationship(s):")
                for _, r in bidir.head(5).iterrows():
                    print(f"     • {r.get('From Table', '?')} ↔ {r.get('To Table', '?')}")
                print("\n   Bidirectional filtering can create ambiguity in DAX generation.")
                print("   ACTION: Review if these are necessary; consider alternatives.")
                issues.append(("important", "1.1 Star Schema", f"{len(bidir)} bidirectional relationship(s)"))
                score -= min(4, len(bidir) * 2)

        # Isolated visible tables
        related_tables = set()
        if 'From Table' in relationships_df.columns:
            related_tables = set(relationships_df['From Table'].dropna()) | set(relationships_df['To Table'].dropna())
        visible_tbl_names = set(visible_tables['Name'].dropna())
        isolated = visible_tbl_names - related_tables
        # Exclude parameter tables
        isolated = {t for t in isolated if not any(x in t.lower() for x in ['parameter', ' parameter', '_param'])}

        if isolated:
            print(f"\n⚠️  {len(isolated)} visible table(s) with no relationships:")
            for t in sorted(isolated)[:8]:
                print(f"     • {t}")
            if len(isolated) > 8:
                print(f"     ... and {len(isolated) - 8} more")
            print("\n   ACTION: Connect these tables to the model or hide them.")
            issues.append(("important", "1.1 Star Schema", f"{len(isolated)} isolated visible table(s)"))
            score -= min(6, len(isolated) * 2)

        if not issues:
            print("✅ PASSED: Relationship structure follows star schema design.")

    score = max(0, score)
    print(f"\n📊 Score: {score}/20")
    check_scores['SM-001'] = {'achieved': score, 'max': 20, 'weight': 3}
    all_issues.extend(issues)


run_semantic_check('SM-001', _sm_check)


### 1.2 ⚡ Best Practice Analyzer · `SM-002`

**Why it matters:** BPA identifies 60+ performance, DAX quality, and error prevention issues that affect Data Agent response time and reliability.

**What's checked:** All BPA rules (performance, DAX, formatting, maintenance)

In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 1.2: BEST PRACTICE ANALYZER")
    print("=" * 70)

    issues = []
    score = 15  # Max points

    try:
        import sempy_labs as labs
        bpa_results = labs.run_model_bpa(dataset=dataset, workspace=workspace)

        if bpa_results is not None and not bpa_results.empty:
            sev_col = 'Severity' if 'Severity' in bpa_results.columns else None
            print(f"⚠️  {len(bpa_results)} BPA recommendation(s) found:\n")

            if sev_col:
                sev_counts = bpa_results[sev_col].value_counts()
                for sev in ['Error', 'Warning', 'Info']:
                    if sev in sev_counts.index:
                        print(f"   {sev}: {sev_counts[sev]}")

            # Score based on severity
            if   len(bpa_results) >= 30: score = 3
            elif len(bpa_results) >= 20: score = 6
            elif len(bpa_results) >= 10: score = 9
            elif len(bpa_results) >= 5:  score = 12
            else:                        score = 14

            print("\nTop issues (click to expand for full details):")
            display(bpa_results.head(20))

            issues.append(("important", "1.2 BPA", f"{len(bpa_results)} recommendations from Best Practice Analyzer"))

            print("\n📚 Learn more: https://learn.microsoft.com/power-bi/transform-model/service-notebooks")
        else:
            print("✅ PASSED: No BPA issues found!")

    except ImportError:
        print("ℹ️  Semantic Link Labs not available for BPA.")
        print("   Install: %pip install semantic-link-labs --upgrade")
        score = 10
        check_status['SM-002'] = (Status.NOT_EVALUATED, "semantic-link-labs not available for BPA")
    except Exception as e:
        print(f"⚠️  Could not run BPA: {e}")
        print("\n   Run manually: labs.run_model_bpa(dataset=dataset, workspace=workspace)")
        score = 10
        check_status['SM-002'] = (Status.NOT_EVALUATED, f"BPA could not run: {e}")

    print(f"\n📊 Score: {score}/15")
    check_scores['SM-002'] = {'achieved': score, 'max': 15, 'weight': 3}
    all_issues.extend(issues)


run_semantic_check('SM-002', _sm_check)


### 1.3 💾 Memory Analyzer · `SM-003`

**Why it matters:** Large model size impacts Data Agent performance and workspace capacity. Optimizing memory usage improves response times.

**What's checked:** Runs `fabric.model_memory_analyzer()` to analyze model memory usage and identify optimization opportunities

**Key insights:**
- Memory footprint by table and column
- High-cardinality columns that may need optimization
- Unused or redundant columns
- Data type optimization opportunities

In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 1.3: MEMORY ANALYZER")
    print("=" * 70)

    print("Running Model Memory Analyzer...\n")

    try:
        # Run the model memory analyzer
        memory_results = fabric.model_memory_analyzer(dataset=dataset, workspace=workspace)

        print("✅ Memory Analyzer completed successfully!")
        print("\n📊 Results displayed above.")
        print("\n✅ Key actions from Memory Analyzer:")
        print("   • Remove unused columns")
        print("   • Optimize high-cardinality columns")
        print("   • Review column data types")
        print("   • Consider aggregations for large fact tables")

        if is_direct_lake:
            print("\n💡 DIRECT LAKE SPECIFIC:")
            print("   • Run V-Order optimization on Delta tables")
            print("   • Review Direct Lake fallback scenarios")
            print("   • Monitor Direct Lake mode in Query Performance Analyzer")
            print("\n   📚 https://learn.microsoft.com/fabric/fundamentals/direct-lake-understand-storage")

        # Full credit for running the analyzer
        score = 5
        issues = []

    except Exception as e:
        print(f"⚠️  Could not run Memory Analyzer: {e}")
        print("\n📚 Documentation:")
        print("   https://learn.microsoft.com/power-bi/transform-model/service-notebooks#model-memory-analyzer")
        print("\n🔧 Manual alternative:")
        print("   Run in a separate cell: fabric.model_memory_analyzer(dataset=dataset, workspace=workspace)")

        # Partial credit
        score = 3
        issues = [("important", "1.3 Memory", "Memory Analyzer could not be executed")]
        check_status['SM-003'] = (Status.NOT_EVALUATED, f"Memory Analyzer could not run: {e}")

    check_scores['SM-003'] = {'achieved': score, 'max': 5, 'weight': 1}
    all_issues.extend(issues)
    print(f"\n📊 Score: {score}/5")


run_semantic_check('SM-003', _sm_check)


### 1.4 🔍 Data Type Validation · `SM-004`

**Why it matters:** Incorrect data types cause DAX errors, incorrect aggregations, and poor performance.

**What's checked:**
- Date columns stored as text
- Numeric columns stored as text
- Text columns with numeric names

In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 1.4: DATA TYPE VALIDATION")
    print("=" * 70)

    issues = []
    score = 10

    # Detect potential date columns stored as text
    date_pattern = r'(date|dt_|dt$|_dt|day|month|year|quarter|fiscal|calendar|timestamp|time)'
    text_date_cols = visible_columns[
        (visible_columns['Data Type'].astype(str).str.contains('string|text', case=False, na=False)) &
        (visible_columns['Column Name'].astype(str).str.lower().str.contains(date_pattern, na=False))
    ]

    numeric_pattern = r'(amount|amt|quantity|qty|count|cnt|price|cost|rate|percent|pct|total|sum|value|val|number|num|nbr|id|key)$'
    text_numeric_cols = visible_columns[
        (visible_columns['Data Type'].astype(str).str.contains('string|text', case=False, na=False)) &
        (visible_columns['Column Name'].astype(str).str.lower().str.contains(numeric_pattern, na=False))
    ]

    if not text_date_cols.empty:
        print(f"⚠️  {len(text_date_cols)} potential date column(s) stored as text:")
        for _, row in text_date_cols.head(10).iterrows():
            print(f"     • {row.get('Table Name')}[{row.get('Column Name')}]")
        if len(text_date_cols) > 10:
            print(f"     ... and {len(text_date_cols) - 10} more")
        print("\n   ACTION: Convert to Date or DateTime type in Power Query.")
        issues.append(("important", "1.4 Data Types", f"{len(text_date_cols)} date-like columns stored as text"))
        score -= min(5, len(text_date_cols))

    if not text_numeric_cols.empty:
        print(f"\n⚠️  {len(text_numeric_cols)} potential numeric column(s) stored as text:")
        for _, row in text_numeric_cols.head(10).iterrows():
            print(f"     • {row.get('Table Name')}[{row.get('Column Name')}]")
        if len(text_numeric_cols) > 10:
            print(f"     ... and {len(text_numeric_cols) - 10} more")
        print("\n   ACTION: Verify these are truly text; convert if numeric.")
        issues.append(("recommended", "1.4 Data Types", f"{len(text_numeric_cols)} numeric-like columns stored as text"))
        score -= min(3, len(text_numeric_cols))

    if not issues:
        print("✅ PASSED: No obvious data type issues detected.")

    score = max(0, score)
    print(f"\n📊 Score: {score}/10")
    check_scores['SM-004'] = {'achieved': score, 'max': 10, 'weight': 2}
    all_issues.extend(issues)


run_semantic_check('SM-004', _sm_check)


### 1.5 🏷️ Business-Friendly Naming · `SM-005`

**Why it matters:** Data Agent interprets object names literally. Technical names like `DIM_CUST_01` or `F_SLS_AMT` provide no context.

**What's checked:** Database-style prefixes/suffixes, abbreviations, ALL_CAPS, cryptic names

**📐 Naming convention guidelines** (applied to tables, columns, and measures):

- **Avoid abbreviations and acronyms** — instead of `Std. Margin` or `S.M.`, write `Standard Margin`.
- **Avoid excessive numbers and grammar, and don't use Unicode characters** — keep names plain ASCII.
- **Avoid technical prefixes** like `DIM_` or `FACT_`.
- **Use standard casing with spaces** instead of `CamelCase` or `snake_case`.
- **Use special prefixes for technical fields or properties** to sort or identify them.
- **Do specify aggregations, units, and periods — but use a consistent syntax.** Choose one convention for each of the following:
  - **Periods** (previous year, previous month, yesterday, etc.) — instead of `Sales Last Year`, write `Sales (1YP)` or `Sales (PY)`. `Sales (1YP)` is preferable because it's easy to extend to `Sales (2YP)`.
  - **Aggregations** (MTD, YTD, Weekly Average, etc.) — instead of `Sales Month To Date PY`, write `Sales MTD (1YP)`.
  - **Units** (currency €, percent %, units (qty), lines, pieces (pcs), etc.) — layering units can get complex, e.g. `Sales (€) MTD (1YP)` alongside `Sales (Units) YTD (2YP)`.
  - **Comparisons** to targets, periods, or aggregates (e.g. `vs Budget`, `vs 1YP`, `vs Average`).

In [ ]:
def _sm_check():
    import re
    import pandas as pd

    # -------------------------------------------------------------------------
    # CONFIGURATION: PATTERNS FOR COPILOT READINESS
    # -------------------------------------------------------------------------

    # 1. Technical/Database Naming Patterns
    TECH_PATTERNS = [
        (r'^(DIM|FACT|FCT|STG|SRC|TBL|VW|RPT|TMP|TEMP|LKP|REF|BRG|BRIDGE|MAP|INT|ODS|SLV|GLD|SILVER|BRONZE|RAW)_', "Database prefix (DIM_, FACT_)"),
        (r'_(DIM|FACT|FCT|TBL|LKP|REF|SK|NK|AK|BK|PK|FK)$', "Database suffix (_DIM, _SK)"),
        (r'_(AMT|QTY|CNT|CT|NUM|NBR|NO|DT|TS|FLG|FLAG|IND|CD|CODE|KEY|ID|VAL)$', "Abbreviation suffix (_AMT, _ID)"),
        (r'^[A-Z][A-Z0-9_]{2,}$', "ALL_CAPS or database style"),
        (r'^[A-Za-z0-9]{1,2}$', "Very short name (1-2 chars)"),
        (r'\d{2,}$', "Ends with numbers (Table01)"),
    ]

    # 2. Natural Language (NLP) Patterns - Critical for Copilot
    NLP_PATTERNS = [
        (r'^[a-z]', "Starts with lowercase"),
        (r'[a-z][A-Z]', "CamelCase detected (use spaces)"),
        (r'_', "Snake_case / underscores detected"),
        (r'[#@$%^&*()\-+=]', "Special characters/symbols"),
        (r'\b(YoY|MoM|QoQ|YTD|MTD|LY|PY|CY)\b', "Vague acronym (use 'Year Over Year')"),
    ]

    def is_unfriendly(name):
        # Check Technical Patterns
        for pattern, reason in TECH_PATTERNS:
            if re.search(pattern, str(name)):
                return True, reason
        # Check NLP Patterns
        for pattern, reason in NLP_PATTERNS:
            if re.search(pattern, str(name)):
                return True, reason
        return False, None

    def is_redundant(table_name, field_name):
        """Flags if 'Customer' table contains 'Customer Name'."""
        t = str(table_name).lower()
        f = str(field_name).lower()
        if f.startswith(t) and f != t:
            return True, f"Redundant table prefix in '{field_name}'"
        return False, None

    # -------------------------------------------------------------------------
    # EXECUTION: AUDIT LOGIC
    # -------------------------------------------------------------------------

    print("=" * 70)
    print("CHECK 1.5: COPILOT-READY BUSINESS NAMING & SEMANTICS")
    print("=" * 70)

    issues = []
    score = 15
    flagged = {'tables': [], 'columns': [], 'measures': [], 'descriptions': []}

    # Audit Tables
    for _, row in visible_tables.iterrows():
        name = row['Name']
        ok, reason = is_unfriendly(name)
        if ok:
            flagged['tables'].append({'Name': name, 'Reason': reason})

    # Audit Columns
    for _, row in visible_columns.iterrows():
        t_name = row.get('Table Name', '')
        c_name = row.get('Column Name', '')
        desc = row.get('Description', '')

        # 1. Check Unfriendly Naming
        ok, reason = is_unfriendly(c_name)

        # 2. Check Redundancy
        if not ok:
            ok, reason = is_redundant(t_name, c_name)

        if ok:
            flagged['columns'].append({'Table': t_name, 'Column': c_name, 'Reason': reason})

        # 3. Check for Missing Descriptions (Copilot Hints)
        if pd.isna(desc) or str(desc).strip() == "":
            flagged['descriptions'].append({'Table': t_name, 'Object': c_name, 'Type': 'Column'})

    # Audit Measures
    for _, row in visible_measures.iterrows():
        t_name = row.get('Table Name', '')
        m_name = row.get('Measure Name', '')
        desc = row.get('Description', '')

        ok, reason = is_unfriendly(m_name)
        if ok:
            flagged['measures'].append({'Table': t_name, 'Measure': m_name, 'Reason': reason})

        if pd.isna(desc) or str(desc).strip() == "":
            flagged['descriptions'].append({'Table': t_name, 'Object': m_name, 'Type': 'Measure'})

    # -------------------------------------------------------------------------
    # REPORTING & SCORING
    # -------------------------------------------------------------------------

    total_flagged = len(flagged['tables']) + len(flagged['columns']) + len(flagged['measures'])
    total_objects = len(visible_tables) + len(visible_columns) + len(visible_measures)

    if total_flagged == 0:
        print("✅ PASSED: All visible names are Copilot-friendly.")
    else:
        # Penalize score based on percentage of bad names
        penalty = (total_flagged / max(total_objects, 1)) * 20 
        score = max(0, int(15 - penalty))

        if flagged['tables']:
            print(f"⚠️  {len(flagged['tables'])} table(s) with technical/NLP issues:")
            for item in flagged['tables'][:5]:
                print(f"     • {item['Name']}  →  {item['Reason']}")

        if flagged['columns']:
            print(f"\n⚠️  {len(flagged['columns'])} column(s) with technical/NLP issues:")
            for item in flagged['columns'][:10]:
                print(f"     • {item['Table']}[{item['Column']}]  →  {item['Reason']}")

        if flagged['measures']:
            print(f"\n⚠️  {len(flagged['measures'])} measure(s) with technical/NLP issues:")
            for item in flagged['measures'][:10]:
                print(f"     • {item['Table']}[{item['Measure']}]  →  {item['Reason']}")

        if flagged['descriptions']:
            print(f"\nℹ️  {len(flagged['descriptions'])} objects missing Descriptions (Copilot context):")
            print(f"     • Example: {flagged['descriptions'][0]['Table']}[{flagged['descriptions'][0]['Object']}]")

        print("\n💡 COPILOT TIP: Use spaces instead of underscores or CamelCase.")
        print("   Instead of 'Total_Sales' or 'TotalSales', use 'Total Sales'.")
        issues.append(("warning", "1.5 Naming", f"{total_flagged} names need NLP optimization"))

    print(f"\n📊 Copilot Readiness Score: {score}/15")
    check_scores['SM-005'] = {'achieved': score, 'max': 15, 'weight': 3}
    all_issues.extend(issues)


run_semantic_check('SM-005', _sm_check)


### 1.6 📝 Descriptions Coverage · `SM-006`

**Why it matters:** Descriptions are CRITICAL for Data Agent accuracy. Every object in your AI Data Schema MUST have a clear, concise description.

**What's checked:** Description coverage for visible tables, columns, and measures

In [ ]:
def _sm_check():
    # =========================================================================
    # CHECK 1.6: DESCRIPTIONS COVERAGE (ORIGINAL CODE)
    # =========================================================================
    print("=" * 70)
    print("CHECK 1.6: DESCRIPTIONS COVERAGE")
    print("=" * 70)

    issues = []
    score = 20  # Highest weight — most critical

    tables_no_desc   = visible_tables[missing_desc_mask(visible_tables)]
    cols_no_desc     = visible_columns[missing_desc_mask(visible_columns)]
    measures_no_desc = visible_measures[missing_desc_mask(visible_measures)]

    tbl_cov = 1 - len(tables_no_desc)   / max(len(visible_tables),   1)
    col_cov = 1 - len(cols_no_desc)     / max(len(visible_columns),  1)
    msr_cov = 1 - len(measures_no_desc) / max(len(visible_measures), 1)

    print("Description Coverage:\n")
    print(f"  Tables   : {len(visible_tables) - len(tables_no_desc):>4}/{len(visible_tables):<4} ({tbl_cov*100:>5.1f}%)")
    print(f"  Columns  : {len(visible_columns) - len(cols_no_desc):>4}/{len(visible_columns):<4} ({col_cov*100:>5.1f}%)")
    print(f"  Measures : {len(visible_measures) - len(measures_no_desc):>4}/{len(visible_measures):<4} ({msr_cov*100:>5.1f}%)")
    print()

    total_missing = len(tables_no_desc) + len(cols_no_desc) + len(measures_no_desc)

    if total_missing == 0:
        print("✅ PASSED: All visible objects have descriptions.")
    else:
        overall_cov = 1 - total_missing / max(len(visible_tables) + len(visible_columns) + len(visible_measures), 1)
        score = max(0, int(20 * overall_cov))

        if not tables_no_desc.empty:
            print(f"❌ {len(tables_no_desc)} table(s) missing descriptions:")
            for name in tables_no_desc['Name'].tolist()[:8]:
                print(f"     • {name}")
            if len(tables_no_desc) > 8:
                print(f"     ... and {len(tables_no_desc) - 8} more")

        if not cols_no_desc.empty:
            print(f"\n⚠️  {len(cols_no_desc)} column(s) missing descriptions (first 15):")
            for _, row in cols_no_desc.head(15).iterrows():
                print(f"     • {row.get('Table Name', '?')}[{row.get('Column Name', '?')}]")
            if len(cols_no_desc) > 15:
                print(f"     ... and {len(cols_no_desc) - 15} more")

        if not measures_no_desc.empty:
            print(f"\n⚠️  {len(measures_no_desc)} measure(s) missing descriptions (first 15):")
            for _, row in measures_no_desc.head(15).iterrows():
                print(f"     • {row.get('Table Name', '?')}[{row.get('Measure Name', '?')}]")
            if len(measures_no_desc) > 15:
                print(f"     ... and {len(measures_no_desc) - 15} more")

        print("\n🚨 CRITICAL: Add descriptions to ALL objects in your AI Data Schema before deployment.")
        issues.append(("critical", "1.6 Coverage", f"{total_missing} visible objects missing descriptions"))

    print(f"\n📊 Coverage Score: {score}/20")
    check_scores['SM-006'] = {'achieved': score, 'max': 20, 'weight': 3}


    # =========================================================================
    # CHECK 1.6.1: DESCRIPTIONS QUALITY (INFORMATIONAL AUDIT)
    # =========================================================================
    print("\n" + "=" * 70)
    print("CHECK 1.6.1: DESCRIPTIONS QUALITY AUDIT")
    print("=" * 70)

    quality_issues = []

    # First check: Are there ANY descriptions at all?
    total_with_descriptions = (len(visible_tables) - len(tables_no_desc)) + \
                              (len(visible_columns) - len(cols_no_desc)) + \
                              (len(visible_measures) - len(measures_no_desc))

    if total_with_descriptions == 0:
        print("❌ FAILED: No descriptions found in the model.")
        print("\n   🚨 CRITICAL: All visible objects must have descriptions for Data Agent to function properly.")
        print("   ACTION: Add descriptions to tables, columns, and measures before proceeding with quality checks.")
    else:
        # Filter for objects that ALREADY have descriptions to audit their content
        tables_with_desc   = visible_tables[~missing_desc_mask(visible_tables)]
        cols_with_desc     = visible_columns[~missing_desc_mask(visible_columns)]
        measures_with_desc = visible_measures[~missing_desc_mask(visible_measures)]

        print(f"✅ Found {total_with_descriptions} descriptions to audit for quality.\n")

        def check_quality(name, desc):
            n = str(name).lower().strip()
            d = str(desc).lower().strip()

            if d == n: 
                return "Redundant: Matches name exactly"
            if len(d) < 15: 
                return "Too brief: Less than 15 characters"
            if any(p in d for p in ['tbd', 'todo', 'placeholder', 'n/a', 'test']): 
                return "Placeholder text detected"
            if d.startswith("this column") or d.startswith("this measure"): 
                return "Circular: Starts with 'This column/measure...'"
            return None

        # Audit Tables Quality
        table_quality_issues = []
        for _, row in tables_with_desc.iterrows():
            issue = check_quality(row['Name'], row['Description'])
            if issue: 
                table_quality_issues.append(f"Table: {row['Name']} → {issue}")
                quality_issues.append(f"Table: {row['Name']} → {issue}")

        # Audit Columns Quality
        column_quality_issues = []
        for _, row in cols_with_desc.iterrows():
            issue = check_quality(row['Column Name'], row['Description'])
            if issue: 
                column_quality_issues.append(f"Column: {row['Table Name']}[{row['Column Name']}] → {issue}")
                quality_issues.append(f"Column: {row['Table Name']}[{row['Column Name']}] → {issue}")

        # Audit Measures Quality
        measure_quality_issues = []
        for _, row in measures_with_desc.iterrows():
            issue = check_quality(row['Measure Name'], row['Description'])
            if issue: 
                measure_quality_issues.append(f"Measure: {row['Table Name']}[{row['Measure Name']}] → {issue}")
                quality_issues.append(f"Measure: {row['Table Name']}[{row['Measure Name']}] → {issue}")

        # Report quality audit results
        if not quality_issues:
            print("✅ PASSED: All descriptions meet AI quality standards.")
            print(f"\n   • Tables: {len(tables_with_desc)} checked ✓")
            print(f"   • Columns: {len(cols_with_desc)} checked ✓")
            print(f"   • Measures: {len(measures_with_desc)} checked ✓")
        else:
            print(f"ℹ️  QUALITY AUDIT RESULTS: {len(quality_issues)} descriptions could be improved:\n")

            if table_quality_issues:
                print(f"   📊 Tables ({len(table_quality_issues)} issues):")
                for item in table_quality_issues[:5]:
                    print(f"      • {item}")

            if column_quality_issues:
                print(f"\n   📋 Columns ({len(column_quality_issues)} issues):")
                for item in column_quality_issues[:5]:
                    print(f"      • {item}")

            if measure_quality_issues:
                print(f"\n   📐 Measures ({len(measure_quality_issues)} issues):")
                for item in measure_quality_issues[:5]:
                    print(f"      • {item}")

            remaining = max(0, len(quality_issues) - 15)
            if remaining > 0:
                print(f"\n   ... and {remaining} more issues")

            print("\n💡 AI TIP: Use descriptions to provide business context that the object name doesn't explain.")
            print("   Avoid: 'Total Sales' → 'This measure calculates total sales'")
            print("   Better: 'Total Sales' → 'Sum of all invoiced sales amounts in USD, excluding returns and discounts'")

    # Final merge of issues found in 1.6 (1.6.1 is informational, so it doesn't append to all_issues)
    all_issues.extend(issues)


run_semantic_check('SM-006', _sm_check)


### 1.7 🔤 Synonyms / Linguistic Schema · `SM-007`

**Why it matters:** Synonyms help Data Agent match natural language variations (revenue/income/sales → Total Revenue).

**What's checked:** Synonym configuration on visible objects

In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 1.7: SYNONYMS / LINGUISTIC SCHEMA")
    print("=" * 70)

    issues = []
    score = 5

    try:
        import sempy_labs as labs

        # Use Semantic Link Labs to list all synonyms
        df_synonyms = labs.list_synonyms(dataset=dataset, workspace=workspace)

        if df_synonyms is not None and not df_synonyms.empty:
            # Count synonyms by object type
            syn_by_type = df_synonyms.groupby('Object Type').size()

            print("✅ Synonyms configured:\n")
            for obj_type, count in syn_by_type.items():
                print(f"   {obj_type:<10} : {count:>4} object(s)")

            print(f"\n   Total: {len(df_synonyms)} synonym(s) across "
                  f"{len(df_synonyms['Object Name'].unique())} object(s)")

            # ----------------------------------------------------------------
            # OVER-SYNONYMIZATION DETECTION
            # Flags identical synonym terms mapped to multiple distinct objects.
            # Per best practice: don't map the same synonym (e.g., 'Sales') to
            # both 'Order Count' and 'Total Revenue' — this confuses the AI.
            # ----------------------------------------------------------------
            syn_col = next((c for c in ['Synonym', 'Synonyms', 'Term'] if c in df_synonyms.columns), None)
            if syn_col:
                exploded = df_synonyms.copy()
                exploded[syn_col] = exploded[syn_col].astype(str).str.split(r'[,;|]')
                exploded = exploded.explode(syn_col)
                exploded[syn_col] = exploded[syn_col].str.strip().str.lower()
                exploded = exploded[exploded[syn_col] != '']

                dup_terms = (
                    exploded.groupby(syn_col)['Object Name']
                    .nunique()
                    .reset_index(name='Object Count')
                )
                dup_terms = dup_terms[dup_terms['Object Count'] > 1]

                if not dup_terms.empty:
                    print(f"\n⚠️  OVER-SYNONYMIZATION: {len(dup_terms)} term(s) mapped to multiple objects:")
                    for _, r in dup_terms.head(8).iterrows():
                        targets = exploded[exploded[syn_col] == r[syn_col]]['Object Name'].unique()
                        print(f"     • '{r[syn_col]}'  →  {', '.join(targets[:5])}")
                    print("\n   ❌ Don't over-synonymize: a single term mapped to multiple objects")
                    print("      (e.g., 'Sales' → Order Count AND Total Revenue) confuses the AI.")
                    issues.append(("important", "1.7 Synonyms",
                                   f"{len(dup_terms)} synonym term(s) mapped to multiple objects"))
                    score = max(2, score - 2)

            # Auto-generated synonym warning
            if 'State' in df_synonyms.columns:
                auto_gen = df_synonyms[
                    df_synonyms['State'].astype(str).str.contains('Auto|Generated', case=False, na=False)
                ]
                if not auto_gen.empty:
                    print(f"\nℹ️  {len(auto_gen)} auto-generated synonym(s) detected.")
                    print("   Manually review and approve — auto-generated synonyms can be wrong")
                    print("   (e.g., Copilot may guess 'Client' = 'Customer'; verify explicitly).")

            # Check coverage of visible measures
            if not measures_df.empty:
                measures_with_synonyms = df_synonyms[
                    df_synonyms['Object Type'] == 'Measure'
                ]['Object Name'].unique()
                vis_msr_names = visible_measures['Measure Name'].tolist()
                msrs_without = [m for m in vis_msr_names if m not in measures_with_synonyms]

                if msrs_without:
                    pct_covered = 1 - len(msrs_without) / max(len(vis_msr_names), 1)
                    score = max(2, int(5 * pct_covered))
                    print(f"\n⚠️  {len(msrs_without)} visible measure(s) without synonyms (showing first 10):")
                    for m in msrs_without[:10]:
                        print(f"     • {m}")
                    if len(msrs_without) > 10:
                        print(f"     ... and {len(msrs_without) - 10} more")
                    issues.append(("recommended", "1.7 Synonyms",
                                   f"{len(msrs_without)} measures without synonyms"))
                else:
                    print("\n✅ All visible measures have synonyms!")

            print("\n📊 Synonym details:")
            display(df_synonyms.head(20))
        else:
            print("⚠️  No synonyms configured on any object.")
            print("\n   ACTION: Define synonyms for key measures and dimensions.")
            print("   PREFERRED METHOD: Use 'AI Instructions' (Prep for AI) in natural language,")
            print("                     e.g., \"VFR means Volume of Fruit Revenue\".")
            score = 1
            issues.append(("recommended", "1.7 Synonyms", "No synonyms configured"))

        print("\n💡 SYNONYM BEST PRACTICES:")
        print("   ✅ DO use AI Instructions to define synonyms in natural language")
        print("       (modern, version-controlled, optimized for the LLM)")
        print("   ✅ DO manually approve synonyms — auto-generated ones can be wrong")
        print("   ❌ DON'T use the legacy Q&A Setup dialog for new definitions")
        print("   ❌ DON'T map the same synonym to multiple distinct objects")
        print("       (e.g., 'Sales' → Order Count AND Total Revenue → AI confusion)")
        print("   ❌ DON'T expose cryptic acronyms (e.g., 'VFR') without defining them")
        print("       in AI Instructions — the LLM has no way to know your jargon.")
        print("\n   Synonym examples (use sparingly, one canonical target each):")
        print("   • Total Revenue → income, turnover, proceeds")
        print("   • Customer → client, account, buyer")

        print("\n📚 Learn more:")
        print("   https://learn.microsoft.com/power-bi/create-reports/copilot-semantic-models#linguistic-schema")
        print("   https://learn.microsoft.com/power-bi/natural-language/q-and-a-best-practices")

    except Exception as e:
        print(f"ℹ️  Could not retrieve synonyms: {e}")
        print("\n   Manual check: Power BI Desktop → select object → Properties pane → Synonyms")
        print("   PREFERRED: Define synonyms in 'AI Instructions' (Prep for AI) instead of legacy Q&A Setup.")
        score = 3
        check_status['SM-007'] = (Status.NOT_EVALUATED, f"Synonyms could not be inspected via TOM: {e}")

    print("\n🔔 PREREQUISITES (per Microsoft Learn):")
    print("   • Q&A must be enabled to add a linguistic schema:")
    print("     Power BI Desktop → File → Options → Current file → Data Load →")
    print("     'Turn on Q&A to ask natural language questions about your data'")
    print("   • To EXCLUDE a technical / redundant field from Copilot:")
    print("     Q&A Setup → Synonyms → toggle OFF 'Include in Q&A' for that field.")
    print("     This is the canonical way to hide a field from Copilot's linguistic context")
    print("     when you can't simply hide it in the model.")
    print("   • You can use Copilot itself in Q&A Setup to SUGGEST synonyms,")
    print("     then curate them manually — do NOT accept auto-generated synonyms blindly.")

    print(f"\n📊 Score: {score}/5")
    check_scores['SM-007'] = {'achieved': score, 'max': 5, 'weight': 1}
    all_issues.extend(issues)


run_semantic_check('SM-007', _sm_check)


### 1.8 🏷️ Row Labels · `SM-008`

**Why it matters:** Row labels define the primary descriptor for dimension tables, helping Data Agent identify which column to display (e.g., "Customer Name" instead of "Customer ID").

**What's checked:** Row label configuration on dimension tables

In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 1.8: ROW LABELS FOR DIMENSION TABLES")
    print("=" * 70)

    issues = []
    score = 5

    try:
        # Using Semantic Link Labs TOM (Tabular Object Model) access
        # Row labels are exposed through Table.RowLabel TOM property
        # This property is fully represented in TMDL and accessible via XMLA
        from sempy_labs.tom import connect_semantic_model

        tables_with_row_labels = []
        tables_without = []
        all_user_tables = []

        with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
            for table in tom.model.Tables:
                # Skip auto-generated date tables and hidden tables
                if table.Name.startswith(('DateTableTemplate_', 'LocalDateTable_')) or table.IsHidden:
                    continue

                all_user_tables.append(table.Name)

                # Column.IsDefaultLabel corresponds to Table.RowLabel TOM property
                # When using PBIP format, this is stored in TMDL and version-controlled
                row_label_column = next((col.Name for col in table.Columns if col.IsDefaultLabel and not col.IsHidden), None)

                if row_label_column:
                    tables_with_row_labels.append((table.Name, row_label_column))
                else:
                    tables_without.append(table.Name)

        print(f"📊 Total user tables analyzed: {len(all_user_tables)}")
        print()

        if tables_with_row_labels:
            print(f"✅ {len(tables_with_row_labels)} table(s) have row labels configured:")
            for tbl, col in tables_with_row_labels[:8]:
                print(f"     • {tbl} → [{col}]")
            if len(tables_with_row_labels) > 8:
                print(f"     ... and {len(tables_with_row_labels) - 8} more")
            print()

        if tables_without:
            print(f"⚠️  {len(tables_without)} table(s) do NOT have row labels set:")
            for tbl in tables_without[:10]:
                print(f"     • {tbl}")
            if len(tables_without) > 10:
                print(f"     ... and {len(tables_without) - 10} more")

            print("\n   📍 WHY THIS MATTERS:")
            print("      Row labels tell Copilot/Data Agent which column to display for each table.")
            print("      Without row labels, the AI may choose the wrong column (e.g., ID instead of Name).")
            print("\n   🎯 RECOMMENDATION:")
            print("      Set row labels on DIMENSION tables (Customer, Product, Store, Employee, etc.)")
            print("      For fact tables (Sales, Orders, Transactions), row labels are typically not needed.")
            print("\n   ⚙️  HOW TO SET:")
            print("      Power BI Desktop → Model view → select table → Advanced properties:")
            print("         1. Row label: Choose the column that describes each row (e.g., 'Customer Name')")
            print("         2. Key column: Choose the column that uniquely identifies each row (e.g., 'CustomerID')")
            print("\n   💡 TIP: Both Row Label AND Key Column should be set on dimension tables for best results.")
            print("\n   💾 Storage: Row labels are stored in TMDL (when using PBIP format)")
            print("      and are version-controlled via Git. Works for Import, DirectQuery, and Direct Lake.")

            issues.append(("important", "1.8 Row Labels", f"{len(tables_without)} tables without row labels"))
            score = max(1, 5 - min(len(tables_without) // 2, 4))
        else:
            print("✅ PASSED: All tables have row labels configured.")

        print("\n📚 Learn more:")
        print("   https://learn.microsoft.com/power-bi/natural-language/q-and-a-tooling-intro#set-a-row-label")
        print("   https://blog.crossjoin.co.uk/2024/08/25/tuning-power-bi-copilot-with-row-labels-and-key-columns/")

    except Exception as e:
        print(f"ℹ️  Could not inspect row labels: {e}")
        print("\n   Manual check: Power BI Desktop → Model view → select table → Advanced → Row label")
        score = 3
        check_status['SM-008'] = (Status.NOT_EVALUATED, f"Row labels could not be inspected via TOM: {e}")

    print(f"\n📊 Score: {score}/5")
    check_scores['SM-008'] = {'achieved': score, 'max': 5, 'weight': 2}
    all_issues.extend(issues)


run_semantic_check('SM-008', _sm_check)


### 1.9 🔢 Explicit Measures (No Implicit Aggregations) · `SM-009`

**Why it matters:** Numeric columns with default summarization create implicit measures that produce unpredictable results.

**What's checked:** Numeric columns with summarization ≠ "None"

In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 1.9: EXPLICIT MEASURES (NO IMPLICIT AGGREGATIONS)")
    print("=" * 70)

    issues = []
    score = 15

    NUMERIC_TYPES = ['Int64', 'Double', 'Decimal', 'Currency', 'Single',
                     'Whole number', 'Decimal number', 'Fixed decimal number']
    SAFE_SUMMARIZE = {'None', 'DoNotSummarize', 'none', 'Do Not Summarize', 'donotsummarize'}

    if 'Summarize By' not in columns_df.columns:
        print("ℹ️  'Summarize By' column not found — skipping check.")
        print("\n   Manual check: Power BI Desktop → Column tools → Summarization")
        score = 10
    else:
        num_cols = visible_columns[visible_columns['Data Type'].isin(NUMERIC_TYPES)]
        implicit = num_cols[
            ~num_cols['Summarize By'].astype(str).isin(SAFE_SUMMARIZE) &
            num_cols['Summarize By'].notna()
        ]

        if implicit.empty:
            print("✅ PASSED: All numeric columns have Summarize By = None.")
        else:
            pct = len(implicit) / max(len(num_cols), 1) * 100
            score = max(0, int(15 - pct / 5))

            print(f"❌ {len(implicit)} numeric column(s) with implicit summarization:\n")
            for tbl, grp in implicit.groupby('Table Name'):
                print(f"   {tbl}:")
                for _, row in grp.iterrows():
                    print(f"      • {row.get('Column Name')} (Type: {row.get('Data Type')}, Summarize: {row.get('Summarize By')})")

            print("\n🚨 CRITICAL: Set all numeric columns to 'Don't summarize'.")
            print("   Then create explicit DAX measures for calculations.")
            print("\n   📍 Power BI Desktop → Column tools → Summarization → Don't summarize")
            issues.append(("critical", "1.9 Implicit Measures", f"{len(implicit)} columns with implicit summarization"))

    print(f"\n📊 Score: {score}/15")
    check_scores['SM-009'] = {'achieved': score, 'max': 15, 'weight': 3}
    all_issues.extend(issues)


run_semantic_check('SM-009', _sm_check)


### 1.10 📊 Report-Scoped Measures · `SM-010`

**Why it matters:** Report-scoped measures are NOT accessible to Data Agent. Migrate them to the semantic model to make them available.

**What's checked:** Detection via TOM (if available)

In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 1.10: REPORT-SCOPED MEASURES")
    print("=" * 70)

    print("ℹ️  This check requires access to the .pbix file or TOM model.\n")
    print("🔍 Manual verification:")
    print("   1. Open your reports in Power BI Desktop")
    print("   2. Check Data pane for measures with a 📊 icon (report-scoped)")
    print("   3. Move these to the semantic model if Data Agent should use them")
    print("\n💡 TIP: Report-scoped measures are NOT visible to Data Agent.")
    print("   Create them in the model instead.")

    sm_manual('SM-010', "Check reports for report-scoped measures (📊 icon) and move them into the semantic model.")


run_semantic_check('SM-010', _sm_check)


### 1.11 📅 Ambiguous Date Fields · `SM-011`

**Why it matters:** Multiple date columns without clear guidance confuse the AI when users ask time-based questions.

**What's checked:**
- Count of visible date columns
- Whether date columns have descriptions
- Risk level based on number of date columns

**Remediation:** Use AI Instructions to specify default date field, hide non-primary date columns, or add clear descriptions.

In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 1.11: AMBIGUOUS DATE FIELDS & MARKED DATE TABLE")
    print("=" * 70)

    issues = []
    score = 5

    DATE_TYPES = ['DateTime', 'Date', 'DateTimeOffset']

    date_cols = visible_columns[
        visible_columns['Data Type'].isin(DATE_TYPES) |
        visible_columns['Data Type'].astype(str).str.lower().str.contains('date', na=False)
    ]

    def desc_status(row):
        d = row.get('Description', '')
        return "has description" if d and str(d).strip() else "NO description"

    # ------------------------------------------------------------------
    # MARKED DATE TABLE CHECK (per MS Learn copilot-semantic-models)
    # Without a marked date table, Copilot may filter time questions by
    # the wrong column (e.g., Customer[Birthday]) instead of the real
    # date table. This produced a documented incorrect-result example.
    # ------------------------------------------------------------------
    marked_date_tables = []
    try:
        from sempy_labs.tom import connect_semantic_model
        with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
            for table in tom.model.Tables:
                if table.Name.startswith(('DateTableTemplate_', 'LocalDateTable_')):
                    continue
                # DataCategory == 'Time' on a date column → table is marked as date table
                for col in table.Columns:
                    try:
                        if str(getattr(col, 'DataCategory', '') or '').lower() == 'time':
                            marked_date_tables.append((table.Name, col.Name))
                            break
                    except Exception:
                        continue
        if marked_date_tables:
            print("✅ Marked date table detected:")
            for tbl, col in marked_date_tables:
                print(f"     • {tbl}[{col}]  (DataCategory = Time)")
        else:
            print("⚠️  NO marked date table detected.")
            print("   Copilot may filter time-based questions by the WRONG date column")
            print("   (e.g., Customer[Birthday] instead of your fact date).")
            print("   ACTION: In Power BI Desktop → Table tools → 'Mark as date table'")
            print("           Choose your dimension date column. (Sets DataCategory='Time')")
            issues.append(("important", "1.11 Date Fields", "No marked date table — Copilot may filter the wrong date column"))
            score = max(1, score - 2)
    except Exception as e:
        print(f"ℹ️  Could not inspect marked date table via TOM: {e}")
        print("   Manual check: Power BI Desktop → Table tools → 'Mark as date table'")

    print()

    # ------------------------------------------------------------------
    # AMBIGUOUS DATE FIELDS CHECK
    # ------------------------------------------------------------------
    if date_cols.empty:
        print("✅ OK: No visible date columns found.")
        print("   Date columns may be hidden (role-playing dimension design).")
    elif len(date_cols) == 1:
        r = date_cols.iloc[0]
        print(f"✅ PASSED: Single visible date column:")
        print(f"   {r.get('Table Name')}[{r.get('Column Name')}] — {desc_status(r)}")
    elif len(date_cols) <= 3:
        print(f"ℹ️  INFO: {len(date_cols)} date columns found (low ambiguity risk):")
        for _, row in date_cols.iterrows():
            print(f"   • {row.get('Table Name')}[{row.get('Column Name')}] — {desc_status(row)}")
        print("\n💡 Consider adding AI Instructions to specify default date for common questions.")
        score = min(score, 4)
    else:
        print(f"⚠️  WARNING: {len(date_cols)} visible date column(s) — HIGH ambiguity risk!\n")
        for tbl, grp in date_cols.groupby('Table Name'):
            print(f"   Table: {tbl}")
            for _, row in grp.iterrows():
                print(f"      • {row.get('Column Name')} ({desc_status(row)})")

        score = max(0, 5 - max(0, len(date_cols) - 3))
        issues.append(("important", "1.11 Date Fields", f"{len(date_cols)} visible date columns without clear AI guidance"))

        print("\n📍 ACTION:")
        print("   1. Hide date columns users should not query directly")
        print("      (especially Customer[Birthday], Employee[HireDate], etc. —")
        print("       per MS Learn, these are common Copilot filter mistakes)")
        print("   2. Add descriptions clarifying each date column's purpose")
        print("   3. Mark your primary date table (Table tools → 'Mark as date table')")
        print("   4. Prep for AI → AI Instructions:")
        print("      Example: 'When the user asks about dates, use [Order Date] by default'")
        print("   5. Create Verified Answers for most common time-based questions")

    print("\n📚 Reference:")
    print("   https://learn.microsoft.com/power-bi/create-reports/copilot-semantic-models")

    print(f"\n📊 Score: {score}/5")
    check_scores['SM-011'] = {'achieved': score, 'max': 5, 'weight': 1}
    all_issues.extend(issues)


run_semantic_check('SM-011', _sm_check)


### 1.12 👁️ Hidden Objects & Private Tables Risk (PK / FK / Sort / GUIDs) · `SM-012`

**Why it matters (per [MS Learn — Copilot grounding](https://learn.microsoft.com/power-bi/create-reports/copilot-semantic-models)):**

Copilot **excludes** the following from its grounding context:
- Hidden columns and measures
- Hidden report pages
- **Tables marked as Private**

Copilot **includes** as grounding context:
- Visible tables/columns/measures
- Descriptions, data types, format strings, data category
- Full linguistic schema (synonyms / verbs)

**Implications:**
- Verified Answers break silently if they reference hidden columns
- **Technical columns** (primary keys, foreign keys, sort columns, GUIDs) MUST be hidden — Copilot never needs to see a relationship key or GUID to answer a business question
- For **internal-only tables** that should be entirely invisible to Copilot, mark them as **Private** (Power BI Desktop → Model view → Table → Properties → Private)

**What's checked:**
- Count of hidden objects
- Hidden columns lacking descriptions
- **Visible technical columns** (key/GUID/sort patterns) that should be hidden
- Verified Answers risk assessment

**Remediation:** Hide all primary keys, foreign keys, sort columns, and GUIDs. Mark internal-only tables as Private. Add descriptions to all remaining hidden columns referenced by Verified Answers.


In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 1.12: HIDDEN OBJECTS, PRIVATE TABLES & TECH COLUMNS")
    print("=" * 70)

    issues = []
    score = 5

    hidden_tables   = tables_df[is_hidden_mask(tables_df)]
    hidden_columns  = columns_df[is_hidden_mask(columns_df)]
    hidden_measures = measures_df[is_hidden_mask(measures_df)]

    print(f"   Hidden objects summary:")
    print(f"      Tables   : {len(hidden_tables):>4}")
    print(f"      Columns  : {len(hidden_columns):>4}")
    print(f"      Measures : {len(hidden_measures):>4}")
    print()

    # ------------------------------------------------------------------
    # DETECT TABLES MARKED AS PRIVATE (excluded from Copilot grounding)
    # Per MS Learn: tables marked as Private are excluded from Copilot's
    # grounding context, even if they are not hidden.
    # ------------------------------------------------------------------
    private_tables = []
    try:
        from sempy_labs.tom import connect_semantic_model
        with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
            for table in tom.model.Tables:
                if table.Name.startswith(('DateTableTemplate_', 'LocalDateTable_')):
                    continue
                # ExcludeFromModelRefresh / IsPrivate vary by TOM version; check both
                is_private = False
                for attr in ('IsPrivate', 'ExcludeFromAutomaticAggregations'):
                    try:
                        if bool(getattr(table, attr, False)):
                            is_private = True
                            break
                    except Exception:
                        continue
                if is_private:
                    private_tables.append(table.Name)
        if private_tables:
            print(f"ℹ️  {len(private_tables)} table(s) marked as Private (hidden from Copilot grounding):")
            for t in private_tables[:8]:
                print(f"      • {t}")
            print("   ✅ Good practice for internal-only tables you don't want Copilot to see.")
            print()
    except Exception:
        pass  # TOM not available or attribute not exposed — silent skip

    # ------------------------------------------------------------------
    # DETECT VISIBLE TECHNICAL COLUMNS THAT SHOULD BE HIDDEN
    # Per best practice: Copilot never needs to see a GUID or relationship
    # key to answer a business question. Hide PKs, FKs, sort columns, GUIDs.
    # ------------------------------------------------------------------
    TECH_COL_PATTERN = (
        r'(^id$|_id$|^key$|_key$|^pk_|_pk$|^fk_|_fk$|guid|^sk_|_sk$|'
        r'sortorder|sort_order|sort_key|_sortby$|_index$|rowversion|etag|hashkey)'
    )
    visible_tech_cols = visible_columns[
        visible_columns['Column Name'].astype(str).str.lower().str.contains(TECH_COL_PATTERN, regex=True, na=False)
    ]

    if not visible_tech_cols.empty:
        print(f"⚠️  {len(visible_tech_cols)} visible column(s) appear to be technical (PK/FK/sort/GUID):")
        for _, row in visible_tech_cols.head(15).iterrows():
            print(f"     • {row.get('Table Name')}[{row.get('Column Name')}]  (Type: {row.get('Data Type')})")
        if len(visible_tech_cols) > 15:
            print(f"     ... and {len(visible_tech_cols) - 15} more")
        print("\n   🚨 ACTION: Hide these columns. Copilot/Data Agent never needs to see")
        print("      a GUID or relationship key to answer a business question.")
        print("      Visible technical columns add noise and may be picked incorrectly")
        print("      during DAX generation.")
        issues.append(("important", "1.12 Hidden Objects",
                       f"{len(visible_tech_cols)} visible technical columns (PK/FK/sort/GUID) should be hidden"))
        score = max(1, score - 2)
    else:
        print("✅ No visible technical (PK/FK/sort/GUID) columns detected.")

    print()

    # Hidden columns without descriptions — double risk
    if not hidden_columns.empty:
        hc_no_desc = hidden_columns[
            hidden_columns['Description'].isna() | (hidden_columns['Description'].astype(str).str.strip() == '')
        ] if 'Description' in hidden_columns.columns else hidden_columns

        pct = len(hc_no_desc) / max(len(hidden_columns), 1) * 100

        if pct > 50:
            print(f"⚠️  WARNING: {len(hc_no_desc)} hidden column(s) lack descriptions ({pct:.0f}% of hidden columns).")
            print("   If any of these are referenced by Verified Answers, the answer will silently fail.")
            score = max(1, score - 2)
            issues.append(("important", "1.12 Hidden Objects", f"{len(hc_no_desc)} hidden columns without descriptions — Verified Answers risk"))
        else:
            print(f"✅ Most hidden columns have descriptions ({100-pct:.0f}% coverage).")
    else:
        print("✅ No hidden columns found.")

    print("\n🚨 REMINDER: Verified Answers will NOT work if they reference hidden columns.")
    print("   Ensure all columns referenced in Verified Answers are VISIBLE in the model.")
    print("\n💡 RULE OF THUMB (per MS Learn — Copilot grounding):")
    print("   • Hide:    PKs, FKs, sort columns, GUIDs, audit timestamps, internal flags")
    print("   • Private: Entire internal-only tables (excluded from Copilot grounding)")
    print("   • Q&A:    Toggle OFF 'Include in Q&A' for fields you want to keep visible")
    print("              in reports but exclude from Copilot/Q&A linguistic context.")
    print("   • Visible: Business-meaningful names users would type in a question")

    print("\n📚 Reference:")
    print("   https://learn.microsoft.com/power-bi/create-reports/copilot-semantic-models")

    print(f"\n📊 Score: {score}/5")
    check_scores['SM-012'] = {'achieved': score, 'max': 5, 'weight': 1}
    all_issues.extend(issues)


run_semantic_check('SM-012', _sm_check)


### 1.13 📦 Model Complexity / Bloat — Copilot Context Truncation Risk · `SM-013`

**Why it matters:** Copilot and Data Agents serialize your semantic model schema (tables, columns, measures, calculated columns) into the LLM prompt as **grounding context**. Large models exceed the prompt budget and the schema gets **truncated before reaching the LLM** — Copilot literally never sees the omitted objects and will hallucinate, miss fields, or pick the wrong table without any visible error.

> 📘 **Official guidance:** [Prepare your data for AI: AI data schemas (Microsoft Learn)](https://learn.microsoft.com/power-bi/create-reports/copilot-prepare-data-ai-data-schema)
>
> *"By using an AI data schema, semantic model authors can define a focused subset of the model's schema for Copilot to prioritize when it generates responses … A streamlined schema reduces ambiguity, which helps Copilot deliver clearer and more accurate responses."*

### What's checked (automated)
- Visible **tables** count (warn > 30, critical > 50)
- Visible **columns** count (warn > 300, critical > 500)
- Visible **measures** count (warn > 100, critical > 150)
- **Calculated columns** count (warn > 50, critical > 100 — recomputed at refresh, add schema noise)
- Total grounding object count (tables + columns + measures)
- Visible helper / intermediate measures (should be hidden)
- **Q&A enabled** prerequisite (Prep data for AI tabs are disabled if Q&A is off)

### Remediation — use *Prep data for AI → Simplify data schema*

Per Microsoft Learn, the official path is:

1. **Power BI Desktop → Home ribbon → Prep data for AI** (or in the service: semantic-model page ribbon)
2. **Turn on Q&A** for the model first — otherwise the Prep-for-AI tabs are disabled
3. Open the **Simplify data schema** tab and select **only** the fields Copilot should reason over — prioritize clean columns with limited ambiguity
4. Click **Apply** to save the AI data schema to the model

### Key nuances from the docs (do not skip)

- 🔸 **Hidden ≠ excluded forever.** Hidden fields are auto-excluded only on *initial* AI-data-schema setup. New columns added to the model later are **not** auto-excluded — re-open Prep for AI after every model change.
- 🔸 **Hierarchies have a gotcha.** If the AI data schema includes a hierarchy but a column of that hierarchy isn't individually selected on the table, the Copilot schema **still includes that column**. Pick fields individually if you need precise control.
- 🔸 **Relationships are always respected** regardless of the AI data schema — Copilot can traverse to related tables even if a related field isn't selected. So the schema scope must still align with the relationship graph.
- 🔸 **AI data schema does NOT apply to every Copilot feature.** Report summarization, "create a report page", "search for data", and DAX-query generation use the **entire** semantic model. Model-level cleanup (hiding tech columns, marking tables Private, removing dead calc columns) is still required.
- 🔸 **Each edit needs a Copilot pane refresh** (close & reopen) to take effect during testing.
- 🔸 **Consumers can't see or disable the AI data schema** — the author owns the trust boundary.

### Additional model-level actions (to keep total schema small)

- Mark internal-only tables as **Private** so Copilot ignores them entirely
- **Hide** technical columns (PKs, FKs, GUIDs, sort columns, audit timestamps)
- Push frequently-used **calculated columns into the source query / dataflow** (ETL)
- Hide **helper / intermediate measures** used only as building blocks
- Split very large models into **domain-specific Data Agents** (Sales, Finance, …) instead of one "everything" agent


In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 1.13: MODEL COMPLEXITY / BLOAT — COPILOT CONTEXT TRUNCATION RISK")
    print("=" * 70)

    issues = []
    score = 5

    # ------------------------------------------------------------------
    # Identify calculated columns (DAX-defined columns added to the model)
    # Different sempy versions expose this via 'Type', 'Column Type', or 'Kind'
    # ------------------------------------------------------------------
    calc_col_total = 0
    calc_col_visible = 0
    calc_col_type_field = next(
        (c for c in ['Type', 'Column Type', 'Kind'] if c in columns_df.columns),
        None
    )
    if calc_col_type_field:
        calc_mask_all = columns_df[calc_col_type_field].astype(str).str.lower().str.contains(
            'calculated', na=False
        )
        calc_col_total = int(calc_mask_all.sum())
        if not visible_columns.empty:
            calc_mask_vis = visible_columns[calc_col_type_field].astype(str).str.lower().str.contains(
                'calculated', na=False
            )
            calc_col_visible = int(calc_mask_vis.sum())
            visible_calc_cols = visible_columns[calc_mask_vis]
        else:
            visible_calc_cols = visible_columns.iloc[0:0]
    else:
        visible_calc_cols = visible_columns.iloc[0:0]

    vis_tbl_n = len(visible_tables)
    vis_col_n = len(visible_columns)
    vis_msr_n = len(visible_measures)
    vis_calc_n = calc_col_visible
    total_grounding_objs = vis_tbl_n + vis_col_n + vis_msr_n  # calc cols already counted in columns

    # ------------------------------------------------------------------
    # Q&A enabled prerequisite — Prep-for-AI tabs are disabled if Q&A is off
    # (per https://learn.microsoft.com/power-bi/create-reports/copilot-prepare-data-ai-data-schema)
    # ------------------------------------------------------------------
    qna_enabled = None
    try:
        from sempy_labs.tom import connect_semantic_model
        with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
            # The model property is typically named 'DiscourageImplicitMeasures' or
            # 'DefaultPowerBIDataSourceVersion'; Q&A enablement lives under
            # model.QueryGroups / model.Model.DataAccessOptions or the
            # 'IsQNAFeatureEnabled' annotation on some versions. We probe defensively.
            for attr in ('IsQNAFeatureEnabled', 'QnaEnabled'):
                if hasattr(tom.model, attr):
                    qna_enabled = bool(getattr(tom.model, attr))
                    break
            if qna_enabled is None:
                # Fall back to annotation lookup
                try:
                    for ann in tom.model.Annotations:
                        if 'qna' in ann.Name.lower() or 'questionsanswers' in ann.Name.lower():
                            qna_enabled = str(ann.Value).lower() in ('true', '1', 'enabled')
                            break
                except Exception:
                    pass
    except Exception:
        pass

    # ------------------------------------------------------------------
    # Display model size summary
    # ------------------------------------------------------------------
    print("   📐 Model size (objects sent to Copilot / LLM as grounding context):")
    print(f"      Visible tables            : {vis_tbl_n:>5}")
    print(f"      Visible columns           : {vis_col_n:>5}")
    print(f"      Visible measures          : {vis_msr_n:>5}")
    if calc_col_type_field:
        print(f"      Calculated columns (vis)  : {vis_calc_n:>5}  (total incl. hidden: {calc_col_total})")
    else:
        print(f"      Calculated columns        :   n/a  (column type metadata not exposed by sempy)")
    print(f"      ─────────────────────────────────")
    print(f"      Total grounding objects   : {total_grounding_objs:>5}")
    print()

    # ------------------------------------------------------------------
    # Thresholds — based on practical Copilot grounding limits
    # (model schema is serialized into the LLM prompt; large models truncate)
    # ------------------------------------------------------------------
    THRESHOLDS = {
        'tables':      {'warn':  30, 'critical':  50},
        'columns':     {'warn': 300, 'critical': 500},
        'measures':    {'warn': 100, 'critical': 150},
        'calc_cols':   {'warn':  50, 'critical': 100},
        'total':       {'warn': 500, 'critical': 800},
    }

    def _bloat_check(label, n, key, unit_hint):
        """Emit warning/critical based on threshold and return score delta."""
        t = THRESHOLDS[key]
        if n > t['critical']:
            print(f"🔴 CRITICAL: {n} {label} — exceeds critical threshold ({t['critical']}).")
            print(f"   Copilot grounding will likely be TRUNCATED before reaching the LLM.")
            print(f"   {unit_hint}")
            issues.append(("critical", "1.13 Complexity",
                           f"{n} {label} exceeds critical threshold ({t['critical']}) — schema truncation risk"))
            return -2
        elif n > t['warn']:
            print(f"🟠 WARNING: {n} {label} — above recommended limit ({t['warn']}).")
            print(f"   Risk of Copilot context truncation; accuracy and latency will degrade.")
            print(f"   {unit_hint}")
            issues.append(("important", "1.13 Complexity",
                           f"{n} {label} exceeds warning threshold ({t['warn']})"))
            return -1
        return 0

    PREP_HINT_TABLE = "→ Mark internal tables as Private; reduce scope via Prep data for AI → Simplify data schema."
    PREP_HINT_COL   = "→ Hide PKs/FKs/GUIDs/sort cols; expose only business columns via Prep data for AI → Simplify data schema."
    PREP_HINT_MSR   = "→ Hide helper/intermediate measures; include only KPI-level measures in the AI data schema."
    PREP_HINT_CALC  = "→ Push calc-column logic into the source query / dataflow (ETL) instead of DAX."
    PREP_HINT_TOTAL = "→ Split into domain-specific Data Agents (Sales / Finance / …) instead of one 'everything' agent."

    score += _bloat_check("visible tables",   vis_tbl_n,  'tables',   PREP_HINT_TABLE)
    score += _bloat_check("visible columns",  vis_col_n,  'columns',  PREP_HINT_COL)
    score += _bloat_check("visible measures", vis_msr_n,  'measures', PREP_HINT_MSR)
    if calc_col_type_field:
        score += _bloat_check("calculated columns", vis_calc_n, 'calc_cols', PREP_HINT_CALC)
    score += _bloat_check("total grounding objects", total_grounding_objs, 'total', PREP_HINT_TOTAL)

    # Show top tables by visible calculated-column count (helps prioritize remediation)
    if calc_col_type_field and vis_calc_n > 0 and 'Table Name' in visible_calc_cols.columns:
        top_calc_tables = (
            visible_calc_cols.groupby('Table Name')['Column Name']
            .count()
            .sort_values(ascending=False)
            .head(5)
        )
        print("\n   Top tables with calculated columns:")
        for tbl, cnt in top_calc_tables.items():
            print(f"      • {tbl}: {cnt} calculated column(s)")

    # ------------------------------------------------------------------
    # Detect helper measures that are visible (they should be hidden)
    # ------------------------------------------------------------------
    HELPER_PATTERN = r'\b(helper|_h$|aux|auxiliary|temp|tmp|working|intermediate|_int$|__[a-z])\b'
    visible_helpers = visible_measures[
        visible_measures['Measure Name'].astype(str).str.lower().str.contains(HELPER_PATTERN, regex=True, na=False)
    ]

    if not visible_helpers.empty:
        print(f"\n⚠️  {len(visible_helpers)} potentially visible helper/intermediate measure(s):")
        for _, row in visible_helpers.head(10).iterrows():
            print(f"      • {row.get('Table Name')}[{row.get('Measure Name')}]")
        if len(visible_helpers) > 10:
            print(f"      ... and {len(visible_helpers) - 10} more")
        print("   → Hide them or exclude them from your AI data schema (Prep data for AI).")
        issues.append(("important", "1.13 Complexity",
                       f"{len(visible_helpers)} visible helper/intermediate measure(s)"))
        score -= 1

    # ------------------------------------------------------------------
    # Q&A enablement prerequisite for Prep-for-AI
    # ------------------------------------------------------------------
    print()
    if qna_enabled is True:
        print("✅ Q&A is enabled — Prep data for AI tabs are accessible.")
    elif qna_enabled is False:
        print("🔴 CRITICAL: Q&A is DISABLED on this model.")
        print("   The Prep data for AI → Simplify data schema tab will be disabled until")
        print("   you enable Q&A (Power BI Desktop → File → Options → Current file → Q&A).")
        issues.append(("critical", "1.13 Complexity",
                       "Q&A disabled — Prep for AI / AI data schema cannot be configured"))
        score -= 2
    else:
        print("ℹ️  Could not determine Q&A enablement programmatically.")
        print("   Verify manually: Q&A must be ON for Prep data for AI tabs to be enabled.")

    if not issues:
        print("\n✅ PASSED: Model complexity is within healthy limits for Copilot grounding.")

    # ------------------------------------------------------------------
    # Always-on guidance about truncation + Prep for AI
    # ------------------------------------------------------------------
    print()
    print("📍 WHY THIS MATTERS FOR COPILOT / DATA AGENT:")
    print("   The semantic model schema is serialized into the LLM prompt as grounding")
    print("   context. Large models exceed the prompt budget and the schema gets")
    print("   TRUNCATED — Copilot literally never sees the omitted tables/columns and")
    print("   will hallucinate, miss fields, or pick the wrong table without any error.")
    print()
    print("✅ HOW TO FIX — Prep data for AI → Simplify data schema:")
    print("   1. Power BI Desktop → Home ribbon → Prep data for AI")
    print("      (or in the service: semantic-model page ribbon)")
    print("   2. Ensure Q&A is ON (otherwise the Prep-for-AI tabs are disabled)")
    print("   3. Open the 'Simplify data schema' tab")
    print("   4. Select ONLY the fields Copilot should reason over — prioritize clean")
    print("      columns with limited ambiguity; remove confusing/duplicate fields")
    print("   5. Click Apply to save")
    print()
    print("⚠️  IMPORTANT NUANCES (per MS Learn):")
    print("   • Hidden fields are auto-excluded only on FIRST setup — re-open Prep for AI")
    print("     after model changes to re-evaluate newly added columns.")
    print("   • Hierarchy gotcha: if a hierarchy is selected but its column isn't picked")
    print("     individually, Copilot's schema STILL includes that column.")
    print("   • Relationships are always respected — Copilot can traverse to related")
    print("     tables even if related fields aren't in the AI data schema.")
    print("   • AI data schema does NOT apply to: report summarization, 'create a report")
    print("     page', 'search for data', or DAX-query generation — those use the FULL")
    print("     model. Model-level cleanup (hiding tech columns, marking Private) is")
    print("     still required.")
    print("   • After editing, close & reopen the Copilot pane to refresh.")
    print()
    print("💡 ADDITIONAL MODEL-LEVEL ACTIONS:")
    print("   • Mark internal-only tables as Private")
    print("   • Hide PKs, FKs, GUIDs, sort columns, audit timestamps")
    print("   • Push calc-column logic into source query / dataflow")
    print("   • Split very large models into domain-specific Data Agents")
    print()
    print("📚 References:")
    print("   • https://learn.microsoft.com/power-bi/create-reports/copilot-prepare-data-ai-data-schema")
    print("   • https://learn.microsoft.com/power-bi/create-reports/copilot-evaluate-data")
    print("   • https://learn.microsoft.com/power-bi/create-reports/copilot-semantic-models")

    score = max(0, score)
    print(f"\n📊 Score: {score}/5")
    check_scores['SM-013'] = {'achieved': score, 'max': 5, 'weight': 1}
    all_issues.extend(issues)


run_semantic_check('SM-013', _sm_check)


### 1.14 ⚡ Performance Analyzer for AI Data Schema Measures · `SM-014`

**Why it matters:** Slow-executing measures in your AI Data Schema cause timeout errors and poor user experience. You must validate performance specifically for the measures selected in Prep for AI.

**What's checked:** Manual — run Performance Analyzer in Power BI Desktop targeting only measures included in AI Data Schema

**Key insights:**
- Measures exceeding 2 seconds warrant optimization before deployment
- Storage Engine vs Formula Engine query ratio
- Direct Lake fallback scenarios that trigger Import mode scans

In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 1.14: PERFORMANCE ANALYZER FOR AI DATA SCHEMA MEASURES")
    print("=" * 70)

    print("🔵 MANUAL CHECK — requires Power BI Desktop.\n")
    print("📍 HOW TO RUN:")
    print("   Power BI Desktop → View tab → Performance Analyzer → Start recording")
    print("   Then create visuals using each measure in your AI Data Schema.\n")
    print("✅ CHECKLIST:")
    print()
    print("   [ ] 1. Open report connected to this semantic model in Power BI Desktop")
    print("   [ ] 2. Enable Performance Analyzer (View → Performance Analyzer)")
    print("   [ ] 3. Click 'Start recording'")
    print("   [ ] 4. For each measure in AI Data Schema, create a simple visual using it")
    print("   [ ] 5. Review DAX query times — target < 2 seconds per measure")
    print("   [ ] 6. Export results: click '...' → 'Export data'")
    print()
    print("⚠️  WARNING THRESHOLDS:")
    print("   • < 500ms  ✅ Excellent — no action needed")
    print("   • 500ms–2s ⚠️  Acceptable — monitor under production load")
    print("   • 2s–5s    🟠 Slow — optimize before deployment")
    print("   • > 5s     🔴 Critical — will cause Data Agent timeouts")
    print()
    print("💡 OPTIMIZATION ACTIONS:")
    print("   • Run BPA (Check 1.2) to identify inefficient DAX patterns")
    print("   • Remove column store scans with SUMMARIZECOLUMNS rewrites")
    print("   • Use aggregation tables for large fact tables")
    print("   • Review Direct Lake fallback if using Direct Lake mode (Check 1.3)")
    print("   • Check if measure uses CALCULATE with complex filter contexts")
    print()
    print("📚 Documentation:")
    print("   https://learn.microsoft.com/power-bi/create-reports/desktop-performance-analyzer")

    sm_manual('SM-014', "Run Performance Analyzer against AI Data Schema measures (target < 2 s per measure).",
              legacy_attested=prep_for_ai_configured)


run_semantic_check('SM-014', _sm_check)


### 1.15 🔁 Duplicate Column Names Across Tables · `SM-015`

**Why it matters:** When two tables expose a column with the same name (e.g., `Sales[Date]` and `Orders[Date]`), the Data Agent's DAX generator can pick the wrong table, leading to incorrect filter context and wrong results without any obvious error.

**What's checked:** Automated — scans all visible columns for names appearing in more than one table

In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 1.15: DUPLICATE COLUMN NAMES ACROSS TABLES")
    print("=" * 70)

    issues = []
    score = 5

    if 'Column Name' in visible_columns.columns and 'Table Name' in visible_columns.columns:
        # Find column names that appear in more than one table
        col_table_counts = (
            visible_columns
            .groupby('Column Name')['Table Name']
            .agg(list)
            .reset_index()
        )
        col_table_counts.columns = ['Column Name', 'Tables']
        col_table_counts['Table Count'] = col_table_counts['Tables'].apply(len)

        duplicates = col_table_counts[col_table_counts['Table Count'] > 1].sort_values('Table Count', ascending=False)

        if duplicates.empty:
            print("✅ PASSED: No duplicate column names found across tables.")
        else:
            print(f"⚠️  {len(duplicates)} column name(s) appear in multiple tables:\n")
            for _, row in duplicates.head(20).iterrows():
                tables_str = ', '.join(row['Tables'][:5])
                if len(row['Tables']) > 5:
                    tables_str += f" ... +{len(row['Tables']) - 5} more"
                print(f"     • [{row['Column Name']}]  →  {row['Table Count']} tables: {tables_str}")

            if len(duplicates) > 20:
                print(f"\n     ... and {len(duplicates) - 20} more duplicate column names")

            print("\n📍 WHY THIS CAUSES ERRORS:")
            print("   When you ask 'Filter by Date', the AI doesn't know which table's")
            print("   [Date] column to use — it may guess wrong, silently returning bad data.")
            print()
            print("✅ REMEDIATION OPTIONS:")
            print("   Option A (Preferred): Rename ambiguous columns to be table-specific")
            print("     Example: Sales[Date] → Sales[Order Date]")
            print("     Example: Orders[Date] → Orders[Delivery Date]")
            print("   Option B: Add clear descriptions to distinguish identical column names")
            print("   Option C: Hide the non-primary instance in the AI Data Schema")
            print("   Option D: Add AI Instructions to specify which table to use by default")
            print()
            print("💡 FOCUS: Prioritize columns included in your AI Data Schema.")

            penalty = min(4, len(duplicates))
            score = max(1, 5 - penalty)
            issues.append(("important", "1.15 Duplicate Columns", f"{len(duplicates)} column name(s) shared across multiple tables"))
    else:
        print("ℹ️  Column metadata not available — skipping check.")

    score = max(0, score)
    print(f"\n📊 Score: {score}/5")
    check_scores['SM-015'] = {'achieved': score, 'max': 5, 'weight': 1}
    all_issues.extend(issues)


run_semantic_check('SM-015', _sm_check)


### 1.16 🧠 Copilot Grounding Data Completeness · `SM-016`

**Why it matters:** Per [MS Learn — Use Copilot with semantic models](https://learn.microsoft.com/power-bi/create-reports/copilot-semantic-models), Copilot uses these model properties as **grounding context**:

- Descriptions (already covered in Check 1.6)
- **Format strings** — currency, %, decimals → tells Copilot how to present numbers
- **Data category** — Geography, Web URL, Image URL, Barcode, Time → tells Copilot which visual / interpretation to pick
- **Calculation group descriptions** — should list calculation item names (e.g., `YTD`, `MTD`, `PY`)
- Synonyms / linguistic schema (covered in Check 1.7)

**What's checked (automated):**
- Visible measures missing format strings
- Geography-like columns without a `Geography` data category set
- URL/Image-like columns without a `Web URL` / `Image URL` data category
- Calculation groups whose descriptions are empty or don't list calculation items


In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 1.16: COPILOT GROUNDING DATA COMPLETENESS")
    print("=" * 70)

    issues = []
    score = 5

    # ------------------------------------------------------------------
    # 1) Format strings on visible measures
    # ------------------------------------------------------------------
    fmt_col = next((c for c in ['Format String', 'FormatString'] if c in measures_df.columns), None)
    if fmt_col:
        msr_no_fmt = visible_measures[
            visible_measures[fmt_col].isna() | (visible_measures[fmt_col].astype(str).str.strip() == '')
        ]
        if not msr_no_fmt.empty:
            print(f"⚠️  {len(msr_no_fmt)} visible measure(s) missing a format string:")
            for _, r in msr_no_fmt.head(10).iterrows():
                print(f"     • {r.get('Table Name')}[{r.get('Measure Name')}]")
            if len(msr_no_fmt) > 10:
                print(f"     ... and {len(msr_no_fmt) - 10} more")
            print("   ACTION: Set currency / % / decimal format strings — Copilot uses them as grounding")
            print("           and will present numbers in the right format.")
            issues.append(("important", "1.16 Grounding",
                           f"{len(msr_no_fmt)} visible measure(s) without format strings"))
            score = max(1, score - 2)
        else:
            print("✅ All visible measures have format strings configured.")
    else:
        print("ℹ️  Format string column not exposed by sempy — skipping format-string check.")

    print()

    # ------------------------------------------------------------------
    # 2) Data category hints for Geography / URL / Image columns
    # ------------------------------------------------------------------
    cat_col = next((c for c in ['Data Category', 'DataCategory'] if c in columns_df.columns), None)

    GEO_PATTERN = r'(country|state|province|region|city|postal|zip|continent|county|address|latitude|longitude)'
    URL_PATTERN = r'(url|link|href|website|webpage)'
    IMG_PATTERN = r'(image|photo|picture|thumbnail|imageurl|imgurl)'

    geo_missing = url_missing = img_missing = pd.DataFrame()
    if cat_col:
        is_text = visible_columns['Data Type'].astype(str).str.contains('string|text', case=False, na=False)
        no_cat = visible_columns[cat_col].isna() | \
                 (visible_columns[cat_col].astype(str).str.strip().isin(['', 'Uncategorized', 'None']))

        geo_missing = visible_columns[
            is_text & no_cat &
            visible_columns['Column Name'].astype(str).str.lower().str.contains(GEO_PATTERN, na=False)
        ]
        url_missing = visible_columns[
            is_text & no_cat &
            visible_columns['Column Name'].astype(str).str.lower().str.contains(URL_PATTERN, na=False)
        ]
        img_missing = visible_columns[
            is_text & no_cat &
            visible_columns['Column Name'].astype(str).str.lower().str.contains(IMG_PATTERN, na=False)
        ]

        if not geo_missing.empty:
            print(f"⚠️  {len(geo_missing)} geography-like column(s) without a Data Category:")
            for _, r in geo_missing.head(8).iterrows():
                print(f"     • {r.get('Table Name')}[{r.get('Column Name')}]")
            print("   ACTION: Set Data Category to Country/Region, State, City, etc.")
            issues.append(("recommended", "1.16 Grounding",
                           f"{len(geo_missing)} geo-like column(s) without Data Category"))

        if not url_missing.empty:
            print(f"\n⚠️  {len(url_missing)} URL-like column(s) without a 'Web URL' Data Category:")
            for _, r in url_missing.head(8).iterrows():
                print(f"     • {r.get('Table Name')}[{r.get('Column Name')}]")
            issues.append(("recommended", "1.16 Grounding",
                           f"{len(url_missing)} URL-like column(s) without Web URL category"))

        if not img_missing.empty:
            print(f"\n⚠️  {len(img_missing)} image-like column(s) without an 'Image URL' Data Category:")
            for _, r in img_missing.head(8).iterrows():
                print(f"     • {r.get('Table Name')}[{r.get('Column Name')}]")
            issues.append(("recommended", "1.16 Grounding",
                           f"{len(img_missing)} image-like column(s) without Image URL category"))

        if geo_missing.empty and url_missing.empty and img_missing.empty:
            print("✅ Geography / URL / Image columns either categorized or absent.")
    else:
        print("ℹ️  Data Category column not exposed by sempy — skipping category check.")

    print()

    # ------------------------------------------------------------------
    # 3) Calculation group descriptions should list calculation item names
    # ------------------------------------------------------------------
    try:
        from sempy_labs.tom import connect_semantic_model
        cg_issues = []
        cg_total = 0
        with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
            for table in tom.model.Tables:
                try:
                    cg = getattr(table, 'CalculationGroup', None)
                except Exception:
                    cg = None
                if cg is None:
                    continue
                cg_total += 1
                items = []
                try:
                    items = [ci.Name for ci in cg.CalculationItems]
                except Exception:
                    pass
                desc = (table.Description or '').strip() if hasattr(table, 'Description') else ''
                if not desc:
                    cg_issues.append((table.Name, "no description"))
                else:
                    # check at least 1 calc item name appears in description
                    desc_lower = desc.lower()
                    if items and not any(it.lower() in desc_lower for it in items):
                        cg_issues.append((table.Name, f"description doesn't list any of: {', '.join(items[:5])}"))

        if cg_total == 0:
            print("ℹ️  No calculation groups found.")
        else:
            if cg_issues:
                print(f"⚠️  {len(cg_issues)}/{cg_total} calculation group(s) need better descriptions:")
                for tbl, reason in cg_issues[:5]:
                    print(f"     • {tbl} → {reason}")
                print("   ACTION: Per MS Learn, list calculation item names (YTD, MTD, PY, …)")
                print("           in the calculation group description so Copilot can use them.")
                issues.append(("important", "1.16 Grounding",
                               f"{len(cg_issues)} calculation group(s) without item names in description"))
                score = max(1, score - 1)
            else:
                print(f"✅ All {cg_total} calculation group(s) have descriptions listing item names.")
    except Exception as e:
        print(f"ℹ️  Could not inspect calculation groups via TOM: {e}")

    print(f"\n📊 Score: {score}/5")
    print("\n📚 Reference:")
    print("   https://learn.microsoft.com/power-bi/create-reports/copilot-semantic-models")

    check_scores['SM-016'] = {'achieved': score, 'max': 5, 'weight': 2}
    all_issues.extend(issues)


run_semantic_check('SM-016', _sm_check)


### 1.17 🪜 Hierarchies (Logical Groupings) · `SM-017`

**Why it matters:** Per [MS Learn — Optimize your semantic model for Copilot](https://learn.microsoft.com/power-bi/create-reports/copilot-evaluate-data) (Model structure), clear hierarchies — especially on dimension tables that support drilling down — help Copilot navigate levels and generate more accurate, drill-aware answers.

**What's checked (automated):**
- Enumerates all hierarchies (and their level counts) via TOM
- Flags drill-down candidate dimension tables (Date, Geography, Product, Organization, Customer, etc.) that have **no hierarchy** defined

**Examples:** `Date`: Year › Quarter › Month › Day · `Geography`: Country/Region › State › City


In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 1.17: HIERARCHIES (LOGICAL GROUPINGS)")
    print("=" * 70)

    issues = []
    score = 5

    # Per MS Learn (Optimize your semantic model for Copilot → Model structure):
    # establish clear hierarchies, especially on dimension tables that support
    # drilling down (e.g., Date: Year > Quarter > Month > Day; Geography:
    # Country/Region > State > City). Hierarchies help Copilot navigate levels.
    DRILL_PATTERN = r'(date|calendar|time|geography|geo|location|product|organi[sz]ation|org|customer|employee|account)'

    try:
        from sempy_labs.tom import connect_semantic_model
        hierarchies = []            # (table, hierarchy, level_count)
        tables_with_hier = set()
        with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
            for table in tom.model.Tables:
                try:
                    for h in table.Hierarchies:
                        lvl = 0
                        try:
                            lvl = len(list(h.Levels))
                        except Exception:
                            pass
                        hierarchies.append((table.Name, h.Name, lvl))
                        tables_with_hier.add(table.Name)
                except Exception:
                    continue

        if hierarchies:
            print(f"✅ {len(hierarchies)} hierarchy(ies) found across {len(tables_with_hier)} table(s):\n")
            for tname, hname, lvl in hierarchies[:12]:
                print(f"     • {tname}[{hname}]  ({lvl} level(s))")
            if len(hierarchies) > 12:
                print(f"     ... and {len(hierarchies) - 12} more")
        else:
            print("⚠️  No hierarchies found in the model.")

        # Flag drill-down-friendly visible tables that have NO hierarchy
        candidate_tables = [
            t for t in visible_tables['Name'].dropna()
            if re.search(DRILL_PATTERN, str(t).lower())
        ]
        missing_hier = [t for t in candidate_tables if t not in tables_with_hier]

        if missing_hier:
            print(f"\n⚠️  {len(missing_hier)} drill-down candidate table(s) without a hierarchy:")
            for t in missing_hier[:10]:
                print(f"     • {t}")
            if len(missing_hier) > 10:
                print(f"     ... and {len(missing_hier) - 10} more")
            print("\n   ACTION: Add hierarchies (e.g., Date: Year > Quarter > Month > Day,")
            print("           Geography: Country/Region > State > City) so Copilot can drill down.")
            issues.append(("recommended", "1.17 Hierarchies",
                           f"{len(missing_hier)} drill-down table(s) without a hierarchy"))
            score = max(2, score - 2)
        elif hierarchies:
            print("\n✅ Key dimension tables have hierarchies configured.")

        if not hierarchies and not candidate_tables:
            print("\nℹ️  No obvious drill-down dimensions detected — hierarchies optional for this model.")

    except Exception as e:
        print(f"ℹ️  Could not inspect hierarchies via TOM: {e}")
        print("   Manual check: Model view → dimension table → right-click column → New hierarchy")
        score = 3
        check_status['SM-017'] = (Status.NOT_EVALUATED, f"Hierarchies could not be inspected via TOM: {e}")

    print(f"\n📊 Score: {score}/5")
    print("\n📚 Reference (Model structure):")
    print("   https://learn.microsoft.com/power-bi/create-reports/copilot-evaluate-data")

    check_scores['SM-017'] = {'achieved': score, 'max': 5, 'weight': 1}
    all_issues.extend(issues)


run_semantic_check('SM-017', _sm_check)


### 1.18 ✂️ Description Length — Copilot 200-Character Window · `SM-018`

**Why it matters:** Per [MS Learn — Optimize your semantic model for Copilot](https://learn.microsoft.com/power-bi/create-reports/copilot-evaluate-data) (DAX query considerations), **Copilot uses only the first 200 characters** of any table, column, or measure description as grounding context. Anything beyond 200 characters is ignored.

**What's checked (automated):**
- Visible tables, columns, and measures whose descriptions exceed 200 characters
- Flags them so you can **front-load** the key business meaning and intended usage into the first 200 characters


In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 1.18: DESCRIPTION LENGTH — COPILOT 200-CHARACTER WINDOW")
    print("=" * 70)

    issues = []
    score = 5

    # Per MS Learn (Optimize your semantic model for Copilot → DAX query
    # considerations): "Copilot uses only the first 200 characters" of a
    # description. Front-load the key business meaning + intended usage so the
    # most important grounding context fits inside the first 200 characters.
    COPILOT_DESC_LIMIT = 200

    def _over_limit(df, name_col):
        hits = []
        if 'Description' not in df.columns:
            return hits
        for _, row in df.iterrows():
            desc = str(row.get('Description') or '').strip()
            if len(desc) > COPILOT_DESC_LIMIT:
                if name_col == 'Name':
                    label = str(row.get('Name', '?'))
                else:
                    label = f"{row.get('Table Name', '?')}[{row.get(name_col, '?')}]"
                hits.append((label, len(desc)))
        return hits

    long_tbls = _over_limit(visible_tables,   'Name')
    long_cols = _over_limit(visible_columns,  'Column Name')
    long_msrs = _over_limit(visible_measures, 'Measure Name')
    total_long = len(long_tbls) + len(long_cols) + len(long_msrs)

    if total_long == 0:
        print(f"✅ PASSED: All descriptions fit within the {COPILOT_DESC_LIMIT}-character Copilot grounding window.")
    else:
        print(f"⚠️  {total_long} description(s) exceed {COPILOT_DESC_LIMIT} characters —")
        print(f"    Copilot only reads the FIRST {COPILOT_DESC_LIMIT} characters; the remainder is ignored.\n")
        for label, items in [("Tables", long_tbls), ("Columns", long_cols), ("Measures", long_msrs)]:
            if items:
                print(f"   {label} ({len(items)}):")
                for name, length in items[:8]:
                    print(f"      • {name}  ({length} chars)")
                if len(items) > 8:
                    print(f"      ... and {len(items) - 8} more")
        print("\n   ACTION: Front-load the business meaning + intended usage in the first")
        print(f"           {COPILOT_DESC_LIMIT} characters; move secondary detail to the end.")
        issues.append(("recommended", "1.18 Desc Length",
                       f"{total_long} description(s) exceed the {COPILOT_DESC_LIMIT}-char Copilot window"))
        score = max(2, 5 - min(3, total_long))

    print(f"\n📊 Score: {score}/5")
    print("\n📚 Reference (DAX query considerations):")
    print("   https://learn.microsoft.com/power-bi/create-reports/copilot-evaluate-data")

    check_scores['SM-018'] = {'achieved': score, 'max': 5, 'weight': 1}
    all_issues.extend(issues)


run_semantic_check('SM-018', _sm_check)


---
## 8.2 PREP FOR AI — AI DATA SCHEMA · `SM-019` … `SM-021`

This section validates the **AI Data Schema** configuration (Prep for AI → Simplify data schema).

### 2.1 🎯 AI Data Schema Selection · `SM-019`

**Why it matters:** This is THE most important configuration. Select ONLY tables/columns/measures relevant to Data Agent questions.

**What's checked:** AI Data Schema tables read live from Prep for AI (Power BI MCP server); manual checklist when unreadable

In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 2.1: AI DATA SCHEMA SELECTION (MANUAL)")
    print("=" * 70)

    print("🚨 CRITICAL: This is a MANUAL configuration in Power BI.\n")
    print("📍 Power BI Desktop or Service → Home → Prep data for AI → Simplify data schema")
    print("\n✅ CHECKLIST:")
    print("\n   [ ] 1. Define Data Agent scope (what questions should it answer?)")
    print("   [ ] 2. Select ONLY relevant tables for those questions")
    print("   [ ] 3. Include ALL dependent objects for selected measures (see Check 2.2)")
    print("   [ ] 4. Exclude helper/intermediate measures")
    print("   [ ] 5. Exclude duplicate measures (keep only canonical versions)")
    print("   [ ] 6. Verify no hidden fields referenced by Verified Answers")
    print("   [ ] 7. Ensure selected tables match Data Agent configuration (see Check 5)")
    print("\n⚠️  IMPORTANT: Start narrow — it's easier to add objects than remove them.")
    print("\n📚 Documentation:")
    print("   https://learn.microsoft.com/power-bi/create-reports/copilot-prepare-data-ai-data-schema")

    if prep4ai is not None:
        n_tbl = len(prep4ai.get('tables') or [])
        print(f"\n🔎 Live read (Prep for AI): {n_tbl} table(s) in the AI Data Schema; {len(visible_tables)} visible table(s) in the model.")
        if n_tbl == 0:
            score = 0
            all_issues.append(("critical", "2.1 AI Data Schema", "AI Data Schema has no tables selected"))
        elif n_tbl > 20:
            score = 12
            all_issues.append(("important", "2.1 AI Data Schema", f"{n_tbl} tables in AI Data Schema — narrow the scope (warning sign: > 20)"))
        else:
            score = 20
        check_scores['SM-019'] = {'achieved': score, 'max': 20, 'weight': 3}
        print(f"\n📊 Score: {score}/20 (read from Prep for AI configuration)")
    else:
        sm_manual('SM-019', f"AI Data Schema not readable ({PREP4AI_ERROR}) — verify Prep data for AI → Simplify data schema.",
                  legacy_attested=prep_for_ai_configured)


run_semantic_check('SM-019', _sm_check)


### 2.2 🔗 Measure Dependencies · `SM-020`

**Why it matters:** If a measure depends on columns/measures not in AI Data Schema, DAX generation will fail or produce incorrect results.

**What's checked:** Measure dependencies via Semantic Link Labs

In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 2.2: MEASURE DEPENDENCIES ANALYSIS")
    print("=" * 70)

    try:
        from sempy_labs import get_measure_dependencies

        print("Retrieving measure dependencies...\n")
        deps = get_measure_dependencies(dataset=dataset, workspace=workspace)

        if deps is not None and not deps.empty:
            n_measures = deps['Measure Name'].nunique() if 'Measure Name' in deps.columns else len(deps)
            print(f"✅ Dependency data retrieved for {n_measures} measure(s).\n")
            print("📋 Sample dependencies (first 30 rows):")
            display(deps.head(30))

            print("\n🚨 CRITICAL ACTION:")
            print("   When configuring AI Data Schema, include ALL dependent objects for each")
            print("   measure you select. Missing dependencies cause incorrect DAX generation.")
            print("\n💡 WORKFLOW:")
            print("   1. Select measure in AI Data Schema")
            print("   2. Find its row in the dependency table above")
            print("   3. Add ALL referenced tables, columns, and measures to AI Data Schema")

            score = 15
            check_scores['SM-020'] = {'achieved': score, 'max': 15, 'weight': 3}
        else:
            print("ℹ️  No dependency data returned.")
            score = 10
            check_scores['SM-020'] = {'achieved': score, 'max': 15, 'weight': 3}

    except ImportError:
        print("ℹ️  get_measure_dependencies not available.")
        print("\n   Install: %pip install semantic-link-labs --upgrade")
        score = 10
        check_scores['SM-020'] = {'achieved': score, 'max': 15, 'weight': 3}
        check_status['SM-020'] = (Status.NOT_EVALUATED, "get_measure_dependencies not available")
    except Exception as e:
        print(f"⚠️  Could not analyze dependencies: {e}")
        score = 10
        check_status['SM-020'] = (Status.NOT_EVALUATED, f"Dependency analysis failed: {e}")
        check_scores['SM-020'] = {'achieved': score, 'max': 15, 'weight': 3}

    print(f"\n📊 Score: {score}/15")


run_semantic_check('SM-020', _sm_check)


### 2.3 🔧 Helper Measures Detection · `SM-021`

**Why it matters:** Helper/intermediate measures visible in the model should be excluded from AI Data Schema to reduce noise.

**What's checked:** Visible measures with helper/intermediate naming patterns

In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 2.3: HELPER MEASURES DETECTION")
    print("=" * 70)

    issues = []
    score = 5

    HELPER_PATTERN = r'\b(helper|_h$|_helper$|aux|auxiliary|temp|tmp|working|intermediate|_int$|__[a-z]|_calc$|internal)\b'
    visible_helpers = visible_measures[
        visible_measures['Measure Name'].astype(str).str.lower().str.contains(HELPER_PATTERN, regex=True, na=False)
    ]

    if visible_helpers.empty:
        print("✅ PASSED: No obvious helper measures found in visible measures.")
    else:
        print(f"⚠️  {len(visible_helpers)} potential helper/intermediate measure(s):\n")
        for _, row in visible_helpers.head(12).iterrows():
            print(f"     • {row.get('Table Name')}[{row.get('Measure Name')}]")
        if len(visible_helpers) > 12:
            print(f"     ... and {len(visible_helpers) - 12} more")

        print("\n📍 ACTION:")
        print("   1. Hide these measures if they're only building blocks")
        print("   2. Or exclude from AI Data Schema if they must stay visible")

        score = max(1, 5 - len(visible_helpers))
        issues.append(("important", "2.3 Helper Measures", f"{len(visible_helpers)} visible helper measures"))

    print(f"\n📊 Score: {score}/5")
    check_scores['SM-021'] = {'achieved': score, 'max': 5, 'weight': 1}
    all_issues.extend(issues)


run_semantic_check('SM-021', _sm_check)


---
## 8.3 PREP FOR AI — VERIFIED ANSWERS · `SM-022`

**Critical configuration:** Create verified answers for your 5-10 most common or complex questions.

> 💡 **DevOps Insight:** Verified Answers travel with your model (PBIP/TMDL) and flow through the model's version control and deployment pipelines. Data agent configuration has its own Git item definition (preview).

> ✅ **Current guidance:** 5–7 complete trigger questions per verified answer and up to **three filters**.

## 🔍 Where Verified Answers Live

- ✅ Are **fully honored by Data Agents** when present in the semantic model

**Verified Answers are stored in your semantic model's `linguisticMetadata` property** — the same location as AI Instructions and synonyms. This means they:

- ✅ Can be reviewed in pull requests
- ✅ Are version-controlled via TMDL when using PBIP format
- ✅ Flow through deployment pipelines with your model

In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 3: VERIFIED ANSWERS (MANUAL)")
    print("=" * 70)

    print("🚨 CRITICAL: Verified Answers stored in linguisticMetadata (version-controlled).\n")
    print("📍 Power BI Desktop or Service → Home → Prep data for AI → Verified answers")
    print("\n💾 STORAGE LOCATION:")
    print("   • Verified Answers are stored in model's linguisticMetadata property")
    print("   • When using PBIP format, they appear in TMDL files")
    print("   • Fully version-controlled via Git when using source control")
    print("   • Travel with the model through deployment pipelines")
    print("\n✅ BEST PRACTICES CHECKLIST:")
    print("\n   [ ] 1. Identify 5-10 most common questions from your team")
    print("   [ ] 2. Create verified answer using appropriate visual type")
    print("   [ ] 3. Add 5-7 complete, robust trigger questions (NOT partial phrases)")
    print("   [ ] 4. Include both formal and conversational phrasings")
    print("        Example triggers for 'YTD Revenue by Region':")
    print("        - What is the year to date revenue by region?")
    print("        - Show revenue YTD broken down by region")
    print("        - YTD sales performance per region")
    print("        - How are regions performing this year?")
    print("   [ ] 5. Configure up to 3 filters for flexible slicing")
    print("   [ ] 6. ⚠️  CRITICAL: Ensure ALL fields used are VISIBLE (not hidden)")
    print("   [ ] 7. Test each trigger question for exact and semantic matching")
    print("   [ ] 8. After renaming objects, re-save affected verified answers")
    print("   [ ] 9. Review TMDL changes in pull requests (PBIP format)")
    print("\n🚫 COMMON MISTAKES:")
    print("   • Using partial phrases as triggers ('revenue by region')")
    print("   • Referencing hidden columns → verified answer will FAIL silently")
    print("   • Too few trigger questions (need 5-7 for good coverage)")
    print("   • Not testing both formal and casual language")
    print("   • Editing in service and losing version control")
    print("\n✅ DEVOPS TIP:")
    print("   Configure Verified Answers in Power BI Desktop, save as PBIP, commit to Git.")
    print("   Changes to verified answers will appear in your TMDL diffs for review.")
    print("\n📚 Documentation:")
    print("   https://learn.microsoft.com/power-bi/create-reports/copilot-prepare-data-ai-verified-answers")

    if prep4ai is not None:
        vas = prep4ai.get('verified_answers') or []
        weak = [va.get('Name', 'Unnamed') for va in vas if not 5 <= len(va.get('TriggerQuestions') or []) <= 7]
        many_filters = [va.get('Name', 'Unnamed') for va in vas if len(va.get('Filters') or []) > 3]
        print(f"\n🔎 Live read (Prep for AI): {len(vas)} verified answer(s); outside 5–7 triggers: {len(weak)}; > 3 filters: {len(many_filters)}")
        if not vas:
            score = 8
            all_issues.append(("important", "3 Verified Answers", "No verified answers configured"))
        else:
            score = 15 if not (weak or many_filters) else 10
            for n in weak[:10]:
                all_issues.append(("recommended", f"Verified answer: {n}", "Use 5–7 complete trigger questions"))
            for n in many_filters[:10]:
                all_issues.append(("recommended", f"Verified answer: {n}", "Use at most 3 filters"))
        check_scores['SM-022'] = {'achieved': score, 'max': 15, 'weight': 3}
        print(f"\n📊 Score: {score}/15 (read from Prep for AI configuration)")
    else:
        sm_manual('SM-022', f"Verified Answers not readable ({PREP4AI_ERROR}) — verify in Prep data for AI → Verified answers.",
                  legacy_attested=prep_for_ai_configured)


run_semantic_check('SM-022', _sm_check)


---
## 8.4 PREP FOR AI — AI INSTRUCTIONS · `SM-023`

**Critical configuration:** Define business terminology, date defaults, and metric preferences.

> 🎯 **DevOps Insight:** When AI instructions live in the semantic model, they become part of PBIP folder structure, can be reviewed in pull requests, and flow through deployment pipelines. Data Agent instructions create configuration drift risk.

## 🔍 The Two Instruction Surfaces

**Push as much instruction content as possible into the semantic model layer** (Prep for AI), where it benefits from version control. Keep Data Agent instructions minimal — focused only on cross-source routing.

There are **two distinct places** where AI instructions can be configured:

## 💡 Best Practice Pattern

### 1️⃣ Semantic Model AI Instructions (Prep for AI)

**Storage:** `linguisticMetadata.CustomInstructions` property in model metadata

**Access:** Power BI Desktop → Prep data for AI → Add AI instructions

**Honored by:** Copilot in Power BI and the data agent's DAX generation tool

**Version Control:** ✅ Stored in TMDL when using PBIP format

**DevOps:** ✅ Diff-able in Git and flows through CI/CD pipelines

### 2️⃣ Data Agent Instructions (Fabric Service)

**Storage:** Data agent item configuration (`stage_config.json` → `aiInstructions`), not in model metadata

**Access:** Fabric Portal → Data Agent → Instructions

**Honored by:** The data agent orchestrator (routing, response format). ⚠️ **Ignored by the semantic-model DAX generation tool**

**Version Control:** ⚠️ No TMDL representation; source-controlled via the data agent Git item definition (preview)

**DevOps:** ⚠️ Separate lifecycle from the model — review both in pull requests to avoid configuration drift

In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 4: AI INSTRUCTIONS (MANUAL)")
    print("=" * 70)

    print("🚨 CRITICAL: AI Instructions at SEMANTIC MODEL level (version-controlled).\n")
    print("📍 Power BI Desktop → Home → Prep data for AI → Add AI instructions")
    print("\n💾 STORAGE & VERSION CONTROL:")
    print("   • Instructions stored in linguisticMetadata.CustomInstructions")
    print("   • linguisticMetadata version: 4.0.0 → 4.2.0 when instructions added")
    print("   • Model annotation PBI_ProTooling includes 'CopilotTooling' flag")
    print("   • When using PBIP format: fully version-controlled via TMDL")
    print("   • Changes appear in Git diffs for review in pull requests")
    print("\n📋 WHAT TO INCLUDE (SEMANTIC MODEL INSTRUCTIONS):")
    print("\n   1️⃣  BUSINESS TERMINOLOGY & SYNONYMS (preferred over legacy Q&A Setup)")
    print("      Define organization-specific terms and abbreviations in natural language:")
    print("      Example: 'TMS is total media spend, calculated using [Total Media Spend]'")
    print("      Example: 'VFR means Volume of Fruit Revenue'")
    print("      Example: 'Top performer = sales rep achieving 110%+ of monthly quota'")
    print("      💡 The AI Instructions pane is now the recommended place for synonyms")
    print("         and obscure acronyms — superior to the legacy Q&A Setup dialog.")
    print("\n   2️⃣  NEGATIVE SCOPE — TELL COPILOT WHAT'S NOT IN THE MODEL ⚠️ HIGH IMPACT")
    print("      Explicitly state what data the model does NOT contain to prevent guessing.")
    print("      Example: 'This model only contains data for the UK. If asked about US sales,")
    print("                state that data is unavailable.'")
    print("      Example: 'Data is limited to FY2023 onward. For prior years, no data exists.'")
    print("      Example: 'Returns and refunds are not tracked in this model.'")
    print("      Example: 'This model does not contain customer PII (no email, phone, address).'")
    print("      → Prevents hallucinations and 'best guess' responses on out-of-scope questions.")
    print("\n   3️⃣  TIME PERIOD DEFINITIONS")
    print("      Clarify fiscal year, seasons, reporting periods:")
    print("      Example: 'Fiscal year starts July 1'")
    print("      Example: 'Peak season is November through January'")
    print("\n   4️⃣  DEFAULT DATE FIELD")
    print("      Specify which date to use when ambiguous:")
    print("      Example: 'Unless specified, use Order Date for date-based questions'")
    print("\n   5️⃣  METRIC PREFERENCES")
    print("      Guide which measure to use for common questions:")
    print("      Example: 'For profitability questions, use Contribution Margin'")
    print("      Example: 'Revenue questions should use Net Revenue after discounts'")
    print("\n   6️⃣  CALCULATION GROUPS & FIELD PARAMETERS")
    print("      If using these features, explain how they work:")
    print("      Example: 'Time intelligence is handled by the [Time Calc] calculation group'")
    print("\n   7️⃣  COMPLEX DAX PATTERNS (optional)")
    print("      Provide example DAX for complex scenarios to guide AI")
    print("\n   8️⃣  DEFAULT GROUPINGS & ANALYSIS PREFERENCES")
    print("      Define how the AI should group and analyze data by default:")
    print("      Example: 'When analyzing sales performance, default grouping is Region then Product Category'")
    print("      Example: 'Comparative analysis should default to prior year as the comparison period'")
    print("      Example: 'Rank analyses should show Top 10 unless the user specifies otherwise'")
    print("      Example: 'Currency values should be displayed in thousands (K) unless specified'")
    print()
    print("   [ ] Add Negative Scope statements (data the model does NOT contain)")
    print("   [ ] Define cryptic acronyms / industry jargon (e.g., VFR, TMS)")
    print("   [ ] Add default grouping preference to AI Instructions")
    print("   [ ] Add default comparison period (e.g., prior year, prior month)")
    print("   [ ] Specify default sort order for ranked questions")
    print("   [ ] Define default aggregation level (product, category, region)")
    print("\n🚫 DO NOT INCLUDE IN MODEL INSTRUCTIONS (use Data Agent instructions):")
    print("   • Response formatting preferences")
    print("   • Cross-data-source routing ('use lakehouse for X, semantic model for Y')")
    print("   • Tone and presentation style")
    print("\n⚠️  WHY THIS MATTERS — CONFIGURATION DRIFT RISK:")
    print("   Your semantic model might be fully version-controlled with AI instructions")
    print("   locked into your release pipeline. But Data Agent instructions sitting on")
    print("   top have separate configuration someone edited last week — and nobody reviewed.")
    print("\n🔄 DEVOPS PATTERN:")
    print("   1. Configure instructions in Power BI Desktop (Prep for AI)")
    print("   2. Save model as PBIP format")
    print("   3. Commit TMDL files to Git")
    print("   4. Deploy through pipeline (instructions travel with model)")
    print("   5. Keep Data Agent instructions minimal (cross-source routing only)")

    print("\n💡 TIP: Test responses BEFORE adding instructions to identify gaps.")
    print("   Instructions should clarify, not contradict verified answers.")
    print("\n📚 Documentation:")
    print("   https://learn.microsoft.com/power-bi/create-reports/copilot-prepare-data-ai-instructions")
    print("   https://learn.microsoft.com/fabric/data-science/data-agent-semantic-model")

    if prep4ai is not None:
        ci = str(prep4ai.get('custom_instructions') or '').strip()
        neg = bool(re.search(r"(does not|doesn't|not) (contain|include|track|have)|no data|unavailable|out of scope|only contains", ci, re.I))
        dflt = bool(re.search(r"\bdate\b|fiscal|default", ci, re.I))
        print(f"\n🔎 Live read (Prep for AI): instructions {len(ci)} chars; negative scope={neg}; date/default guidance={dflt}")
        if not ci:
            score = 0
            all_issues.append(("critical", "4 AI Instructions", "No Prep for AI AI Instructions configured"))
        else:
            score = 15 if (neg and dflt) else 10
            if not neg:
                all_issues.append(("important", "4 AI Instructions", "Add negative-scope statements (what the model does NOT contain)"))
            if not dflt:
                all_issues.append(("recommended", "4 AI Instructions", "Add default date field / fiscal calendar guidance"))
        check_scores['SM-023'] = {'achieved': score, 'max': 15, 'weight': 3}
        print(f"\n📊 Score: {score}/15 (read from Prep for AI configuration)")
    else:
        sm_manual('SM-023', f"AI Instructions not readable ({PREP4AI_ERROR}) — verify in Prep data for AI → Add AI instructions.",
                  legacy_attested=prep_for_ai_configured)


run_semantic_check('SM-023', _sm_check)


### 4.1 🔍 Prep for AI Configuration Validator · `SM-024`

**Why it matters:** Programmatically verify what's actually configured in Prep for AI (AI Data Schema, Custom Instructions, Verified Answers).

**What's checked:** Retrieves and displays actual Prep for AI configuration from the semantic model using Power BI MCP Server

> 💡 **Source:** Code adapted from [Fabric.guru by Sandeep Pawar](https://fabric.guru/programmatically-retrieve-prep-data-for-ai-configuration-of-semantic-models)

In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 4.1: PREP FOR AI CONFIGURATION VALIDATOR")
    print("=" * 70)

    print("📥 Prep for AI configuration (retrieved once per model by the SemanticModelAdapter via the Power BI MCP server)\n")

    if prep4ai is None:
        print(f"❌ Could not retrieve Prep for AI configuration for model '{dataset}': {PREP4AI_ERROR}")
        print("\n   This check requires a Fabric notebook with notebookutils and access to the Power BI MCP server.")
        all_issues.append(("important", "4.1 Prep for AI Config", f"Prep for AI configuration not retrievable: {PREP4AI_ERROR}"))
        check_status['SM-024'] = (Status.NOT_EVALUATED, f"Prep for AI configuration not retrievable: {PREP4AI_ERROR}")
        score = 0
    else:
        print(f"✅ Successfully retrieved Prep for AI configuration for: {prep4ai['name']}\n")

        # AI Data Schema - Tables
        print("=" * 70)
        print("📊 AI DATA SCHEMA - TABLES")
        print("=" * 70)
        if prep4ai['tables']:
            print(f"\n✅ {len(prep4ai['tables'])} table(s) configured in AI Data Schema:\n")
            for table in prep4ai['tables']:
                table_name = table.get('Name', 'Unknown')
                columns = table.get('Columns', [])
                measures = table.get('Measures', [])
                print(f"   📋 {table_name}")
                print(f"      • Columns: {len(columns)}")
                print(f"      • Measures: {len(measures)}")
        else:
            print("\n⚠️  No tables found in AI Data Schema!")
            all_issues.append(("critical", "4.1 Prep for AI Config", "No tables found in AI Data Schema"))

        # Custom Instructions
        print("\n" + "=" * 70)
        print("📝 CUSTOM AI INSTRUCTIONS")
        print("=" * 70)
        if prep4ai['custom_instructions']:
            print(f"\n✅ Custom instructions configured:\n")
            print(prep4ai['custom_instructions'])
        else:
            print("\n⚠️  No custom AI instructions found.")
            all_issues.append(("important", "4.1 Prep for AI Config", "No custom AI instructions found in Prep for AI"))

        # Verified Answers
        print("\n" + "=" * 70)
        print("✔️  VERIFIED ANSWERS")
        print("=" * 70)
        if prep4ai['verified_answers']:
            print(f"\n✅ {len(prep4ai['verified_answers'])} verified answer(s) configured:\n")
            for idx, va in enumerate(prep4ai['verified_answers'], 1):
                print(f"   {idx}. Name: {va.get('Name', 'Unnamed')}")
                triggers = va.get('TriggerQuestions', [])
                print(f"      • Trigger questions: {len(triggers)}")
                if triggers:
                    print(f"      • Example: \"{triggers[0]}\"")

                # Show what fields are used (projections)
                projections = va.get('Projections', {})
                if projections:
                    tables_used = projections.get('Tables', [])
                    columns_used = projections.get('Columns', [])
                    measures_used = projections.get('Measures', [])
                    print(f"      • Uses: {len(tables_used)} table(s), {len(columns_used)} column(s), {len(measures_used)} measure(s)")
        else:
            print("\n⚠️  No verified answers found.")
            all_issues.append(("recommended", "4.1 Prep for AI Config", "No verified answers found in Prep for AI"))

        # Scoring
        score = 0
        if prep4ai['tables']:
            score += 5
        if prep4ai['custom_instructions']:
            score += 3
        if prep4ai['verified_answers']:
            score += 2

        print(f"\n💡 NOTE: This retrieves current configuration. You cannot modify Prep for AI programmatically.")
        print("   Use this to verify, monitor, or validate your configurations.\n")

    print(f"\n📊 Configuration Score: {score}/10")
    check_scores['SM-024'] = {'achieved': score, 'max': 10, 'weight': 2}

    print("\n📚 Learn more:")
    print("   https://fabric.guru/programmatically-retrieve-prep-data-for-ai-configuration-of-semantic-models")
    print("   https://learn.microsoft.com/power-bi/developer/mcp/remote-mcp-server-get-started")


run_semantic_check('SM-024', _sm_check)


---
## 8.5 🔗 Data Agent ↔ Prep for AI Alignment · `SM-025` · `SM-026`

**Critical requirement:** select the **same tables** in the data agent that you defined in Prep for AI → AI Data Schema. This check compares the agent's semantic-model table selection with the AI Data Schema that the Power BI MCP server returns. It also flags selected tables that no longer exist in the model (drift). It then records whether standard or advanced (preview runtime) DAX generation is active.

> 🔄 This replaces v2.2.2's manual "Data Agent Configuration" check, which scored zero until you configured it by hand.

In [ ]:
def _sm_alignment():
    print("=" * 70)
    print("CHECK 8.5: DATA AGENT TABLE SELECTION ↔ AI DATA SCHEMA (SM-025)")
    print("=" * 70)
    src = CURRENT_SEMANTIC_SOURCE
    if AGENT_PROFILE.standalone_semantic:
        check_status['SM-025'] = (Status.NOT_APPLICABLE, "Standalone mode — no data agent selection to compare")
        print("➖ Standalone semantic-model mode — no data agent table selection to compare.")
        return
    model_tables = set(tables_df['Name'].astype(str))
    selected = {o["name"] for o in src.selected_objects if o.get("name")}
    drift = sorted(selected - model_tables)
    for t in drift:
        all_issues.append(("critical", t, "Selected in the data agent but not found in the model (refresh the agent schema)"))
    if not selected:
        check_status['SM-025'] = (Status.UNKNOWN if not src.metadata.get("element_tree_available") else Status.FAIL,
                                  "Agent table selection not exposed" if not src.metadata.get("element_tree_available") else "No tables selected in the data agent")
        print(f"❔ {check_status['SM-025'][1]}")
        return
    if prep4ai is None:
        check_status['SM-025'] = (Status.NOT_EVALUATED, f"AI Data Schema not readable ({PREP4AI_ERROR}); agent selects {len(selected)} table(s)")
        print(f"⏸️ {check_status['SM-025'][1]}")
        return
    ai_tables = {t.get('Name') for t in prep4ai.get('tables') or [] if t.get('Name')}
    extra = sorted(selected - ai_tables)
    missing = sorted(ai_tables - selected)
    print(f"  Agent-selected tables : {len(selected)}")
    print(f"  AI Data Schema tables : {len(ai_tables)}")
    for t in extra:
        print(f"  ❌ Selected in agent but NOT in AI Data Schema: {t}")
        all_issues.append(("critical", t, "Selected in the data agent but not in the AI Data Schema"))
    for t in missing:
        print(f"  ⚠️  In AI Data Schema but NOT selected in agent: {t}")
        all_issues.append(("important", t, "In the AI Data Schema but not selected in the data agent"))
    if drift or extra:
        score = 0 if drift else 10
    elif missing:
        score = 15
    else:
        score = 20
        print("  ✅ Agent selection matches the AI Data Schema.")
    check_scores['SM-025'] = {'achieved': score, 'max': 20, 'weight': 3}
    print("\n✅ CHECKLIST (manual follow-ups):")
    print("   [ ] Mark the semantic model as 'Prepped for AI' in the Power BI service")
    print("   [ ] Test responses BEFORE adding data agent instructions; review generated DAX")
    print("   [ ] Keep semantic guidance in Prep for AI — the DAX tool ignores agent-level instructions")
    print(f"\n📊 Score: {score}/20")


def _sm_dax_mode():
    print("\n" + "=" * 70)
    print("CHECK 8.6: DAX GENERATION MODE & VALUE INDEXING (SM-026)")
    print("=" * 70)
    if AGENT_PROFILE.standalone_semantic:
        check_status['SM-026'] = (Status.NOT_APPLICABLE, "Standalone mode — runtime belongs to a data agent")
        print("➖ Standalone mode — runtime not applicable.")
        return
    rt = AGENT_PROFILE.runtime
    if rt == "UNKNOWN":
        check_status['SM-026'] = (Status.UNKNOWN, "Runtime not exposed — cannot tell standard vs advanced DAX generation")
    else:
        mode = "advanced DAX generation (preview runtime)" if rt == "PREVIEW" else "standard DAX generation"
        qa_gate = datetime.now(timezone.utc) >= datetime(2026, 12, 1, tzinfo=timezone.utc)
        note = ("Q&A retirement date reached — re-verify the current advanced-DAX value-indexing requirement"
                if qa_gate else "Advanced-DAX value indexing currently documented with Q&A (retirement scheduled Dec 2026) — version-gated")
        status = Status.WARN if (rt == "PREVIEW" and (qa_gate or not PREVIEW_ACCEPTED)) else Status.PASS
        check_status['SM-026'] = (status, f"Runtime {rt}: {mode}. {note if rt == 'PREVIEW' else ''}".strip())
    print(f"  {STATUS_ICON[check_status['SM-026'][0].value]} {check_status['SM-026'][1]}")


run_semantic_check('SM-025', _sm_alignment)
run_semantic_check('SM-026', _sm_dax_mode)

---
# 9️⃣ GRAPH MODEL *(preview)*

Graph Model is a **preview** data-agent source that's queried through GQL/NL2GQL. This adapter is preview-aware. Node and edge metadata are read from the source configuration: element types `node`/`edge`, or keys such as `nodeTypes`, `edgeTypes`, `nodes`, `edges`, or `schema`. Anything the adapter can't inspect is reported as `UNKNOWN` or `NOT_EVALUATED`.

**Graph modeling principles encoded:**

- Model an entity as a **node** when users need to traverse to it, group by it, or share relationships through it.
- Keep descriptive values as **properties** when they don't need independent traversal.
- Model only the relationships that answer the agent's intended questions. Give edges clear directional semantics and verify both endpoint mappings.
- Plan schema changes carefully: current graph behavior **requires a full reload after structural changes**.

In [ ]:
class GraphSourceAdapter(SourceAdapter):
    source_family = "GRAPH"
    primary_rule = "GRAPH-005"

    def live_rules(self):
        return ["GRAPH-008"]

    def collect_metadata(self, s):
        raw = s.metadata.get("raw_config", {})
        objs = s.metadata.get("all_objects", []) or s.selected_objects
        nodes = _schema_part(raw, "nodeTypes", "node_types", "nodes", "vertexTypes") or \
            [dict(name=o["name"], properties=o.get("columns", [])) for o in objs if o.get("object_type") in ("node", "nodetype")]
        edges = _schema_part(raw, "edgeTypes", "edge_types", "edges", "relationshipTypes") or \
            [dict(name=o["name"], properties=o.get("columns", [])) for o in objs if o.get("object_type") in ("edge", "edgetype", "relationship", "relationshiptype")]
        return {"nodes": nodes, "edges": edges, "exposed": bool(nodes or edges)}

    def run_schema_checks(self, s, md):
        out = [preview_acceptance("GRAPH-001", s)]
        if not md["exposed"]:
            for rid in ("GRAPH-002", "GRAPH-003", "GRAPH-004", "GRAPH-005", "GRAPH-006", "GRAPH-007"):
                out.append(make_finding(rid, Status.UNKNOWN, "Graph node/edge metadata not exposed by the current contract", source=s, confidence="LOW"))
            return out
        nodes, edges = md["nodes"], md["edges"]
        nlabel = lambda n: _text(_first(n, "label", "name", "displayName", "display_name"))
        tech = [nlabel(n) for n in nodes if _TECH_LABEL.search(nlabel(n))]
        out.append(make_finding("GRAPH-002", Status.PASS if not tech else Status.WARN, f"{len(nodes)} node types; technical labels: {tech[:8] or 'none'}", source=s))
        generic = [nlabel(e) for e in edges if _GENERIC_EDGE.search(nlabel(e))]
        out.append(make_finding("GRAPH-003", Status.NOT_APPLICABLE if not edges else Status.PASS if not generic else Status.WARN,
                                f"{len(edges)} edge types; generic labels: {generic[:8] or 'none'}", source=s))
        nokey = [nlabel(n) for n in nodes if not _first(n, "key", "keys", "keyProperty", "keyProperties", "primaryKey", "id_property")]
        out.append(make_finding("GRAPH-004", Status.PASS if not nokey else Status.FAIL, f"Node types without a key: {nokey or 'none'}", source=s))
        nomap = [nlabel(x) for x in nodes + edges if not _first(x, "source", "sourceTable", "source_table", "mapping", "table", "binding", "dataSource")]
        out.append(make_finding("GRAPH-005", Status.PASS if not nomap else Status.FAIL if len(nomap) == len(nodes + edges) else Status.WARN,
                                f"Types without a source mapping: {nomap[:8] or 'none'} (column existence needs live metadata)", source=s, confidence="MEDIUM"))
        labels = {nlabel(n).lower() for n in nodes}
        bad_edges = []
        for e in edges:
            src = _text(_first(e, "sourceNodeType", "source_node", "from", "origin", "sourceLabel", "startNode")).lower()
            dst = _text(_first(e, "targetNodeType", "target_node", "to", "destination", "targetLabel", "endNode")).lower()
            if not src or not dst or (labels and (src not in labels or dst not in labels)):
                bad_edges.append(f"{nlabel(e)}({src or '?'}→{dst or '?'})")
        out.append(make_finding("GRAPH-006", Status.NOT_APPLICABLE if not edges else Status.PASS if not bad_edges else Status.FAIL,
                                f"Edges with missing/unknown endpoints: {bad_edges[:8] or 'none'}", source=s))
        wide = [f"{nlabel(n)}({len(_first(n, 'properties', default=[]) or [])})" for n in nodes if len(_first(n, "properties", default=[]) or []) > 30]
        out.append(make_finding("GRAPH-007", Status.PASS if not wide else Status.WARN, f"Node types with > 30 properties: {wide or 'none'}", source=s))
        return out

    def run_configuration_checks(self, s, md):
        return [
            manual_check("GRAPH-009", "After structural schema changes, verify the full graph reload completed before testing.", source=s),
            manual_check("GRAPH-010", "Add one-hop, multi-hop, optional, and aggregation questions to the evaluation set (section 14).", source=s),
            manual_check("GRAPH-015", "Record current graph size, timeout, and feature limits for the active release.", source=s),
            manual_check("GRAPH-016", "Test representative users against the graph item and its underlying data (section 15).", source=s),
        ]

    def validate_examples(self, s, md):
        out = []
        if not s.example_queries:
            return [make_finding(r, Status.NOT_APPLICABLE, "No GQL examples exposed", source=s) for r in ("GRAPH-011", "GRAPH-012", "GRAPH-013", "GRAPH-014")]
        hits = {r: [] for r in ("GRAPH-011", "GRAPH-012", "GRAPH-013", "GRAPH-014")}
        for e in s.example_queries:
            for rid, msg in lint_gql(e.get("query", "")):
                hits[rid].append(detail(rid, Status.WARN, msg, source=s, object_path=e["id"]))
        n = len(s.example_queries)
        for rid, h in hits.items():
            bad = len({x.object_path for x in h})
            out.append(make_finding(rid, Status.PASS if not bad else Status.WARN, f"{n - bad}/{n} GQL examples pass", ratio=(n - bad) / n if bad else None, source=s, confidence="MEDIUM"))
            out += h
        return out

    def smoke_test(self, s, md):
        return [make_finding("GRAPH-008", Status.NOT_EVALUATED, "Orphan/dangling-edge measurement needs source-table access; run a GQL/SQL probe on mapped keys",
                             source=s, confidence="LOW")]


ADAPTERS["GRAPH_MODEL"] = GraphSourceAdapter()
GRAPH_FINDINGS = run_family("GRAPH", "graph", "GRAPH MODEL (PREVIEW) — GRAPH-001 … GRAPH-016")

---
# 🔟 FABRIC ONTOLOGY *(preview)*

Ontology grounding gives the agent entity types, relationships, definitions, rules, source bindings, provenance, and access controls. This adapter checks **semantic completeness and binding integrity**, not just raw table metadata.

> ⚠️ **Ontology-specific warning:** don't run the semantic-model analyzer against an ontology just because the ontology was generated from a semantic model. Analyze (1) the semantic model with `SemanticModelAdapter` only when it's an independently connected source, (2) the ontology's entities, keys, relationships, definitions, and bindings here, and (3) cross-source consistency only when both are connected to the same agent.

**Current known issues (version-gated, snapshot 2026-09-29):**

- The data agent **doesn't work with ontologies bound to semantic models** (`ONT-011`, blocker).
- Current guidance recommends adding the agent instruction **`Support group by in GQL`** to work around an aggregation issue (`ONT-012`).

In [ ]:
class OntologySourceAdapter(SourceAdapter):
    source_family = "ONTOLOGY"
    primary_rule = "ONT-005"

    def collect_metadata(self, s):
        raw = s.metadata.get("raw_config", {})
        objs = s.metadata.get("all_objects", []) or s.selected_objects
        entities = _schema_part(raw, "entityTypes", "entity_types", "entities") or \
            [dict(name=o["name"], description=o.get("description"), properties=o.get("columns", [])) for o in objs if o.get("object_type") in ("entity", "entitytype")]
        rels = _schema_part(raw, "relationshipTypes", "relationship_types", "relationships") or \
            [dict(name=o["name"]) for o in objs if o.get("object_type") in ("relationship", "relationshiptype")]
        bindings = _schema_part(raw, "bindings", "dataBindings", "data_bindings")
        for e in entities:
            for b in _items(e, "bindings", "dataBindings", "data_bindings"):
                bindings.append({**b, "_entity": _text(_first(e, "name", "label"))})
        return {"entities": entities, "relationships": rels, "bindings": bindings, "exposed": bool(entities)}

    def run_schema_checks(self, s, md):
        out = [preview_acceptance("ONT-001", s)]
        if not md["exposed"]:
            for rid in ("ONT-002", "ONT-003", "ONT-004", "ONT-005", "ONT-006", "ONT-007", "ONT-008", "ONT-011", "ONT-015"):
                out.append(make_finding(rid, Status.UNKNOWN, "Ontology entity/binding metadata not exposed by the current contract", source=s, confidence="LOW"))
            return out
        ents, rels, binds = md["entities"], md["relationships"], md["bindings"]
        name = lambda x: _text(_first(x, "name", "label", "displayName", "display_name"))
        tech = [name(e) for e in ents if _TECH_LABEL.search(name(e)) or re.match(r"^(dim|fact)", name(e), re.I)]
        out.append(make_finding("ONT-002", Status.PASS if not tech else Status.WARN, f"{len(ents)} entities; technical names: {tech[:8] or 'none'}", source=s))
        nodef = [name(e) for e in ents if len(_text(_first(e, "definition", "description"))) < 20]
        cov = 1 - len(nodef) / len(ents)
        out.append(make_finding("ONT-003", Status.PASS if cov >= 0.9 else Status.WARN if cov >= 0.5 else Status.FAIL,
                                f"{len(ents) - len(nodef)}/{len(ents)} entities have definitions", ratio=cov if cov < 0.9 else None, source=s))
        nokey = [name(e) for e in ents if not _first(e, "key", "keys", "keyProperty", "keyProperties", "entityIdParts", "identifier")]
        out.append(make_finding("ONT-004", Status.PASS if not nokey else Status.FAIL, f"Entities without a key: {nokey or 'none'}", source=s))
        unbound = []
        for e in ents:
            for p in _items(e, "properties", "attributes"):
                if not _first(p, "binding", "column", "sourceColumn", "source_column", "boundTo") and not _first(e, "bindings", "dataBindings"):
                    unbound.append(f"{name(e)}.{name(p)}")
        out.append(make_finding("ONT-005", Status.PASS if not unbound else Status.FAIL, f"Unbound properties: {unbound[:8] or 'none'}", source=s, confidence="MEDIUM"))
        if not rels:
            out.append(make_finding("ONT-006", Status.NOT_APPLICABLE if len(ents) <= 1 else Status.WARN, "No relationship types exposed", source=s))
            out.append(make_finding("ONT-007", Status.NOT_APPLICABLE, "No relationship types exposed", source=s))
        else:
            gen = [name(r) for r in rels if _GENERIC_EDGE.search(name(r)) or re.search(r"_[A-Za-z]+_to_|^\w+_\w+$", name(r))]
            out.append(make_finding("ONT-006", Status.PASS if not gen else Status.WARN, f"Generated/technical relationship names: {gen[:8] or 'none'}", source=s))
            def _keyset(e):
                k = _first(e, "key", "keys", "keyProperty", "keyProperties", default=[])
                return {str(_first(x, "name") if isinstance(x, dict) else x).lower() for x in (k if isinstance(k, list) else [k]) if x}
            keys = {name(e).lower(): _keyset(e) for e in ents}
            misaligned = []
            for r in rels:
                tgt = _text(_first(r, "target", "targetEntity", "to", "targetEntityType")).lower()
                cols = _first(r, "targetKeyColumns", "matchedColumns", "target_key_columns", "joinColumns", default=[]) or []
                cols = {str(c).lower() for c in (cols if isinstance(cols, list) else [cols])}
                if tgt and cols and keys.get(tgt) and not cols & keys[tgt]:
                    misaligned.append(name(r))
            out.append(make_finding("ONT-007", Status.PASS if not misaligned else Status.FAIL, f"Relationships whose matched columns miss the target key: {misaligned or 'none'}", source=s, confidence="MEDIUM"))
        related = {_text(_first(r, "source", "sourceEntity", "from")).lower() for r in rels} | {_text(_first(r, "target", "targetEntity", "to")).lower() for r in rels}
        isolated = [name(e) for e in ents if name(e).lower() not in related]
        out.append(make_finding("ONT-008", Status.NOT_APPLICABLE if len(ents) <= 1 else Status.PASS if not isolated else Status.WARN,
                                f"Entities without relationships: {isolated[:8] or 'none'} (orphan keys need live data)", source=s, confidence="MEDIUM"))
        btypes = [normalize_source_type(_first(b, "sourceType", "source_type", "type", "itemType", "dataSourceType")) for b in binds]
        sm_bound = [f"{b.get('_entity', '')}:{_text(_first(b, 'sourceName', 'source', 'item', 'itemName'))}" for b, t in zip(binds, btypes) if t == "SEMANTIC_MODEL"]
        if not binds:
            out.append(make_finding("ONT-011", Status.UNKNOWN, "Binding source types not exposed — verify no binding targets a semantic model", source=s, confidence="LOW"))
            out.append(make_finding("ONT-015", Status.WARN, "Underlying source bindings not visible", source=s))
        else:
            out.append(make_finding("ONT-011", Status.FAIL if sm_bound else Status.PASS,
                                    f"Semantic-model bindings (known issue): {sm_bound or 'none'}; binding kinds: {sorted(set(btypes))}", source=s))
            out.append(make_finding("ONT-015", Status.PASS, f"{len(binds)} binding(s) across kinds {sorted(set(btypes))}", source=s))
        md["bound_kinds"] = sorted(set(btypes))
        return out

    def run_configuration_checks(self, s, md):
        has_wa = bool(re.search(r"support\s+group\s+by\s+in\s+gql", AGENT_PROFILE.instructions or "", re.I))
        others = [x for x in SOURCES if x is not s]
        return [
            make_finding("ONT-012", Status.PASS if has_wa else Status.WARN,
                         f"'Support group by in GQL' in agent instructions = {has_wa} (temporary workaround, docs snapshot {DOC_SNAPSHOT_DATE})", source=s),
            make_finding("ONT-009", Status.NOT_APPLICABLE, "Ontology is the only connected source", source=s) if not others else
            manual_check("ONT-009", f"Compare ontology definitions/rules/metrics with: {[o.display_name for o in others]}", source=s),
            manual_check("ONT-010", f"Grant Read on the ontology and on every bound source (kinds: {md.get('bound_kinds') or 'unknown'}).", source=s),
            manual_check("ONT-013", "Include traversal, grouping, and aggregation prompts in the evaluation set (section 14).", source=s),
            manual_check("ONT-014", "Confirm ontology initialization/reload completed before treating 'no data' as a schema failure.", source=s),
        ]


ADAPTERS["ONTOLOGY"] = OntologySourceAdapter()
ONTOLOGY_FINDINGS = run_family("ONTOLOGY", "ontology", "FABRIC ONTOLOGY (PREVIEW) — ONT-001 … ONT-015")

---
# 1️⃣1️⃣ AZURE AI SEARCH INDEX *(preview)*

Azure AI Search is the **unstructured-data** source. It needs a different readiness model: retrieval quality, field configuration, citations, permissions, and grounded-answer evaluation. The adapter reads the endpoint, index name, search mode, document count, and fields from the source configuration. Add `sample_documents` to a manual manifest to evaluate the index-quality checks (`AIS-010` … `AIS-014`). Those are proposed checks, not product requirements.

<details><summary>Azure AI Search source context template</summary>

```markdown
## Content
This index contains ...

## Intended questions
Use this index for policy, procedure, document, and narrative questions about ...

## Key fields
- Content: `content`
- Title: `title`
- Citation: `sourceUrl`
- Updated timestamp: `lastModified`

## Retrieval behavior
Use `<hybrid|semantic|full-text>` search and retrieve `<3-20>` documents.

## Boundaries
Do not use this source for authoritative numerical aggregation when a governed structured source is available.
```
</details>

In [ ]:
class AzureAiSearchAdapter(SourceAdapter):
    source_family = "AI_SEARCH"
    primary_rule = "AIS-002"
    CITATION_FIELDS = ("url", "sourceUrl", "filePath", "path", "folderPath")

    def collect_metadata(self, s):
        raw = s.metadata.get("raw_config", {})
        fields = _items(raw, "fields", "indexFields", "index_fields")
        if not fields:
            fields = [{"name": c["name"], "type": c.get("data_type")} for o in s.selected_objects + s.metadata.get("all_objects", []) for c in o.get("columns", [])]
        return {
            "endpoint": _text(_first(raw, "endpoint", "url", "resource_url", "resourceUrl", "searchServiceUrl", "search_endpoint", "serviceEndpoint")),
            "index": _text(_first(raw, "index_name", "indexName", "index")),
            "search_type": _text(_first(raw, "search_type", "searchType", "queryType", "query_type", "searchMode")).lower(),
            "doc_count": _first(raw, "document_count", "documentCount", "top_k", "topK", "top", "retrievalCount"),
            "semantic_config": _first(raw, "semantic_configuration", "semanticConfiguration", "semanticConfigurationName"),
            "fields": fields,
            "samples": _items(raw, "sample_documents", "sampleDocuments"),
            "context": f"{s.description or ''}\n{s.instructions or ''}",
        }

    def run_schema_checks(self, s, md):
        out = [preview_acceptance("AIS-001", s)]
        ep, idx = md["endpoint"], md["index"]
        if not ep and not idx:
            out.append(make_finding("AIS-002", Status.UNKNOWN, "Search endpoint/index not exposed in the source configuration", source=s, confidence="LOW"))
        else:
            ok = bool(re.match(r"^https://[a-z0-9\-]+\.search\.(windows\.net|azure\.us|azure\.cn)/?$", ep, re.I)) and bool(idx)
            out.append(make_finding("AIS-002", Status.PASS if ok else Status.FAIL, f"endpoint='{ep or '—'}', index='{idx or '—'}' (format check; reachability needs live access)",
                                    source=s, confidence="MEDIUM"))
        st = md["search_type"]
        known = {"full-text", "fulltext", "simple", "full", "keyword", "hybrid", "semantic", "vector", "hybrid-semantic", "semantichybrid"}
        if not st:
            out.append(make_finding("AIS-007", Status.UNKNOWN, "Search mode not exposed", source=s, confidence="LOW"))
        else:
            warn = "semantic" in st and not md["semantic_config"]
            out.append(make_finding("AIS-007", Status.WARN if (warn or st not in known) else Status.PASS,
                                    f"Search mode '{st}'" + ("; no semantic configuration recorded" if warn else ""), source=s))
        dc = md["doc_count"]
        try:
            dci = int(dc)
            out.append(make_finding("AIS-008", Status.PASS if 3 <= dci <= 20 else Status.FAIL, f"Document count = {dci} (supported 3–20)", source=s))
        except (TypeError, ValueError):
            out.append(make_finding("AIS-008", Status.UNKNOWN, f"Document count not exposed ({dc!r})", source=s, confidence="LOW"))
        names = [_text(_first(f, "name")) for f in md["fields"]]
        if not names:
            out.append(make_finding("AIS-009", Status.UNKNOWN, "Index fields not exposed", source=s, confidence="LOW"))
        else:
            exact = [n for n in names if n in self.CITATION_FIELDS]
            ci = [n for n in names if n.lower() in {c.lower() for c in self.CITATION_FIELDS} and n not in exact]
            out.append(make_finding("AIS-009", Status.PASS if exact else Status.FAIL,
                                    f"Citation fields (exact, case-sensitive): {exact or 'none'}" + (f"; case mismatch: {ci}" if ci else ""), source=s))
        cues = [c for c, rx in (("content", r"\b(contains?|documents?|policies|procedures|manuals?)\b"), ("intended questions", r"\b(questions?|use (this|for))\b"),
                                ("key fields", r"\b(field|content|title|sourceUrl|url)\b"), ("boundaries", r"\b(do not|don't|not for|only for|boundar)")) if re.search(rx, md["context"], re.I)]
        out.append(make_finding("AIS-006", Status.PASS if len(cues) >= 3 else Status.WARN if cues else Status.FAIL, f"Source-context cues: {cues}", source=s, confidence="MEDIUM"))
        return out

    def run_configuration_checks(self, s, md):
        out = [
            manual_check("AIS-003", "Confirm role-based access control is enabled on the search service.", source=s),
            manual_check("AIS-004", "Confirm Search Index Data Reader and Search Index Data Contributor for the connecting identity.", source=s),
            manual_check("AIS-005", "Verify query-time access respects the asking user's identity and index permissions.", source=s),
            manual_check("AIS-015", "Verify expected documents/chunks are retrieved for ground-truth prompts (section 14).", source=s),
            manual_check("AIS-016", "Verify answer facts are supported by retrieved content and citations point to the right document.", source=s),
        ]
        structured = [x for x in SOURCES if x.source_family in ("SQL", "KQL", "SEMANTIC", "GRAPH", "ONTOLOGY")]
        if not structured:
            out.append(make_finding("AIS-017", Status.NOT_APPLICABLE, "No structured sources connected", source=s))
        else:
            t = (AGENT_PROFILE.instructions or "").lower()
            ok = s.display_name.lower() in t and bool(re.search(r"polic|document|procedure|narrative|unstructured|prose", t))
            out.append(make_finding("AIS-017", Status.PASS if ok else Status.WARN, f"Agent instructions route document questions to '{s.display_name}' = {ok}", source=s, confidence="MEDIUM"))
        return out

    def validate_examples(self, s, md):
        samples = md["samples"]
        if not samples:
            return [make_finding(r, Status.NOT_EVALUATED, "No sample documents available (add sample_documents to the manifest or sample the index)", source=s, confidence="LOW")
                    for r in ("AIS-010", "AIS-011", "AIS-012", "AIS-013", "AIS-014")]
        out = []
        cit = [f for f in self.CITATION_FIELDS if any(f in d for d in samples)]
        bad_cit = [i for i, d in enumerate(samples) if not any(re.match(r"^(https?://|/|[a-z]:\\|\w+/)", _text(d.get(f)), re.I) for f in cit)]
        out.append(make_finding("AIS-010", Status.PASS if cit and not bad_cit else Status.FAIL, f"{len(samples) - len(bad_cit)}/{len(samples)} samples have usable citation values", source=s))
        content_key = next((k for k in ("content", "chunk", "text", "body", "merged_content") if any(k in d for d in samples)), None)
        empty = [i for i, d in enumerate(samples) if not _text(d.get(content_key))] if content_key else list(range(len(samples)))
        out.append(make_finding("AIS-011", Status.PASS if not empty else Status.FAIL, f"Content field '{content_key}': {len(empty)} empty of {len(samples)}", source=s))
        trace = [i for i, d in enumerate(samples) if not any(k in d for k in ("parent_id", "parentId", "chunk_id", "chunkId", "page", "page_number", "section", "title"))]
        out.append(make_finding("AIS-012", Status.PASS if not trace else Status.WARN, f"{len(samples) - len(trace)}/{len(samples)} chunks traceable to document/section/page", source=s))
        texts = [_text(d.get(content_key)) for d in samples] if content_key else []
        dups = len(texts) - len(set(texts))
        lens = pd.Series([len(t) for t in texts]) if texts else pd.Series(dtype=float)
        outliers = int(((lens < 50) | (lens > lens.median() * 5)).sum()) if len(lens) else 0
        out.append(make_finding("AIS-013", Status.PASS if not dups and not outliers else Status.WARN, f"Duplicates={dups}; size outliers={outliers}", source=s))
        ts_key = next((k for k in ("lastModified", "last_modified", "metadata_storage_last_modified", "updated", "indexedAt") if any(k in d for d in samples)), None)
        if ts_key:
            latest = pd.to_datetime([d.get(ts_key) for d in samples], utc=True, errors="coerce").max()
            age_h = (datetime.now(timezone.utc) - latest.to_pydatetime()).total_seconds() / 3600 if pd.notna(latest) else None
            out.append(make_finding("AIS-014", Status.NOT_EVALUATED if age_h is None else Status.PASS if age_h <= FRESHNESS_MAX_AGE_HOURS else Status.WARN,
                                    f"Latest '{ts_key}' age = {age_h if age_h is None else round(age_h, 1)} h (threshold {FRESHNESS_MAX_AGE_HOURS} h)", source=s))
        else:
            out.append(make_finding("AIS-014", Status.NOT_EVALUATED, "No indexed/modified timestamp on samples", source=s, confidence="LOW"))
        return out


ADAPTERS["AZURE_AI_SEARCH"] = AzureAiSearchAdapter()
AIS_FINDINGS = run_family("AI_SEARCH", "ai_search", "AZURE AI SEARCH (PREVIEW) — AIS-001 … AIS-017")

---
# 1️⃣2️⃣ CROSS-SOURCE ROUTING & CONSISTENCY

A multi-source agent can have healthy individual sources and still answer incorrectly when routing or shared definitions conflict. Most of these checks are `NOT_APPLICABLE` for single-source agents.

Provide a **routing matrix** (`ROUTING_MATRIX_PATH`) to make routing decisions explicit and testable:

```csv
topic,primary_source,secondary_source,reason,default_date,grain,metric_owner,clarification_rule
Revenue,SalesSemanticModel,SalesWarehouse,Certified measures,FiscalDate,Month,Finance,"Ask gross vs net when unspecified"
Delivery status,OperationsEventhouse,,Real-time events,EventTime,Event,Operations,
Policy interpretation,PolicySearchIndex,,Approved documents,PublishedDate,Document,Legal,
Supplier relationships,SupplierGraph,,Relationship traversal,AsOfDate,Entity,Procurement,
```

`DA-021` (source drift) is also rolled up here from the adapter resolution checks (`SQL-002`, `KQL-002`, `GRAPH-005`, `ONT-005`, `SM-025`).

In [ ]:
print("=" * 70)
print("CROSS-SOURCE ROUTING & CONSISTENCY — XSR-001 … XSR-012, DA-021")
print("=" * 70)

begin_section("cross")
_X = []


def load_optional_csv(path, required=()):
    if not str(path).strip():
        return None, "not provided"
    try:
        df = pd.read_csv(path)
    except Exception as e:
        return None, f"could not read: {short(e, 120)}"
    df.columns = [c.strip() for c in df.columns]
    missing = [c for c in required if c not in df.columns]
    return df, (f"missing columns {missing}" if missing else "ok")


ROUTING_MATRIX, _rm_note = load_optional_csv(ROUTING_MATRIX_PATH, ("topic", "primary_source"))
multi = len(SOURCES) > 1 and not AGENT_PROFILE.standalone_semantic
fams = {s.source_family for s in SOURCES}
T = (AGENT_PROFILE.instructions or "").lower()
names_lower = {s.display_name.lower(): s for s in SOURCES}


def _na(rule_id, why="Single-source agent"):
    return make_finding(rule_id, Status.NOT_APPLICABLE, why)


if not multi:
    for rid in [f"XSR-{i:03d}" for i in range(1, 13)]:
        _X.append(_na(rid, "Standalone semantic-model mode" if AGENT_PROFILE.standalone_semantic else "Single-source agent"))
else:
    # XSR-001 / XSR-002 routing matrix
    if ROUTING_MATRIX is None or _rm_note != "ok":
        _X.append(manual_check("XSR-001", f"Provide ROUTING_MATRIX_PATH mapping topics to candidate sources (routing matrix {_rm_note})."))
        _X.append(manual_check("XSR-002", "Name one authoritative source (or reconciliation rule) per topic in the routing matrix."))
    else:
        display(ROUTING_MATRIX)
        prim = {str(x).strip().lower() for x in ROUTING_MATRIX["primary_source"].dropna()}
        secs = {str(x).strip().lower() for x in ROUTING_MATRIX.get("secondary_source", pd.Series(dtype=str)).dropna()}
        uncovered = [s.display_name for s in SOURCES if s.display_name.lower() not in prim | secs]
        unknown = sorted((prim | secs) - set(names_lower))
        _X.append(make_finding("XSR-001", Status.PASS if not uncovered and not unknown else Status.WARN,
                               f"{len(ROUTING_MATRIX)} topics; sources without a topic: {uncovered or 'none'}; unknown source names: {unknown or 'none'}"))
        dup_topics = ROUTING_MATRIX["topic"].str.lower().value_counts()
        dup_topics = dup_topics[dup_topics > 1].index.tolist()
        needs_rule = ROUTING_MATRIX[ROUTING_MATRIX.get("secondary_source", pd.Series(index=ROUTING_MATRIX.index, dtype=str)).notna()]
        no_reason = needs_rule[needs_rule.get("reason", pd.Series(index=needs_rule.index, dtype=str)).isna()]["topic"].tolist() if not needs_rule.empty else []
        _X.append(make_finding("XSR-002", Status.PASS if not dup_topics and not no_reason else Status.WARN,
                               f"Topics with multiple primaries: {dup_topics or 'none'}; overlaps without a reason: {no_reason or 'none'}"))

    # XSR-003 metric-definition consistency (shared metric vocabulary across sources)
    metric_words = ["revenue", "sales", "margin", "profit", "orders", "customers", "cost", "units", "quantity", "discount", "returns"]
    per_src = {s.display_name: {w for w in metric_words if re.search(rf"\b{w}\b", f"{s.instructions or ''} {s.description or ''} "
               + " ".join(o['name'] for o in s.selected_objects), re.I)} for s in SOURCES}
    shared = sorted({w for w in metric_words if sum(w in v for v in per_src.values()) > 1})
    if shared and not (ROUTING_MATRIX is not None and "metric_owner" in ROUTING_MATRIX.columns and ROUTING_MATRIX["metric_owner"].notna().all()):
        _X.append(manual_check("XSR-003", f"Metrics appearing in more than one source: {shared} — confirm formula, grain, currency, status, and date logic agree."))
    else:
        _X.append(make_finding("XSR-003", Status.PASS, f"Shared metric vocabulary: {shared or 'none'}; metric owners recorded in routing matrix", confidence="MEDIUM"))

    # XSR-004 time semantics
    tz = {s.display_name: ("utc" if re.search(r"\butc\b", s.instructions or "", re.I) else "local" if re.search(r"local time|time ?zone", s.instructions or "", re.I) else None)
          for s in SOURCES if s.source_family in ("SQL", "KQL")}
    undocumented = [k for k, v in tz.items() if v is None]
    conflict = len({v for v in tz.values() if v}) > 1
    _X.append(make_finding("XSR-004", Status.NOT_APPLICABLE if not tz else Status.FAIL if conflict else Status.WARN if undocumented else Status.PASS,
                           f"Timezone per source: {tz}; conflicting={conflict}", confidence="MEDIUM"))

    # XSR-005 entity-key consistency
    keycols = {s.display_name: {c["name"].lower() for o in s.selected_objects for c in o.get("columns", []) if re.search(r"(id|key|code)$", c.get("name", ""), re.I)} for s in SOURCES}
    with_cols = {k: v for k, v in keycols.items() if v}
    if len(with_cols) < 2:
        _X.append(manual_check("XSR-005", "Column metadata not available for two or more sources — confirm shared entity identifiers map across sources."))
    else:
        common = set.intersection(*with_cols.values())
        _X.append(make_finding("XSR-005", Status.PASS if common else Status.WARN, f"Key columns shared by all sources with metadata: {sorted(common)[:10] or 'none'}", confidence="LOW"))

    # XSR-006 structured vs unstructured / XSR-007 real-time vs historical / XSR-008 graph-ontology routing
    def _route(rule_id, applies, target_fams, cue):
        if not applies:
            return _na(rule_id, "Required source combination not present")
        targets = [s.display_name for s in SOURCES if s.source_family in target_fams]
        ok = any(t.lower() in T for t in targets) and bool(re.search(cue, T))
        return make_finding(rule_id, Status.PASS if ok else Status.WARN if AGENT_PROFILE.instructions else Status.FAIL,
                            f"Agent instructions route to {targets} with routing cue = {ok}", confidence="MEDIUM")
    structured = fams & {"SQL", "KQL", "SEMANTIC", "GRAPH", "ONTOLOGY"}
    _X.append(_route("XSR-006", "AI_SEARCH" in fams and bool(structured), {"AI_SEARCH"}, r"polic|document|procedure|unstructured|narrative"))
    _X.append(_route("XSR-007", "KQL" in fams and bool(fams & {"SQL", "SEMANTIC"}), {"KQL"}, r"real[- ]time|latest|live|stream|events?|last \d+ (minutes|hours)"))
    _X.append(_route("XSR-008", bool(fams & {"GRAPH", "ONTOLOGY"}) and len(fams) > 1, {"GRAPH", "ONTOLOGY"}, r"relationship|travers|connected|path|network|hierarch"))

    # XSR-009 ambiguity handling
    amb = bool(re.search(r"clarif|ask (the user|a follow[- ]up|for)|if (the question is )?(ambiguous|unclear)", T))
    _X.append(make_finding("XSR-009", Status.PASS if amb else Status.FAIL if AGENT_PROFILE.instructions else Status.UNKNOWN,
                           f"Clarification rule present in agent instructions = {amb}"))

    # XSR-010 cross-source reconciliation — needs multi-source ground-truth cases
    _ev, _ = load_optional_csv(EVALUATION_DATASET_PATH, ("question", "expected_answer"))
    multi_cases = 0
    if _ev is not None and "expected_source" in _ev.columns:
        multi_cases = int(_ev["expected_source"].astype(str).str.contains(r"[|+;]").sum())
    _X.append(make_finding("XSR-010", Status.PASS, f"{multi_cases} multi-source ground-truth case(s)") if multi_cases else
              manual_check("XSR-010", "Add ground-truth cases that need two sources (expected_source 'A|B') and verify units, keys, and time alignment."))
    _X.append(manual_check("XSR-011", "Verify a persona lacking one source receives a clear authorization outcome (see section 15)."))
    _X.append(manual_check("XSR-012", "Verify the agent discloses unavailable sources and does not silently substitute a non-authoritative one."))

# DA-021 source refresh / drift roll-up from adapter resolution checks
_res_rules = {"SQL-002", "KQL-002", "GRAPH-005", "ONT-005", "SM-025"}
_res = [f for f in all_findings() if f.rule_id in _res_rules and f.score_possible > 0]
if AGENT_PROFILE.standalone_semantic:
    _X.append(_na("DA-021", "Standalone semantic-model mode"))
elif not _res:
    _X.append(make_finding("DA-021", Status.UNKNOWN, "No source-resolution evidence collected", confidence="LOW"))
else:
    fails = [f"{f.source_name}:{f.rule_id}" for f in _res if f.status == Status.FAIL]
    evaluated = [f for f in _res if f.status in EVALUATED_STATUSES]
    _X.append(make_finding("DA-021", Status.FAIL if fails else Status.PASS if evaluated else Status.NOT_EVALUATED,
                           f"Resolution checks evaluated {len(evaluated)}/{len(_res)}; drifted: {fails or 'none'}"))

record("cross", _X)
show_findings(_X, title="\nCross-source findings:")

---
# 1️⃣3️⃣ EXAMPLE-QUERY VALIDATION SUMMARY

| Source | Example queries (current dedicated documentation) |
|---|---|
| Lakehouse · Warehouse · Eventhouse KQL Database | Supported |
| Power BI semantic model · Ontology | **Not supported** |
| Fabric SQL Database · Mirrored Database · Graph Model · Azure AI Search | Not explicitly listed. Missing examples are **never failed**. |

**Quality rules applied to every source that exposes examples:** unique question and matching query, validation against the selected schema, question literals mapped to predicates, no duplicate intent or conflicting logic, read-only statements, and bounded detail queries. Also verify in **run steps** which examples were actually retrieved. Current documentation disagrees on whether the **top 3** or **typically the top 4** examples reach the query tool, so the analyzer doesn't enforce a fixed count. Re-run validation after every schema change.

In [ ]:
print("=" * 70)
print("EXAMPLE-QUERY VALIDATION SUMMARY")
print("=" * 70)

_ex_rules = {"SQL": ["SQL-012", "SQL-013", "SQL-014", "SQL-015", "SQL-016"], "KQL": ["KQL-006", "KQL-007", "KQL-008", "KQL-009", "KQL-014", "KQL-015"],
             "GRAPH": ["GRAPH-011", "GRAPH-012", "GRAPH-013", "GRAPH-014"]}
_fdf = findings_df()
rows = []
for s in SOURCES:
    rules = _ex_rules.get(s.source_family, [])
    sub = _fdf[(_fdf["source_id"] == s.source_id) & (_fdf["rule_id"].isin(rules))] if not _fdf.empty else _fdf
    ro = _fdf[(_fdf["source_id"] == s.source_id) & (_fdf["rule_id"] == "DA-014") & (_fdf["status"] == "FAIL")] if not _fdf.empty else _fdf
    rows.append({
        "source": s.display_name, "kind": s.source_kind, "language": query_language_for(s),
        "support": {True: "supported", False: "not supported", None: "not listed"}[capability(s, "example_queries")],
        "examples": len(s.example_queries),
        "lint/validation issues": int(((sub["score_possible"] == 0) & sub["status"].isin(["WARN", "FAIL"])).sum()) if len(sub) else 0,
        "read-only violations": len(ro),
        "evaluator": "evaluate_few_shots (SQL)" if s.source_kind in ("LAKEHOUSE", "WAREHOUSE") else "static lint + execution + E2E",
    })
if rows:
    display(pd.DataFrame(rows))
else:
    print("No sources discovered.")
print("\nℹ️  Verify retrieved examples in run steps (DA-024); do not rely on a fixed top-3/top-4 retrieval count.")

---
# 1️⃣4️⃣ END-TO-END EVALUATION

Use the Fabric evaluation SDK (`fabric.dataagent.evaluation`) against a **published `sandbox` or `production`** stage. Keep the summary and step-level tables so you can compare results across commits and runtime changes. Only `question` and `expected_answer` are required by the SDK. The other columns are analyzer metadata for routing and diagnostics.

```csv
case_id,question,expected_answer,expected_source,expected_language,expected_objects,persona,category,criticality
Q001,"...","...",SalesWarehouse,SQL,"dbo.FactSales|dbo.DimDate",FinanceReader,aggregation,critical
Q002,"...","...",OperationsEventhouse,KQL,"DeliveryEvents",OperationsReader,realtime,critical
Q003,"...","...",PolicySearchIndex,SEARCH,"Policy-123",Employee,document,recommended
```

**Required test suites (`category`):** happy path · routing · ambiguity · boundary · security · freshness · complex query · regression · adversarial phrasing · performance.

**Failure taxonomy:** `ROUTING_ERROR` · `SCHEMA_SELECTION_ERROR` · `SCHEMA_DRIFT` · `METADATA_AMBIGUITY` · `INSTRUCTION_GAP` · `INSTRUCTION_CONFLICT` · `EXAMPLE_NOT_RETRIEVED` · `EXAMPLE_CONFLICT` · `QUERY_SYNTAX_ERROR` · `QUERY_SEMANTIC_ERROR` · `WRONG_FILTER_VALUE` · `WRONG_DATE_LOGIC` · `WRONG_GRAIN` · `JOIN_FANOUT` · `PERMISSION_ERROR` · `SOURCE_UNAVAILABLE` · `STALE_DATA` · `RETRIEVAL_MISS` · `CITATION_MISSING` · `CITATION_WRONG` · `ANSWER_FORMAT_ERROR` · `LATENCY_THRESHOLD_EXCEEDED` · `PREVIEW_REGRESSION` · `UNKNOWN`

> ℹ️ If you query agents from code with the Fabric OpenAI client, migrate from the Assistants API to the **Responses API** (August 2026). Management and configuration code isn't affected.

In [ ]:
print("=" * 70)
print("END-TO-END EVALUATION — DA-022 · DA-023 · DA-024")
print("=" * 70)

begin_section("evaluation")
_E = []
FAILURE_TAXONOMY = ["ROUTING_ERROR", "SCHEMA_SELECTION_ERROR", "SCHEMA_DRIFT", "METADATA_AMBIGUITY", "INSTRUCTION_GAP",
                    "INSTRUCTION_CONFLICT", "EXAMPLE_NOT_RETRIEVED", "EXAMPLE_CONFLICT", "QUERY_SYNTAX_ERROR", "QUERY_SEMANTIC_ERROR",
                    "WRONG_FILTER_VALUE", "WRONG_DATE_LOGIC", "WRONG_GRAIN", "JOIN_FANOUT", "PERMISSION_ERROR", "SOURCE_UNAVAILABLE",
                    "STALE_DATA", "RETRIEVAL_MISS", "CITATION_MISSING", "CITATION_WRONG", "ANSWER_FORMAT_ERROR",
                    "LATENCY_THRESHOLD_EXCEEDED", "PREVIEW_REGRESSION", "UNKNOWN"]
TEST_SUITES = ["happy path", "routing", "ambiguity", "boundary", "security", "freshness", "complex", "regression", "adversarial", "performance"]
EVAL_DATASET, _ev_note = load_optional_csv(EVALUATION_DATASET_PATH, ("question", "expected_answer"))
EVAL_SUMMARY, EVAL_DETAILS = None, None

# DA-022 ground-truth set
if AGENT_PROFILE.standalone_semantic:
    _E.append(make_finding("DA-022", Status.NOT_APPLICABLE, "Standalone semantic-model mode — no data agent to evaluate"))
elif EVAL_DATASET is None or _ev_note != "ok":
    _E.append(make_finding("DA-022", Status.FAIL, f"Ground-truth dataset {_ev_note} (EVALUATION_DATASET_PATH)"))
else:
    n = len(EVAL_DATASET)
    extras = [c for c in ("expected_source", "persona", "category", "criticality") if c in EVAL_DATASET.columns]
    routed = set(EVAL_DATASET["expected_source"].dropna().astype(str).str.split(r"[|+;]").explode().str.strip().str.lower()) if "expected_source" in EVAL_DATASET.columns else set()
    unrouted = [s.display_name for s in SOURCES if s.display_name.lower() not in routed] if routed else [s.display_name for s in SOURCES]
    cats = set(EVAL_DATASET["category"].dropna().astype(str).str.lower()) if "category" in EVAL_DATASET.columns else set()
    suites = [t for t in TEST_SUITES if any(t.split()[0] in c for c in cats)]
    st = Status.PASS if n >= 10 and "expected_source" in extras and not unrouted else Status.WARN
    _E.append(make_finding("DA-022", st, f"{n} cases; metadata columns {extras}; sources without cases: {unrouted or 'none'}; suites covered: {suites or 'unknown'}",
                           ratio=0.7 if st == Status.WARN else None))
    bad_q = [q for q in EVAL_DATASET.get("expected_query", pd.Series(dtype=str)).dropna() if read_only_violations(q, "SQL") or read_only_violations(q, "KQL")]
    if bad_q:
        _E.append(detail("DA-014", Status.FAIL, f"{len(bad_q)} evaluation query/queries contain data-changing statements", severity="BLOCKER"))

def _default_lakehouse_attached() -> bool:
    """The evaluation SDK reads/writes its results table in the notebook's default lakehouse."""
    try:
        from fabric.dataagent._fabric_runtime import get_fabric_context
        ctx = get_fabric_context() or {}
        return bool(ctx.get("fs.defaultFS") and ctx.get("trident.lakehouse.id"))
    except Exception:
        return os.path.isdir("/lakehouse/default/Tables")


_eval_lakehouse = _default_lakehouse_attached()

# DA-023 end-to-end evaluation (SDK; published stage)
if AGENT_PROFILE.standalone_semantic:
    _E.append(make_finding("DA-023", Status.NOT_APPLICABLE, "Standalone semantic-model mode — test in Copilot / Power BI Desktop (see 14.2)"))
else:
    try:
        from fabric.dataagent.evaluation import evaluate_data_agent, get_evaluation_summary, get_evaluation_details
        _eval_sdk = True
    except Exception as e:
        _eval_sdk, _eval_err = False, short(e, 160)
    if RUN_EVALUATION and _eval_sdk and not _eval_lakehouse:
        _E.append(make_finding("DA-023", Status.NOT_EVALUATED,
                               "RUN_EVALUATION = True but no default lakehouse is attached; the evaluation SDK stores results there",
                               confidence="LOW"))
    elif RUN_EVALUATION and _eval_sdk and EVAL_DATASET is not None and _ev_note == "ok":
        try:
            eval_id = evaluate_data_agent(EVAL_DATASET[["question", "expected_answer"]], AGENT_PROFILE.name,
                                          workspace_name=WORKSPACE_ID_OR_NAME or None, table_name=EVALUATION_TABLE_NAME,
                                          data_agent_stage=DATA_AGENT_STAGE)
            EVAL_SUMMARY = get_evaluation_summary(EVALUATION_TABLE_NAME)
            EVAL_DETAILS = get_evaluation_details(eval_id, EVALUATION_TABLE_NAME, get_all_rows=True)
        except Exception as e:
            _E.append(make_finding("DA-023", Status.NOT_EVALUATED, f"evaluate_data_agent failed: {short(e, 200)}", confidence="LOW"))
    elif _eval_sdk and _eval_lakehouse and str(EVALUATION_TABLE_NAME).strip():
        try:
            EVAL_SUMMARY = get_evaluation_summary(EVALUATION_TABLE_NAME)   # read-only: latest stored results
        except Exception:
            EVAL_SUMMARY = None

    if EVAL_DETAILS is not None and isinstance(EVAL_DETAILS, pd.DataFrame) and not EVAL_DETAILS.empty:
        jcol = next((c for c in EVAL_DETAILS.columns if re.search(r"judg|correct|evaluation_result|is_true|result", c, re.I)), None)
        if jcol:
            verdict = EVAL_DETAILS[jcol].astype(str).str.lower()
            passed = verdict.isin(["true", "yes", "correct", "1", "pass"]).sum()
            rate = passed / len(verdict)
            crit_fail = 0
            if "criticality" in EVAL_DATASET.columns and "question" in EVAL_DETAILS.columns:
                crit_q = set(EVAL_DATASET.loc[EVAL_DATASET["criticality"].astype(str).str.lower() == "critical", "question"])
                crit_fail = int(((EVAL_DETAILS["question"].isin(crit_q)) & ~verdict.isin(["true", "yes", "correct", "1", "pass"])).sum())
            st = Status.FAIL if crit_fail else Status.PASS if rate >= 0.9 else Status.WARN if rate >= 0.7 else Status.FAIL
            _E.append(make_finding("DA-023", st, f"{passed}/{len(verdict)} correct ({rate:.0%}); critical failures: {crit_fail}; stage {DATA_AGENT_STAGE}", ratio=rate))
        else:
            _E.append(make_finding("DA-023", Status.NOT_EVALUATED, f"Evaluation ran; judgement column not recognized ({list(EVAL_DETAILS.columns)[:8]})", confidence="LOW"))
        display(EVAL_DETAILS.head(25))
    elif not any(f.rule_id == "DA-023" for f in _E):
        if EVAL_SUMMARY is not None and isinstance(EVAL_SUMMARY, pd.DataFrame) and not EVAL_SUMMARY.empty:
            print("📋 Latest stored evaluation summary:")
            display(EVAL_SUMMARY.tail(5))
            _E.append(manual_check("DA-023", f"Stored results found in '{EVALUATION_TABLE_NAME}' — confirm they reflect the current published version, then attest."))
        else:
            why = (f"evaluation SDK unavailable ({_eval_err})" if not _eval_sdk
                   else "RUN_EVALUATION = False" + ("" if _eval_lakehouse else "; no default lakehouse attached, so stored results were not read"))
            _E.append(manual_check("DA-023", f"No evaluation run ({why}). Evaluate the published stage and classify failures."))

_E.append(manual_check("DA-024", "Review run steps: selected source, generated SQL/DAX/KQL/GQL, retrieved examples/topics, intermediate steps."))
record("evaluation", _E)
show_findings(_E, title="\nEvaluation findings:")
print(f"\n🧭 Classify failures with the taxonomy: {', '.join(FAILURE_TAXONOMY[:8])}, …")

## 14.2 🧪 Semantic-model testing workflow (guidance)

This checklist is kept from v2.2.2. Scoring now comes from `DA-022` (ground-truth set), `DA-023` (end-to-end evaluation), and `DA-024` (run-step review) instead of an automatic zero.

In [ ]:
print("=" * 70)
print("14.2 SEMANTIC-MODEL TESTING WORKFLOW (GUIDANCE — scored by DA-022/DA-023/DA-024)")
print("=" * 70)

print("🚨 CRITICAL: Test systematically BEFORE adding AI Instructions.\n")
print("⏱️  IMPORTANT — DON'T EXPECT IMMEDIATE INDEXING:")
print("   After publishing a model, the text index (which helps Copilot understand")
print("   string values like 'West' or 'Apple') may take a few minutes to build.")
print("   If Copilot fails immediately after publish, WAIT ~5 MINUTES and try again.")
print()
print("✅ TESTING WORKFLOW:")
print("\n   1️⃣  BASELINE TESTING (before AI Instructions)")
print("      • Ask 10-20 common questions")
print("      • Review DAX query in each response")
print("      • Verify data accuracy")
print("      • Identify which questions fail or return incorrect results")
print("      → Use results to guide Prep for AI configuration tweaks")
print("\n   2️⃣  VERIFIED ANSWERS TESTING")
print("      • Test each trigger question individually")
print("      • Verify exact matches return the verified answer")
print("      • Test semantic variations")
print("      • Confirm filters work as expected")
print("\n   3️⃣  AI DATA SCHEMA VALIDATION")
print("      • Test questions using objects IN the AI Data Schema → should work")
print("      • Test questions using objects NOT in schema → should fail gracefully")
print("      → Confirms schema is correctly configured")
print("\n   4️⃣  ISOLATE CAPABILITIES (Power BI Desktop)")
print("      • Use the 'Select Skills' dropdown to force Copilot to test ONE capability")
print("        at a time (e.g., only 'Create Report Pages', only 'Summarize').")
print("      • Helps pinpoint whether failures are model, instruction, or skill issues.")
print("\n   5️⃣  USE DAX QUERY VIEW FOR DIAGNOSIS")
print("      • Open DAX Query View in Power BI Desktop and use Copilot there to:")
print("        - Generate test queries against your AI Data Schema measures")
print("        - EXPLAIN complex DAX (e.g., 'Explain how KEEPFILTERS works in this measure')")
print("      • Great for validating measure logic before exposing to Data Agent.")
print("\n   6️⃣  PERFORMANCE TESTING")
print("      • Measure response times for complex queries")
print("      • If slow: analyze DAX performance, optimize model, simplify instructions")
print("      • Use Performance Analyzer in Power BI Desktop")
print("\n   7️⃣  AUTOMATED EVALUATION (Python SDK)")
print("      • Create ground truth Q&A dataset")
print("      • Run batch evaluation")
print("      • Track accuracy metrics over time")
print("\n      📚 SDK: https://learn.microsoft.com/fabric/data-science/fabric-data-agent-sdk")
print("      📚 Evaluation: https://learn.microsoft.com/fabric/data-science/evaluate-data-agent")
print("\n   8️⃣  DIAGNOSTICS REVIEW")
print("      • Download diagnostics logs from failed/incorrect responses")
print("      • Identify which stage caused the issue:")
print("        - Intent recognition")
print("        - DAX generation")
print("        - Data retrieval")
print("      → Guides which configuration needs adjustment")
print("\n      📚 Diagnostics: https://learn.microsoft.com/fabric/data-science/evaluate-data-agent#diagnostics-button")
print("\n   9️⃣  ITERATION")
print("      • Based on testing results, adjust:")
print("        - AI Data Schema (add/remove objects)")
print("        - AI Instructions (clarify terminology, add Negative Scope)")
print("        - Verified Answers (add more triggers)")
print("      • Re-test after each change (allow ~5 min for re-indexing)")
print("\n💡 DEPLOYMENT:")
print("   [ ] Use Git integration for version control")
print("   [ ] Use Deployment Pipelines for dev → test → prod")
print("   [ ] Add Data Agent description before publishing")
print("   [ ] For M365 Copilot: add publishing instructions")
print("\n      📚 Git: https://learn.microsoft.com/fabric/data-science/data-agent-source-control")
print("      📚 M365: https://learn.microsoft.com/fabric/data-science/data-agent-microsoft-365-copilot")


---
# 1️⃣5️⃣ SECURITY, PERMISSIONS & GOVERNANCE

**Analyzer behavior**

- Test with the creator identity **and** at least one consumer persona. Don't infer access from workspace membership alone.
- Classify each failure as schema, query, authentication, authorization, RLS/CLS, outbound protection, or source availability.
- Don't request broader permissions as the first fix. Verify least privilege instead.
- Never log bearer tokens, connection secrets, raw sensitive values, or full query results. Exports are redacted.
- For Azure AI Search, validate role-based access and user-identity propagation. For ontology, validate the ontology item **and** every bound source.
- Treat cross-geo AI settings and outbound-access rules as **release gates** when they apply.

Provide persona test results in `PERSONA_TEST_RESULTS_PATH`:

```csv
persona,source_name,expected_access,observed_access,notes
FinanceReader,SalesWarehouse,allow,allow,
FinanceReader,OperationsEventhouse,deny,deny,"Clear authorization message"
RegionalManager,SalesSemanticModel,allow,allow,"RLS: only own region returned"
```

In [ ]:
print("=" * 70)
print("SECURITY & PERMISSIONS — DA-015 · DA-016 · DA-017 · DA-018")
print("=" * 70)

begin_section("security")
_S = []
MIN_PERMISSIONS = {
    "SEMANTIC_MODEL": "Read (workspace access not required for data-agent interaction)",
    "LAKEHOUSE": "Read on the lakehouse item + table access if enforced",
    "WAREHOUSE": "Read / SELECT on relevant tables",
    "FABRIC_SQL_DATABASE": "Read/query access sufficient for metadata and data retrieval",
    "MIRRORED_DATABASE": "Read/query access sufficient for metadata and data retrieval",
    "KQL_DATABASE": "Reader role on the database",
    "ONTOLOGY": "Read on the ontology + Read on each bound semantic model / lakehouse / KQL database",
    "GRAPH_MODEL": "Read on the graph item and underlying data",
    "AZURE_AI_SEARCH": "Role-based access (Search Index Data Reader / Contributor for the connecting identity)",
}
if SOURCES:
    display(pd.DataFrame([{"source": s.display_name, "kind": s.source_kind, "minimum documented permission": MIN_PERMISSIONS.get(s.source_kind, "Query/read-level access")}
                          for s in SOURCES]))

PERSONA_RESULTS, _p_note = load_optional_csv(PERSONA_TEST_RESULTS_PATH, ("persona", "source_name", "expected_access", "observed_access"))
if PERSONA_RESULTS is None or _p_note != "ok":
    _S.append(manual_check("DA-015", f"Verify a consumer identity reads required metadata/data without elevated permissions (persona results {_p_note})."))
    _S.append(manual_check("DA-016", "Test RLS, CLS, source ACLs, and user-identity behavior with representative personas."))
else:
    display(PERSONA_RESULTS)
    pr = PERSONA_RESULTS.assign(exp=PERSONA_RESULTS["expected_access"].astype(str).str.lower().str.strip(),
                                obs=PERSONA_RESULTS["observed_access"].astype(str).str.lower().str.strip())
    over = pr[(pr.exp == "deny") & (pr.obs == "allow")]
    under = pr[(pr.exp == "allow") & (pr.obs != "allow")]
    for _, r in over.iterrows():
        _S.append(detail("DA-015", Status.FAIL, f"{r.persona} unexpectedly allowed on {r.source_name}", object_path=f"{r.persona}@{r.source_name}", severity="BLOCKER"))
    for _, r in under.iterrows():
        _S.append(detail("DA-015", Status.FAIL, f"{r.persona} denied/errored on {r.source_name} (observed '{r.obs}')", object_path=f"{r.persona}@{r.source_name}"))
    untested = [s.display_name for s in SOURCES if s.display_name.lower() not in set(pr["source_name"].astype(str).str.lower())]
    _S.append(make_finding("DA-015", Status.FAIL if len(over) or len(under) else Status.WARN if untested else Status.PASS,
                           f"{len(pr)} persona tests; over-privileged={len(over)}; access gaps={len(under)}; sources untested: {untested or 'none'}"))
    personas = pr["persona"].nunique()
    _S.append(make_finding("DA-016", Status.PASS if personas >= 2 else Status.WARN, f"{personas} persona(s) tested; deny cases={int((pr.exp == 'deny').sum())}"))
    if len(SOURCES) > 1 and (pr.exp == "deny").any() and not len(over):
        _FINDINGS_BY_SECTION["cross"] = [f for f in _FINDINGS_BY_SECTION.get("cross", []) if f.rule_id != "XSR-011"]
        record("cross", make_finding("XSR-011", Status.PASS, "Deny personas observed as denied (verify the agent's message is clear)", confidence="MEDIUM"))

_S.append(manual_check("DA-017", "Confirm data agent, Copilot/Azure OpenAI, and cross-geo processing/storing tenant settings are enabled and approved."))
_ext = [s.display_name for s in SOURCES if s.source_kind in ("MIRRORED_DATABASE", "AZURE_AI_SEARCH")]
_S.append(manual_check("DA-018", f"Confirm workspace outbound data-connection rules allow required external connections: {_ext}.") if _ext else
          make_finding("DA-018", Status.NOT_APPLICABLE, "No external connections detected (mirrored / Azure AI Search)"))
record("security", _S)
show_findings(_S, title="\nSecurity findings:")

---
# 1️⃣6️⃣ ALM, DEPLOYMENT & OPERATIONS

- **Draft vs. published:** consumers use the **published** version. Draft changes don't reach production until you re-publish. A published agent keeps the runtime it was published with until it's re-published.
- **Source control (preview):** Fabric Git integration stores the agent under `files/config/` as `data_agent.json`, `publish_info.json`, `draft/`, and `published/`. Each source folder holds `datasource.json` and `fewshots.json`. Make changes in `draft/` and don't edit `published/` directly.
- **Deployment pipelines:** promote dev → test → prod and review the deployment plan before applying it.
- **Operations:** record the owner, review cadence, incident path, and change approvers. Monitor capacity and latency before rollout.

> 🔄 **Updated guidance:** v2.x said data agent instructions had "no Git story." Data agent configuration can now be source-controlled (preview), so review it in pull requests alongside the semantic model's TMDL.

In [ ]:
print("=" * 70)
print("ALM, DEPLOYMENT & OPERATIONS — DA-019 · DA-020 · DA-025 · DA-026")
print("=" * 70)

begin_section("alm")
_A = []
P = AGENT_PROFILE
if P.standalone_semantic:
    for rid in ("DA-019", "DA-020"):
        _A.append(make_finding(rid, Status.NOT_APPLICABLE, "Standalone semantic-model mode — use PBIP/TMDL + deployment pipelines for the model"))
else:
    snap = P.published_snapshot
    if snap is None:
        _A.append(make_finding("DA-019", Status.UNKNOWN, "Other-stage (draft/published) configuration not available in this collection mode", confidence="LOW"))
    elif snap.get("error"):
        _A.append(make_finding("DA-019", Status.WARN, f"Could not read '{snap.get('stage')}' stage — the agent may not be published: {snap['error']}", confidence="LOW"))
    else:
        cur_names = sorted(s.display_name for s in SOURCES)
        diffs = []
        if _text(snap.get("instructions")) != _text(P.instructions):
            diffs.append("agent instructions")
        if snap.get("source_names") is not None and sorted(snap.get("source_names", [])) != cur_names:
            diffs.append(f"sources {snap.get('source_names')} vs {cur_names}")
        if snap.get("runtime") not in (None, "UNKNOWN") and P.runtime != "UNKNOWN" and snap.get("runtime") != P.runtime:
            diffs.append(f"runtime {snap.get('runtime')} vs {P.runtime}")
        st = Status.PASS if not diffs else Status.WARN
        _A.append(make_finding("DA-019", st, f"Compared {P.stage} with '{snap.get('stage')}': " + (f"differences in {diffs} — publish only after evaluation" if diffs else "no differences")))

    if P.collection_mode == "GIT_EXPORT":
        _A.append(make_finding("DA-020", Status.PASS, f"Analyzed from the Git item definition ({P.metadata.get('config_dir')})"))
    else:
        try:
            ws_id = resolve_workspace_id(P.workspace) if P.workspace else (fabric.get_workspace_id() if fabric else None)
            conn = fabric_rest("GET", f"/v1/workspaces/{ws_id}/git/connection")
            state = _text(_first(conn, "gitConnectionState", "state"))
            prov = _first(conn, "gitProviderDetails", default={}) or {}
            _A.append(make_finding("DA-020", Status.PASS if state.lower().startswith("connected") else Status.WARN,
                                   f"Workspace Git state = {state or 'unknown'}; provider = {_text(_first(prov, 'gitProviderType'))}; branch = {_text(_first(prov, 'branchName'))}"))
        except Exception as e:
            _A.append(manual_check("DA-020", f"Confirm the workspace is Git-connected and promoted via deployment pipelines (Git status not readable: {short(e, 100)})."))

own = {k: P.metadata.get(k) for k in ("owner", "review_cadence", "incident_path", "change_approvers") if P.metadata.get(k)}
_A.append(make_finding("DA-025", Status.PASS if len(own) >= 3 else Status.WARN, f"USER_SUPPLIED ownership fields: {list(own)}", confidence="MEDIUM")
          if own else manual_check("DA-025", "Document owner, review cadence, incident path, and change approvers (manifest fields or MANUAL_ATTESTATIONS)."))
_A.append(manual_check("DA-026", "Monitor capacity consumption (Capacity Metrics app) and query latency before production rollout."))
record("alm", _A)
show_findings(_A, title="\nALM & operations findings:")

---
## 16.2 🛠️ POWER BI SERVICE & ADMIN (semantic model sources) · `SM-027`

**Tenant- and capacity-level configuration that affects every Data Agent and Copilot user.**

These items are typically configured by Power BI / Fabric admins, not model authors. They round out the readiness picture.

### 7.1 ✅ Mark Semantic Model as "Prepped for AI"
- Power BI Service → semantic model → **Settings → Prepped for AI**
- Signals to **Standalone Copilot** (Power BI Home page) that this model should be **prioritized in search results**
- Without this flag, your model competes with every other model in the tenant

### 7.2 🔍 Standalone Copilot — "Needle in a Haystack" Search
- Standalone Copilot (Power BI Home page) can find specific reports or visuals across the entire tenant
- Example query: *"Find the funnel chart about sales"*
- Only works if content is **well-indexed** and models are marked **Prepped for AI**
- Encourage users to use it instead of manually browsing workspaces

### 7.3 ⚙️ Copilot Capacities (Admin)
- **Problem:** "Noisy neighbors" — heavy AI usage can slow down ETL and reporting pipelines on a shared capacity
- **Solution:** Admins can designate a **dedicated Fabric capacity solely for Copilot usage**
- Mission-critical pipelines stay isolated from interactive AI workloads
- Configure: Fabric Admin Portal → Capacity settings → Copilot

### 7.4 💎 F64 Capacity Requirement (per [MS Learn](https://learn.microsoft.com/power-bi/create-reports/copilot-semantic-models))
- Copilot in Power BI Desktop requires the desktop client to be **configured to consume Copilot from an F64+ workspace**
- This requirement applies **regardless of which semantic model** you connect to (Pro / PPU / F64)
- Configure: Power BI Desktop → File → Options → Preview features → Copilot → choose F64 workspace

### 7.5 🏷️ Endorsement & Tags as a Copilot Readiness Gate
- Per MS Learn, treat **Copilot readiness as a criteria for endorsement** (Promoted / Certified)
- Use **Fabric tags** to label semantic models as ready for Copilot consumption
- Helps data consumers identify which models they can confidently query with Copilot
- Configure: Workspace → semantic model → Endorsement / Tags

### 7.6 🧪 Use a Dedicated Testing Copy
- If your production model is messy, **create a copy** specifically for testing Copilot features
- Refine the copy first, then port refinements back to production
- Avoids destabilizing existing reports while you iterate

### 7.7 📐 Confirm Q&A Is Enabled (linguistic schema prerequisite)
- Power BI Desktop → File → Options → Current file → **Data Load** → *"Turn on Q&A to ask natural language questions about your data"*
- Required to add a linguistic schema (synonyms, verbs, "Include in Q&A" toggle)
- Copilot reuses the Q&A linguistic schema as grounding context


In [ ]:
def _sm_check():
    print("=" * 70)
    print("CHECK 7: SERVICE & ADMIN BEST PRACTICES (MANUAL)")
    print("=" * 70)

    print("✅ TENANT / CAPACITY / SERVICE CHECKLIST:\n")
    print("   [ ] 7.1 Mark semantic model as 'Prepped for AI'")
    print("           Power BI Service → semantic model → Settings → Prepped for AI")
    print("           → Boosts visibility in Standalone Copilot search results.\n")
    print("   [ ] 7.2 Educate users on Standalone Copilot (Power BI Home page)")
    print("           → Use natural language to find reports/visuals tenant-wide")
    print("           → Example: 'Find the funnel chart about sales'\n")
    print("   [ ] 7.3 (Admin) Designate a dedicated Copilot capacity")
    print("           Fabric Admin Portal → Capacity settings → Copilot")
    print("           → Prevents 'noisy neighbor' impact on ETL / reporting pipelines.\n")
    print("   [ ] 7.4 Confirm F64+ capacity is configured for Copilot in Power BI Desktop")
    print("           Power BI Desktop → File → Options → Copilot → choose F64 workspace")
    print("           → Required regardless of which semantic model you connect to.\n")
    print("   [ ] 7.5 Use Endorsement (Promoted / Certified) and Fabric Tags as a")
    print("           Copilot readiness gate so consumers know which models to trust.\n")
    print("   [ ] 7.6 Use a dedicated TESTING COPY of production models")
    print("           → Refine the copy first, then port refinements back to production.\n")
    print("   [ ] 7.7 Confirm Q&A is enabled (linguistic schema prerequisite)")
    print("           Power BI Desktop → Options → Current file → Data Load →")
    print("           'Turn on Q&A to ask natural language questions about your data'\n")
    print("   [ ] 7.8 Confirm tenant-level Copilot settings allow your workload")
    print("           Fabric Admin Portal → Tenant settings → Copilot and Azure OpenAI Service\n")

    print("⏱️  REMINDER: After publishing changes, allow ~5 minutes for the text index")
    print("    to rebuild before testing Copilot — early failures are often just timing.\n")

    print("📚 Documentation:")
    print("   https://learn.microsoft.com/power-bi/create-reports/copilot-semantic-models")
    print("   https://learn.microsoft.com/fabric/admin/copilot-fabric-admin-settings")
    print("   https://learn.microsoft.com/power-bi/create-reports/copilot-introduction")
    print("   https://learn.microsoft.com/fabric/governance/endorsement-overview")

    sm_manual('SM-027', "Service/admin items not yet validated (Prepped for AI, endorsement/tags, Copilot capacity, F64, Q&A, tenant settings).",
              legacy_attested=prep_for_ai_configured)


run_semantic_check('SM-027', _sm_check)


---
# 1️⃣7️⃣ SCORECARD, RELEASE GATES & EXPORTS

The scorecard reports two numbers:

1. **Readiness:** points earned ÷ points possible, counting only **applicable, evaluated** checks (`PASS`/`WARN`/`FAIL`).
2. **Evaluation coverage:** evaluated applicable weight ÷ total applicable weight. `MANUAL`, `UNKNOWN`, and `NOT_EVALUATED` checks lower coverage, not readiness.

A high readiness score with low coverage **isn't production-ready**. **Release gates override the average:** a single open blocker makes the agent *Not ready*, whatever the score. Severity weights are configurable (`SEVERITY_WEIGHTS`) and are published in the report.

In [ ]:
DF = findings_df()
P = AGENT_PROFILE


def readiness_coverage(df):
    applicable = df[df["status"] != "NOT_APPLICABLE"]
    evaluated = applicable[applicable["status"].isin(["PASS", "WARN", "FAIL"])]
    ep, ap = evaluated["score_possible"].sum(), applicable["score_possible"].sum()
    readiness = evaluated["score_earned"].sum() / ep if ep > 0 else None
    coverage = ep / ap if ap > 0 else None
    return readiness, coverage


def _pct(x):
    return "N/A" if x is None else f"{x * 100:.0f}%"


def _scope_row(label, df):
    r, c = readiness_coverage(df)
    scored = df[df["score_possible"] > 0]
    counts = scored["status"].value_counts().to_dict()
    return {"scope": label, "readiness": _pct(r), "coverage": _pct(c), "_r": r, "_c": c,
            **{k: counts.get(k, 0) for k in ["PASS", "WARN", "FAIL", "MANUAL", "UNKNOWN", "NOT_EVALUATED", "NOT_APPLICABLE"]}}


scope_rows = [_scope_row("OVERALL", DF), _scope_row("Common agent checks", DF[DF["source_family"] == "AGENT"]),
              _scope_row("Cross-source", DF[DF["source_family"] == "CROSS_SOURCE"])]
for s in SOURCES:
    scope_rows.append(_scope_row(f"{s.display_name} ({s.source_kind})", DF[DF["source_id"] == s.source_id]))
SCOPE_SCORES = pd.DataFrame(scope_rows)
OVERALL_READINESS, OVERALL_COVERAGE = SCOPE_SCORES.iloc[0]["_r"], SCOPE_SCORES.iloc[0]["_c"]


def _rule_status(rule_id, source_id=None):
    sub = DF[(DF["rule_id"] == rule_id) & (DF["score_possible"] > 0)]
    if source_id is not None:
        sub = sub[sub["source_id"] == source_id]
    if sub.empty:
        return None
    for st in ("FAIL", "UNKNOWN", "NOT_EVALUATED", "MANUAL", "WARN", "PASS", "NOT_APPLICABLE"):
        if (sub["status"] == st).any():
            return st
    return None


def _gate(name, status, evidence):
    return {"gate": name, "status": status, "evidence": evidence}


_SMOKE_RULE = {"SQL": "SQL-002", "KQL": "KQL-002", "SEMANTIC": "SM-000", "GRAPH": "GRAPH-005", "ONTOLOGY": "ONT-005", "AI_SEARCH": "AIS-002"}
gates = []
blockers = DF[(DF["severity"] == "BLOCKER") & (DF["status"] == "FAIL")]
gates.append(_gate("No open BLOCKER findings", "FAIL" if len(blockers) else "PASS", f"{len(blockers)} open blocker row(s)"))
unk = DF[(DF["rule_id"] == "DA-004") & (DF["status"] == "UNKNOWN")]
gates.append(_gate("No unknown connected source types", "FAIL" if len(unk) else "PASS", f"{len(unk)} unknown"))
standalone = P.standalone_semantic
gates.append(_gate("Source count is 1–5", "N/A" if standalone else (_rule_status("DA-003") or "UNKNOWN"), f"{len(SOURCES)} source(s)"))
smoke = {s.display_name: (_rule_status(_SMOKE_RULE.get(s.source_family, ""), s.source_id) or "UNKNOWN") for s in SOURCES}
g = "FAIL" if "FAIL" in smoke.values() else "PASS" if smoke and all(v == "PASS" for v in smoke.values()) else "UNKNOWN"
gates.append(_gate("Every source passes a read/query smoke test", g, str(smoke)))
perm = [_rule_status("DA-015"), _rule_status("DA-016")]
gates.append(_gate("Permissions tested for representative consumers", "N/A" if standalone else
                   "FAIL" if "FAIL" in perm else "PASS" if all(p == "PASS" for p in perm) else "UNKNOWN", f"DA-015={perm[0]}, DA-016={perm[1]}"))
xsr = DF[(DF["source_family"] == "CROSS_SOURCE") & (DF["score_possible"] > 0)]
xs = "N/A" if (standalone or len(SOURCES) <= 1) else "FAIL" if (xsr["status"] == "FAIL").any() or _rule_status("DA-009") == "FAIL" else \
    "PASS" if (xsr["status"].isin(["PASS", "NOT_APPLICABLE"]).all() and _rule_status("DA-009") == "PASS") else "UNKNOWN"
gates.append(_gate("Cross-source routing tests pass", xs, f"XSR statuses: {xsr['status'].value_counts().to_dict()}"))
gates.append(_gate("All critical ground-truth questions pass", "N/A" if standalone else {"PASS": "PASS", "FAIL": "FAIL"}.get(_rule_status("DA-023"), "UNKNOWN"),
                   f"DA-023={_rule_status('DA-023')}"))
d13 = _rule_status("DA-013")
gates.append(_gate("Preview usage explicitly accepted", "N/A" if standalone else "PASS" if d13 in ("PASS", "NOT_APPLICABLE") else "FAIL" if d13 in ("WARN", "FAIL") else "UNKNOWN",
                   f"DA-013={d13}"))
d19 = _rule_status("DA-019")
gates.append(_gate("Published version/runtime match the evaluated version", "N/A" if standalone else {"PASS": "PASS", "WARN": "FAIL"}.get(d19, "UNKNOWN"), f"DA-019={d19}"))
gates.append(_gate(f"Evaluation coverage ≥ {COVERAGE_RELEASE_THRESHOLD:.0%}", "UNKNOWN" if OVERALL_COVERAGE is None else
                   "PASS" if OVERALL_COVERAGE >= COVERAGE_RELEASE_THRESHOLD else "FAIL", _pct(OVERALL_COVERAGE)))
GATES = pd.DataFrame(gates)

if (GATES["status"] == "FAIL").any():
    RATING, COMMENT = "🔴 NOT READY", "Release gate(s) failed — resolve blockers before deployment (averages do not override gates)"
elif (GATES["status"] == "UNKNOWN").any():
    RATING, COMMENT = "🟠 INCOMPLETE EVIDENCE", "No failed gates, but some gates lack evidence — complete manual/live checks"
elif OVERALL_READINESS is not None and OVERALL_READINESS >= 0.9:
    RATING, COMMENT = "🟢 READY FOR DEPLOYMENT", "Excellent — monitor and iterate based on user feedback"
elif OVERALL_READINESS is not None and OVERALL_READINESS >= 0.75:
    RATING, COMMENT = "🟡 MOSTLY READY", "Good foundation — address flagged items before deployment"
else:
    RATING, COMMENT = "🟠 NEEDS WORK", "Significant improvements needed before production use"

print()
print("=" * 78)
print(f"   🤖  FABRIC DATA AGENT ANALYZER SCORECARD v{ANALYZER_VERSION}")
print(f"       Agent: {P.name}  |  Workspace: {P.workspace or 'current'}  |  Stage: {P.stage}  |  Runtime: {P.runtime}")
print(f"       Mode: {P.collection_mode}  |  Docs snapshot: {DOC_SNAPSHOT_DATE}  |  Weights: {SEVERITY_WEIGHTS}")
print("=" * 78)
print(f"\n  Readiness : {_pct(OVERALL_READINESS)}      Evaluation coverage : {_pct(OVERALL_COVERAGE)}")
print(f"  Rating    : {RATING}")
print(f"  Status    : {COMMENT}\n")
display(SCOPE_SCORES.drop(columns=["_r", "_c"]))

print("\n🚦 RELEASE GATES")
display(GATES.assign(status=GATES["status"].map(lambda s: {"PASS": "✅ PASS", "FAIL": "❌ FAIL", "UNKNOWN": "❔ UNKNOWN", "N/A": "➖ N/A"}.get(s, s))))

sev_rank = DF["severity"].map(SEVERITY_ORDER)
BACKLOG = DF[DF["status"].isin(["FAIL", "WARN"])].assign(_sev=sev_rank, _scored=(DF["score_possible"] == 0).astype(int),
                                                         _st=DF["status"].map({"FAIL": 0, "WARN": 1}))
BACKLOG = BACKLOG.sort_values(["_sev", "_st", "_scored", "rule_id"]).drop(columns=["_sev", "_scored", "_st"])
if len(blockers):
    print("\n⛔ OPEN BLOCKERS")
    display(blockers[["rule_id", "source_name", "object_path", "title", "evidence", "recommendation"]])
crit = BACKLOG[(BACKLOG["severity"] == "CRITICAL") & (BACKLOG["status"] == "FAIL")]
if len(crit):
    print("\n🔴 CRITICAL FAILURES")
    display(crit[["rule_id", "source_name", "object_path", "title", "evidence"]].head(30))
print("\n🛠️ PRIORITIZED REMEDIATION BACKLOG (top 30)")
display(BACKLOG[["rule_id", "severity", "status", "source_name", "object_path", "title", "recommendation", "docs_url"]].head(30))

sem = DF[(DF["source_family"] == "SEMANTIC") & (DF["score_possible"] > 0) & (DF["rule_id"].str.startswith("SM-"))]
for sid, grp in sem.groupby("source_id"):
    print(f"\n📊 Semantic model checks — {grp['source_name'].iloc[0]}")
    for _, r in grp.sort_values("rule_id").iterrows():
        pct = r.score_earned / r.score_possible if r.status in ("PASS", "WARN", "FAIL") and r.score_possible else None
        bar = ("█" * int((pct or 0) * 20)).ljust(20, "░") if pct is not None else " " * 20
        print(f"  {STATUS_ICON.get(r.status, '')} {r.rule_id:<7} {r.title[:48]:<48} {r.status:<14} {bar}")

limits = DF[DF["status"].isin(["UNKNOWN", "NOT_EVALUATED", "MANUAL"]) & (DF["score_possible"] > 0)]
print(f"\n🔎 EVIDENCE LIMITATIONS: {len(limits)} scored check(s) not evaluated → {limits['status'].value_counts().to_dict()}")
print("   Attest manual checks via MANUAL_ATTESTATIONS, enable RUN_LIVE_QUERY_TESTS / RUN_EVALUATION, or supply CSV inputs to raise coverage.")

In [ ]:
print("=" * 70)
print("EXPORTS — HTML · Markdown · CSV · JSON (one normalized findings set)")
print("=" * 70)

import html as _html


def _safe(name):
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(name)).strip("_")[:80] or "agent"


def _df_records(df, limit=None):
    if df is None or not isinstance(df, pd.DataFrame):
        return None
    d = df.head(limit) if limit else df
    return json.loads(d.to_json(orient="records", date_format="iso", default_handler=str))


EXPORT_DF = DF.copy()
for col in ("evidence", "recommendation", "object_path"):
    EXPORT_DF[col] = EXPORT_DF[col].map(redact)

source_snapshot = [{
    "source_id": s.source_id, "display_name": s.display_name, "source_type_raw": s.source_type_raw, "source_kind": s.source_kind,
    "source_family": s.source_family, "workspace_id": s.workspace_id, "provenance": s.provenance,
    "selected_objects": [o.get("path") for o in s.selected_objects], "example_query_count": len(s.example_queries),
    "has_instructions": bool(s.instructions), "has_description": bool(s.description), "preview": s.source_kind in PREVIEW_SOURCE_KINDS,
} for s in SOURCES]
BUNDLE = {
    "analyzer_version": ANALYZER_VERSION, "docs_snapshot_date": DOC_SNAPSHOT_DATE,
    "run_started_utc": RUN_STARTED_UTC.isoformat(timespec="seconds"), "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "agent": {"name": P.name, "id": P.agent_id, "workspace": P.workspace, "stage": P.stage, "runtime": P.runtime,
              "collection_mode": P.collection_mode, "provenance": P.provenance, "standalone_semantic": P.standalone_semantic,
              "has_instructions": bool(P.instructions), "discovery_errors": [redact(e) for e in P.errors]},
    "parameters": {"RUN_LIVE_QUERY_TESTS": RUN_LIVE_QUERY_TESTS, "RUN_PERFORMANCE_TESTS": RUN_PERFORMANCE_TESTS, "RUN_EVALUATION": RUN_EVALUATION,
                   "FAIL_ON_PREVIEW_SOURCE": FAIL_ON_PREVIEW_SOURCE, "PREVIEW_ACCEPTED": PREVIEW_ACCEPTED, "SEVERITY_WEIGHTS": SEVERITY_WEIGHTS,
                   "COVERAGE_RELEASE_THRESHOLD": COVERAGE_RELEASE_THRESHOLD, "attested_rules": sorted(MANUAL_ATTESTATIONS)},
    "sources": source_snapshot,
    "capability_matrix": CAPABILITIES, "example_query_support": EXAMPLE_QUERY_SUPPORT,
    "scorecard": {"readiness": OVERALL_READINESS, "coverage": OVERALL_COVERAGE, "rating": RATING, "comment": COMMENT,
                  "scopes": _df_records(SCOPE_SCORES.drop(columns=["_r", "_c"])), "gates": _df_records(GATES)},
    "evaluation_summary": _df_records(EVAL_SUMMARY, 50) if isinstance(EVAL_SUMMARY, pd.DataFrame) else None,
    "routing_matrix": _df_records(ROUTING_MATRIX) if isinstance(ROUTING_MATRIX, pd.DataFrame) else None,
    "known_limitations": _df_records(KNOWN_LIMITATIONS),
    "findings": _df_records(EXPORT_DF),
}


def build_markdown():
    L = [f"# Fabric Data Agent Analyzer Report — {P.name}", "",
         f"- **Analyzer:** v{ANALYZER_VERSION} · docs snapshot {DOC_SNAPSHOT_DATE} · generated {BUNDLE['generated_utc']}",
         f"- **Agent:** {P.name} · workspace {P.workspace or 'current'} · stage {P.stage} · runtime {P.runtime} · mode {P.collection_mode}",
         f"- **Readiness:** {_pct(OVERALL_READINESS)} · **Coverage:** {_pct(OVERALL_COVERAGE)} · **Rating:** {RATING}", "", "## Release gates", "",
         "| Gate | Status | Evidence |", "|---|---|---|"]
    L += [f"| {g['gate']} | {g['status']} | {redact(g['evidence']).replace('|', '/')} |" for g in gates]
    L += ["", "## Scores by scope", "", "| Scope | Readiness | Coverage | PASS | WARN | FAIL | MANUAL | UNKNOWN | NOT_EVALUATED |", "|---|---|---|---|---|---|---|---|---|"]
    L += [f"| {r['scope']} | {r['readiness']} | {r['coverage']} | {r['PASS']} | {r['WARN']} | {r['FAIL']} | {r['MANUAL']} | {r['UNKNOWN']} | {r['NOT_EVALUATED']} |" for r in scope_rows]
    L += ["", "## Source inventory", "", "| Source | Kind | Raw type | Selected | Examples | Preview |", "|---|---|---|---|---|---|"]
    L += [f"| {s['display_name']} | {s['source_kind']} | {s['source_type_raw']} | {len(s['selected_objects'])} | {s['example_query_count']} | {'yes' if s['preview'] else ''} |" for s in source_snapshot]
    L += ["", "## Prioritized remediation backlog", "", "| Rule | Severity | Status | Source | Object | Recommendation |", "|---|---|---|---|---|---|"]
    for _, r in BACKLOG.head(50).iterrows():
        L.append(f"| [{r.rule_id}]({r.docs_url}) | {r.severity} | {r.status} | {r.source_name or ''} | {redact(r.object_path or '')} | {redact(r.recommendation).replace('|', '/')} |")
    lim = EXPORT_DF[EXPORT_DF["status"].isin(["UNKNOWN", "NOT_EVALUATED", "MANUAL"]) & (EXPORT_DF["score_possible"] > 0)]
    L += ["", "## Evidence limitations", "", f"{len(lim)} scored checks were not evaluated: " + ", ".join(sorted(set(lim["rule_id"])))]
    L += ["", "## Known limitations watch list", ""] + [f"- **{r.limitation_id}** {r.limitation} ({r.rule_ids})" for r in KNOWN_LIMITATIONS.itertuples()]
    return "\n".join(L)


def build_html():
    css = ("body{font-family:Segoe UI,Arial,sans-serif;margin:24px;color:#222}h1{font-size:22px}table{border-collapse:collapse;margin:8px 0 18px}"
           "td,th{border:1px solid #ddd;padding:4px 8px;font-size:12px;vertical-align:top}th{background:#f3f3f3}.PASS{color:#107c10}.FAIL{color:#a80000}"
           ".WARN{color:#c19c00}.kpi{display:inline-block;margin-right:28px;font-size:18px}")
    def tbl(df):
        return df.to_html(index=False, escape=True, classes="t")
    parts = [f"<html><head><meta charset='utf-8'><title>Fabric Data Agent Analyzer — {_html.escape(P.name)}</title><style>{css}</style></head><body>",
             f"<h1>🤖 Fabric Data Agent Analyzer — {_html.escape(P.name)}</h1>",
             f"<p>Analyzer v{ANALYZER_VERSION} · docs snapshot {DOC_SNAPSHOT_DATE} · stage {P.stage} · runtime {P.runtime} · mode {P.collection_mode}</p>",
             f"<div class='kpi'>Readiness <b>{_pct(OVERALL_READINESS)}</b></div><div class='kpi'>Coverage <b>{_pct(OVERALL_COVERAGE)}</b></div>"
             f"<div class='kpi'>{_html.escape(RATING)}</div><p>{_html.escape(COMMENT)}</p>",
             "<h2>Release gates</h2>", tbl(GATES.assign(evidence=GATES["evidence"].map(redact))),
             "<h2>Scores by scope</h2>", tbl(SCOPE_SCORES.drop(columns=["_r", "_c"])),
             "<h2>Source inventory</h2>", tbl(pd.DataFrame(source_snapshot).drop(columns=["selected_objects"]) if source_snapshot else pd.DataFrame()),
             "<h2>Prioritized remediation backlog</h2>",
             tbl(BACKLOG[["rule_id", "severity", "status", "source_name", "object_path", "title", "recommendation"]].head(60).assign(
                 object_path=lambda d: d["object_path"].map(redact))),
             "<h2>Known limitations</h2>", tbl(KNOWN_LIMITATIONS[["limitation_id", "limitation", "rule_ids", "last_reviewed"]]),
             "</body></html>"]
    return "\n".join(parts)


EXPORT_PATHS = {}
if EXPORT_REPORTS:
    stamp = RUN_STARTED_UTC.strftime("%Y%m%dT%H%M%SZ")
    candidates = [os.path.join(OUTPUT_ROOT, _safe(P.name), stamp), os.path.join(os.getcwd(), "DataAgentReadiness", _safe(P.name), stamp)]
    out_dir = None
    for c in candidates:
        try:
            os.makedirs(c, exist_ok=True)
            out_dir = c
            break
        except Exception as e:
            print(f"   ℹ️ Cannot write to {c}: {short(e, 100)}")
    if out_dir:
        base = os.path.join(out_dir, f"readiness_{_safe(P.name)}")
        EXPORT_DF.to_csv(base + "_findings.csv", index=False)
        with open(base + "_evidence.json", "w", encoding="utf-8") as fh:
            json.dump(BUNDLE, fh, indent=2, default=str)
        with open(base + "_report.md", "w", encoding="utf-8") as fh:
            fh.write(build_markdown())
        with open(base + "_report.html", "w", encoding="utf-8") as fh:
            fh.write(build_html())
        EXPORT_PATHS = {k: base + sfx for k, sfx in (("csv", "_findings.csv"), ("json", "_evidence.json"), ("markdown", "_report.md"), ("html", "_report.html"))}
        for k, v in EXPORT_PATHS.items():
            print(f"   ✅ {k:<8} → {v}")
    else:
        print("   ❌ No writable output folder — exports skipped.")
else:
    print("   ➖ EXPORT_REPORTS = False — exports skipped.")

print(f"\n✅ Analysis complete — {len(DF)} finding rows ({int((DF['score_possible'] > 0).sum())} scored). Rating: {RATING}")
print("🔒 No configuration or data was modified. Re-run after each change and compare exports across commits.")

---
### 📝 Notes & Migration Log

**v3.0.0 (2026-09-29): Fabric Data Agent Analyzer**
- Renamed to **FabricDataAgentAnalyzer** (formerly *Semantic Model Data Agent Readiness Analyzer*, `SemanticModel_DataAgent_Readiness.ipynb`). One notebook now assesses a data agent, its shared configuration, and every connected source family.
- Added `LIVE_SDK`, `GIT_EXPORT`, and `MANUAL_MANIFEST` collection modes. Standalone semantic-model mode (the v2.x contract: `dataset` + `workspace`) is kept.
- Replaced `check_scores` / `all_issues` tuples with normalized `Finding` records, a versioned source-type alias table, a capability registry, and a `KNOWN_LIMITATIONS` watch list.
- Added common agent checks (`DA-*`) and adapters for SQL (`SQL-*`, `LH-*`, `WH-*`, `SQLDB-*`, `MIR-*`), KQL (`KQL-*`), Graph (`GRAPH-*`), Ontology (`ONT-*`), and Azure AI Search (`AIS-*`). Added cross-source routing (`XSR-*`), evaluation, security, and ALM sections.
- Scoring now reports readiness and coverage separately, with blocker-first **release gates**. Manual checks are `MANUAL` (or attested), not zero.
- Fixed the v2.2.2 version drift (title v2.2.2 vs scorecard v2.2.1) and rule-key drift (`1.12_date_fields`, `1.13_hidden_objects`, `1.14_model_bloat`, `1.17_perf_analyzer`, `1.18_duplicate_cols`). These now use stable `SM-*` IDs.
- Restored the business-friendly naming check (1.5 → `SM-005`), which was missing from v2.2.2.
- The Prep for AI validator now uses a synchronous client and a proper bearer header. The token is never printed. Its result also drives `SM-019`, `SM-022`, `SM-023`, and `SM-025`.
- Updated guidance: data agent configuration can be source-controlled (preview), Verified Answers use 5–7 trigger questions and up to 3 filters, and the Q&A dependency for advanced DAX is version-gated (Q&A retirement is scheduled for December 2026).
- Exports: HTML, Markdown, CSV, and JSON evidence bundle.
- Added the **Source Run Plan** (4.1) and `SKIP_UNSELECTED_SOURCES`. Moved the adapter framework to 4.2 and moved the shared adapter helpers (`_mentioned`, `_items`, `_schema_part`, `preview_acceptance`, `_TECH_LABEL`, `_GENERIC_EDGE`) out of the SQL and Graph cells. Each source section now runs, or can be skipped, on its own.

**v2.x history (condensed):** v2.4 added the Copilot grounding completeness check, marked date table and private tables detection, and the service/admin section. v2.3 added negative-scope guidance, the over-synonymization check, and the technical-column check. v2.2 added the Performance Analyzer check, duplicate column names, and the Prep for AI validator. v2.1 added weighted scoring and alignment with the official Microsoft checklist.

**What stays manual (attest with `MANUAL_ATTESTATIONS`):** tenant and cross-geo settings, outbound rules, preview acceptance, operational ownership, capacity monitoring, run-step review, report-scoped measures, Performance Analyzer, Power BI service/admin items, Azure AI Search RBAC, ontology bound-source permissions, and graph reloads.

**Key references:** [Data agent data sources](https://learn.microsoft.com/en-us/fabric/data-science/data-agent-add-datasources) · [Configuration best practices](https://learn.microsoft.com/en-us/fabric/data-science/data-agent-configuration-best-practices) · [Example queries](https://learn.microsoft.com/en-us/fabric/data-science/data-agent-example-queries) · [Runtime](https://learn.microsoft.com/en-us/fabric/data-science/data-agent-runtime) · [Python SDK](https://learn.microsoft.com/en-us/fabric/data-science/fabric-data-agent-sdk) · [Evaluate](https://learn.microsoft.com/en-us/fabric/data-science/evaluate-data-agent) · [Source control & ALM](https://learn.microsoft.com/en-us/fabric/data-science/data-agent-source-control) · [SQL sources](https://learn.microsoft.com/en-us/fabric/data-science/data-agent-sql-sources) · [KQL best practices](https://learn.microsoft.com/en-us/kusto/query/best-practices?view=microsoft-fabric) · [Graph schema design](https://learn.microsoft.com/en-us/fabric/graph/design-graph-schema) · [Ontology data agent](https://learn.microsoft.com/en-us/fabric/iq/ontology/how-to-create-data-agent) · [Azure AI Search index](https://learn.microsoft.com/en-us/fabric/data-science/data-agent-ai-search-index) · [Semantic model best practices](https://learn.microsoft.com/en-us/fabric/data-science/semantic-model-best-practices)

---
*This notebook is read-only by default. It doesn't modify the data agent, its sources, or your semantic model. Run it after each change and compare the exports.*